# ESAP RSSD v2.10
## Scalable, spatially balanced sensor-directed response surface sampling

**Developed and modernized by A.J. Brown, Agricultural Data Scientist**

ESAP RSSD v2.10 is an independent Python modernization and extension of the sensor-directed response surface sampling methodology developed by Lesch and colleagues for selecting limited calibration samples from dense, spatially referenced sensor surveys.

The statistical foundation of this notebook draws primarily from the regression-directed spatial sampling framework of **Lesch, Strauss, and Rhoades (1995)** and the spatial response surface sampling methodology of **Lesch (2005)**. The 1995 algorithm emphasized simultaneous representation of the survey region and efficient estimation of regression-model parameters, while the 2005 SRS framework formalized predictor-space response-surface targeting and geographic separation of selected calibration sites.

This implementation also acknowledges the important modern software and methodological work of **Jayanta Banik (2024)**. Banik's M.S. thesis, *Sensor Directed Sampling*, and associated Python-based SDSampling work explored the use of central composite response-surface designs and modern computational workflows in the ESAP RSSD context. That work provided an important practical point of departure for the present modernization.

### What ESAP RSSD v2.10 changes

The notebook preserves the core ESAP objective of selecting calibration sites suitable for a low-order ordinary regression model, while adapting the workflow for modern dense and irregular sensor data such as drone imagery, electromagnetic-induction transects, vehicle sensor paths, and spatial point clouds.

The v2.10 workflow:

1. transforms and standardizes selected sensor predictors and constructs decorrelated, standardized principal-component scores;
2. defines a central composite response-surface design in coded PC space;
3. identifies observed survey locations representing each theoretical predictor-space condition;
4. measures field-wide geographic coverage with **Spatially Balanced Average Distance (SBAD)**, a generalization of the classical ESAP average-distance criterion that separates geographic support from raw sensor observation density;
5. finds a near-optimal field-coverage envelope and maximizes the exact geometric mean minimum separation distance (**geoMSD**) within that envelope;
6. adds a limited number of spatial support sites to reduce remaining geographic coverage gaps; and
7. reports regression-design, spatial-coverage, optimizer-stability, candidate-saturation, and proxy spatial-scale diagnostics.

The central design principle is:

> **ESAP 2 separates geographic support from sensor observation density.**

The method is intended to give a subsequent ordinary linear or polynomial calibration regression the best reasonable chance of satisfying its independent-error assumption. Geographic separation and proxy spatial-scale diagnostics do **not** prove residual independence. Residual spatial dependence must still be evaluated after response measurements are collected and the calibration model is fitted.

This implementation avoids survey-wide pairwise distance matrices, Cartesian enumeration of candidate combinations, and regular-raster assumptions. It is designed for field-scale use on ordinary computing hardware, including multi-million-observation sensor surveys.

> **Project status:** ESAP RSSD v2.10 is an independent research implementation and modernization. It is not an official USDA ESAP software release and should not be represented as an exact reproduction of every undocumented behavior or default in the original ESAP software.

### Primary methodological references

Lesch, S. M., Strauss, D. J., & Rhoades, J. D. (1995). Spatial prediction of soil salinity using electromagnetic induction techniques: 2. An efficient spatial sampling algorithm suitable for multiple linear regression model identification and estimation. *Water Resources Research, 31*(2), 387–398. https://doi.org/10.1029/94WR02180

Lesch, S. M. (2005). Sensor-directed response surface sampling designs for characterizing spatial variation in soil properties. *Computers and Electronics in Agriculture, 46*, 153–179. https://doi.org/10.1016/j.compag.2004.11.004

Banik, J. (2024). *Sensor Directed Sampling* [M.S. thesis, University of California, Riverside]. UC Riverside Electronic Theses and Dissertations. https://escholarship.org/uc/item/77f4z65n

### Additional historical context

Lesch, S. M., Rhoades, J. D., & Corwin, D. L. (2000). *ESAP-95 Version 2.01R: User Manual and Tutorial Guide*. USDA Agricultural Research Service, Research Report No. 146.

---

**Suggested citation for this implementation:**  
Brown, A. J. ESAP RSSD v2.10: Scalable, spatially balanced sensor-directed response surface sampling. Independent Python research implementation.


In [ ]:
# @title Initialize ESAP RSSD v2.10 engine { display-mode: "form" }
# Core dependencies available in a standard Google Colab runtime.
from __future__ import annotations

import ast
import gc
import hashlib
import json
import math
import os
import platform
import sys
import time
import warnings
from collections import OrderedDict
from dataclasses import asdict, dataclass, field
from importlib import metadata as importlib_metadata
import json
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform
from scipy.stats import chi2, skew
from sklearn.decomposition import IncrementalPCA, PCA
from sklearn.preprocessing import PowerTransformer, StandardScaler

NOTEBOOK_VERSION = "2.10.0"
np.set_printoptions(precision=4, suppress=True)


# v2.10 default configuration
@dataclass
class RSSDConfig:
    # Input. The defaults make the notebook executable without external files.
    INPUT_FILE: Optional[str] = None
    EXCEL_SHEET_NAME: Any = 0
    USE_SYNTHETIC_DEMO: bool = True
    SYNTHETIC_ROWS: int = 6000

    # Required column roles. Change these for a real file.
    ID_COLUMN: str = "sample_id"
    X_COLUMN: str = "x"
    Y_COLUMN: str = "y"
    FEATURE_COLUMNS: List[str] = field(
        default_factory=lambda: ["sensor_1", "sensor_2", "sensor_3", "sensor_4"]
    )
    FEATURE_TRANSFORMS: Dict[str, str] = field(default_factory=dict)

    # Response-surface design and sample budget.
    N_SAMPLES: int = 20
    N_DESIGN_COMPONENTS: int = 2
    DESIGN_COVERAGE: float = 0.80
    CENTER_REPLICATES: int = 2
    SAMPLE_BUDGET_MODE: str = "rssd_with_support"  # rssd_with_support, ccd_exact, or balanced_target_replication
    SUPPORT_SITE_MODE: str = "legacy_inspired_auto"  # legacy_inspired_auto, manual, or none
    N_SUPPORT_SITES: int = 0

    # Statistical screening and candidate discovery.
    OUTLIER_COVERAGE: float = 0.999
    CANDIDATE_EXPLORATION_MODE: str = "adaptive"
    CANDIDATES_PER_TARGET: int = 3
    CANDIDATE_SATURATION_START_K: int = 3
    CANDIDATE_SATURATION_GROWTH_FACTOR: int = 2
    CANDIDATE_SATURATION_MAX_K: int = 64
    CANDIDATE_SATURATION_STABLE_STAGES_REQUIRED: int = 2
    CANDIDATE_STAGE_EXPLORATION_STARTS: int = 5
    CANDIDATE_SATURATION_AD_REL_TOL: float = 0.005
    CANDIDATE_SATURATION_GEOMSD_REL_TOL: float = 0.01
    CANDIDATE_SATURATION_MINSEP_REL_TOL: float = 0.02
    PC_CANDIDATE_TOLERANCE: float = 0.15
    MAX_PC_CANDIDATE_TOLERANCE: float = 1.50
    CANDIDATE_TOLERANCE_GROWTH: float = 1.5

    # Optimizer. N_OPTIMIZER_STARTS is a maximum when stable-search early stopping is enabled.
    N_OPTIMIZER_STARTS: int = 50
    MIN_OPTIMIZER_STARTS: int = 12
    OPTIMIZER_NO_IMPROVEMENT_PATIENCE: int = 12
    EARLY_STOP_ON_STABLE_SEARCH: bool = True
    VERIFY_REPRODUCIBILITY_BY_RERUN: bool = False
    OPTIMIZER_INITIALIZATION_MODE: str = "adaptive"
    OPTIMIZER_START_MODE: str = "adaptive"
    OPTIMIZER_START_BATCH_SIZE: int = 10
    OPTIMIZER_MIN_STARTS: int = 20
    OPTIMIZER_MAX_STARTS: int = 150
    OPTIMIZER_BEST_IMPROVEMENT_REL_TOL: float = 0.001
    OPTIMIZER_NEAR_BEST_REL_TOL: float = 0.01
    OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION: float = 0.80
    OPTIMIZER_STABLE_BATCHES_REQUIRED: int = 2
    MAX_OPTIMIZER_SWEEPS: int = 100
    AD_SUPPORT_MODE: str = "adaptive"
    AD_SUPPORT_START_SIZE: int = 5000
    AD_SUPPORT_GROWTH_FACTOR: int = 2
    AD_SUPPORT_MAX_SIZE: int = 40000
    AD_SUPPORT_FIXED_SIZE: int = 20000
    AD_SUPPORT_REL_TOL: float = 0.005
    AD_SUPPORT_RANK_CORRELATION_MIN: float = 0.995
    AD_SUPPORT_STABLE_STAGES_REQUIRED: int = 2
    AD_SUPPORT_RANK_PANEL_SIZE: int = 20
    SUPPORT_STAGE_INITIAL_STARTS: int = 10
    SUPPORT_STAGE_REFINEMENT_STARTS: int = 3
    AD_SUPPORT_PRACTICAL_EQUIV_REL_TOL: float = 0.0025
    AD_SUPPORT_DECISIVE_PAIR_AGREEMENT_MIN: float = 0.95
    AD_SUPPORT_NEAR_BEST_REL_TOL: float = 0.005
    SUPPORT_HYBRID_GEOMSD_REL_TOL: float = 0.01
    SUPPORT_HYBRID_MINSEP_REL_TOL: float = 0.02
    SUPPORT_HYBRID_COVERAGE_RATIO_ABS_TOL: float = 0.005
    SUPPORT_GAP_ANCHORS: int = 500
    SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR: int = 5
    AD_COVERAGE_ENVELOPE_REL_TOL: float = 0.05
    AD_DISTANCE_CACHE_MAX_MIB: int = 512
    RUN_REFERENCE_DESIGNS: bool = False
    RANDOM_SEED: int = 20250717
    GEOMSD_TIE_REL_TOL: float = 1e-6
    PCA_TIE_TOL: float = 1e-12

    # Memory behavior.
    MEMORY_MODE: str = "auto"  # full, auto, or incremental
    MAX_WORKING_MEMORY_FRACTION: float = 0.45
    INCREMENTAL_BATCH_SIZE: int = 50000
    NUMERIC_DTYPE: str = "float32"

    # Optional approximation for extreme candidate-discovery workloads.
    ALLOW_APPROXIMATE_PREFILTER: bool = False
    APPROX_PREFILTER_TRIGGER_ROWS: int = 750000
    APPROX_PREFILTER_MAX_ROWS: int = 250000
    APPROX_PREFILTER_BINS_PER_PC: int = 8

    # Diagnostics and testing.
    PLOT_MAX_POINTS: int = 50000
    SURVEY_COLOR: str = "C0"
    TARGET_COLOR: str = "C3"
    OUTLIER_COLOR: str = "C1"
    CANDIDATE_COLOR: str = "C2"
    SELECTED_COLOR: str = "C3"
    FOOTPRINT_COLOR: str = "0.5"
    SUPPORT_COLOR: str = "C4"
    DIAGNOSTIC_CHUNK_SIZE: int = 100000
    RUN_SPACING_DIAGNOSTIC: bool = True
    SPACING_DIAGNOSTIC_MAX_POINTS: int = 6000
    SPACING_DIAGNOSTIC_NEIGHBORS: int = 24
    SPACING_DIAGNOSTIC_RANDOM_PAIRS: int = 120000
    SPACING_DIAGNOSTIC_BINS: int = 16
    SPACING_TARGET_SEMIVARIANCE_FRACTION: float = 0.90
    SPACING_CAUTION_RATIO: float = 1.00
    SPACING_WARNING_RATIO: float = 0.75
    SPACING_MIN_PAIRS_PER_BIN: int = 250
    SPACING_RANGE_STABILITY_BINS: int = 3
    SPACING_PLATEAU_TAIL_BINS: int = 4
    SPACING_PLATEAU_REL_SLOPE_TOL: float = 0.05


cfg = RSSDConfig()

# Development timing benchmarks are opt-in and are not part of normal field-run initialization.
RUN_DEVELOPMENT_BENCHMARKS = False
DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS = 10000

# Explicit transform examples for a real run:
# cfg.FEATURE_TRANSFORMS = {"ECa_vertical": "log", "NDVI": "yeo-johnson"}
# cfg.USE_SYNTHETIC_DEMO = False
# cfg.INPUT_FILE = "/content/my_survey.csv"  # or None for Colab upload




# @title Load internal data and PCA functions (run; no editing required) { display-mode: "form" }
def make_synthetic_survey(n: int, seed: int) -> pd.DataFrame:
    """Create correlated, spatially structured proxy variables for a runnable demonstration."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 2400.0, n)
    y = rng.uniform(0.0, 1600.0, n)
    z1 = 0.9 * np.sin(x / 310.0) + 0.7 * np.cos(y / 240.0) + rng.normal(0, 0.35, n)
    z2 = 0.75 * z1 + 0.5 * np.sin((x + y) / 420.0) + rng.normal(0, 0.3, n)
    z3 = -0.35 * z1 + 0.9 * np.cos(x / 520.0) + rng.normal(0, 0.45, n)
    z4 = 0.45 * z2 + 0.5 * z3 + rng.normal(0, 0.4, n)
    return pd.DataFrame(
        {
            "sample_id": np.arange(1, n + 1, dtype=np.int64),
            "x": x,
            "y": y,
            "sensor_1": z1,
            "sensor_2": z2,
            "sensor_3": z3,
            "sensor_4": z4,
        }
    )


def load_survey(config: RSSDConfig) -> Tuple[pd.DataFrame, str]:
    """Load CSV/XLS/XLSX input or the deterministic synthetic demonstration."""
    if config.USE_SYNTHETIC_DEMO:
        return make_synthetic_survey(config.SYNTHETIC_ROWS, config.RANDOM_SEED), "synthetic_demo"

    input_path = config.INPUT_FILE
    if input_path is None:
        try:
            from google.colab import files  # type: ignore
        except ImportError as exc:
            raise ValueError(
                "Set cfg.INPUT_FILE to a CSV/XLS/XLSX path outside Colab, or enable the synthetic demo."
            ) from exc
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one CSV or Excel survey file.")
        input_path = next(iter(uploaded))

    path = Path(input_path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xls", ".xlsx", ".xlsm"}:
        try:
            df = pd.read_excel(path, sheet_name=config.EXCEL_SHEET_NAME)
        except ImportError as exc:
            raise ImportError("Excel input requires openpyxl (xlsx/xlsm) or xlrd (xls).") from exc
    else:
        raise ValueError(f"Unsupported input type {suffix!r}; use CSV, XLS, XLSX, or XLSM.")
    return df, str(path)




def coordinates_look_geographic(x: np.ndarray, y: np.ndarray, x_name: str, y_name: str) -> bool:
    """Conservatively detect decimal-degree longitude/latitude coordinates."""
    xn, yn = x_name.lower(), y_name.lower()
    name_signal = ("lon" in xn or "longitude" in xn) and ("lat" in yn or "latitude" in yn)
    bounded = np.mean((x >= -180) & (x <= 180) & (y >= -90) & (y <= 90)) >= 0.99
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        return False
    xf, yf = x[finite], y[finite]
    decimal_signal = np.mean(np.abs(xf - np.round(xf)) > 1e-6) > 0.8 and np.mean(
        np.abs(yf - np.round(yf)) > 1e-6
    ) > 0.8
    typical_lonlat = (
        np.nanmedian(np.abs(xf)) >= 20
        and np.nanmedian(np.abs(yf)) >= 10
        and np.ptp(xf) <= 20
        and np.ptp(yf) <= 20
    )
    return bool(name_signal or (bounded and decimal_signal and typical_lonlat))


def validate_and_index_data(
    df: pd.DataFrame, config: RSSDConfig
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, Dict[str, Any]]:
    """Validate required data and return valid-row and coordinate-eligibility masks."""
    required = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]
    missing_columns = [c for c in required if c not in df.columns]
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")
    if len(set(config.FEATURE_COLUMNS)) != len(config.FEATURE_COLUMNS):
        raise ValueError("FEATURE_COLUMNS contains duplicates.")
    if config.N_DESIGN_COMPONENTS > len(config.FEATURE_COLUMNS):
        raise ValueError("N_DESIGN_COMPONENTS cannot exceed the number of feature columns.")

    duplicate_id_count = int(df[config.ID_COLUMN].duplicated(keep=False).sum())
    if duplicate_id_count:
        raise ValueError(
            f"Found {duplicate_id_count} rows with duplicated IDs. IDs must be unique before RSSD selection."
        )

    numeric = df[[config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]].apply(
        pd.to_numeric, errors="coerce"
    )
    arr = numeric.to_numpy(dtype=np.float64, copy=False)
    finite_mask = np.isfinite(arr).all(axis=1)
    missing_nonfinite_counts = {
        c: int((~np.isfinite(numeric[c].to_numpy(dtype=np.float64))).sum())
        for c in numeric.columns
    }
    n_invalid = int((~finite_mask).sum())
    if finite_mask.sum() < max(config.N_SAMPLES, 20):
        raise ValueError("Too few complete finite rows remain for the requested design.")

    valid_features = numeric.loc[finite_mask, config.FEATURE_COLUMNS]
    zero_variance = [c for c in config.FEATURE_COLUMNS if float(valid_features[c].var(ddof=0)) <= 0.0]
    if zero_variance:
        raise ValueError(f"Zero-variance feature columns are not usable: {zero_variance}")

    x = numeric.loc[finite_mask, config.X_COLUMN].to_numpy(dtype=np.float64)
    y = numeric.loc[finite_mask, config.Y_COLUMN].to_numpy(dtype=np.float64)
    if coordinates_look_geographic(x, y, config.X_COLUMN, config.Y_COLUMN):
        raise ValueError(
            "Coordinates strongly resemble longitude/latitude in decimal degrees. Reproject to an appropriate linear-unit CRS (for example, the relevant UTM zone) before computing geoMSD."
        )

    duplicated_coordinates_all = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep=False).to_numpy()
    later_coordinate_duplicate = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep="first").to_numpy()
    coordinate_eligible = finite_mask & ~later_coordinate_duplicate

    out = df.copy()
    out["rssd_complete_finite"] = finite_mask
    out["rssd_duplicate_coordinate"] = duplicated_coordinates_all
    out["rssd_coordinate_candidate_eligible"] = coordinate_eligible
    report = {
        "input_rows": int(len(df)),
        "complete_finite_rows": int(finite_mask.sum()),
        "invalid_rows_excluded": n_invalid,
        "duplicated_id_rows": duplicate_id_count,
        "duplicated_coordinate_rows": int(duplicated_coordinates_all.sum()),
        "later_duplicate_coordinate_rows_excluded_from_candidates": int(later_coordinate_duplicate.sum()),
        "missing_or_nonfinite_by_column": missing_nonfinite_counts,
        "duplicate_coordinate_handling": (
            "All records are retained; among exact coordinate duplicates, only the first valid record is candidate-eligible."
        ),
    }
    return out, finite_mask, coordinate_eligible, report


def transform_features(
    values: np.ndarray, feature_names: Sequence[str], transforms: Mapping[str, str]
) -> Tuple[np.ndarray, Dict[str, Dict[str, Any]]]:
    """Apply explicit per-feature transformations without automatic choices."""
    transformed = np.asarray(values, dtype=np.float64).copy()
    details: Dict[str, Dict[str, Any]] = {}
    allowed = {"none", "log", "yeo-johnson"}
    unknown_keys = sorted(set(transforms) - set(feature_names))
    if unknown_keys:
        raise KeyError(f"FEATURE_TRANSFORMS contains unknown features: {unknown_keys}")
    for j, name in enumerate(feature_names):
        method = transforms.get(name, "none").lower()
        if method not in allowed:
            raise ValueError(f"Unsupported transform {method!r} for {name}; choose {sorted(allowed)}.")
        if method == "none":
            details[name] = {"method": "none"}
        elif method == "log":
            if np.any(transformed[:, j] <= 0):
                raise ValueError(f"Log transformation requires strictly positive values in {name!r}.")
            transformed[:, j] = np.log(transformed[:, j])
            details[name] = {"method": "log", "base": "natural"}
        else:
            pt = PowerTransformer(method="yeo-johnson", standardize=False)
            transformed[:, [j]] = pt.fit_transform(transformed[:, [j]])
            details[name] = {"method": "yeo-johnson", "lambda": float(pt.lambdas_[0])}
    return transformed.astype(np.float32, copy=False), details


def available_memory_bytes() -> int:
    """Return available RAM when psutil is present, otherwise use a conservative Colab fallback."""
    try:
        import psutil

        return int(psutil.virtual_memory().available)
    except ImportError:
        return 8 * 1024**3


def choose_pca_mode(n_rows: int, n_features: int, config: RSSDConfig) -> Tuple[str, Dict[str, float]]:
    """Choose full or incremental PCA from an explicit working-memory estimate."""
    # Approximate transformed, standardized, work, and score arrays plus estimator overhead.
    estimated = int(n_rows * n_features * (4 + 4 + 8 + 4) + n_rows * config.N_DESIGN_COMPONENTS * 4)
    available = available_memory_bytes()
    safe = int(config.MAX_WORKING_MEMORY_FRACTION * available)
    if config.MEMORY_MODE == "auto":
        mode = "full" if estimated <= safe else "incremental"
    else:
        mode = config.MEMORY_MODE
    return mode, {
        "estimated_working_bytes": estimated,
        "available_memory_bytes": available,
        "safe_memory_bytes": safe,
        "estimated_working_mib": estimated / 1024**2,
    }


def fit_standardized_pca(
    transformed: np.ndarray, config: RSSDConfig
) -> Tuple[np.ndarray, StandardScaler, Any, str, Dict[str, float]]:
    """Standardize features, fit PCA, and divide retained scores by sqrt(explained variance)."""
    n, f = transformed.shape
    mode, memory_report = choose_pca_mode(n, f, config)
    scaler = StandardScaler(copy=True)
    batch = max(config.INCREMENTAL_BATCH_SIZE, f + 1)

    def batch_slices() -> Iterable[Tuple[int, int]]:
        """Yield batches while merging a final batch smaller than the feature count."""
        start = 0
        while start < n:
            end = min(start + batch, n)
            if 0 < n - end < f:
                end = n
            yield start, end
            start = end

    if mode == "full":
        standardized = scaler.fit_transform(transformed).astype(np.float32, copy=False)
        estimator: Any = PCA(n_components=f, svd_solver="full", random_state=config.RANDOM_SEED)
        estimator.fit(standardized)
        retained_raw = (standardized - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T
    else:
        for start, end in batch_slices():
            scaler.partial_fit(transformed[start:end])
        estimator = IncrementalPCA(n_components=f, batch_size=batch)
        for start, end in batch_slices():
            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)
            estimator.partial_fit(xb)
        retained_raw = np.empty((n, config.N_DESIGN_COMPONENTS), dtype=np.float32)
        for start, end in batch_slices():
            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)
            retained_raw[start:end] = (
                (xb - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T
            )

    scales = np.sqrt(np.asarray(estimator.explained_variance_[: config.N_DESIGN_COMPONENTS]))
    if np.any(scales <= 0):
        raise ValueError("PCA produced a nonpositive explained variance in a retained component.")
    design_scores = (retained_raw / scales).astype(np.float32, copy=False)
    return design_scores, scaler, estimator, mode, memory_report


def pca_validation_table(scores: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray]:
    """Return means, sample standard deviations, and the correlation matrix."""
    names = [f"PC{i + 1}" for i in range(scores.shape[1])]
    table = pd.DataFrame(
        {"mean": np.mean(scores, axis=0), "sample_sd": np.std(scores, axis=0, ddof=1)}, index=names
    )
    corr = np.corrcoef(scores, rowvar=False)
    corr = np.atleast_2d(corr)
    return table, corr


# @title Load internal spatial-design functions (run; no editing required) { display-mode: "form" }

def selected_pairwise_distance_matrix(coords: np.ndarray) -> np.ndarray:
    """Return the exact small m by m selected-site Euclidean distance matrix."""
    coords = np.asarray(coords, dtype=np.float64)
    if coords.ndim != 2 or coords.shape[1] != 2:
        raise ValueError("Selected coordinates must be an m by 2 array.")
    if coords.shape[0] < 2:
        raise ValueError("At least two selected coordinate pairs are required.")
    delta = coords[:, None, :] - coords[None, :, :]
    d2 = np.einsum("ijk,ijk->ij", delta, delta, dtype=np.float64)
    np.fill_diagonal(d2, np.inf)
    return np.sqrt(d2, dtype=np.float64)


def selected_nearest_neighbor_distances_ckdtree_reference(coords: np.ndarray, workers: int = -1) -> np.ndarray:
    """Validation-only reference for the previous tiny selected-set cKDTree implementation."""
    coords = np.asarray(coords, dtype=np.float64)
    if coords.ndim != 2 or coords.shape[0] < 2:
        raise ValueError("At least two selected coordinate pairs are required.")
    distances, _ = cKDTree(coords).query(coords, k=2, workers=workers)
    return np.asarray(distances[:, 1], dtype=np.float64)


def selected_nearest_neighbor_distances(coords: np.ndarray) -> np.ndarray:
    """Calculate exact nearest selected-site distances with a direct small m by m array."""
    return np.min(selected_pairwise_distance_matrix(coords), axis=1)


def exact_geo_msd(coords: np.ndarray) -> float:
    """Compute exp(mean(log(d_j))) for exact nearest-selected-neighbor distances d_j."""
    d = selected_nearest_neighbor_distances(coords)
    if np.any(d <= 0):
        return 0.0
    return float(np.exp(np.mean(np.log(d))))


def make_base_ccd(p: int, radius: float, center_replicates: int) -> pd.DataFrame:
    """Construct cube, axial, and replicated-center targets on common outer radius R."""
    if p > 4:
        raise ValueError("p > 4 requires a separately specified hybrid or small composite design.")
    rows: List[Dict[str, Any]] = []
    cube_level = radius / math.sqrt(p)
    for number in range(2**p):
        signs = np.array([1.0 if (number >> j) & 1 else -1.0 for j in range(p)])
        row = {"base_target_id": f"cube_{number + 1:02d}", "target_type": "cube"}
        row.update({f"target_PC{j + 1}": float(signs[j] * cube_level) for j in range(p)})
        rows.append(row)
    for axis in range(p):
        for sign, label in [(-1.0, "minus"), (1.0, "plus")]:
            values = np.zeros(p)
            values[axis] = sign * radius
            row = {
                "base_target_id": f"axial_PC{axis + 1}_{label}",
                "target_type": "axial",
            }
            row.update({f"target_PC{j + 1}": float(values[j]) for j in range(p)})
            rows.append(row)
    for c in range(center_replicates):
        row = {"base_target_id": f"center_{c + 1:02d}", "target_type": "center"}
        row.update({f"target_PC{j + 1}": 0.0 for j in range(p)})
        rows.append(row)
    return pd.DataFrame(rows)




def approximate_pca_prefilter(
    eligible_indices: np.ndarray,
    scores: np.ndarray,
    targets: np.ndarray,
    config: RSSDConfig,
) -> Tuple[np.ndarray, bool, Dict[str, Any]]:
    """Optional reproducible occupancy prefilter that explicitly preserves tails and target neighborhoods."""
    eligible_indices = np.asarray(eligible_indices, dtype=int)
    if (
        not config.ALLOW_APPROXIMATE_PREFILTER
        or len(eligible_indices) <= config.APPROX_PREFILTER_TRIGGER_ROWS
    ):
        return eligible_indices, False, {"reason": "disabled_or_below_trigger"}

    rng = np.random.default_rng(config.RANDOM_SEED)
    z = scores[eligible_indices]
    p = z.shape[1]
    bins = config.APPROX_PREFILTER_BINS_PER_PC
    bin_ids = np.zeros((len(z), p), dtype=np.int16)
    for j in range(p):
        edges = np.quantile(z[:, j], np.linspace(0, 1, bins + 1)[1:-1])
        bin_ids[:, j] = np.digitize(z[:, j], np.unique(edges), right=False)
    multipliers = (bins ** np.arange(p)).astype(np.int64)
    codes = (bin_ids.astype(np.int64) * multipliers).sum(axis=1)
    occupied = np.unique(codes)
    per_bin_cap = max(1, math.ceil(config.APPROX_PREFILTER_MAX_ROWS / len(occupied)))
    kept_local: List[np.ndarray] = []
    for code_value in occupied:
        members = np.flatnonzero(codes == code_value)
        if len(members) > per_bin_cap:
            members = np.sort(rng.choice(members, size=per_bin_cap, replace=False))
        kept_local.append(members)

    # Explicitly preserve univariate tails.
    tail_n = min(1000, len(z))
    tail_local = []
    for j in range(p):
        order = np.argsort(z[:, j], kind="stable")
        tail_local.extend([order[:tail_n], order[-tail_n:]])

    # Explicitly preserve every observation in each initial target neighborhood.
    tree = cKDTree(z)
    target_local: List[int] = []
    for target in np.unique(targets, axis=0):
        target_local.extend(tree.query_ball_point(target, r=config.PC_CANDIDATE_TOLERANCE))

    local = np.unique(
        np.concatenate([*kept_local, *tail_local, np.asarray(target_local, dtype=int)])
    )
    kept = eligible_indices[local]
    report = {
        "input_eligible_rows": int(len(eligible_indices)),
        "prefilter_rows": int(len(kept)),
        "occupied_pca_strata": int(len(occupied)),
        "per_stratum_cap": int(per_bin_cap),
        "target_neighborhood_rows_explicitly_preserved": int(len(np.unique(target_local))),
    }
    return kept, True, report






def assignment_from_costs(
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    rng: Optional[np.random.Generator] = None,
) -> Optional[np.ndarray]:
    """Use a minimum-cost bipartite assignment; random costs produce reproducible alternative starts."""
    union = np.unique(np.concatenate(pools))
    if len(union) < len(pools):
        return None
    column = {int(index): j for j, index in enumerate(union)}
    prohibitive = 1e12
    cost = np.full((len(pools), len(union)), prohibitive, dtype=np.float64)
    for i, (pool, distances) in enumerate(zip(pools, pool_distances)):
        values = distances if rng is None else rng.random(len(pool))
        for index, value in zip(pool, values):
            cost[i, column[int(index)]] = float(value)
    rows, cols = linear_sum_assignment(cost)
    if len(rows) != len(pools) or np.any(cost[rows, cols] >= prohibitive):
        return None
    selected = np.empty(len(pools), dtype=int)
    selected[rows] = union[cols]
    return selected


def matching_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray) -> Tuple[float, float]:
    """Return mean and maximum Euclidean target mismatch in standardized PC units."""
    distances = np.linalg.norm(scores[selected] - targets, axis=1)
    return float(np.mean(distances)), float(np.max(distances))


def design_rank(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray) -> Tuple[float, float, float]:
    """Return geoMSD, mean mismatch, and maximum mismatch for lexicographic comparison."""
    mean_pc, max_pc = matching_metrics(selected, targets, scores)
    return exact_geo_msd(coords[selected]), mean_pc, max_pc


def rank_is_better(
    candidate: Tuple[float, float, float],
    incumbent: Tuple[float, float, float],
    config: RSSDConfig,
) -> bool:
    """Compare valid designs: geoMSD first, then mean and maximum PCA mismatch only on ties."""
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate[0]), abs(incumbent[0]))
    if candidate[0] > incumbent[0] + geo_tol:
        return True
    if abs(candidate[0] - incumbent[0]) <= geo_tol:
        if candidate[1] < incumbent[1] - config.PCA_TIE_TOL:
            return True
        if abs(candidate[1] - incumbent[1]) <= config.PCA_TIE_TOL:
            return candidate[2] < incumbent[2] - config.PCA_TIE_TOL
    return False



def candidate_pool_summary_table(
    target_instances: pd.DataFrame,
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    target_tolerances: Sequence[float],
    config: RSSDConfig,
) -> pd.DataFrame:
    """Summarize how broad each target-specific candidate search actually was."""
    records: List[Dict[str, Any]] = []
    for position, (pool, distances, tolerance) in enumerate(
        zip(pools, pool_distances, target_tolerances)
    ):
        distances = np.asarray(distances, dtype=float)
        row = target_instances.iloc[position]
        records.append(
            {
                "target_position": int(position),
                "target_instance_id": row["target_instance_id"],
                "base_target_id": row["base_target_id"],
                "target_type": row["target_type"],
                "candidate_pool_size": int(len(pool)),
                "unique_candidate_pool_size": int(len(np.unique(pool))),
                "minimum_pca_target_distance": float(np.min(distances)) if len(distances) else None,
                "median_pca_target_distance": float(np.median(distances)) if len(distances) else None,
                "maximum_pca_target_distance": float(np.max(distances)) if len(distances) else None,
                "final_tolerance_used": float(tolerance),
                "tolerance_ratio_to_configured_max": float(tolerance / config.MAX_PC_CANDIDATE_TOLERANCE)
                if config.MAX_PC_CANDIDATE_TOLERANCE > 0
                else None,
                "nearest_k_fallback_implied": bool(tolerance > config.MAX_PC_CANDIDATE_TOLERANCE * (1.0 + 1e-12)),
            }
        )
    return pd.DataFrame(records)


def search_sufficiency_diagnostics(
    optimizer_summary: pd.DataFrame,
    candidate_pool_report: pd.DataFrame,
    assignment_attempts: int,
    config: RSSDConfig,
) -> Dict[str, Any]:
    """Describe whether the bounded candidate/exchange search appears saturated."""
    if optimizer_summary.empty:
        return {"status": "not_evaluated", "recommendation": "No optimizer starts were recorded."}

    best_geo = float(optimizer_summary["final_geoMSD"].max())
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(best_geo))
    near_best = optimizer_summary["final_geoMSD"] >= best_geo - geo_tol
    best_rows = optimizer_summary.loc[near_best]
    starts_run = int(optimizer_summary["start_number"].max())
    first_best_start = int(best_rows["start_number"].iloc[0])
    last_best_start = int(best_rows["start_number"].iloc[-1])
    starts_after_last_best = starts_run - last_best_start
    stop_values = optimizer_summary.get("stop_after_this_start", pd.Series(dtype=object)).replace("", np.nan).dropna()
    stop_reason = str(stop_values.iloc[-1]) if len(stop_values) else "configured_max_starts_completed"

    min_pool = int(candidate_pool_report["candidate_pool_size"].min()) if len(candidate_pool_report) else 0
    median_pool = float(candidate_pool_report["candidate_pool_size"].median()) if len(candidate_pool_report) else 0.0
    maxed_tolerance_targets = int(
        (candidate_pool_report["tolerance_ratio_to_configured_max"] >= 0.999).sum()
    ) if len(candidate_pool_report) else 0
    fallback_targets = int(candidate_pool_report["nearest_k_fallback_implied"].sum()) if len(candidate_pool_report) else 0
    low_pool_targets = int(
        (candidate_pool_report["candidate_pool_size"] < max(3, config.CANDIDATES_PER_TARGET)).sum()
    ) if len(candidate_pool_report) else 0

    flags: List[str] = []
    if assignment_attempts > 1:
        flags.append("candidate pools had to be enlarged before a unique assignment existed")
    if fallback_targets:
        flags.append("one or more targets required nearest-k fallback beyond the configured PCA tolerance")
    if maxed_tolerance_targets:
        flags.append("one or more targets reached the configured maximum PCA tolerance")
    if low_pool_targets:
        flags.append("one or more targets retained fewer candidates than the requested pool size")

    if flags:
        status = "candidate_space_constrained"
        recommendation = "Review target tolerances, candidate counts, forced locations, and PCA feature choices before treating the search as saturated."
    elif stop_reason == "stable_no_improvement" or starts_after_last_best >= config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE:
        status = "plausibly_saturated_local_search"
        recommendation = "The best design survived the configured no-improvement patience; additional starts are optional unless the field plan is high stakes."
    elif starts_run >= config.N_OPTIMIZER_STARTS and starts_after_last_best < max(3, config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE // 2):
        status = "late_improvement_possible"
        recommendation = "The best design appeared late; increase the maximum starts or candidate pools if runtime allows."
    else:
        status = "review_optimizer_trace"
        recommendation = "Inspect the convergence table; more starts may be useful if geoMSD is still improving materially."

    return {
        "status": status,
        "recommendation": recommendation,
        "best_geoMSD": best_geo,
        "starts_run": starts_run,
        "configured_max_starts": int(config.N_OPTIMIZER_STARTS),
        "first_best_start": first_best_start,
        "last_best_start": last_best_start,
        "starts_after_last_best": int(starts_after_last_best),
        "near_best_start_count": int(near_best.sum()),
        "optimizer_stop_reason": stop_reason,
        "assignment_attempts": int(assignment_attempts),
        "minimum_candidate_pool_size": min_pool,
        "median_candidate_pool_size": median_pool,
        "targets_at_maximum_tolerance": maxed_tolerance_targets,
        "targets_with_nearest_k_fallback": fallback_targets,
        "targets_below_requested_pool_size": low_pool_targets,
        "constraint_flags": flags,
        "interpretation": "This is a bounded-search diagnostic, not a mathematical proof of global optimality.",
    }


def coordinate_exchange(
    start: np.ndarray,
    pools: Sequence[np.ndarray],
    targets: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Optimize one site at a time; no Cartesian product of candidate pools is generated."""
    selected = np.asarray(start, dtype=int).copy()
    if len(np.unique(selected)) != len(selected):
        raise ValueError("Coordinate exchange requires a unique starting assignment.")
    initial_rank = design_rank(selected, targets, scores, coords)
    current_rank = initial_rank
    accepted = 0
    sweeps = 0
    for sweep in range(1, config.MAX_OPTIMIZER_SWEEPS + 1):
        accepted_this_sweep = 0
        for position in rng.permutation(len(selected)):
            used_elsewhere = set(np.delete(selected, position).tolist())
            best_index = int(selected[position])
            best_rank = current_rank
            for candidate_index in pools[position]:
                candidate_index = int(candidate_index)
                if candidate_index == selected[position] or candidate_index in used_elsewhere:
                    continue
                proposal = selected.copy()
                proposal[position] = candidate_index
                proposal_rank = design_rank(proposal, targets, scores, coords)
                if rank_is_better(proposal_rank, best_rank, config):
                    best_index, best_rank = candidate_index, proposal_rank
            if best_index != selected[position]:
                selected[position] = best_index
                current_rank = best_rank
                accepted += 1
                accepted_this_sweep += 1
        sweeps = sweep
        if accepted_this_sweep == 0:
            break
    return selected, {
        "initial_geoMSD": initial_rank[0],
        "final_geoMSD": current_rank[0],
        "accepted_swaps": accepted,
        "sweeps": sweeps,
        "final_mean_pca_target_distance": current_rank[1],
        "final_max_pca_target_distance": current_rank[2],
    }


def optimize_multiple_starts(
    minimum_distance_start: np.ndarray,
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    targets: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """Run minimum-mismatch and reproducible random starts with optional stable-search stopping."""
    rng = np.random.default_rng(config.RANDOM_SEED)
    summaries: List[Dict[str, Any]] = []
    best: Optional[np.ndarray] = None
    best_rank: Optional[Tuple[float, float, float]] = None
    starts_since_best = 0
    stop_reason = "configured_max_starts_completed"
    for start_number in range(1, config.N_OPTIMIZER_STARTS + 1):
        start = minimum_distance_start.copy() if start_number == 1 else assignment_from_costs(
            pools, pool_distances, rng
        )
        if start is None:
            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")
        optimized, summary = coordinate_exchange(start, pools, targets, scores, coords, config, rng)
        summary["start_number"] = start_number
        summary["start_type"] = "minimum_pca_distance" if start_number == 1 else "random_unique_assignment"
        rank = design_rank(optimized, targets, scores, coords)
        new_best = best_rank is None or rank_is_better(rank, best_rank, config)
        if new_best:
            best, best_rank = optimized.copy(), rank
            starts_since_best = 0
        else:
            starts_since_best += 1
        summary["new_best_design"] = bool(new_best)
        summary["best_so_far_geoMSD"] = float(best_rank[0]) if best_rank is not None else None
        summary["starts_since_best"] = int(starts_since_best)
        summary["stop_after_this_start"] = ""
        summaries.append(summary)
        if (
            config.EARLY_STOP_ON_STABLE_SEARCH
            and start_number >= config.MIN_OPTIMIZER_STARTS
            and starts_since_best >= config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE
        ):
            stop_reason = "stable_no_improvement"
            summaries[-1]["stop_after_this_start"] = stop_reason
            break
    if best is None:
        raise RuntimeError("Optimizer produced no design.")
    summary_df = pd.DataFrame(summaries)
    summary_df["optimizer_stop_reason"] = stop_reason
    return best, summary_df


# @title Load diagnostics and acceptance tests (run; no editing required) { display-mode: "form" }
def second_order_matrix(scores: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    """Build intercept, linear, squared, and pairwise-interaction columns."""
    z = np.asarray(scores, dtype=np.float64)
    n, p = z.shape
    columns = [np.ones(n)]
    names = ["Intercept"]
    for j in range(p):
        columns.append(z[:, j])
        names.append(f"PC{j + 1}")
    for j in range(p):
        columns.append(z[:, j] ** 2)
        names.append(f"PC{j + 1}^2")
    for j in range(p):
        for k in range(j + 1, p):
            columns.append(z[:, j] * z[:, k])
            names.append(f"PC{j + 1}:PC{k + 1}")
    return np.column_stack(columns), names


def regression_design_diagnostics(
    selected_scores: np.ndarray,
    population_scores: np.ndarray,
    chunk_size: int,
) -> Tuple[Dict[str, Any], np.ndarray]:
    """Calculate rank, condition, leverage, and chunked average relative prediction variance."""
    x, names = second_order_matrix(selected_scores)
    rank = int(np.linalg.matrix_rank(x))
    n_columns = x.shape[1]
    condition = float(np.linalg.cond(x))
    result: Dict[str, Any] = {
        "model_terms": names,
        "matrix_rows": int(x.shape[0]),
        "matrix_columns": int(n_columns),
        "matrix_rank": rank,
        "full_rank": rank == n_columns,
        "condition_number": condition,
    }
    if rank != n_columns:
        result.update(
            {
                "maximum_leverage": None,
                "average_relative_prediction_variance": None,
                "warning": "Rank deficient: inverse-based leverage and avePVar were not calculated.",
            }
        )
        return result, np.full(len(selected_scores), np.nan)

    xtx_inv = np.linalg.inv(x.T @ x)
    leverage = np.einsum("ij,jk,ik->i", x, xtx_inv, x)
    total = 0.0
    count = 0
    for start in range(0, len(population_scores), chunk_size):
        xp, _ = second_order_matrix(population_scores[start : start + chunk_size])
        h = np.einsum("ij,jk,ik->i", xp, xtx_inv, xp)
        total += float(np.sum(1.0 + h))
        count += len(h)
    result.update(
        {
            "maximum_leverage": float(np.max(leverage)),
            "average_relative_prediction_variance": total / count,
        }
    )
    return result, leverage





def json_ready(value: Any) -> Any:
    """Recursively convert NumPy/Pandas values to JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    return value


def package_versions() -> Dict[str, str]:
    """Record software versions needed to reproduce a run."""
    packages = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "openpyxl"]
    versions = {"python": platform.python_version()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not_installed"
    return versions










# --- ESAP RSSD v2.10 targeted overrides ---
























































# V210_OVERRIDES






# --- ESAP RSSD v2.10 overrides inserted below ---


@dataclass
class RSSDRunResult:
    config: RSSDConfig
    selected_sites: pd.DataFrame
    candidate_sites: pd.DataFrame
    target_instances: pd.DataFrame
    support_sequence: pd.DataFrame
    ad_support_resolution: pd.DataFrame
    candidate_saturation: pd.DataFrame
    optimizer_stability: pd.DataFrame
    spatial_support_sites: pd.DataFrame
    field_coverage_distances: pd.DataFrame
    proxy_spatial_scale: pd.DataFrame
    pca_validation: pd.DataFrame
    feature_skewness: pd.DataFrame
    metadata: Dict[str, Any]
    run_summary: str
    diagnostic_data: Dict[str, Any] = field(default_factory=dict)



def validate_config(config: RSSDConfig) -> None:
    if config.MEMORY_MODE not in {"full", "auto", "incremental"}:
        raise ValueError("MEMORY_MODE must be 'full', 'auto', or 'incremental'.")
    if config.SAMPLE_BUDGET_MODE not in {"rssd_with_support", "ccd_exact", "balanced_target_replication"}:
        raise ValueError("SAMPLE_BUDGET_MODE must be rssd_with_support, ccd_exact, or balanced_target_replication.")
    if config.SUPPORT_SITE_MODE not in {"legacy_inspired_auto", "manual", "none"}:
        raise ValueError("SUPPORT_SITE_MODE must be legacy_inspired_auto, manual, or none.")
    if config.AD_SUPPORT_MODE not in {"adaptive", "fixed"}:
        raise ValueError("AD_SUPPORT_MODE must be adaptive or fixed.")
    if config.OPTIMIZER_START_MODE not in {"adaptive", "fixed"}:
        raise ValueError("OPTIMIZER_START_MODE must be adaptive or fixed.")
    if not (1 <= config.N_DESIGN_COMPONENTS <= 4):
        raise ValueError("This version supports 1-4 design PCs.")
    if config.N_DESIGN_COMPONENTS == 4:
        warnings.warn("A full 4-D central composite design contains many targets; Lesch notes that hybrid or small composite designs may be preferable.")
    for name, value in {"OUTLIER_COVERAGE": config.OUTLIER_COVERAGE, "DESIGN_COVERAGE": config.DESIGN_COVERAGE}.items():
        if not 0.0 < value < 1.0:
            raise ValueError(f"{name} must lie strictly between 0 and 1.")
    if config.CANDIDATES_PER_TARGET < 2:
        raise ValueError("CANDIDATES_PER_TARGET must be at least 2.")
    if int(config.CANDIDATE_SATURATION_START_K) < 1 or int(config.CANDIDATE_SATURATION_MAX_K) < int(config.CANDIDATE_SATURATION_START_K):
        raise ValueError("Candidate saturation K settings are inconsistent.")
    if int(config.CANDIDATE_SATURATION_GROWTH_FACTOR) < 2:
        raise ValueError("CANDIDATE_SATURATION_GROWTH_FACTOR must be at least 2.")
    if int(config.OPTIMIZER_START_BATCH_SIZE) < 1 or int(config.OPTIMIZER_MIN_STARTS) < 1 or int(config.OPTIMIZER_MAX_STARTS) < int(config.OPTIMIZER_MIN_STARTS):
        raise ValueError("Batched optimizer start settings are inconsistent.")
    if config.N_OPTIMIZER_STARTS < 1 or not (1 <= config.MIN_OPTIMIZER_STARTS <= config.N_OPTIMIZER_STARTS):
        raise ValueError("Legacy optimizer start settings are inconsistent.")
    if config.AD_SUPPORT_START_SIZE < 1 or config.AD_SUPPORT_MAX_SIZE < 1 or config.AD_SUPPORT_GROWTH_FACTOR < 2:
        raise ValueError("AD support-size settings are inconsistent.")
    if not (0.0 <= config.AD_COVERAGE_ENVELOPE_REL_TOL < 1.0):
        raise ValueError("AD_COVERAGE_ENVELOPE_REL_TOL must be nonnegative and less than 1.")
    if int(config.SUPPORT_GAP_ANCHORS) < 1:
        raise ValueError("SUPPORT_GAP_ANCHORS must be at least 1.")
    if int(config.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR) < 1:
        raise ValueError("SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR must be at least 1.")
    if config.SPACING_DIAGNOSTIC_MAX_POINTS < 10:
        raise ValueError("SPACING_DIAGNOSTIC_MAX_POINTS must be at least 10.")
    if config.SPACING_DIAGNOSTIC_NEIGHBORS < 1:
        raise ValueError("SPACING_DIAGNOSTIC_NEIGHBORS must be positive.")
    if config.SPACING_DIAGNOSTIC_RANDOM_PAIRS < 100:
        raise ValueError("SPACING_DIAGNOSTIC_RANDOM_PAIRS must be at least 100.")
    if config.SPACING_DIAGNOSTIC_BINS < 4:
        raise ValueError("SPACING_DIAGNOSTIC_BINS must be at least 4.")
    if int(config.SPACING_MIN_PAIRS_PER_BIN) < 1:
        raise ValueError("SPACING_MIN_PAIRS_PER_BIN must be at least 1.")
    if int(config.SPACING_RANGE_STABILITY_BINS) < 1 or int(config.SPACING_PLATEAU_TAIL_BINS) < 2:
        raise ValueError("Proxy spatial-scale reliability settings are inconsistent.")


def allocate_response_surface_instances(base_targets: pd.DataFrame, response_surface_count: int, p: int) -> Tuple[pd.DataFrame, pd.Series]:
    base_n = len(base_targets)
    if response_surface_count < base_n:
        raise ValueError(f"The full base CCD requires {base_n} response-surface sites; requested {response_surface_count}.")
    counts = np.ones(base_n, dtype=int)
    extras = response_surface_count - base_n
    if extras > 0:
        counts += extras // base_n
        counts[: extras % base_n] += 1
    rows: List[Dict[str, Any]] = []
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    for i, base in base_targets.iterrows():
        for replicate in range(1, int(counts[i]) + 1):
            row = {"target_instance_id": f"T{len(rows) + 1:03d}", "base_target_id": base["base_target_id"], "target_type": base["target_type"], "target_replicate_number": replicate, "sample_role": "response_surface"}
            row.update({c: float(base[c]) for c in target_cols})
            rows.append(row)
    instances = pd.DataFrame(rows)
    return instances, instances.groupby("base_target_id", sort=False).size().rename("replication_count")


def _legacy_inspired_desired_support_count(n_samples: int) -> int:
    if n_samples <= 6:
        return 1
    if n_samples <= 20:
        return 2
    return max(3, int(math.ceil(n_samples / 10.0)))


def allocate_sample_budget(base_targets: pd.DataFrame, n_samples: int, config: RSSDConfig, p: int) -> Tuple[pd.DataFrame, pd.Series, Dict[str, Any]]:
    base_n = len(base_targets)
    if config.SAMPLE_BUDGET_MODE == "ccd_exact":
        response_count, support_count = base_n, 0
        if n_samples != base_n:
            print(f"ccd_exact mode uses {base_n} response-surface samples; configured N_SAMPLES={n_samples} is ignored.")
    elif config.SAMPLE_BUDGET_MODE == "balanced_target_replication":
        response_count, support_count = n_samples, 0
    else:
        if n_samples < base_n:
            raise ValueError(f"rssd_with_support requires N_SAMPLES >= {base_n}, the base CCD size.")
        desired = 0 if config.SUPPORT_SITE_MODE == "none" else (max(0, int(config.N_SUPPORT_SITES)) if config.SUPPORT_SITE_MODE == "manual" else _legacy_inspired_desired_support_count(n_samples))
        available_extra_sites = n_samples - base_n
        support_count = min(desired, max(0, available_extra_sites))
        response_count = n_samples - support_count
        if desired > support_count:
            print(f"Spatial support sites capped at {support_count} because the full {base_n}-site response-surface core must be preserved.")
    target_instances, replication = allocate_response_surface_instances(base_targets, response_count, p)
    report = {"sample_budget_mode": config.SAMPLE_BUDGET_MODE, "support_site_mode": config.SUPPORT_SITE_MODE, "requested_total_samples": int(n_samples), "base_CCD_target_count": int(base_n), "response_surface_sites": int(response_count), "spatial_support_sites": int(support_count), "full_response_surface_core_preserved": bool(response_count >= base_n), "support_allocation_label": "legacy-inspired automatic support allocation" if config.SUPPORT_SITE_MODE == "legacy_inspired_auto" else config.SUPPORT_SITE_MODE}
    return target_instances, replication, report


def allocate_target_instances(base_targets: pd.DataFrame, n_samples: int, mode: str, p: int) -> Tuple[pd.DataFrame, pd.Series]:
    temp = RSSDConfig(N_SAMPLES=n_samples, SAMPLE_BUDGET_MODE=mode)
    if mode == "rssd_with_support":
        temp.SUPPORT_SITE_MODE = "none"
    instances, replication, _ = allocate_sample_budget(base_targets, n_samples, temp, p)
    return instances, replication


def _nearest_observed_coordinate_to_center(coords: np.ndarray, ids: np.ndarray, indices: np.ndarray, center: np.ndarray) -> int:
    d2 = np.einsum("ij,ij->i", coords[indices] - center, coords[indices] - center)
    best = np.flatnonzero(d2 == np.min(d2))
    return int(indices[int(best[np.argmin(ids[indices[best]])])])


def _leaf_bbox(coords: np.ndarray, indices: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    subset = coords[indices]
    lo = np.min(subset, axis=0); hi = np.max(subset, axis=0); span = hi - lo
    return lo, hi, span, float(np.sqrt(np.sum(span * span)))


def build_spatially_balanced_support_sequence(domain_coords: np.ndarray, stable_ids: Optional[np.ndarray] = None, max_size: int = 40000) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    import heapq
    coords_in = np.asarray(domain_coords, dtype=np.float64)
    finite = np.isfinite(coords_in).all(axis=1)
    coords_in = coords_in[finite]
    ids_in = (np.arange(len(domain_coords), dtype=np.int64) if stable_ids is None else np.asarray(stable_ids, dtype=np.int64))[finite]
    if len(coords_in) == 0:
        raise ValueError("No finite coordinates are available for the spatial support sequence.")
    order = np.lexsort((ids_in, coords_in[:, 1], coords_in[:, 0]))
    sorted_coords = coords_in[order]; sorted_ids = ids_in[order]
    unique_mask = np.ones(len(sorted_coords), dtype=bool)
    if len(sorted_coords) > 1:
        unique_mask[1:] = np.any(np.diff(sorted_coords, axis=0) != 0.0, axis=1)
    coords = sorted_coords[unique_mask]; ids = sorted_ids[unique_mask]
    max_len = min(int(max_size), len(coords))
    all_indices = np.arange(len(coords), dtype=np.int64)
    lo, hi, span, diagonal = _leaf_bbox(coords, all_indices)
    root_rep = _nearest_observed_coordinate_to_center(coords, ids, all_indices, 0.5 * (lo + hi))
    rows = [{"support_rank": 1, "analysis_index": int(ids[root_rep]), "X": float(coords[root_rep, 0]), "Y": float(coords[root_rep, 1])}]
    leaves = {0: {"indices": all_indices, "representative": int(root_rep), "lo": lo, "hi": hi, "span": span, "diagonal": diagonal}}
    heap: List[Tuple[float, int]] = []
    if len(all_indices) > 1 and diagonal > 0:
        heapq.heappush(heap, (-diagonal, 0))
    leaf_counter = 0
    while len(rows) < max_len and heap:
        _, leaf_id = heapq.heappop(heap)
        leaf = leaves.pop(leaf_id, None)
        if leaf is None:
            continue
        indices = leaf["indices"]; lo = leaf["lo"]; hi = leaf["hi"]; span = leaf["span"]; diagonal = leaf["diagonal"]
        if len(indices) <= 1 or diagonal <= 0 or np.max(span) <= 0:
            continue
        axis = int(np.argmax(span)); midpoint = float(0.5 * (lo[axis] + hi[axis]))
        left_indices = indices[coords[indices, axis] <= midpoint]; right_indices = indices[coords[indices, axis] > midpoint]
        if len(left_indices) == 0 or len(right_indices) == 0:
            continue
        rep = int(leaf["representative"])
        for child_indices in (left_indices, right_indices):
            child_lo, child_hi, child_span, child_diag = _leaf_bbox(coords, child_indices)
            append_rep = not np.any(child_indices == rep)
            child_rep = rep if not append_rep else _nearest_observed_coordinate_to_center(coords, ids, child_indices, 0.5 * (child_lo + child_hi))
            leaf_counter += 1
            leaves[leaf_counter] = {"indices": child_indices, "representative": int(child_rep), "lo": child_lo, "hi": child_hi, "span": child_span, "diagonal": child_diag}
            if len(child_indices) > 1 and child_diag > 0 and np.max(child_span) > 0:
                heapq.heappush(heap, (-float(child_diag), leaf_counter))
            if append_rep and len(rows) < max_len:
                rows.append({"support_rank": len(rows) + 1, "analysis_index": int(ids[child_rep]), "X": float(coords[child_rep, 0]), "Y": float(coords[child_rep, 1])})
    table = pd.DataFrame(rows)
    report = {"spatial_domain_rows": int(len(domain_coords)), "finite_spatial_domain_rows": int(np.sum(finite)), "unique_spatial_domain_coordinates": int(len(coords)), "support_sequence_maximum_length": int(len(table)), "algorithm": "deterministic recursive occupied-space bisection", "uses_only_xy": True, "uses_pc_scores": False, "uses_row_count_as_geographic_weight": False}
    return table, report


def exact_sbad(support_coords: np.ndarray, selected_coords: np.ndarray) -> float:
    support = np.asarray(support_coords, dtype=np.float64); selected = np.asarray(selected_coords, dtype=np.float64)
    if len(support) == 0 or len(selected) == 0:
        return float("nan")
    d, _ = cKDTree(selected).query(support, k=1, workers=-1)
    return float(np.mean(np.asarray(d, dtype=np.float64)))








def minimum_selected_separation(coords: np.ndarray) -> float:
    coords = np.asarray(coords, dtype=np.float64)
    return float(np.min(selected_nearest_neighbor_distances(coords))) if len(coords) >= 2 else float("inf")


def run_selected_set_development_benchmark(rng: np.random.Generator, enabled: bool, evaluations: Optional[int] = None) -> Dict[str, Any]:
    """Optionally run the slow old-versus-new selected-set timing benchmark."""
    configured_evaluations = int(DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS if evaluations is None else evaluations)
    if not bool(enabled):
        return {
            "development_selected_set_benchmark_run": False,
            "configured_evaluations": int(configured_evaluations),
            "evaluations": 0,
            "selected_sites": 20,
            "old_ckdtree_workers_minus_1_seconds": None,
            "new_vectorized_seconds": None,
            "speedup_ratio": None,
            "geoMSD_equality": None,
        }

    bench_coords = rng.uniform(0.0, 1000.0, size=(20, 2))
    old_total = 0.0
    new_total = 0.0
    t0 = time.perf_counter()
    for _ in range(configured_evaluations):
        old_total += float(np.sum(selected_nearest_neighbor_distances_ckdtree_reference(bench_coords, workers=-1)))
    old_elapsed = time.perf_counter() - t0
    t0 = time.perf_counter()
    for _ in range(configured_evaluations):
        new_total += float(np.sum(selected_nearest_neighbor_distances(bench_coords)))
    new_elapsed = time.perf_counter() - t0
    np.testing.assert_allclose(old_total, new_total, rtol=1e-12, atol=1e-8)
    return {
        "development_selected_set_benchmark_run": True,
        "configured_evaluations": int(configured_evaluations),
        "evaluations": int(configured_evaluations),
        "selected_sites": 20,
        "old_ckdtree_workers_minus_1_seconds": float(old_elapsed),
        "new_vectorized_seconds": float(new_elapsed),
        "speedup_ratio": float(old_elapsed / max(new_elapsed, 1e-12)),
        "geoMSD_equality": True,
    }


def design_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, sbad: Optional[float] = None) -> Dict[str, float]:
    mean_pc, max_pc = matching_metrics(selected, targets, scores)
    selected_coords = coords[selected]
    return {"SBAD": float(sbad) if sbad is not None else float("nan"), "geoMSD": exact_geo_msd(selected_coords), "minimum_separation": minimum_selected_separation(selected_coords), "mean_pca_target_distance": float(mean_pc), "max_pca_target_distance": float(max_pc)}


def _tol_equal(a: float, b: float, rel_tol: float = 1e-9, abs_tol: float = 1e-12) -> bool:
    return abs(a - b) <= max(abs_tol, rel_tol * max(1.0, abs(a), abs(b)))


def ad_reference_is_better(candidate: Dict[str, float], incumbent: Dict[str, float], config: RSSDConfig) -> bool:
    if candidate["SBAD"] < incumbent["SBAD"] - config.PCA_TIE_TOL:
        return True
    if _tol_equal(candidate["SBAD"], incumbent["SBAD"], abs_tol=config.PCA_TIE_TOL):
        geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate["geoMSD"]), abs(incumbent["geoMSD"]))
        if candidate["geoMSD"] > incumbent["geoMSD"] + geo_tol:
            return True
        if abs(candidate["geoMSD"] - incumbent["geoMSD"]) <= geo_tol:
            if candidate["minimum_separation"] > incumbent["minimum_separation"] + config.PCA_TIE_TOL:
                return True
            if _tol_equal(candidate["minimum_separation"], incumbent["minimum_separation"], abs_tol=config.PCA_TIE_TOL):
                if candidate["mean_pca_target_distance"] < incumbent["mean_pca_target_distance"] - config.PCA_TIE_TOL:
                    return True
                if _tol_equal(candidate["mean_pca_target_distance"], incumbent["mean_pca_target_distance"], abs_tol=config.PCA_TIE_TOL):
                    return candidate["max_pca_target_distance"] < incumbent["max_pca_target_distance"] - config.PCA_TIE_TOL
    return False


def hybrid_is_better(candidate: Dict[str, float], incumbent: Dict[str, float], config: RSSDConfig, sbad_limit: float) -> bool:
    cand_feasible = candidate["SBAD"] <= sbad_limit + config.PCA_TIE_TOL
    inc_feasible = incumbent["SBAD"] <= sbad_limit + config.PCA_TIE_TOL
    if cand_feasible and not inc_feasible:
        return True
    if inc_feasible and not cand_feasible:
        return False
    if not cand_feasible and not inc_feasible:
        return ad_reference_is_better(candidate, incumbent, config)
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate["geoMSD"]), abs(incumbent["geoMSD"]))
    if candidate["geoMSD"] > incumbent["geoMSD"] + geo_tol:
        return True
    if abs(candidate["geoMSD"] - incumbent["geoMSD"]) <= geo_tol:
        if candidate["minimum_separation"] > incumbent["minimum_separation"] + config.PCA_TIE_TOL:
            return True
        if _tol_equal(candidate["minimum_separation"], incumbent["minimum_separation"], abs_tol=config.PCA_TIE_TOL):
            if candidate["mean_pca_target_distance"] < incumbent["mean_pca_target_distance"] - config.PCA_TIE_TOL:
                return True
            if _tol_equal(candidate["mean_pca_target_distance"], incumbent["mean_pca_target_distance"], abs_tol=config.PCA_TIE_TOL):
                return candidate["max_pca_target_distance"] < incumbent["max_pca_target_distance"] - config.PCA_TIE_TOL
    return False


def candidate_pool_for_target_incremental(target: np.ndarray, pc_tree: cKDTree, search_scores: np.ndarray, search_coords: np.ndarray, search_global_indices: np.ndarray, desired: int, config: RSSDConfig) -> Tuple[np.ndarray, np.ndarray, float, int, bool]:
    tolerance = float(config.PC_CANDIDATE_TOLERANCE)
    local = pc_tree.query_ball_point(target, r=tolerance)
    expansions = 0
    while len(local) < desired and tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:
        tolerance = min(config.MAX_PC_CANDIDATE_TOLERANCE, tolerance * config.CANDIDATE_TOLERANCE_GROWTH)
        local = pc_tree.query_ball_point(target, r=tolerance); expansions += 1
    nearest_k_fallback = False
    if len(local) < desired:
        k = min(max(desired, 1), len(search_scores))
        nearest_d, nearest_i = pc_tree.query(target, k=k)
        local = np.atleast_1d(nearest_i).astype(int).tolist()
        tolerance = max(tolerance, float(np.max(np.atleast_1d(nearest_d))))
        nearest_k_fallback = True
    if len(local) == 0:
        raise RuntimeError("No candidate observations were found for a response-surface target.")
    local_arr = np.asarray(local, dtype=int)
    pc_dist = np.linalg.norm(search_scores[local_arr] - target, axis=1)
    stable = np.lexsort((search_global_indices[local_arr], pc_dist))
    local_arr = local_arr[stable]; pc_dist = pc_dist[stable]
    selected_positions = [0]
    remaining = np.ones(len(local_arr), dtype=bool); remaining[0] = False
    min_geo = np.linalg.norm(search_coords[local_arr] - search_coords[local_arr[0]], axis=1); min_geo[0] = -np.inf
    while len(selected_positions) < min(desired, len(local_arr)):
        rem_pos = np.flatnonzero(remaining)
        order = np.lexsort((search_global_indices[local_arr[rem_pos]], pc_dist[rem_pos], -min_geo[rem_pos]))
        chosen = int(rem_pos[order[0]])
        selected_positions.append(chosen); remaining[chosen] = False
        new_dist = np.linalg.norm(search_coords[local_arr] - search_coords[local_arr[chosen]], axis=1)
        min_geo = np.minimum(min_geo, new_dist); min_geo[~remaining] = -np.inf
    positions = np.asarray(selected_positions, dtype=int)
    return search_global_indices[local_arr[positions]], pc_dist[positions], float(tolerance), int(expansions), bool(nearest_k_fallback)


def candidate_pool_for_target(target, pc_tree, search_scores, search_coords, search_global_indices, desired, config):
    indices, distances, tolerance, expansions, _ = candidate_pool_for_target_incremental(target, pc_tree, search_scores, search_coords, search_global_indices, desired, config)
    return indices, distances, tolerance, expansions


def discover_candidate_pools(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig, pool_multiplier: int = 1, requested_k: Optional[int] = None) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:
    p = config.N_DESIGN_COMPONENTS
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    targets = target_instances[target_cols].to_numpy(dtype=np.float64)
    search_scores = scores[search_indices]; search_coords = coords[search_indices]; pc_tree = cKDTree(search_scores)
    target_keys = [tuple(row) for row in np.round(targets, 12)]
    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}
    unique_sequences: Dict[Tuple[float, ...], Tuple[np.ndarray, np.ndarray, float, int, bool]] = {}
    pools: List[np.ndarray] = []; pool_distances: List[np.ndarray] = []; tolerances: List[float] = []; records: List[Dict[str, Any]] = []
    base_k = int(requested_k if requested_k is not None else config.CANDIDATES_PER_TARGET * pool_multiplier)
    for t, target in enumerate(targets):
        key = target_keys[t]; desired = max(base_k, multiplicities[key])
        if key not in unique_sequences or len(unique_sequences[key][0]) < desired:
            unique_sequences[key] = candidate_pool_for_target_incremental(target, pc_tree, search_scores, search_coords, search_indices, desired, config)
        indices, distances, tolerance, expansions, fallback = unique_sequences[key]
        pools.append(indices[:desired]); pool_distances.append(distances[:desired]); tolerances.append(float(tolerance))
        for rank, (index, distance) in enumerate(zip(indices[:desired], distances[:desired]), start=1):
            row = {"target_position": t, "target_instance_id": target_instances.iloc[t]["target_instance_id"], "base_target_id": target_instances.iloc[t]["base_target_id"], "target_type": target_instances.iloc[t]["target_type"], "sample_role": target_instances.iloc[t].get("sample_role", "response_surface"), "candidate_analysis_index": int(index), "candidate_rank": rank, "pca_target_distance": float(distance), "final_tolerance_used": float(tolerance), "tolerance_expansions": int(expansions), "nearest_k_fallback_implied": bool(fallback), "candidate_sequence_shared_for_replicated_target": bool(multiplicities[key] > 1)}
            row.update({c: float(target_instances.iloc[t][c]) for c in target_cols})
            records.append(row)
    return pools, pool_distances, pd.DataFrame(records), tolerances


def discover_candidate_pools_saturated(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float], pd.DataFrame, int, int]:
    if config.CANDIDATE_EXPLORATION_MODE == "fixed":
        k_values = [int(config.CANDIDATES_PER_TARGET)]
    else:
        k = max(int(config.CANDIDATE_SATURATION_START_K), int(config.CANDIDATES_PER_TARGET)); k_values = []
        while k <= int(config.CANDIDATE_SATURATION_MAX_K):
            k_values.append(k); k *= int(config.CANDIDATE_SATURATION_GROWTH_FACTOR)
        if k_values[-1] != int(config.CANDIDATE_SATURATION_MAX_K):
            k_values = sorted(set(k_values + [int(config.CANDIDATE_SATURATION_MAX_K)]))
    previous_unique = None; final = None; records = []; assignment_attempts = 0
    for stage, k in enumerate(k_values, start=1):
        pools, distances, associations, tolerances = discover_candidate_pools(target_instances, search_indices, scores, coords, config, requested_k=int(k))
        assignment_possible = assignment_from_costs(pools, distances) is not None
        if assignment_possible and assignment_attempts == 0:
            assignment_attempts = stage
        unique_candidates = int(len(np.unique(np.concatenate(pools)))) if pools else 0
        added = None if previous_unique is None else unique_candidates - previous_unique
        stable = bool(previous_unique is not None and added == 0 and assignment_possible)
        records.append({"stage": int(stage), "requested_K": int(k), "candidate_pools_nested": True, "unique_candidate_observations": unique_candidates, "assignment_possible": bool(assignment_possible), "targets_below_requested_K": int(sum(len(pool) < k for pool in pools)), "added_unique_candidates_from_previous": added, "stable_stage": stable, "confirmation_stage": stable, "maximum_K_reached": bool(k == max(k_values))})
        final = (pools, distances, associations, tolerances); previous_unique = unique_candidates
        if stable and stage < len(k_values):
            break
    if final is None or assignment_from_costs(final[0], final[1]) is None:
        raise RuntimeError("No unique target-to-site assignment was possible after adaptive candidate expansion.")
    table = pd.DataFrame(records)
    return final[0], final[1], final[2], final[3], table, max(1, assignment_attempts), int(table["requested_K"].iloc[-1])



def _start_designs(minimum_distance_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], rng: np.random.Generator, warm_starts: Optional[Sequence[np.ndarray]] = None) -> Iterable[Tuple[str, np.ndarray]]:
    seen: set[Tuple[int, ...]] = set()
    for label, design in [("minimum_pca_distance", minimum_distance_start), *[("warm_start", w) for w in (warm_starts or [])]]:
        if design is None:
            continue
        arr = np.asarray(design, dtype=int); key = tuple(arr.tolist())
        if len(np.unique(arr)) == len(arr) and key not in seen:
            seen.add(key); yield label, arr.copy()
    while True:
        arr = assignment_from_costs(pools, pool_distances, rng)
        if arr is None:
            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")
        key = tuple(arr.tolist())
        if key not in seen:
            seen.add(key); yield "random_unique_assignment", arr.copy()




def _panel_sort_key(record: Dict[str, Any], objective_type: str) -> Tuple[Any, ...]:
    m = record["metrics"]
    return (m["SBAD"], -m["geoMSD"], -m["minimum_separation"], m["mean_pca_target_distance"], m["max_pca_target_distance"]) if objective_type == "ad_reference" else (-m["geoMSD"], -m["minimum_separation"], m["mean_pca_target_distance"], m["max_pca_target_distance"], m["SBAD"])




def _support_sizes(config: RSSDConfig, max_available: int) -> List[int]:
    if config.AD_SUPPORT_MODE == "fixed":
        return [min(int(config.AD_SUPPORT_FIXED_SIZE), max_available)]
    sizes = []; size = min(int(config.AD_SUPPORT_START_SIZE), max_available); limit = min(int(config.AD_SUPPORT_MAX_SIZE), max_available)
    while True:
        sizes.append(size)
        if size >= limit:
            break
        size = min(size * int(config.AD_SUPPORT_GROWTH_FACTOR), limit)
        if size == sizes[-1]:
            break
    return sorted(set(int(s) for s in sizes))


def export_selected_sites_kmz(selected_sites: pd.DataFrame, config: RSSDConfig, source_epsg: int, output_path: str = "ESAP_RSSD_selected_sites.kmz") -> Dict[str, Any]:
    import html, zipfile
    try:
        from pyproj import CRS, Transformer
    except ImportError:
        return {"created": False, "reason": "pyproj is not installed"}
    if not source_epsg or int(source_epsg) <= 0:
        raise ValueError("A valid projected source EPSG code is required for KMZ export.")
    source_crs = CRS.from_epsg(int(source_epsg))
    if not source_crs.is_projected:
        raise ValueError("KMZ source EPSG must describe a projected CRS.")
    transformer = Transformer.from_crs(source_crs, CRS.from_epsg(4326), always_xy=True)
    longitude, latitude = transformer.transform(pd.to_numeric(selected_sites[config.X_COLUMN]).to_numpy(float), pd.to_numeric(selected_sites[config.Y_COLUMN]).to_numpy(float))
    placemarks: List[str] = []
    for position, (_, row) in enumerate(selected_sites.iterrows(), start=1):
        site_id = str(row.get(config.ID_COLUMN, position)); role = str(row.get("sample_role", "response_surface")); role_label = "Spatial support site" if role == "spatial_support" else "Response-surface site"
        label = f"{position:02d} - {site_id}"
        details = {"Sample order": position, "Sample ID": site_id, "Sample role": role_label, "Target instance": row.get("target_instance_id", ""), "Target type": row.get("target_type", ""), "PCA target distance": row.get("pca_target_distance", ""), "Nearest selected neighbor": row.get("nearest_selected_geographic_neighbor_distance", ""), "Projected X": row.get(config.X_COLUMN, ""), "Projected Y": row.get(config.Y_COLUMN, ""), "Source EPSG": int(source_epsg)}
        description_rows = "".join(f"<tr><th align='left'>{html.escape(str(k))}</th><td>{html.escape(str(v))}</td></tr>" for k, v in details.items())
        placemarks.append("<Placemark>" f"<name>{html.escape(label)}</name><styleUrl>#rssdSample</styleUrl>" f"<description><![CDATA[<table>{description_rows}</table>]]></description>" f"<ExtendedData><Data name='sample_id'><value>{html.escape(site_id)}</value></Data><Data name='sample_role'><value>{html.escape(role_label)}</value></Data></ExtendedData>" f"<Point><coordinates>{float(longitude[position - 1]):.10f},{float(latitude[position - 1]):.10f},0</coordinates></Point></Placemark>")
    kml = "<?xml version='1.0' encoding='UTF-8'?><kml xmlns='http://www.opengis.net/kml/2.2'><Document><name>ESAP RSSD v2.10 selected sites</name><Style id='rssdSample'><IconStyle><color>ff0000ff</color><scale>1.1</scale><Icon><href>http://maps.google.com/mapfiles/kml/pushpin/red-pushpin.png</href></Icon></IconStyle><LabelStyle><color>ff000000</color><scale>0.9</scale></LabelStyle></Style>" + "".join(placemarks) + "</Document></kml>"
    destination = Path(output_path)
    with zipfile.ZipFile(destination, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("doc.kml", kml.encode("utf-8"))
    return {"created": True, "file": str(destination), "source_epsg": int(source_epsg), "output_crs": "EPSG:4326", "placemark_count": int(len(selected_sites))}







def _save_or_show_figure(fig: Any, name: str, output_dir: Optional[Path], show: bool, saved: List[str]) -> None:
    fig.tight_layout()
    if output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        path = output_dir / name
        fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
        saved.append(str(path))
    if show:
        plt.show()
    else:
        plt.close(fig)


def create_esap_rssd_figures(result: RSSDRunResult, output_dir: Optional[Any] = None, show: bool = True) -> List[str]:
    """Create live and bundle figures for v2.10 maps, legacy diagnostics, and SBAD diagnostics."""
    cfg_local = result.config
    output_path = Path(output_dir) if output_dir is not None else None
    saved: List[str] = []
    selected = result.selected_sites.copy()
    candidates = result.candidate_sites.copy()
    support = result.support_sequence.copy()
    coverage = result.field_coverage_distances.copy()
    diag = result.diagnostic_data or {}
    coords_all = diag.get("coords")
    scores_all = diag.get("design_scores")
    eligible = diag.get("eligible_indices")
    original_features = diag.get("original_features")
    xcol, ycol = cfg_local.X_COLUMN, cfg_local.Y_COLUMN
    p = int(cfg_local.N_DESIGN_COMPONENTS)

    # 01: full survey/support footprint and final selected sites.
    fig, ax = plt.subplots(figsize=(9, 7))
    if coords_all is not None:
        ax.scatter(coords_all[:, 0], coords_all[:, 1], s=2, alpha=0.10, color=cfg_local.FOOTPRINT_COLOR, rasterized=True, label="Survey footprint")
    if len(support):
        final_support = support[support.get("used_in_final_support_prefix", False).astype(bool)] if "used_in_final_support_prefix" in support else support
        ax.scatter(final_support["X"], final_support["Y"], s=8, alpha=0.20, color=cfg_local.SUPPORT_COLOR, label="SBAD support prefix")
    for role, marker, color, label in [("response_surface", "x", cfg_local.SELECTED_COLOR, "Response-surface sites"), ("spatial_support", "^", cfg_local.SUPPORT_COLOR, "Spatial support sites")]:
        part = selected[selected["sample_role"] == role]
        if len(part):
            ax.scatter(part[xcol], part[ycol], marker=marker, s=90 if marker == "x" else 70, color=color, edgecolor="black" if marker != "x" else None, label=label)
    if len(selected) <= 60:
        for _, row in selected.iterrows():
            ax.annotate(str(row["sample_order"]), (row[xcol], row[ycol]), xytext=(3, 3), textcoords="offset points", fontsize=7)
    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Final ESAP RSSD v2.10 field map")
    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")
    _save_or_show_figure(fig, "01_final_field_map.png", output_path, show, saved)

    # 02: candidates and selected sites.
    fig, ax = plt.subplots(figsize=(9, 7))
    if len(candidates):
        unique_candidates = candidates.drop_duplicates("candidate_analysis_index")
        ax.scatter(unique_candidates[xcol], unique_candidates[ycol], s=12, alpha=0.35, color=cfg_local.CANDIDATE_COLOR, label="Candidate observations")
    ax.scatter(selected[xcol], selected[ycol], marker="x", s=80, color=cfg_local.SELECTED_COLOR, label="Final selected")
    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Candidate pool and final selected locations")
    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")
    _save_or_show_figure(fig, "02_candidates_and_selected_map.png", output_path, show, saved)

    # 03: response-surface target matches in PC space.
    response = selected[selected["sample_role"] == "response_surface"].copy()
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    selected_pc_cols = [f"selected_PC{j + 1}" for j in range(p)]
    for col in target_cols + selected_pc_cols:
        if col in response:
            response[col] = pd.to_numeric(response[col], errors="coerce")
    fig, ax = plt.subplots(figsize=(8, 6 if p > 1 else 4.5))
    if p == 1:
        ax.scatter(response["target_PC1"], np.zeros(len(response)), marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")
        ax.scatter(response["selected_PC1"], np.zeros(len(response)), s=40, color=cfg_local.SELECTED_COLOR, label="Selected")
        for _, row in response.iterrows():
            ax.plot([row["target_PC1"], row["selected_PC1"]], [0, 0], linewidth=0.8, alpha=0.55)
        ax.set_yticks([]); ax.set_xlabel("Standardized PC1")
    else:
        ax.scatter(response["target_PC1"], response["target_PC2"], marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")
        ax.scatter(response["selected_PC1"], response["selected_PC2"], s=42, color=cfg_local.SELECTED_COLOR, label="Selected response-surface sites")
        for _, row in response.iterrows():
            ax.plot([row["target_PC1"], row["selected_PC1"]], [row["target_PC2"], row["selected_PC2"]], linewidth=0.8, alpha=0.55)
        ax.set_xlabel("Standardized PC1"); ax.set_ylabel("Standardized PC2"); ax.set_aspect("equal", adjustable="box")
    ax.set_title("Response-surface target matches"); ax.legend(loc="best")
    _save_or_show_figure(fig, "03_response_surface_target_matches.png", output_path, show, saved)

    # 04: selected versus whole eligible PC boxplots, restored from v2.8 style.
    if scores_all is not None and eligible is not None:
        fig, axes = plt.subplots(1, p, figsize=(4.8 * p, 5), squeeze=False)
        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)
        for j in range(p):
            axes[0, j].boxplot([scores_all[eligible, j], scores_all[selected_indices, j]], patch_artist=True)
            axes[0, j].set_xticks([1, 2], ["Whole eligible", "Selected"])
            axes[0, j].set_title(f"Standardized PC{j + 1}")
        fig.suptitle("PC distributions: whole eligible survey vs selected")
        _save_or_show_figure(fig, "04_pc_boxplots_whole_vs_selected.png", output_path, show, saved)

    # 05: original feature boxplots, restored from v2.8 style.
    if original_features is not None and eligible is not None:
        feature_names = list(diag.get("feature_columns", cfg_local.FEATURE_COLUMNS))
        n_features = len(feature_names); ncols = min(3, n_features); nrows = math.ceil(n_features / ncols)
        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows), squeeze=False)
        for j, feature in enumerate(feature_names):
            axes.flat[j].boxplot([original_features[eligible, j], original_features[selected_indices, j]], patch_artist=True)
            axes.flat[j].set_xticks([1, 2], ["Whole eligible", "Selected"])
            axes.flat[j].set_title(str(feature))
        for ax in axes.flat[n_features:]:
            ax.set_visible(False)
        fig.suptitle("Sensor features: whole eligible survey vs selected")
        _save_or_show_figure(fig, "05_feature_boxplots_whole_vs_selected.png", output_path, show, saved)

    # 06: SBAD support-resolution diagnostics.
    ad = result.ad_support_resolution.copy()
    if len(ad):
        fig, ax1 = plt.subplots(figsize=(8.5, 5.2))
        sbad_star_col = "SBAD_star" if "SBAD_star" in ad.columns else "SBAD_star_screen"
        ax1.plot(ad["support_size"], ad[sbad_star_col], marker="o", label="AD-reference SBAD*")
        ax1.plot(ad["support_size"], ad["hybrid_core_SBAD"], marker="o", label="Hybrid core SBAD")
        ax1.set_xlabel("Nested support prefix size"); ax1.set_ylabel("SBAD (projected units)")
        ax2 = ax1.twinx()
        ax2.plot(ad["support_size"], ad["core_coverage_ratio"], marker="s", linestyle="--", color="black", label="Core coverage ratio")
        ax2.axhline(1.0 + cfg_local.AD_COVERAGE_ENVELOPE_REL_TOL, linestyle=":", color="black", linewidth=1)
        ax2.set_ylabel("SBAD / SBAD*")
        lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
        ax1.set_title("Adaptive SBAD support-resolution diagnostics")
        _save_or_show_figure(fig, "06_sbad_support_resolution.png", output_path, show, saved)

    # 07: final field coverage distance distribution.
    if len(coverage):
        d = pd.to_numeric(coverage["nearest_final_site_distance"], errors="coerce").dropna().to_numpy(float)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
        ax1.hist(d, bins=30, color=cfg_local.SUPPORT_COLOR, alpha=0.8)
        ax1.axvline(np.mean(d), linestyle="--", color="black", label=f"SBAD={np.mean(d):.3g}")
        ax1.set_xlabel("Nearest final-site distance"); ax1.set_ylabel("Support coordinates"); ax1.set_title("Field coverage distances"); ax1.legend()
        sorted_d = np.sort(d); ax2.plot(sorted_d, np.linspace(0, 1, len(sorted_d), endpoint=True))
        ax2.set_xlabel("Nearest final-site distance"); ax2.set_ylabel("Cumulative fraction"); ax2.set_title("Coverage-distance CDF")
        _save_or_show_figure(fig, "07_field_coverage_distance_distribution.png", output_path, show, saved)

    # 08: spatial support additions.
    additions = result.spatial_support_sites.copy()
    if len(additions):
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        x = additions["support_addition_order"].to_numpy(int)
        ax.plot(x - 1, additions["SBAD_before_addition"], marker="o", label="Before addition")
        ax.plot(x, additions["SBAD_after_addition"], marker="o", label="After addition")
        ax.set_xlabel("Spatial support addition step"); ax.set_ylabel("Final-design SBAD")
        ax.set_title("Sequential spatial support-site SBAD reduction"); ax.legend()
        _save_or_show_figure(fig, "08_spatial_support_sbad_reduction.png", output_path, show, saved)

    # 09: nearest selected-neighbor distances.
    if "nearest_selected_geographic_neighbor_distance" in selected:
        nn = pd.to_numeric(selected["nearest_selected_geographic_neighbor_distance"], errors="coerce").dropna().to_numpy(float)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(nn, bins=min(15, max(5, len(nn) // 2)), color=cfg_local.SELECTED_COLOR, alpha=0.85)
        ax.axvline(float(np.exp(np.mean(np.log(nn[nn > 0])))), linestyle="--", color="black", label="geoMSD")
        ax.set_xlabel("Nearest selected-neighbor distance"); ax.set_ylabel("Selected sites"); ax.set_title("Selected-site nearest-neighbor distances"); ax.legend()
        _save_or_show_figure(fig, "09_nearest_neighbor_distances.png", output_path, show, saved)

    # 10: optimizer stability traces.
    opt = result.optimizer_stability.copy()
    if len(opt):
        final_support = opt["support_size"].max()
        opt_final = opt[opt["support_size"] == final_support]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
        for objective, part in opt_final.groupby("objective_type"):
            ax1.plot(part["start_number"], part["final_SBAD"], marker=".", label=objective)
            ax2.plot(part["start_number"], part["final_geoMSD"], marker=".", label=objective)
        ax1.set_xlabel("Optimizer start"); ax1.set_ylabel("Final SBAD"); ax1.set_title("Optimizer SBAD trace")
        ax2.set_xlabel("Optimizer start"); ax2.set_ylabel("Final geoMSD"); ax2.set_title("Optimizer geoMSD trace")
        ax1.legend(); ax2.legend()
        _save_or_show_figure(fig, "10_optimizer_stability_traces.png", output_path, show, saved)

    # 11: proxy spatial-scale diagnostic.
    proxy = result.proxy_spatial_scale.copy()
    if len(proxy):
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for pc_name, part in proxy.groupby("PC"):
            ax.plot(part["distance_median"], part["fraction_of_far_distance_semivariance"], marker="o", label=pc_name)
        ax.axhline(cfg_local.SPACING_TARGET_SEMIVARIANCE_FRACTION, linestyle="--", color="black", linewidth=1, label="Target fraction")
        ax.set_xlabel("Pair distance (projected units)"); ax.set_ylabel("Fraction of far-distance semivariance")
        ax.set_title("Per-PC proxy spatial-scale diagnostic")
        ax.legend(loc="best")
        _save_or_show_figure(fig, "11_proxy_spatial_scale.png", output_path, show, saved)

    return saved


# --- ESAP RSSD v2.10 targeted overrides ---


class SupportDistanceCache:
    """Shared lazy cache of maximum-support-to-candidate distance vectors."""
    def __init__(self, support_coords: np.ndarray, coords: np.ndarray, max_mib: int = 512):
        self.support_coords = np.asarray(support_coords, dtype=np.float64)
        self.coords = np.asarray(coords, dtype=np.float64)
        self.max_bytes = int(max_mib) * 1024 * 1024
        self.cache: "OrderedDict[int, np.ndarray]" = OrderedDict()
        self.bytes = 0
        self.cache_hits = 0
        self.cache_misses = 0
        self.cache_evictions = 0
        self.peak_cached_vectors = 0
        self.peak_cache_bytes = 0
    def get(self, candidate_index: int) -> np.ndarray:
        key = int(candidate_index)
        if key in self.cache:
            self.cache_hits += 1
            value = self.cache.pop(key)
            self.cache[key] = value
            return value
        self.cache_misses += 1
        delta = self.support_coords - self.coords[key]
        dist = np.sqrt(np.einsum("ij,ij->i", delta, delta)).astype(np.float32, copy=False)
        self.cache[key] = dist
        self.bytes += dist.nbytes
        while self.bytes > self.max_bytes and len(self.cache) > 1:
            _, old = self.cache.popitem(last=False)
            self.bytes -= old.nbytes
            self.cache_evictions += 1
        self.peak_cached_vectors = max(self.peak_cached_vectors, len(self.cache))
        self.peak_cache_bytes = max(self.peak_cache_bytes, self.bytes)
        return dist
    def get_prefix(self, candidate_index: int, prefix_size: int) -> np.ndarray:
        return self.get(int(candidate_index))[:int(prefix_size)]
    def snapshot(self) -> Dict[str, Any]:
        return {
            "cache_hits": int(self.cache_hits),
            "cache_misses": int(self.cache_misses),
            "cache_evictions": int(self.cache_evictions),
            "peak_cached_vectors": int(self.peak_cached_vectors),
            "peak_cache_mib": float(self.peak_cache_bytes / (1024 * 1024)),
            "current_cached_vectors": int(len(self.cache)),
            "current_cache_mib": float(self.bytes / (1024 * 1024)),
        }


class SBADNearestState:
    """Nearest/second-nearest fixed-prefix support distances backed by the shared cache."""
    def __init__(self, cache: SupportDistanceCache, selected: np.ndarray, prefix_size: int):
        self.cache = cache
        self.selected = np.asarray(selected, dtype=int).copy()
        self.prefix_size = int(prefix_size)
        self.recompute()
    def recompute(self) -> None:
        dmat = np.vstack([self.cache.get_prefix(int(idx), self.prefix_size).astype(np.float64, copy=False) for idx in self.selected]).T
        if dmat.shape[1] == 1:
            self.nearest_distance = dmat[:, 0]
            self.second_distance = np.full(self.prefix_size, np.inf, dtype=np.float64)
            self.nearest_position = np.zeros(self.prefix_size, dtype=int)
        else:
            order = np.argpartition(dmat, kth=1, axis=1)[:, :2]
            first = order[np.arange(self.prefix_size), np.argmin(dmat[np.arange(self.prefix_size)[:, None], order], axis=1)]
            nearest_vals = dmat[np.arange(self.prefix_size), first]
            dmat2 = dmat.copy()
            dmat2[np.arange(self.prefix_size), first] = np.inf
            second = np.min(dmat2, axis=1)
            self.nearest_distance = nearest_vals
            self.second_distance = second
            self.nearest_position = first.astype(int)
        self.sbad = float(np.mean(self.nearest_distance, dtype=np.float64))
    def evaluate_swap(self, position: int, candidate_index: int) -> float:
        candidate_distance = self.cache.get_prefix(int(candidate_index), self.prefix_size).astype(np.float64, copy=False)
        baseline = np.where(self.nearest_position == int(position), self.second_distance, self.nearest_distance)
        return float(np.mean(np.minimum(baseline, candidate_distance), dtype=np.float64))


def exact_sbad_cached(cache: SupportDistanceCache, selected: np.ndarray, prefix_size: int) -> float:
    return SBADNearestState(cache, np.asarray(selected, dtype=int), int(prefix_size)).sbad


def _objective_seed_offset(objective_type: str) -> int:
    return {"ad_reference": 101, "hybrid": 211, "candidate_ad": 307, "candidate_hybrid": 401}.get(str(objective_type), 503)


class OptimizationStartBank:
    """Reusable start assignments independent of SBAD support size."""
    def __init__(self, minimum_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], random_seed: int, candidate_k: int, objective_type: str, max_starts: int):
        self.minimum_start = np.asarray(minimum_start, dtype=int).copy()
        self.pools = pools
        self.pool_distances = pool_distances
        self.random_seed = int(random_seed)
        self.candidate_k = int(candidate_k)
        self.objective_type = str(objective_type)
        self.max_starts = int(max_starts)
        self.base_starts: List[np.ndarray] = []
        self._generate()
    def _generate(self) -> None:
        rng = np.random.default_rng(self.random_seed + 1009 * self.candidate_k + _objective_seed_offset(self.objective_type))
        seen: set[Tuple[int, ...]] = set()
        def add(arr: np.ndarray) -> None:
            key = tuple(int(x) for x in arr.tolist())
            if len(np.unique(arr)) == len(arr) and key not in seen:
                seen.add(key); self.base_starts.append(np.asarray(arr, dtype=int).copy())
        add(self.minimum_start)
        attempts = 0
        while len(self.base_starts) < self.max_starts and attempts < self.max_starts * 50:
            attempts += 1
            arr = assignment_from_costs(self.pools, self.pool_distances, rng)
            if arr is not None:
                add(arr)
    def starts(self, requested: int, warm_starts: Optional[Sequence[np.ndarray]] = None) -> List[Tuple[str, np.ndarray]]:
        out: List[Tuple[str, np.ndarray]] = []
        seen: set[Tuple[int, ...]] = set()
        for w in warm_starts or []:
            if w is None: continue
            arr = np.asarray(w, dtype=int)
            key = tuple(int(x) for x in arr.tolist())
            if len(np.unique(arr)) == len(arr) and key not in seen:
                seen.add(key); out.append(("warm_start", arr.copy()))
        for i, arr in enumerate(self.base_starts):
            key = tuple(int(x) for x in arr.tolist())
            if key not in seen:
                seen.add(key); out.append(("minimum_pca_distance" if i == 0 else "random_unique_assignment", arr.copy()))
            if len(out) >= int(requested):
                break
        return out[:int(requested)]
    def base_random_keys(self, n: int) -> List[Tuple[int, ...]]:
        return [tuple(int(x) for x in arr.tolist()) for arr in self.base_starts[:int(n)]]


def design_metrics_cached(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, prefix_size: int, sbad: Optional[float] = None) -> Dict[str, float]:
    mean_pc, max_pc = matching_metrics(selected, targets, scores)
    selected_coords = coords[np.asarray(selected, dtype=int)]
    return {
        "SBAD": float(sbad) if sbad is not None else exact_sbad_cached(cache, selected, prefix_size),
        "geoMSD": exact_geo_msd(selected_coords),
        "minimum_separation": minimum_selected_separation(selected_coords),
        "mean_pca_target_distance": float(mean_pc),
        "max_pca_target_distance": float(max_pc),
    }


def _relative_change(new: Optional[float], old: Optional[float]) -> Optional[float]:
    if new is None or old is None or not np.isfinite(new) or not np.isfinite(old):
        return None
    return float(abs(float(new) - float(old)) / max(1e-12, abs(float(old))))


def focused_ad_refinement_improvement(pre_refinement_best_sbad: Optional[float], post_refinement_best_sbad: Optional[float], numerical_epsilon: float = 1e-12) -> float:
    """Relative improvement from focused AD refinement at a fixed support prefix."""
    if pre_refinement_best_sbad is None or post_refinement_best_sbad is None:
        return 0.0
    pre = float(pre_refinement_best_sbad)
    post = float(post_refinement_best_sbad)
    if not np.isfinite(pre) or not np.isfinite(post):
        return 0.0
    return float(max(0.0, (pre - post) / max(abs(pre), float(numerical_epsilon))))


def support_confirmation_transition(stable_stage: bool, stable_count: int, confirmation_pending: bool, has_larger_prefix: bool, required_stable_stages: int) -> Dict[str, Any]:
    """Advance the adaptive SBAD support-resolution confirmation state machine."""
    required = max(1, int(required_stable_stages))
    confirmation_stage = bool(confirmation_pending)
    stable_count_after = int(stable_count)
    confirmation_pending_next = False
    stop = False
    resolution_stable = False

    if confirmation_stage:
        if stable_stage:
            stable_count_after += 1
            stop = True
            resolution_stable = True
            status = "confirmed"
        else:
            stable_count_after = 0
            status = "confirmation_failed_reset" if has_larger_prefix else "confirmation_failed_at_max"
            stop = not bool(has_larger_prefix)
        return {
            "confirmation_stage": True,
            "stable_count": int(stable_count_after),
            "confirmation_pending_next": bool(confirmation_pending_next),
            "stop": bool(stop),
            "ad_support_resolution_stable": bool(resolution_stable),
            "support_resolution_status": status,
        }

    stable_count_after = stable_count_after + 1 if stable_stage else 0
    if stable_count_after >= required:
        if has_larger_prefix:
            confirmation_pending_next = True
            status = "provisional_stability_reached"
        else:
            stop = True
            status = "stable_at_max_without_separate_confirmation"
    elif not has_larger_prefix:
        stop = True
        status = "support_limit_reached_without_provisional_stability"
    else:
        status = "expanding"

    return {
        "confirmation_stage": False,
        "stable_count": int(stable_count_after),
        "confirmation_pending_next": bool(confirmation_pending_next),
        "stop": bool(stop),
        "ad_support_resolution_stable": False,
        "support_resolution_status": status,
    }


def support_confirmation_state_machine_preview(sizes: Sequence[int], stable_flags: Sequence[bool], required_stable_stages: int) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Small deterministic preview used by unit validations for the support-confirmation state machine."""
    stable_count = 0
    confirmation_pending = False
    rows: List[Dict[str, Any]] = []
    final_size = int(sizes[-1]) if sizes else 0
    final_stable = False
    status = "not_started"
    for i, size in enumerate(sizes):
        transition = support_confirmation_transition(
            bool(stable_flags[i]),
            stable_count,
            confirmation_pending,
            i < len(sizes) - 1,
            required_stable_stages,
        )
        stable_count = int(transition["stable_count"])
        confirmation_pending = bool(transition["confirmation_pending_next"])
        status = str(transition["support_resolution_status"])
        final_size = int(size)
        final_stable = bool(transition["ad_support_resolution_stable"])
        rows.append({
            "support_size": int(size),
            "stable_stage": bool(stable_flags[i]),
            "confirmation_stage": bool(transition["confirmation_stage"]),
            "provisional_stable_count": int(stable_count),
            "support_resolution_status": status,
        })
        if transition["stop"]:
            break
    return rows, {
        "final_support_size": int(final_size),
        "ad_support_resolution_stable": bool(final_stable),
        "support_resolution_status": status,
    }


def _site_set_overlap(a: Optional[np.ndarray], b: Optional[np.ndarray]) -> Optional[float]:
    if a is None or b is None: return None
    sa, sb = set(map(int, np.asarray(a).tolist())), set(map(int, np.asarray(b).tolist()))
    return float(len(sa & sb) / max(1, len(sa | sb)))




def _candidate_k_values(config: RSSDConfig) -> List[int]:
    if config.CANDIDATE_EXPLORATION_MODE == "fixed":
        return [int(config.CANDIDATES_PER_TARGET)]
    vals=[]; k=max(int(config.CANDIDATE_SATURATION_START_K), int(config.CANDIDATES_PER_TARGET))
    while k <= int(config.CANDIDATE_SATURATION_MAX_K):
        vals.append(k); k *= int(config.CANDIDATE_SATURATION_GROWTH_FACTOR)
    if vals[-1] != int(config.CANDIDATE_SATURATION_MAX_K):
        vals=sorted(set(vals+[int(config.CANDIDATE_SATURATION_MAX_K)]))
    return vals



def candidate_sequence_bruteforce_reference(target: np.ndarray, search_indices: np.ndarray, search_scores: np.ndarray, search_coords: np.ndarray, config: RSSDConfig, kmax: int) -> Dict[str, Any]:
    """Validation-only brute-force greedy candidate sequence with preserved tolerance shells."""
    target = np.asarray(target, dtype=np.float64)
    pc_tree = cKDTree(search_scores)
    retained_local: List[int] = []
    retained_set: set[int] = set()
    available_set: set[int] = set()
    available_tol: Dict[int, float] = {}
    available_stage: Dict[int, int] = {}
    added_tol: Dict[int, float] = {}
    added_stage: Dict[int, int] = {}
    tolerance = float(config.PC_CANDIDATE_TOLERANCE)
    expansion_stage = 0
    fallback_used = False

    def add_available(local_indices: np.ndarray, tol: float, stage: int) -> None:
        for value in np.asarray(local_indices, dtype=int).tolist():
            value = int(value)
            if value not in retained_set and value not in available_set:
                available_set.add(value)
                available_tol[value] = float(tol)
                available_stage[value] = int(stage)

    add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)
    while len(retained_local) < int(kmax):
        choices = np.array(sorted(available_set - retained_set), dtype=int)
        while len(choices) == 0:
            if tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:
                tolerance = min(float(config.MAX_PC_CANDIDATE_TOLERANCE), tolerance * float(config.CANDIDATE_TOLERANCE_GROWTH))
                expansion_stage += 1
                add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)
                choices = np.array(sorted(available_set - retained_set), dtype=int)
                continue
            kq = min(len(search_scores), max(int(kmax), len(retained_local) + 1))
            _, nearest_i = pc_tree.query(target, k=kq)
            add_available(np.atleast_1d(nearest_i).astype(int), tolerance, expansion_stage)
            fallback_used = True
            choices = np.array(sorted(available_set - retained_set), dtype=int)
            if len(choices) == 0:
                break
        if len(choices) == 0:
            break
        pc_dist = np.linalg.norm(search_scores[choices] - target, axis=1)
        if not retained_local:
            order = np.lexsort((search_indices[choices], pc_dist))
        else:
            retained_coords = search_coords[np.asarray(retained_local, dtype=int)]
            d = np.sqrt(((search_coords[choices, None, :] - retained_coords[None, :, :]) ** 2).sum(axis=2))
            min_geo = np.min(d, axis=1)
            order = np.lexsort((search_indices[choices], pc_dist, -min_geo))
        chosen = int(choices[int(order[0])])
        retained_local.append(chosen)
        retained_set.add(chosen)
        added_tol[chosen] = float(available_tol.get(chosen, tolerance))
        added_stage[chosen] = int(available_stage.get(chosen, expansion_stage))
    return {
        "indices": search_indices[np.asarray(retained_local, dtype=int)].astype(int),
        "distances": np.linalg.norm(search_scores[np.asarray(retained_local, dtype=int)] - target, axis=1).astype(float) if retained_local else np.array([], dtype=float),
        "tolerances": np.array([added_tol[i] for i in retained_local], dtype=float),
        "expansion_stages": np.array([added_stage[i] for i in retained_local], dtype=int),
        "fallback_used": bool(fallback_used),
    }


def build_nested_candidate_sequences(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig) -> Tuple[Dict[Tuple[float, ...], Dict[str, Any]], pd.DataFrame, Dict[str, Any]]:
    """Build one exact incremental min_geo candidate sequence per unique target coordinate."""
    p = config.N_DESIGN_COMPONENTS
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    targets = target_instances[target_cols].to_numpy(dtype=np.float64)
    search_indices = np.asarray(search_indices, dtype=int)
    search_scores = scores[search_indices]
    search_coords = coords[search_indices]
    pc_tree = cKDTree(search_scores)
    target_keys = [tuple(row) for row in np.round(targets, 12)]
    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}
    kmax = max(int(config.CANDIDATE_SATURATION_MAX_K), max(multiplicities.values()))
    sequences: Dict[Tuple[float, ...], Dict[str, Any]] = {}
    records: List[Dict[str, Any]] = []

    for key in sorted(multiplicities.keys()):
        target = np.asarray(key, dtype=np.float64)
        retained_local: List[int] = []
        retained_set: set[int] = set()
        available_set: set[int] = set()
        available_tol: Dict[int, float] = {}
        available_stage: Dict[int, int] = {}
        min_geo: Dict[int, float] = {}
        added_tol: Dict[int, float] = {}
        added_stage: Dict[int, int] = {}
        tolerance = float(config.PC_CANDIDATE_TOLERANCE)
        expansion_stage = 0
        fallback_used = False

        def add_available(local_indices: np.ndarray, tol: float, stage: int) -> None:
            new_values: List[int] = []
            for value in np.asarray(local_indices, dtype=int).tolist():
                value = int(value)
                if value not in retained_set and value not in available_set:
                    available_set.add(value)
                    available_tol[value] = float(tol)
                    available_stage[value] = int(stage)
                    new_values.append(value)
            if new_values and retained_local:
                new_arr = np.asarray(new_values, dtype=int)
                retained_coords = search_coords[np.asarray(retained_local, dtype=int)]
                d = np.sqrt(((search_coords[new_arr, None, :] - retained_coords[None, :, :]) ** 2).sum(axis=2))
                vals = np.min(d, axis=1)
                for local_idx, value in zip(new_arr.tolist(), vals.tolist()):
                    min_geo[int(local_idx)] = float(value)
            elif new_values:
                for local_idx in new_values:
                    min_geo[int(local_idx)] = float("inf")

        add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)
        while len(retained_local) < kmax:
            choices = np.array(sorted(available_set - retained_set), dtype=int)
            while len(choices) == 0:
                if tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:
                    tolerance = min(float(config.MAX_PC_CANDIDATE_TOLERANCE), tolerance * float(config.CANDIDATE_TOLERANCE_GROWTH))
                    expansion_stage += 1
                    add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)
                    choices = np.array(sorted(available_set - retained_set), dtype=int)
                    continue
                kq = min(len(search_scores), max(kmax, len(retained_local) + 1))
                _, nearest_i = pc_tree.query(target, k=kq)
                add_available(np.atleast_1d(nearest_i).astype(int), tolerance, expansion_stage)
                fallback_used = True
                choices = np.array(sorted(available_set - retained_set), dtype=int)
                if len(choices) == 0:
                    break
            if len(choices) == 0:
                break
            pc_dist_choices = np.linalg.norm(search_scores[choices] - target, axis=1)
            if not retained_local:
                order = np.lexsort((search_indices[choices], pc_dist_choices))
            else:
                current_min_geo = np.array([min_geo[int(i)] for i in choices], dtype=np.float64)
                order = np.lexsort((search_indices[choices], pc_dist_choices, -current_min_geo))
            chosen = int(choices[int(order[0])])
            retained_local.append(chosen)
            retained_set.add(chosen)
            added_tol[chosen] = float(available_tol.get(chosen, tolerance))
            added_stage[chosen] = int(available_stage.get(chosen, expansion_stage))
            remaining = np.array(sorted(available_set - retained_set), dtype=int)
            if len(remaining):
                d_new = np.sqrt(np.sum((search_coords[remaining] - search_coords[chosen]) ** 2, axis=1))
                for local_idx, dist_value in zip(remaining.tolist(), d_new.tolist()):
                    local_idx = int(local_idx)
                    min_geo[local_idx] = float(min(min_geo.get(local_idx, float("inf")), dist_value))

        retained_arr = np.asarray(retained_local, dtype=int)
        indices = search_indices[retained_arr]
        distances = np.linalg.norm(search_scores[retained_arr] - target, axis=1) if len(retained_arr) else np.array([], dtype=float)
        seq = {
            "target": target,
            "indices": indices.astype(int),
            "distances": distances.astype(float),
            "tolerances": np.array([added_tol[i] for i in retained_local], dtype=float),
            "expansion_stages": np.array([added_stage[i] for i in retained_local], dtype=int),
            "fallback_used": bool(fallback_used),
            "multiplicity": int(multiplicities[key]),
        }
        sequences[key] = seq
        for rank, (idx, dist, tol, stage) in enumerate(zip(seq["indices"], seq["distances"], seq["tolerances"], seq["expansion_stages"]), start=1):
            records.append({
                "target_key": str(key),
                "candidate_analysis_index": int(idx),
                "candidate_sequence_rank": int(rank),
                "pca_target_distance": float(dist),
                "candidate_added_at_tolerance": float(tol),
                "candidate_tolerance_expansion_stage": int(stage),
            })

    k_values = _candidate_k_values(config)
    nested = True
    for seq in sequences.values():
        prev: Optional[np.ndarray] = None
        for k in k_values:
            cur = seq["indices"][:min(k, len(seq["indices"]))]
            if prev is not None and not np.array_equal(prev, cur[:len(prev)]):
                nested = False
            prev = cur.copy()
    report = {"candidate_sequences_verified_nested": bool(nested), "unique_base_target_sequences": int(len(sequences)), "candidate_sequence_max_K": int(kmax)}
    return sequences, pd.DataFrame(records), report


def candidate_pools_from_sequences(target_instances: pd.DataFrame, sequences: Dict[Tuple[float, ...], Dict[str, Any]], requested_k: int, config: RSSDConfig) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:
    p = config.N_DESIGN_COMPONENTS
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    targets = target_instances[target_cols].to_numpy(dtype=np.float64)
    target_keys = [tuple(row) for row in np.round(targets, 12)]
    pools=[]; distances=[]; tolerances=[]; records=[]
    for pos, key in enumerate(target_keys):
        seq=sequences[key]
        k=max(int(requested_k), int(seq.get("multiplicity", 1)))
        idx=seq["indices"][:k]; dist=seq["distances"][:k]
        pools.append(idx.astype(int)); distances.append(dist.astype(float)); tolerances.append(float(np.max(seq["tolerances"][:len(idx)])) if len(idx) else np.nan)
        for rank, (candidate_idx, candidate_dist) in enumerate(zip(idx, dist), start=1):
            row={"target_position": int(pos), "target_instance_id": target_instances.iloc[pos]["target_instance_id"], "base_target_id": target_instances.iloc[pos]["base_target_id"], "target_type": target_instances.iloc[pos]["target_type"], "sample_role": target_instances.iloc[pos].get("sample_role", "response_surface"), "candidate_analysis_index": int(candidate_idx), "candidate_rank": int(rank), "candidate_sequence_rank": int(rank), "pca_target_distance": float(candidate_dist), "final_tolerance_used": tolerances[-1], "candidate_added_at_tolerance": float(seq["tolerances"][rank-1]), "candidate_tolerance_expansion_stage": int(seq["expansion_stages"][rank-1])}
            row.update({c: float(target_instances.iloc[pos][c]) for c in target_cols})
            records.append(row)
    return pools, distances, pd.DataFrame(records), tolerances


def _candidate_pool_overlap(pools: Sequence[np.ndarray]) -> int:
    flat=np.concatenate(pools) if pools else np.array([], dtype=int)
    return int(len(flat)-len(np.unique(flat)))


def screen_candidate_saturation(target_instances: pd.DataFrame, sequences: Dict[Tuple[float, ...], Dict[str, Any]], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, support_prefix_size: int, config: RSSDConfig) -> Tuple[int, List[np.ndarray], List[np.ndarray], pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    records=[]; all_opt=[]; previous=None; stable_count=0; selected_k=None; final_pools=None; final_dists=None; final_assoc=None; warm_ad=[]; warm_hybrid=[]
    for stage, k in enumerate(_candidate_k_values(config), start=1):
        stage_start=time.perf_counter()
        pools, pool_dists, assoc, tolerances = candidate_pools_from_sequences(target_instances, sequences, int(k), config)
        assignment=assignment_from_costs(pools, pool_dists)
        possible=assignment is not None
        if not possible:
            records.append({"stage": stage, "requested_K": int(k), "candidate_sequences_verified_nested": True, "unique_candidate_observations": int(len(np.unique(np.concatenate(pools))) if pools else 0), "assignment_possible": False, "targets_below_requested_K": int(sum(len(pool)<k for pool in pools)), "targets_requiring_tolerance_expansion": int(sum(np.nan_to_num(tolerances)>config.PC_CANDIDATE_TOLERANCE)), "maximum_target_tolerance": float(np.nanmax(tolerances)), "SBAD_star": None, "hybrid_core_SBAD": None, "core_coverage_ratio": None, "hybrid_geoMSD": None, "hybrid_minimum_separation": None, "mean_pca_target_distance": None, "maximum_pca_target_distance": None, "relative_change_in_SBAD_star": None, "relative_change_in_hybrid_geoMSD": None, "relative_change_in_hybrid_minimum_separation": None, "AD_exploration_starts": 0, "hybrid_exploration_starts": 0, "stage_runtime_seconds": time.perf_counter()-stage_start, "stable_stage": False, "confirmation_stage": False, "maximum_K_reached": bool(k==_candidate_k_values(config)[-1]), "candidate_space_status": "assignment_not_possible"})
            continue
        ad_best, ad_table, ad_panel = optimize_multiple_starts_sbad(assignment, pools, pool_dists, targets, scores, coords, cache, int(support_prefix_size), config, "ad_reference", candidate_k=int(k), optimization_phase="candidate_screen", start_limit=int(config.CANDIDATE_STAGE_EXPLORATION_STARTS), warm_starts=warm_ad, adaptive=False)
        sbad_star=exact_sbad_cached(cache, ad_best, int(support_prefix_size)); sbad_limit=(1+config.AD_COVERAGE_ENVELOPE_REL_TOL)*sbad_star
        hybrid_best, hy_table, hy_panel = optimize_multiple_starts_sbad(assignment, pools, pool_dists, targets, scores, coords, cache, int(support_prefix_size), config, "hybrid", sbad_limit=sbad_limit, candidate_k=int(k), optimization_phase="candidate_screen", start_limit=int(config.CANDIDATE_STAGE_EXPLORATION_STARTS), warm_starts=[ad_best,*warm_hybrid], adaptive=False)
        hmet=design_metrics_cached(hybrid_best, targets, scores, coords, cache, int(support_prefix_size))
        all_opt.extend([ad_table, hy_table])
        rel_sbad=_relative_change(sbad_star, previous["SBAD_star"] if previous else None)
        rel_geo=_relative_change(hmet["geoMSD"], previous["hybrid_geoMSD"] if previous else None)
        rel_min=_relative_change(hmet["minimum_separation"], previous["hybrid_minimum_separation"] if previous else None)
        stable=bool(previous is not None and (rel_sbad is not None and rel_sbad <= config.CANDIDATE_SATURATION_AD_REL_TOL) and (rel_geo is not None and rel_geo <= config.CANDIDATE_SATURATION_GEOMSD_REL_TOL) and (rel_min is not None and rel_min <= config.CANDIDATE_SATURATION_MINSEP_REL_TOL))
        stable_count = stable_count + 1 if stable else 0
        confirmation=bool(stable_count >= int(config.CANDIDATE_SATURATION_STABLE_STAGES_REQUIRED))
        status="objective_saturated" if confirmation else ("candidate_limit_reached_without_objective_saturation" if k==_candidate_k_values(config)[-1] else "expanding")
        rec={"stage": stage, "requested_K": int(k), "candidate_sequences_verified_nested": True, "unique_candidate_observations": int(len(np.unique(np.concatenate(pools))) if pools else 0), "assignment_possible": True, "targets_below_requested_K": int(sum(len(pool)<k for pool in pools)), "targets_requiring_tolerance_expansion": int(sum(np.nan_to_num(tolerances)>config.PC_CANDIDATE_TOLERANCE)), "maximum_target_tolerance": float(np.nanmax(tolerances)), "SBAD_star": float(sbad_star), "hybrid_core_SBAD": float(hmet["SBAD"]), "core_coverage_ratio": float(hmet["SBAD"]/sbad_star) if sbad_star>0 else np.nan, "hybrid_geoMSD": float(hmet["geoMSD"]), "hybrid_minimum_separation": float(hmet["minimum_separation"]), "mean_pca_target_distance": float(hmet["mean_pca_target_distance"]), "maximum_pca_target_distance": float(hmet["max_pca_target_distance"]), "candidate_pool_overlap": _candidate_pool_overlap(pools), "relative_change_in_SBAD_star": rel_sbad, "relative_change_in_hybrid_geoMSD": rel_geo, "relative_change_in_hybrid_minimum_separation": rel_min, "AD_exploration_starts": int(len(ad_table)), "hybrid_exploration_starts": int(len(hy_table)), "stage_runtime_seconds": float(time.perf_counter()-stage_start), "stable_stage": bool(stable), "confirmation_stage": bool(confirmation), "maximum_K_reached": bool(k==_candidate_k_values(config)[-1]), "candidate_space_status": status}
        records.append(rec)
        selected_k=int(k); final_pools=pools; final_dists=pool_dists; final_assoc=assoc; previous=rec; warm_ad=[ad_best]; warm_hybrid=[hybrid_best]
        if confirmation:
            break
    if selected_k is None or final_pools is None or final_dists is None or final_assoc is None:
        raise RuntimeError("Candidate saturation screening did not produce feasible candidate pools.")
    return selected_k, final_pools, final_dists, final_assoc, pd.DataFrame(records), {"candidate_space_saturated": bool(records[-1].get("candidate_space_status") == "objective_saturated"), "candidate_space_status": records[-1].get("candidate_space_status"), "optimizer_rows": _concat_preserving_columns(all_opt)}




def coordinate_exchange_sbad(start: np.ndarray, pools: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, prefix_size: int, config: RSSDConfig, rng: np.random.Generator, objective_type: str, sbad_limit: Optional[float] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
    selected=np.asarray(start,dtype=int).copy()
    if len(np.unique(selected)) != len(selected): raise ValueError("Coordinate exchange requires a unique starting assignment.")
    state=SBADNearestState(cache, selected, int(prefix_size))
    current=design_metrics_cached(selected, targets, scores, coords, cache, int(prefix_size), state.sbad); initial=dict(current)
    accepted=0; sweeps=0
    for sweep in range(1, int(config.MAX_OPTIMIZER_SWEEPS)+1):
        accepted_this=0
        for position in rng.permutation(len(selected)):
            used=set(np.delete(selected, position).tolist()); best_idx=int(selected[position]); best_metrics=current
            for candidate_index in pools[position]:
                candidate_index=int(candidate_index)
                if candidate_index == selected[position] or candidate_index in used: continue
                proposal=selected.copy(); proposal[position]=candidate_index
                psbad=state.evaluate_swap(int(position), candidate_index)
                pmet=design_metrics_cached(proposal, targets, scores, coords, cache, int(prefix_size), psbad)
                better=ad_reference_is_better(pmet,best_metrics,config) if objective_type=="ad_reference" else hybrid_is_better(pmet,best_metrics,config,float(sbad_limit))
                if better: best_idx=candidate_index; best_metrics=pmet
            if best_idx != selected[position]:
                selected[position]=best_idx; state=SBADNearestState(cache, selected, int(prefix_size)); current=design_metrics_cached(selected, targets, scores, coords, cache, int(prefix_size), state.sbad); accepted+=1; accepted_this+=1
        sweeps=sweep
        if accepted_this==0: break
    return selected, {"objective_type": objective_type, "initial_SBAD": initial["SBAD"], "final_SBAD": current["SBAD"], "initial_geoMSD": initial["geoMSD"], "final_geoMSD": current["geoMSD"], "initial_minimum_separation": initial["minimum_separation"], "final_minimum_separation": current["minimum_separation"], "accepted_swaps": int(accepted), "sweeps": int(sweeps), "coordinate_exchange_converged": bool(sweeps < int(config.MAX_OPTIMIZER_SWEEPS)), "final_mean_pca_target_distance": current["mean_pca_target_distance"], "final_max_pca_target_distance": current["max_pca_target_distance"]}


def _metric_better(metrics: Dict[str,float], best: Optional[Dict[str,float]], objective_type: str, config: RSSDConfig, sbad_limit: Optional[float]) -> bool:
    if best is None: return True
    return ad_reference_is_better(metrics,best,config) if objective_type=="ad_reference" else hybrid_is_better(metrics,best,config,float(sbad_limit))


def _near_best(metrics: Dict[str,float], best: Dict[str,float], objective_type: str, config: RSSDConfig, sbad_limit: Optional[float]) -> bool:
    tol=float(config.OPTIMIZER_NEAR_BEST_REL_TOL)
    if objective_type=="ad_reference": return metrics["SBAD"] <= best["SBAD"]*(1+tol)
    if metrics["SBAD"] > float(sbad_limit) + config.PCA_TIE_TOL: return False
    return metrics["geoMSD"] >= best["geoMSD"]*(1-tol)


def _best_objective_value(metrics: Dict[str,float], objective_type: str) -> float:
    return float(metrics["SBAD"] if objective_type=="ad_reference" else metrics["geoMSD"])



def coordinate_exchange_start_seed(random_seed: int, objective_type: str, candidate_k: int, start: np.ndarray) -> int:
    """Stable per-start visitation seed independent of support size and list position."""
    arr = np.ascontiguousarray(np.asarray(start, dtype=np.int64))
    digest = hashlib.blake2b(digest_size=8)
    digest.update(arr.tobytes())
    digest.update(str(objective_type).encode("utf-8"))
    digest.update(np.asarray([int(candidate_k)], dtype=np.int64).tobytes())
    seed_offset = int.from_bytes(digest.digest(), byteorder="little", signed=False)
    return int((int(random_seed) + seed_offset) % (2**32 - 1))


def coordinate_exchange_permutation_preview(random_seed: int, objective_type: str, candidate_k: int, start: np.ndarray, n_positions: int, n_sweeps: int = 4) -> List[Tuple[int, ...]]:
    rng = np.random.default_rng(coordinate_exchange_start_seed(random_seed, objective_type, candidate_k, start))
    return [tuple(int(x) for x in rng.permutation(int(n_positions)).tolist()) for _ in range(int(n_sweeps))]


def optimize_multiple_starts_sbad(minimum_distance_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, prefix_size: int, config: RSSDConfig, objective_type: str, sbad_limit: Optional[float] = None, warm_starts: Optional[Sequence[np.ndarray]] = None, candidate_k: Optional[int] = None, optimization_phase: str = "final", start_limit: Optional[int] = None, adaptive: bool = True, start_bank: Optional[OptimizationStartBank] = None) -> Tuple[np.ndarray, pd.DataFrame, List[Dict[str, Any]]]:
    candidate_k = int(candidate_k if candidate_k is not None else max(len(p) for p in pools))
    max_starts = int(start_limit if start_limit is not None else (config.OPTIMIZER_MAX_STARTS if adaptive else config.N_OPTIMIZER_STARTS))
    batch_size = max(1, min(int(config.OPTIMIZER_START_BATCH_SIZE), max_starts))
    min_starts = min(max_starts, int(config.OPTIMIZER_MIN_STARTS if adaptive else max_starts))
    if start_bank is None:
        start_bank = OptimizationStartBank(minimum_distance_start, pools, pool_distances, config.RANDOM_SEED, candidate_k, objective_type, max_starts + len(warm_starts or []))
    starts = start_bank.starts(max_starts, warm_starts)
    rows = []
    panel_by_key = {}
    best = None
    best_metrics = None
    previous_batch_best = None
    stable_batches = 0
    stop_reason = "configured_max_starts_completed"
    cumulative = 0
    batch_number = 0
    for batch_start in range(0, len(starts), batch_size):
        batch_number += 1
        batch = starts[batch_start:batch_start + batch_size]
        batch_metrics = []
        for label, start in batch:
            cumulative += 1
            start_seed = coordinate_exchange_start_seed(config.RANDOM_SEED, objective_type, candidate_k, start)
            start_rng = np.random.default_rng(start_seed)
            opt, summary = coordinate_exchange_sbad(start, pools, targets, scores, coords, cache, int(prefix_size), config, start_rng, objective_type, sbad_limit)
            metrics = design_metrics_cached(opt, targets, scores, coords, cache, int(prefix_size), summary["final_SBAD"])
            key = tuple(int(x) for x in opt.tolist())
            panel_by_key[key] = {"design_key": key, "selected": opt.copy(), "metrics": metrics, "objective_type": objective_type}
            if _metric_better(metrics, best_metrics, objective_type, config, sbad_limit):
                best = opt.copy()
                best_metrics = metrics
            batch_metrics.append(metrics)
            rows.append({
                **summary,
                "optimization_phase": optimization_phase,
                "support_size": int(prefix_size),
                "candidate_K": int(candidate_k),
                "start_number": int(cumulative),
                "start_type": label,
                "start_fingerprint_seed": int(start_seed),
                "batch_number": int(batch_number),
            })
        current_value = _best_objective_value(best_metrics, objective_type)
        if previous_batch_best is None:
            rel_improve = None
        elif objective_type == "ad_reference":
            rel_improve = max(0.0, (previous_batch_best - current_value) / max(1e-12, abs(previous_batch_best)))
        else:
            rel_improve = max(0.0, (current_value - previous_batch_best) / max(1e-12, abs(previous_batch_best)))
        near_fraction = float(np.mean([_near_best(m, best_metrics, objective_type, config, sbad_limit) for m in batch_metrics])) if batch_metrics else 0.0
        stable_batch = bool(cumulative >= min_starts and rel_improve is not None and rel_improve <= config.OPTIMIZER_BEST_IMPROVEMENT_REL_TOL and near_fraction >= config.OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION)
        stable_batches = stable_batches + 1 if stable_batch else 0
        init_stable = bool(adaptive and stable_batches >= int(config.OPTIMIZER_STABLE_BATCHES_REQUIRED))
        for r in rows[-len(batch):]:
            r.update({
                "starts_in_batch": int(len(batch)),
                "cumulative_starts": int(cumulative),
                "best_SBAD": float(best_metrics["SBAD"]),
                "best_geoMSD": float(best_metrics["geoMSD"]),
                "best_minimum_separation": float(best_metrics["minimum_separation"]),
                "relative_best_objective_improvement": rel_improve,
                "recent_near_best_fraction": near_fraction,
                "stable_batch": bool(stable_batch),
                "optimizer_initialization_stable": bool(init_stable),
                "stop_reason": "",
            })
        previous_batch_best = current_value
        if init_stable:
            stop_reason = "optimizer_initialization_stable"
            rows[-1]["stop_reason"] = stop_reason
            break
        if cumulative >= max_starts:
            break
    if best is None:
        raise RuntimeError("Optimizer produced no design.")
    if rows and not rows[-1].get("stop_reason"):
        rows[-1]["stop_reason"] = stop_reason
    panel = sorted(panel_by_key.values(), key=lambda r: _panel_sort_key(r, objective_type))[:int(config.AD_SUPPORT_RANK_PANEL_SIZE)]
    return best, pd.DataFrame(rows), panel


def practical_support_stability(prev_designs: Sequence[np.ndarray], curr_designs: Sequence[np.ndarray], prev_prefix: int, curr_prefix: int, cache: SupportDistanceCache, config: RSSDConfig) -> Dict[str, Any]:
    union={tuple(d.tolist()):np.asarray(d,dtype=int) for d in prev_designs}; union.update({tuple(d.tolist()):np.asarray(d,dtype=int) for d in curr_designs})
    keys=list(union.keys()); designs=[union[k] for k in keys]
    prev_vals=np.array([exact_sbad_cached(cache,d,prev_prefix) for d in designs], dtype=float); curr_vals=np.array([exact_sbad_cached(cache,d,curr_prefix) for d in designs], dtype=float)
    best_prev=float(np.min(prev_vals)); best_curr=float(np.min(curr_vals)); decisive=0; agree=0; equiv=0
    for i in range(len(designs)):
        for j in range(i+1,len(designs)):
            prev_dec=abs(prev_vals[i]-prev_vals[j])/max(1e-12,best_prev) > config.AD_SUPPORT_PRACTICAL_EQUIV_REL_TOL
            curr_dec=abs(curr_vals[i]-curr_vals[j])/max(1e-12,best_curr) > config.AD_SUPPORT_PRACTICAL_EQUIV_REL_TOL
            if prev_dec and curr_dec:
                decisive += 1
                agree += int(np.sign(prev_vals[i]-prev_vals[j]) == np.sign(curr_vals[i]-curr_vals[j]))
            else:
                equiv += 1
    prev_near={keys[i] for i,v in enumerate(prev_vals) if v <= best_prev*(1+config.AD_SUPPORT_NEAR_BEST_REL_TOL)}; curr_near={keys[i] for i,v in enumerate(curr_vals) if v <= best_curr*(1+config.AD_SUPPORT_NEAR_BEST_REL_TOL)}
    overlap=len(prev_near & curr_near); union_near=len(prev_near | curr_near)
    return {"design_panel_size": int(len(designs)), "practically_equivalent_pair_count": int(equiv), "decisive_pair_count": int(decisive), "decisive_pair_order_agreement": float(agree/decisive) if decisive else None, "near_best_set_size_previous": int(len(prev_near)), "near_best_set_size_current": int(len(curr_near)), "near_best_set_overlap": int(overlap), "near_best_set_jaccard": float(overlap/union_near) if union_near else None}


def screen_support_resolution(minimum_assignment: np.ndarray, pools: Sequence[np.ndarray], pool_dists: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, config: RSSDConfig, candidate_k: int) -> Tuple[int, pd.DataFrame, pd.DataFrame, Dict[str, Any], List[np.ndarray]]:
    sizes = _support_sizes(config, len(cache.support_coords))
    records: List[Dict[str, Any]] = []
    opt_tables: List[pd.DataFrame] = []
    prev_panel: List[np.ndarray] = []
    prev_size: Optional[int] = None
    prev_sbad: Optional[float] = None
    prev_hmet: Optional[Dict[str, float]] = None
    prev_hybrid: Optional[np.ndarray] = None
    stable_count = 0
    confirmation_pending = False
    final_size = int(sizes[-1])
    final_status = "not_started"
    final_stable = bool(config.AD_SUPPORT_MODE == "fixed")
    panel: List[np.ndarray] = []

    for si, size in enumerate(sizes):
        stage_start = time.perf_counter()
        size = int(size)
        initial = si == 0
        has_larger_prefix = si < len(sizes) - 1

        pre_refinement_best_sbad: Optional[float] = None
        best_re_evaluated_panel: List[np.ndarray] = []
        if prev_panel:
            re_evaluated = sorted(
                ((exact_sbad_cached(cache, np.asarray(d, dtype=int), size), np.asarray(d, dtype=int).copy()) for d in prev_panel),
                key=lambda x: (x[0], tuple(int(v) for v in x[1].tolist())),
            )
            pre_refinement_best_sbad = float(re_evaluated[0][0])
            best_re_evaluated_panel = [d for _, d in re_evaluated[:max(1, int(config.SUPPORT_STAGE_REFINEMENT_STARTS))]]

        starts = int(config.SUPPORT_STAGE_INITIAL_STARTS if initial else max(1, config.SUPPORT_STAGE_REFINEMENT_STARTS))
        ad_best, ad_tab, ad_panel = optimize_multiple_starts_sbad(
            minimum_assignment, pools, pool_dists, targets, scores, coords, cache, size, config,
            "ad_reference", candidate_k=candidate_k, optimization_phase="support_screen",
            start_limit=starts, warm_starts=best_re_evaluated_panel, adaptive=False,
        )
        sbad_star = exact_sbad_cached(cache, ad_best, size)
        post_refinement_best_sbad = float(sbad_star)
        focused_improve = focused_ad_refinement_improvement(pre_refinement_best_sbad, post_refinement_best_sbad)
        limit = (1 + config.AD_COVERAGE_ENVELOPE_REL_TOL) * sbad_star

        hybrid_warm_starts = [ad_best, *best_re_evaluated_panel]
        if prev_hybrid is not None:
            hybrid_warm_starts.append(prev_hybrid)
        hy_best, hy_tab, hy_panel = optimize_multiple_starts_sbad(
            minimum_assignment, pools, pool_dists, targets, scores, coords, cache, size, config,
            "hybrid", sbad_limit=limit, candidate_k=candidate_k, optimization_phase="support_screen",
            start_limit=starts, warm_starts=hybrid_warm_starts, adaptive=False,
        )
        hmet = design_metrics_cached(hy_best, targets, scores, coords, cache, size)
        opt_tables.extend([ad_tab, hy_tab])

        panel_by: Dict[Tuple[int, ...], np.ndarray] = {}
        for d in [*prev_panel, *best_re_evaluated_panel]:
            arr = np.asarray(d, dtype=int)
            panel_by[tuple(int(v) for v in arr.tolist())] = arr.copy()
        for row in ad_panel + hy_panel:
            arr = np.asarray(row["selected"], dtype=int)
            panel_by[tuple(int(v) for v in arr.tolist())] = arr.copy()
        panel_candidates = list(panel_by.values())
        panel = sorted(
            panel_candidates,
            key=lambda d: (exact_sbad_cached(cache, d, size), tuple(int(v) for v in d.tolist())),
        )[:int(config.AD_SUPPORT_RANK_PANEL_SIZE)]

        stab = {
            "design_panel_size": len(panel),
            "practically_equivalent_pair_count": None,
            "decisive_pair_count": None,
            "decisive_pair_order_agreement": None,
            "near_best_set_size_previous": None,
            "near_best_set_size_current": None,
            "near_best_set_overlap": None,
            "near_best_set_jaccard": None,
        }
        rel_ad = rel_hsbad = rel_hgeo = rel_hmin = rel_cov = None
        site_overlap = None
        stable = False

        if prev_size is not None and prev_sbad is not None and prev_hmet is not None and prev_hybrid is not None:
            stab = practical_support_stability(prev_panel, panel, int(prev_size), size, cache, config)
            rel_ad = _relative_change(sbad_star, prev_sbad)
            rel_hsbad = _relative_change(hmet["SBAD"], prev_hmet["SBAD"])
            rel_hgeo = _relative_change(hmet["geoMSD"], prev_hmet["geoMSD"])
            rel_hmin = _relative_change(hmet["minimum_separation"], prev_hmet["minimum_separation"])
            prev_ratio = prev_hmet["SBAD"] / prev_sbad if prev_sbad else np.nan
            curr_ratio = hmet["SBAD"] / sbad_star if sbad_star else np.nan
            rel_cov = abs(curr_ratio - prev_ratio) if np.isfinite(prev_ratio) and np.isfinite(curr_ratio) else None
            site_overlap = _site_set_overlap(prev_hybrid, hy_best)
            decisive_ok = (
                stab["decisive_pair_count"] == 0
                or (
                    stab["decisive_pair_order_agreement"] is not None
                    and stab["decisive_pair_order_agreement"] >= config.AD_SUPPORT_DECISIVE_PAIR_AGREEMENT_MIN
                )
            )
            stable = bool(
                rel_ad is not None
                and rel_ad <= config.AD_SUPPORT_REL_TOL
                and decisive_ok
                and focused_improve <= config.AD_SUPPORT_REL_TOL
                and (rel_hsbad is None or rel_hsbad <= config.AD_SUPPORT_REL_TOL)
                and (rel_hgeo is None or rel_hgeo <= config.SUPPORT_HYBRID_GEOMSD_REL_TOL)
                and (rel_hmin is None or rel_hmin <= config.SUPPORT_HYBRID_MINSEP_REL_TOL)
                and (rel_cov is None or rel_cov <= config.SUPPORT_HYBRID_COVERAGE_RATIO_ABS_TOL)
            )

        if config.AD_SUPPORT_MODE == "fixed":
            transition = {
                "confirmation_stage": False,
                "stable_count": 0,
                "confirmation_pending_next": False,
                "stop": True,
                "ad_support_resolution_stable": True,
                "support_resolution_status": "fixed_support_size",
            }
        elif prev_size is None:
            transition = {
                "confirmation_stage": False,
                "stable_count": stable_count,
                "confirmation_pending_next": False,
                "stop": not has_larger_prefix,
                "ad_support_resolution_stable": False,
                "support_resolution_status": "initial_support_screen" if has_larger_prefix else "support_limit_reached_without_provisional_stability",
            }
        else:
            transition = support_confirmation_transition(
                stable, stable_count, confirmation_pending, has_larger_prefix, config.AD_SUPPORT_STABLE_STAGES_REQUIRED
            )

        stable_count = int(transition["stable_count"])
        confirmation_pending = bool(transition["confirmation_pending_next"])
        final_status = str(transition["support_resolution_status"])
        final_stable = bool(transition["ad_support_resolution_stable"])

        rec = {
            "support_size": int(size),
            "SBAD_star_screen": float(sbad_star),
            "pre_refinement_best_SBAD": pre_refinement_best_sbad,
            "post_refinement_best_SBAD": post_refinement_best_sbad,
            "focused_AD_refinement_improvement": float(focused_improve),
            "relative_change_in_best_AD_reference_SBAD": rel_ad,
            **stab,
            "hybrid_core_SBAD": float(hmet["SBAD"]),
            "core_coverage_ratio": float(hmet["SBAD"] / sbad_star) if sbad_star else np.nan,
            "hybrid_geoMSD": float(hmet["geoMSD"]),
            "hybrid_minimum_separation": float(hmet["minimum_separation"]),
            "relative_change_in_hybrid_SBAD": rel_hsbad,
            "relative_change_in_hybrid_geoMSD": rel_hgeo,
            "relative_change_in_hybrid_minimum_separation": rel_hmin,
            "relative_change_in_core_coverage_ratio": rel_cov,
            "site_set_overlap_with_previous": site_overlap,
            "stage_runtime_seconds": float(time.perf_counter() - stage_start),
            "stable_stage": bool(stable),
            "confirmation_stage": bool(transition["confirmation_stage"]),
            "provisional_stable_count": int(stable_count),
            "confirmation_pending_next": bool(confirmation_pending),
            "support_resolution_status": final_status,
            "stable_at_max_without_separate_confirmation": bool(final_status == "stable_at_max_without_separate_confirmation"),
            "maximum_support_size_reached": bool(size == sizes[-1]),
        }
        records.append(rec)

        final_size = int(size)
        prev_panel = [np.asarray(d, dtype=int).copy() for d in panel]
        prev_size = int(size)
        prev_sbad = float(sbad_star)
        prev_hmet = dict(hmet)
        prev_hybrid = hy_best.copy()

        if transition["stop"]:
            break

    meta = {
        "ad_support_resolution_stable": bool(final_stable),
        "final_support_size": int(final_size),
        "support_screening_only": True,
        "support_resolution_status": final_status,
        "max_support_resolution_reached": bool(final_size == sizes[-1]),
    }
    return int(final_size), pd.DataFrame(records), _concat_preserving_columns(opt_tables), meta, panel


def final_core_optimization(minimum_assignment: np.ndarray, pools: Sequence[np.ndarray], pool_dists: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, final_B: int, config: RSSDConfig, candidate_k: int, warm_panel: Optional[Sequence[np.ndarray]]=None) -> Tuple[np.ndarray,np.ndarray,pd.DataFrame,Dict[str,Any]]:
    ad_best, ad_tab, _=optimize_multiple_starts_sbad(minimum_assignment,pools,pool_dists,targets,scores,coords,cache,int(final_B),config,"ad_reference",candidate_k=candidate_k,optimization_phase="final",warm_starts=warm_panel,adaptive=(config.OPTIMIZER_START_MODE=="adaptive"))
    sbad_star=exact_sbad_cached(cache,ad_best,int(final_B)); limit=(1+config.AD_COVERAGE_ENVELOPE_REL_TOL)*sbad_star
    hy_best, hy_tab, _=optimize_multiple_starts_sbad(minimum_assignment,pools,pool_dists,targets,scores,coords,cache,int(final_B),config,"hybrid",sbad_limit=limit,candidate_k=candidate_k,optimization_phase="final",warm_starts=[ad_best,*(warm_panel or [])],adaptive=(config.OPTIMIZER_START_MODE=="adaptive"))
    hmet=design_metrics_cached(hy_best,targets,scores,coords,cache,int(final_B))
    meta={"SBAD_star":float(sbad_star),"SBAD_limit":float(limit),"hybrid_core_SBAD":float(hmet["SBAD"]),"core_coverage_ratio":float(hmet["SBAD"]/sbad_star) if sbad_star else np.nan,"hybrid_geoMSD":float(hmet["geoMSD"]),"hybrid_minimum_separation":float(hmet["minimum_separation"]),"AD_reference_starts":int(len(ad_tab)),"hybrid_starts":int(len(hy_tab)),"AD_initialization_stable":bool(ad_tab["optimizer_initialization_stable"].iloc[-1]) if len(ad_tab) else False,"hybrid_initialization_stable":bool(hy_tab["optimizer_initialization_stable"].iloc[-1]) if len(hy_tab) else False}
    return hy_best, ad_best, _concat_preserving_columns([ad_tab, hy_tab]), meta




def confirm_candidate_saturation_final_B(selected_k: int, target_instances: pd.DataFrame, sequences: Dict[Tuple[float,...],Dict[str,Any]], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, final_B: int, config: RSSDConfig, candidate_table: pd.DataFrame) -> Tuple[int,List[np.ndarray],List[np.ndarray],pd.DataFrame,pd.DataFrame,pd.DataFrame,Dict[str,Any]]:
    k_values=_candidate_k_values(config); opt_rows=[]; records=[]; current_k=int(selected_k); status="final_B_confirmed_at_screened_K"
    start_idx=k_values.index(current_k) if current_k in k_values else 0
    previous=None; warm_ad=[]; warm_hybrid=[]
    for k in k_values[start_idx:]:
        stage_start=time.perf_counter(); pools,dists,assoc,tols=candidate_pools_from_sequences(target_instances,sequences,int(k),config); assignment=assignment_from_costs(pools,dists)
        if assignment is None: continue
        ad_best,ad_tab,_=optimize_multiple_starts_sbad(assignment,pools,dists,targets,scores,coords,cache,int(final_B),config,"ad_reference",candidate_k=int(k),optimization_phase="final_B_candidate_confirm",start_limit=max(3,int(config.CANDIDATE_STAGE_EXPLORATION_STARTS)),warm_starts=warm_ad,adaptive=False)
        sbad_star=exact_sbad_cached(cache,ad_best,int(final_B)); limit=(1+config.AD_COVERAGE_ENVELOPE_REL_TOL)*sbad_star
        hy_best,hy_tab,_=optimize_multiple_starts_sbad(assignment,pools,dists,targets,scores,coords,cache,int(final_B),config,"hybrid",sbad_limit=limit,candidate_k=int(k),optimization_phase="final_B_candidate_confirm",start_limit=max(3,int(config.CANDIDATE_STAGE_EXPLORATION_STARTS)),warm_starts=[ad_best,*warm_hybrid],adaptive=False)
        hmet=design_metrics_cached(hy_best,targets,scores,coords,cache,int(final_B)); opt_rows.extend([ad_tab,hy_tab])
        rel_sbad=_relative_change(sbad_star,previous["SBAD_star"] if previous else None); rel_geo=_relative_change(hmet["geoMSD"],previous["hybrid_geoMSD"] if previous else None); rel_min=_relative_change(hmet["minimum_separation"],previous["hybrid_minimum_separation"] if previous else None)
        stable=bool(previous is not None and rel_sbad is not None and rel_sbad<=config.CANDIDATE_SATURATION_AD_REL_TOL and rel_geo is not None and rel_geo<=config.CANDIDATE_SATURATION_GEOMSD_REL_TOL and rel_min is not None and rel_min<=config.CANDIDATE_SATURATION_MINSEP_REL_TOL)
        status="objective_saturated_at_final_B" if stable else ("candidate_limit_reached_without_objective_saturation" if k==k_values[-1] else "expanding_at_final_B")
        records.append({"stage": int((candidate_table["stage"].max() if len(candidate_table) else 0)+len(records)+1), "requested_K":int(k), "candidate_sequences_verified_nested":True, "unique_candidate_observations":int(len(np.unique(np.concatenate(pools))) if pools else 0), "assignment_possible":True, "targets_below_requested_K":int(sum(len(pool)<k for pool in pools)), "targets_requiring_tolerance_expansion":int(sum(np.nan_to_num(tols)>config.PC_CANDIDATE_TOLERANCE)), "maximum_target_tolerance":float(np.nanmax(tols)), "SBAD_star":float(sbad_star), "hybrid_core_SBAD":float(hmet["SBAD"]), "core_coverage_ratio":float(hmet["SBAD"]/sbad_star) if sbad_star else np.nan, "hybrid_geoMSD":float(hmet["geoMSD"]), "hybrid_minimum_separation":float(hmet["minimum_separation"]), "mean_pca_target_distance":float(hmet["mean_pca_target_distance"]), "maximum_pca_target_distance":float(hmet["max_pca_target_distance"]), "relative_change_in_SBAD_star":rel_sbad, "relative_change_in_hybrid_geoMSD":rel_geo, "relative_change_in_hybrid_minimum_separation":rel_min, "AD_exploration_starts":len(ad_tab), "hybrid_exploration_starts":len(hy_tab), "stage_runtime_seconds":float(time.perf_counter()-stage_start), "stable_stage":bool(stable), "confirmation_stage":bool(stable), "maximum_K_reached":bool(k==k_values[-1]), "candidate_space_status":status})
        current_k=int(k); previous=records[-1]; warm_ad=[ad_best]; warm_hybrid=[hy_best]
        if stable: break
    pools,dists,assoc,tols=candidate_pools_from_sequences(target_instances,sequences,current_k,config)
    combined=_concat_preserving_columns([candidate_table, pd.DataFrame(records)]) if records else candidate_table
    meta={"final_candidate_K":int(current_k),"candidate_space_status":status,"candidate_space_saturated":bool("saturated" in status)}
    return current_k,pools,dists,assoc,combined,_concat_preserving_columns(opt_rows),meta



def bounded_support_site_candidates(state: SBADNearestState, candidate_tree: cKDTree, candidate_indices: np.ndarray, selected: Sequence[int], cache: SupportDistanceCache, support_prefix_size: int, config: RSSDConfig) -> Tuple[np.ndarray, np.ndarray]:
    gap_order = np.argsort(-state.nearest_distance, kind="stable")[:min(int(config.SUPPORT_GAP_ANCHORS), int(support_prefix_size))]
    bounded: List[int] = []
    for support_pos in gap_order:
        k = min(len(candidate_indices), int(config.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR))
        _, local = candidate_tree.query(cache.support_coords[int(support_pos)], k=k)
        bounded.extend(candidate_indices[np.atleast_1d(local).astype(int)].tolist())
    selected_set = set(map(int, selected))
    bounded_arr = np.array(sorted(set(map(int, bounded)) - selected_set), dtype=int)
    return bounded_arr, np.asarray(gap_order, dtype=int)


def add_spatial_support_sites(core_selected: np.ndarray, n_support_sites: int, support_coords: np.ndarray, candidate_indices: np.ndarray, coords: np.ndarray, config: RSSDConfig, cache: Optional[SupportDistanceCache] = None, support_prefix_size: Optional[int] = None) -> Tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:
    selected = list(map(int, np.asarray(core_selected, dtype=int).tolist()))
    records = []
    candidate_indices = np.asarray(candidate_indices, dtype=int)
    support_prefix_size = int(support_prefix_size or len(support_coords))
    local_cache = cache if cache is not None else SupportDistanceCache(support_coords, coords, config.AD_DISTANCE_CACHE_MAX_MIB)
    candidate_tree = cKDTree(coords[candidate_indices])
    state = SBADNearestState(local_cache, np.asarray(selected, dtype=int), support_prefix_size)
    for step in range(1, int(n_support_sites) + 1):
        before = state.sbad
        before_hits = local_cache.cache_hits
        before_misses = local_cache.cache_misses
        bounded, gap_order = bounded_support_site_candidates(state, candidate_tree, candidate_indices, selected, local_cache, support_prefix_size, config)
        if len(bounded) == 0:
            break
        best = None
        best_sbad = np.inf
        best_minsep = -np.inf
        best_rank = None
        ranked = []
        for cand in bounded:
            cand = int(cand)
            cand_dist = local_cache.get_prefix(cand, support_prefix_size).astype(np.float64, copy=False)
            proposal_nearest = np.minimum(state.nearest_distance, cand_dist)
            psbad = float(np.mean(proposal_nearest, dtype=np.float64))
            pcoords = coords[np.asarray([*selected, cand], dtype=int)]
            pmin = minimum_selected_separation(pcoords)
            ranked.append((psbad, -pmin, cand, pmin))
        ranked.sort(key=lambda item: (item[0], item[1], item[2]))
        if ranked:
            best_sbad, neg_minsep, best, best_minsep = ranked[0]
            best = int(best)
            best_rank = 1
        if best is None:
            break
        selected.append(best)
        state = SBADNearestState(local_cache, np.asarray(selected, dtype=int), support_prefix_size)
        records.append({
            "support_addition_order": int(step),
            "analysis_index": int(best),
            "gap_anchors_considered": int(len(gap_order)),
            "unique_support_candidates_evaluated": int(len(bounded)),
            "bounded_candidate_analysis_indices": ";".join(str(int(x)) for x in bounded.tolist()),
            "selected_candidate_rank_by_exact_SBAD": int(best_rank),
            "SBAD_before_addition": float(before),
            "best_candidate_SBAD_after": float(best_sbad),
            "SBAD_after_addition": float(state.sbad),
            "SBAD_reduction": float(before - state.sbad),
            "minimum_separation_after_addition": float(best_minsep),
            "cache_hits": int(local_cache.cache_hits - before_hits),
            "cache_misses": int(local_cache.cache_misses - before_misses),
            "sample_role": "spatial_support",
        })
    final_selected = np.asarray(selected, dtype=int)
    final_state = SBADNearestState(local_cache, final_selected, support_prefix_size)
    dmat = np.vstack([local_cache.get_prefix(int(idx), support_prefix_size) for idx in final_selected]).T
    nearest_site = np.argmin(dmat, axis=1)
    coverage = pd.DataFrame({
        "support_rank": np.arange(1, support_prefix_size + 1, dtype=int),
        "support_X": local_cache.support_coords[:support_prefix_size, 0],
        "support_Y": local_cache.support_coords[:support_prefix_size, 1],
        "nearest_final_site_position": nearest_site + 1,
        "nearest_final_site_analysis_index": final_selected[nearest_site],
        "nearest_final_site_distance": final_state.nearest_distance,
    })
    return final_selected, pd.DataFrame(records), coverage


def proxy_spacing_diagnostic(eligible_coords: np.ndarray, eligible_scores: np.ndarray, selected_nearest_distances: np.ndarray, selected_geo_msd: float, config: RSSDConfig, support_coords: Optional[np.ndarray] = None, support_scores: Optional[np.ndarray] = None) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if not config.RUN_SPACING_DIAGNOSTIC: return pd.DataFrame(), {"status":"disabled"}
    coords_arr=np.asarray(support_coords if support_coords is not None else eligible_coords,dtype=np.float64); scores_arr=np.asarray(support_scores if support_scores is not None else eligible_scores,dtype=np.float64); selected_nn=np.asarray(selected_nearest_distances,dtype=np.float64)
    n=len(coords_arr)
    if n<3: return pd.DataFrame(), {"status":"not_estimable","interpretation":"Too few support points."}
    rng=np.random.default_rng(config.RANDOM_SEED+280); sample_n=min(int(config.SPACING_DIAGNOSTIC_MAX_POINTS),n); sample_coords=coords_arr[:sample_n]; sample_scores=scores_arr[:sample_n]
    k=min(sample_n,int(config.SPACING_DIAGNOSTIC_NEIGHBORS)+1); _,nbr=cKDTree(sample_coords).query(sample_coords,k=k,workers=-1); nbr=np.atleast_2d(nbr); left=[np.repeat(np.arange(sample_n),nbr.shape[1]-1)]; right=[nbr[:,1:].reshape(-1)]
    a=rng.integers(0,sample_n,size=min(int(config.SPACING_DIAGNOSTIC_RANDOM_PAIRS),max(1000,sample_n*25)),endpoint=False); b=rng.integers(0,sample_n,size=len(a),endpoint=False); same=a==b
    while np.any(same): b[same]=rng.integers(0,sample_n,size=int(np.sum(same)),endpoint=False); same=a==b
    left.append(a); right.append(b); left=np.concatenate(left); right=np.concatenate(right)
    delta=sample_coords[left]-sample_coords[right]; distances=np.sqrt(np.einsum("ij,ij->i",delta,delta)); valid=np.isfinite(distances)&(distances>0); distances=distances[valid]; left=left[valid]; right=right[valid]
    edges=np.unique(np.quantile(distances,np.linspace(0,1,int(config.SPACING_DIAGNOSTIC_BINS)+1)))
    if len(edges)<3: edges=np.unique(np.linspace(float(np.min(distances)),float(np.max(distances)),int(config.SPACING_DIAGNOSTIC_BINS)+1))
    bin_id=np.digitize(distances,edges[1:-1],right=False); records=[]; pc_ranges={}
    for pc in range(sample_scores.shape[1]):
        semivar=0.5*(sample_scores[left,pc]-sample_scores[right,pc])**2; bins=[]
        for b_id in range(len(edges)-1):
            mask=bin_id==b_id; count=int(np.sum(mask))
            if count==0: continue
            bins.append({"PC":f"PC{pc+1}","bin":b_id+1,"pair_count":count,"distance_min":float(np.min(distances[mask])),"distance_median":float(np.median(distances[mask])),"distance_max":float(np.max(distances[mask])),"mean_proxy_semivariance":float(np.mean(semivar[mask]))})
        tab=pd.DataFrame(bins)
        if len(tab)==0: pc_ranges[f"PC{pc+1}"]={"proxy_range":None,"range_reliable":False,"reason":"no_valid_bins"}; continue
        tab["interpreted_bin"]=tab["pair_count"]>=int(config.SPACING_MIN_PAIRS_PER_BIN); valid_tab=tab[tab["interpreted_bin"]].copy()
        if len(valid_tab)<max(3,int(config.SPACING_PLATEAU_TAIL_BINS)):
            reason="insufficient_interpretable_bins"; proxy_range=None; reliable=False; ref=np.nan
        else:
            valid_tab["smoothed_semivariance"]=valid_tab["mean_proxy_semivariance"].rolling(3,center=True,min_periods=1).median()
            tail=valid_tab.tail(int(config.SPACING_PLATEAU_TAIL_BINS)); ref=float(tail["smoothed_semivariance"].mean()); threshold=float(config.SPACING_TARGET_SEMIVARIANCE_FRACTION*ref)
            above=valid_tab["smoothed_semivariance"]>=threshold; run=above.rolling(int(config.SPACING_RANGE_STABILITY_BINS),min_periods=int(config.SPACING_RANGE_STABILITY_BINS)).sum()>=int(config.SPACING_RANGE_STABILITY_BINS)
            reached_idx=np.flatnonzero(run.to_numpy())
            slope=float((tail["smoothed_semivariance"].iloc[-1]-tail["smoothed_semivariance"].iloc[0])/max(1e-12,abs(ref)))
            diffs=np.diff(tail["smoothed_semivariance"].to_numpy()); oscillatory=bool(np.any(diffs>0) and np.any(diffs<0) and (np.max(tail["smoothed_semivariance"])-np.min(tail["smoothed_semivariance"]))/max(1e-12,abs(ref))>config.SPACING_PLATEAU_REL_SLOPE_TOL)
            if len(reached_idx)==0: reason="threshold_not_reached"; reliable=False; proxy_range=None
            elif slope>config.SPACING_PLATEAU_REL_SLOPE_TOL: reason="semivariance_tail_still_increasing"; reliable=False; proxy_range=None
            elif oscillatory: reason="semivariance_tail_unstable_or_oscillatory"; reliable=False; proxy_range=None
            else: reason="threshold_reached_and_tail_plateaued"; reliable=True; proxy_range=float(valid_tab.iloc[int(reached_idx[0])]["distance_median"])
        tab["far_distance_semivariance_reference"]=ref; tab["fraction_of_far_distance_semivariance"]=tab["mean_proxy_semivariance"]/ref if np.isfinite(ref) and ref>0 else np.nan; tab["range_reliable"]=bool(reliable); tab["range_failure_reason"]=reason; records.extend(tab.to_dict(orient="records")); pc_ranges[f"PC{pc+1}"]={"proxy_range":proxy_range,"range_reliable":bool(reliable),"reason":reason}
    reliable_ranges=[v["proxy_range"] for v in pc_ranges.values() if v["range_reliable"] and v["proxy_range"] is not None]
    summary={"status":"reliable_proxy_range_estimated" if reliable_ranges else "proxy_range_unreliable","support_sequence_used":True,"simple_random_raw_row_sample_used":False,"sampled_support_points":int(sample_n),"sampled_pairs":int(len(distances)),"per_pc_range_reliability":pc_ranges,"proxy_decorrelation_distance":float(np.nanmax(reliable_ranges)) if reliable_ranges else None,"selected_absolute_minimum_spacing":float(np.min(selected_nn)),"selected_geoMSD":float(selected_geo_msd),"selected_median_nearest_neighbor_spacing":float(np.median(selected_nn)),"interpretation":"This diagnostic never changes the hybrid design objective."}
    return pd.DataFrame(records), summary


# V210_OVERRIDES




# --- ESAP RSSD v2.10 active orchestration override ---


def _concat_preserving_columns(tables: Sequence[pd.DataFrame]) -> pd.DataFrame:
    """Concat nonempty frames without pandas empty/all-NA dtype FutureWarnings."""
    valid = [table for table in tables if isinstance(table, pd.DataFrame) and len(table)]
    if not valid:
        return pd.DataFrame()
    columns: List[Any] = []
    for table in valid:
        for col in table.columns:
            if col not in columns:
                columns.append(col)
    cleaned = []
    for table in valid:
        part = table.dropna(axis=1, how="all")
        if len(part.columns):
            cleaned.append(part)
    if cleaned:
        out = pd.concat(cleaned, ignore_index=True)
    else:
        out = pd.DataFrame(index=range(sum(len(table) for table in valid)))
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out.reindex(columns=columns)


def _concat_diagnostic_tables(tables: Sequence[pd.DataFrame]) -> pd.DataFrame:
    return _concat_preserving_columns(tables)


def _add_candidate_saturation_aliases(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    if len(out) == 0:
        return out
    if "starts" not in out:
        ad = pd.to_numeric(out.get("AD_exploration_starts", 0), errors="coerce").fillna(0)
        hy = pd.to_numeric(out.get("hybrid_exploration_starts", 0), errors="coerce").fillna(0)
        out["starts"] = (ad + hy).astype(int)
    if "runtime" not in out and "stage_runtime_seconds" in out:
        out["runtime"] = out["stage_runtime_seconds"]
    if "stable" not in out and "stable_stage" in out:
        out["stable"] = out["stable_stage"]
    if "confirmation" not in out and "confirmation_stage" in out:
        out["confirmation"] = out["confirmation_stage"]
    if "max" not in out and "maximum_K_reached" in out:
        out["max"] = out["maximum_K_reached"]
    if "status" not in out and "candidate_space_status" in out:
        out["status"] = out["candidate_space_status"]
    preferred = [
        "stage", "requested_K", "candidate_sequences_verified_nested", "unique_candidate_observations",
        "assignment_possible", "targets_below_requested_K", "targets_requiring_tolerance_expansion",
        "maximum_target_tolerance", "SBAD_star", "hybrid_core_SBAD", "core_coverage_ratio",
        "hybrid_geoMSD", "hybrid_minimum_separation", "mean_pca_target_distance",
        "maximum_pca_target_distance", "relative_change_in_SBAD_star",
        "relative_change_in_hybrid_geoMSD", "relative_change_in_hybrid_minimum_separation",
        "starts", "runtime", "stable", "confirmation", "max", "status",
    ]
    return out[[c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]]


def _add_support_resolution_aliases(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    if len(out) == 0:
        return out
    if "runtime" not in out and "stage_runtime_seconds" in out:
        out["runtime"] = out["stage_runtime_seconds"]
    preferred = [
        "support_size", "SBAD_star_screen", "relative_change_in_best_AD_reference_SBAD",
        "design_panel_size", "practically_equivalent_pair_count", "decisive_pair_count",
        "decisive_pair_order_agreement", "near_best_set_size_previous", "near_best_set_size_current",
        "near_best_set_overlap", "near_best_set_jaccard", "focused_AD_refinement_improvement",
        "hybrid_core_SBAD", "core_coverage_ratio", "hybrid_geoMSD", "hybrid_minimum_separation",
        "relative_change_in_hybrid_SBAD", "relative_change_in_hybrid_geoMSD",
        "relative_change_in_hybrid_minimum_separation", "relative_change_in_core_coverage_ratio",
        "site_set_overlap_with_previous", "runtime", "stable_stage", "confirmation_stage",
    ]
    return out[[c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]]


def _optimizer_phase_summary(table: pd.DataFrame, phase: str, objective: str) -> Dict[str, Any]:
    if not isinstance(table, pd.DataFrame) or len(table) == 0:
        return {"starts": 0, "initialization_stable": False, "stop_reason": "not_run"}
    part = table[(table.get("optimization_phase") == phase) & (table.get("objective_type") == objective)]
    if len(part) == 0:
        return {"starts": 0, "initialization_stable": False, "stop_reason": "not_run"}
    stop_values = part.get("stop_reason", pd.Series(dtype=object)).replace("", np.nan).dropna()
    return {
        "starts": int(pd.to_numeric(part["start_number"], errors="coerce").max()),
        "initialization_stable": bool(part.get("optimizer_initialization_stable", pd.Series([False])).iloc[-1]),
        "stop_reason": str(stop_values.iloc[-1]) if len(stop_values) else "completed",
    }


def run_esap_rssd(config: RSSDConfig, source_df: Optional[pd.DataFrame] = None, workflow_state: Optional[Dict[str, Any]] = None) -> RSSDRunResult:
    run_started = time.perf_counter()
    validate_config(config)
    workflow_state = workflow_state or {}
    stage_times: Dict[str, float] = {}

    print("1/10 Validating survey data")
    t0 = time.perf_counter()
    if source_df is None:
        source_df_raw, input_label = load_survey(config)
    else:
        source_df_raw = source_df.copy()
        input_label = str(workflow_state.get("input_label", "provided_dataframe"))
    source_df, valid_mask, coordinate_eligible_mask, validation_report = validate_and_index_data(source_df_raw, config)
    valid_source_positions = np.flatnonzero(valid_mask)
    analysis_to_source = valid_source_positions.copy()
    analysis_coordinate_eligible = coordinate_eligible_mask[analysis_to_source]
    coords = source_df.iloc[analysis_to_source][[config.X_COLUMN, config.Y_COLUMN]].to_numpy(dtype=np.float64)
    original_features = source_df.iloc[analysis_to_source][config.FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    transformed_features, transformation_details = transform_features(original_features, config.FEATURE_COLUMNS, config.FEATURE_TRANSFORMS)
    skewness_summary = pd.DataFrame(
        {
            "configured_transform": [config.FEATURE_TRANSFORMS.get(c, "none") for c in config.FEATURE_COLUMNS],
            "raw_skewness": skew(original_features, axis=0, bias=False),
            "transformed_skewness": skew(transformed_features, axis=0, bias=False),
        },
        index=config.FEATURE_COLUMNS,
    )
    stage_times["validate_transform"] = time.perf_counter() - t0

    print("2/10 Standardizing sensor variables and fitting PCA")
    t0 = time.perf_counter()
    design_scores, feature_scaler, pca_estimator, pca_mode, memory_report = fit_standardized_pca(transformed_features, config)
    pca_validation, pca_correlation = pca_validation_table(design_scores)
    stage_times["pca"] = time.perf_counter() - t0

    print("3/10 Constructing the response-surface design")
    t0 = time.perf_counter()
    p_dim = int(config.N_DESIGN_COMPONENTS)
    pc_radius = np.linalg.norm(design_scores, axis=1)
    design_radius = float(np.quantile(pc_radius, config.DESIGN_COVERAGE))
    pca_d2 = np.einsum("ij,ij->i", design_scores, design_scores)
    outlier_threshold_d2 = float(chi2.ppf(config.OUTLIER_COVERAGE, df=p_dim))
    pca_outlier = pca_d2 > outlier_threshold_d2
    candidate_eligible = analysis_coordinate_eligible & ~pca_outlier
    if int(candidate_eligible.sum()) < min(config.N_SAMPLES, len(candidate_eligible)):
        raise ValueError("Outlier and duplicate-coordinate masks leave too few candidate-eligible observations.")
    source_df["pca_space_outlier_flag"] = False
    source_df.iloc[analysis_to_source, source_df.columns.get_loc("pca_space_outlier_flag")] = pca_outlier
    base_targets = make_base_ccd(p_dim, design_radius, config.CENTER_REPLICATES)
    target_instances, target_replication_counts, sample_budget_report = allocate_sample_budget(base_targets, config.N_SAMPLES, config, p_dim)
    target_cols = [f"target_PC{j + 1}" for j in range(p_dim)]
    target_array = target_instances[target_cols].to_numpy(dtype=np.float64)
    stage_times["response_surface_design"] = time.perf_counter() - t0

    print("4/10 Building spatially balanced geographic support")
    t0 = time.perf_counter()
    support_max_request = int(config.AD_SUPPORT_FIXED_SIZE if config.AD_SUPPORT_MODE == "fixed" else config.AD_SUPPORT_MAX_SIZE)
    support_sequence, support_report = build_spatially_balanced_support_sequence(coords, np.arange(len(coords), dtype=np.int64), support_max_request)
    support_sizes = _support_sizes(config, len(support_sequence))
    max_support_size = int(max(support_sizes))
    support_all = support_sequence[["X", "Y"]].to_numpy(dtype=np.float64)[:max_support_size]
    support_cache = SupportDistanceCache(support_all, coords, config.AD_DISTANCE_CACHE_MAX_MIB)
    support_report.update({
        "candidate_eligible_rows": int(candidate_eligible.sum()),
        "maximum_cached_support_prefix": int(max_support_size),
        "support_sizes_screened": [int(x) for x in support_sizes],
    })
    stage_times["support_sequence"] = time.perf_counter() - t0

    print("5/10 Building nested base-target candidate sequences")
    t0 = time.perf_counter()
    eligible_indices = np.flatnonzero(candidate_eligible)
    search_indices, approximate_prefilter_used, prefilter_report = approximate_pca_prefilter(eligible_indices, design_scores, target_array, config)
    sequences, candidate_sequence_records, sequence_report = build_nested_candidate_sequences(target_instances, search_indices, design_scores, coords, config)
    stage_times["candidate_sequence_build"] = time.perf_counter() - t0

    print("6/10 Screening candidate-space saturation")
    t0 = time.perf_counter()
    screen_support_size = int(support_sizes[0])
    selected_k, candidate_pools, candidate_pool_distances, candidate_associations, candidate_saturation, candidate_screen_meta = screen_candidate_saturation(
        target_instances, sequences, target_array, design_scores, coords, support_cache, screen_support_size, config
    )
    candidate_optimizer_rows = candidate_screen_meta.pop("optimizer_rows", pd.DataFrame())
    candidate_saturation = _add_candidate_saturation_aliases(candidate_saturation)
    stage_times["candidate_saturation_screen"] = time.perf_counter() - t0

    print("7/10 Stabilizing SBAD geographic support resolution")
    t0 = time.perf_counter()
    minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)
    if minimum_assignment is None:
        raise RuntimeError("No unique target-to-site assignment was possible after v2.10 candidate screening.")
    final_B, ad_support_resolution, support_optimizer_rows, support_screen_meta, warm_panel = screen_support_resolution(
        minimum_assignment, candidate_pools, candidate_pool_distances, target_array, design_scores, coords, support_cache, config, int(selected_k)
    )
    ad_support_resolution = _add_support_resolution_aliases(ad_support_resolution)
    stage_times["support_resolution_screen"] = time.perf_counter() - t0

    print("8/10 Confirming candidate saturation and optimizing the final hybrid RSSD core")
    t0 = time.perf_counter()
    final_k, candidate_pools, candidate_pool_distances, candidate_associations, candidate_saturation, confirm_optimizer_rows, final_k_meta = confirm_candidate_saturation_final_B(
        int(selected_k), target_instances, sequences, target_array, design_scores, coords, support_cache, int(final_B), config, candidate_saturation
    )
    candidate_saturation = _add_candidate_saturation_aliases(candidate_saturation)
    minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)
    if minimum_assignment is None:
        raise RuntimeError("Final-B candidate confirmation left no unique target-to-site assignment.")
    _, _, _, final_tolerances = candidate_pools_from_sequences(target_instances, sequences, int(final_k), config)
    core_selected, ad_reference_selected, final_optimizer_rows, final_meta = final_core_optimization(
        minimum_assignment, candidate_pools, candidate_pool_distances, target_array, design_scores, coords, support_cache, int(final_B), config, int(final_k), warm_panel
    )
    optimizer_stability = _concat_diagnostic_tables([candidate_optimizer_rows, support_optimizer_rows, confirm_optimizer_rows, final_optimizer_rows])
    support_opt_report = {
        **support_screen_meta,
        **final_meta,
        **final_k_meta,
        "screened_candidate_K": int(selected_k),
        "final_candidate_K": int(final_k),
        "final_support_size": int(final_B),
        "support_cache": support_cache.snapshot(),
    }
    stage_times["final_candidate_confirmation_and_optimization"] = time.perf_counter() - t0

    print("9/10 Adding spatial support sites")
    t0 = time.perf_counter()
    final_support_coords = support_all[: int(final_B)]
    support_sequence = support_sequence.copy()
    support_sequence["used_in_final_support_prefix"] = support_sequence["support_rank"] <= int(final_B)
    final_selected, spatial_support_sites, field_coverage_distances = add_spatial_support_sites(
        core_selected,
        int(sample_budget_report["spatial_support_sites"]),
        final_support_coords,
        eligible_indices,
        coords,
        config,
        cache=support_cache,
        support_prefix_size=int(final_B),
    )
    stage_times["spatial_support_site_addition"] = time.perf_counter() - t0

    print("10/10 Calculating diagnostics and writing outputs")
    t0 = time.perf_counter()
    selected_coords = coords[final_selected].astype(np.float64)
    selected_scores = design_scores[final_selected].astype(np.float64)
    nearest_selected = selected_nearest_neighbor_distances(selected_coords)
    final_geo_msd = exact_geo_msd(selected_coords)
    final_sbad = exact_sbad_cached(support_cache, final_selected, int(final_B))
    response_surface_count = len(core_selected)
    core_sbad = exact_sbad_cached(support_cache, core_selected, int(final_B))
    core_geo_msd = exact_geo_msd(coords[core_selected])
    core_min_sep = minimum_selected_separation(coords[core_selected])
    mean_target_distance, max_target_distance = matching_metrics(core_selected, target_array, design_scores)
    regression_diagnostics, leverage = regression_design_diagnostics(selected_scores, design_scores[eligible_indices], config.DIAGNOSTIC_CHUNK_SIZE)
    support_analysis_idx = support_sequence.loc[support_sequence["used_in_final_support_prefix"], "analysis_index"].to_numpy(dtype=int)
    support_scores = design_scores[support_analysis_idx]
    proxy_spatial_scale, spacing_diagnostic = proxy_spacing_diagnostic(
        coords[eligible_indices],
        design_scores[eligible_indices],
        nearest_selected,
        final_geo_msd,
        config,
        support_coords=final_support_coords,
        support_scores=support_scores,
    )

    selected_source_rows = analysis_to_source[final_selected]
    base_columns = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]
    selected_export = source_df.iloc[selected_source_rows][base_columns].reset_index(drop=True).copy()
    selected_export.insert(0, "sample_order", np.arange(1, len(selected_export) + 1, dtype=int))
    selected_export["analysis_index"] = final_selected
    selected_export["sample_role"] = ["response_surface"] * response_surface_count + ["spatial_support"] * (len(final_selected) - response_surface_count)
    for j in range(p_dim):
        selected_export[f"selected_PC{j + 1}"] = selected_scores[:, j]
        selected_export[f"target_PC{j + 1}"] = None
    for column in ["target_instance_id", "base_target_id", "target_type", "target_replicate_number", "pca_target_distance"]:
        selected_export[column] = None
    for i in range(response_surface_count):
        tr = target_instances.iloc[i]
        selected_export.loc[i, "target_instance_id"] = tr["target_instance_id"]
        selected_export.loc[i, "base_target_id"] = tr["base_target_id"]
        selected_export.loc[i, "target_type"] = tr["target_type"]
        selected_export.loc[i, "target_replicate_number"] = tr["target_replicate_number"]
        for j in range(p_dim):
            selected_export.loc[i, f"target_PC{j + 1}"] = float(tr[f"target_PC{j + 1}"])
        selected_export.loc[i, "pca_target_distance"] = float(np.linalg.norm(selected_scores[i] - target_array[i]))
    selected_export.loc[selected_export["sample_role"] == "spatial_support", "target_type"] = "support"
    selected_export["nearest_selected_geographic_neighbor_distance"] = nearest_selected
    if len(leverage) == len(selected_export):
        selected_export["second_order_model_leverage"] = leverage

    candidate_export = candidate_associations.copy()
    if len(candidate_export):
        cand_idx = candidate_export["candidate_analysis_index"].to_numpy(dtype=int)
        candidate_export[config.ID_COLUMN] = source_df.iloc[analysis_to_source[cand_idx]][config.ID_COLUMN].to_numpy()
        candidate_export[config.X_COLUMN] = coords[cand_idx, 0]
        candidate_export[config.Y_COLUMN] = coords[cand_idx, 1]
    candidate_pool_report = candidate_pool_summary_table(target_instances, candidate_pools, candidate_pool_distances, final_tolerances, config)

    output_tables = {
        "ESAP_RSSD_spatial_support_sequence.csv": support_sequence,
        "ESAP_RSSD_ad_support_resolution.csv": ad_support_resolution,
        "ESAP_RSSD_candidate_saturation.csv": candidate_saturation,
        "ESAP_RSSD_optimizer_stability.csv": optimizer_stability,
        "ESAP_RSSD_spatial_support_sites.csv": spatial_support_sites,
        "ESAP_RSSD_field_coverage_distances.csv": field_coverage_distances,
        "ESAP_RSSD_proxy_spatial_scale.csv": proxy_spatial_scale,
        "ESAP_RSSD_selected_sites.csv": selected_export,
        "ESAP_RSSD_candidate_sites.csv": candidate_export,
        "ESAP_RSSD_candidate_pool_report.csv": candidate_pool_report,
        "ESAP_RSSD_candidate_sequences.csv": candidate_sequence_records,
        "ESAP_RSSD_target_instances.csv": target_instances,
        "ESAP_RSSD_pca_standardization_validation.csv": pca_validation,
        "ESAP_RSSD_feature_skewness.csv": skewness_summary,
    }
    for filename, table in output_tables.items():
        table.to_csv(filename, index=False)

    kmz_export_metadata = {"created": False, "reason": "source projected EPSG was not supplied"}
    source_epsg = workflow_state.get("coordinate_epsg")
    if source_epsg:
        try:
            kmz_export_metadata = export_selected_sites_kmz(selected_export, config, int(source_epsg))
        except Exception as exc:
            kmz_export_metadata = {"created": False, "reason": str(exc)}

    total_runtime = time.perf_counter() - run_started
    final_ad_summary = _optimizer_phase_summary(optimizer_stability, "final", "ad_reference")
    final_hybrid_summary = _optimizer_phase_summary(optimizer_stability, "final", "hybrid")
    support_stable = bool(support_opt_report.get("ad_support_resolution_stable", False))
    candidate_status = str(support_opt_report.get("candidate_space_status", "unknown"))
    candidate_saturated = bool(support_opt_report.get("candidate_space_saturated", False))
    support_site_count = len(final_selected) - response_surface_count
    support_sbad_reduction = float(core_sbad - final_sbad)
    cache_stats = support_cache.snapshot()
    support_opt_report["support_cache"] = cache_stats

    run_summary = f"""# ESAP RSSD v2.10 run summary

{int(config.N_SAMPLES)} calibration locations were requested. The selected design used {response_surface_count} PCA response-surface sites and {support_site_count} spatial support sites.

Final SBAD geographic support prefix: {int(final_B):,}. {'Support resolution met the practical-equivalence stability rule.' if support_stable else 'Support resolution reached the configured limit before practical-equivalence confirmation; review ESAP_RSSD_ad_support_resolution.csv.'}

Final candidate K: {int(final_k)}. Candidate-space status: `{candidate_status}`; objective saturation = {candidate_saturated}.

Full final optimizer starts: AD-reference {final_ad_summary['starts']} starts (stable={final_ad_summary['initialization_stable']}, stop={final_ad_summary['stop_reason']}), hybrid {final_hybrid_summary['starts']} starts (stable={final_hybrid_summary['initialization_stable']}, stop={final_hybrid_summary['stop_reason']}).

SBAD_star = {support_opt_report['SBAD_star']:.6g}; SBAD_limit = {support_opt_report['SBAD_limit']:.6g}. The final hybrid response-surface core achieved SBAD = {core_sbad:.6g}, coverage ratio = {core_sbad / support_opt_report['SBAD_star']:.6g}, exact geoMSD = {core_geo_msd:.6g}, and minimum separation = {core_min_sep:.6g}.

{support_site_count} spatial support site(s) were added after the response-surface core, reducing final-design SBAD by {support_sbad_reduction:.6g} to {final_sbad:.6g}.

The proxy spatial-scale diagnostic status was `{spacing_diagnostic.get('status')}`. It is diagnostic only and does not alter the hybrid optimizer.

Shared support-distance cache: {cache_stats['cache_hits']:,} hits, {cache_stats['cache_misses']:,} misses, {cache_stats['cache_evictions']:,} evictions, peak {cache_stats['peak_cached_vectors']:,} vectors / {cache_stats['peak_cache_mib']:.2f} MiB.

Total runtime: {total_runtime:.1f} seconds.

ESAP 2 separates geographic support from sensor observation density. Adaptive diagnostics should vary one source of uncertainty at a time.
"""
    Path("ESAP_RSSD_run_summary.md").write_text(run_summary, encoding="utf-8")

    metadata = {
        "notebook_version": NOTEBOOK_VERSION,
        "input_label": input_label,
        "configuration": asdict(config),
        "validation_report": validation_report,
        "transformation_details": transformation_details,
        "pca_mode": pca_mode,
        "pca_explained_variance_ratio": np.asarray(pca_estimator.explained_variance_ratio_).tolist(),
        "memory_report": memory_report,
        "sample_budget": sample_budget_report,
        "support_sequence": support_report,
        "support_optimization": support_opt_report,
        "candidate_prefilter": {"used": approximate_prefilter_used, **prefilter_report},
        "candidate_sequence_report": sequence_report,
        "candidate_screening": candidate_screen_meta,
        "candidate_final_K": int(final_k),
        "regression_design_diagnostics": regression_diagnostics,
        "spacing_diagnostic": spacing_diagnostic,
        "kmz_export": kmz_export_metadata,
        "stage_runtime_seconds": stage_times,
        "total_runtime_seconds": total_runtime,
        "package_versions": package_versions(),
    }
    diagnostic_data = {
        "coords": coords,
        "design_scores": design_scores,
        "eligible_indices": eligible_indices,
        "pca_outlier": pca_outlier,
        "target_array": target_array,
        "selected_indices": final_selected,
        "core_selected_indices": core_selected,
        "ad_reference_indices": ad_reference_selected,
        "original_features": original_features,
        "feature_columns": list(config.FEATURE_COLUMNS),
        "candidate_indices": np.unique(candidate_export["candidate_analysis_index"].to_numpy(dtype=int)) if len(candidate_export) else np.array([], dtype=int),
    }
    Path("ESAP_RSSD_run_metadata.json").write_text(json.dumps(json_ready(metadata), indent=2, allow_nan=False), encoding="utf-8")
    stage_times["diagnostics_and_exports"] = time.perf_counter() - t0
    print(f"Completed ESAP RSSD v2.10 in {total_runtime:.1f} seconds.")
    print(run_summary)
    return RSSDRunResult(config, selected_export, candidate_export, target_instances, support_sequence, ad_support_resolution, candidate_saturation, optimizer_stability, spatial_support_sites, field_coverage_distances, proxy_spatial_scale, pca_validation, skewness_summary, metadata, run_summary, diagnostic_data)

def duplicate_top_level_definition_report(source: str) -> Dict[str, Any]:
    if not source:
        return {"duplicate_top_level_definition_count": 0, "duplicate_top_level_definition_names": []}
    tree = ast.parse(source)
    names: List[str] = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            names.append(node.name)
    counts = {name: names.count(name) for name in sorted(set(names))}
    duplicates = [name for name, count in counts.items() if count > 1]
    return {"duplicate_top_level_definition_count": int(len(duplicates)), "duplicate_top_level_definition_names": duplicates}



def run_unit_validations(seed: int) -> None:
    global V210_VALIDATION_RESULTS
    V210_VALIDATION_RESULTS = {}
    coords_small = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0], [3.0, 4.0], [8.0, 1.0]])
    kd_value = float(np.exp(np.mean(np.log(selected_nearest_neighbor_distances_ckdtree_reference(coords_small)))))
    vec_value = exact_geo_msd(coords_small)
    dense = squareform(pdist(coords_small))
    np.fill_diagonal(dense, np.inf)
    brute_value = float(np.exp(np.mean(np.log(np.min(dense, axis=1)))))
    np.testing.assert_allclose(vec_value, brute_value, rtol=1e-13, atol=1e-13)
    np.testing.assert_allclose(vec_value, kd_value, rtol=1e-13, atol=1e-13)

    rng = np.random.default_rng(seed)
    selected_distance_cases = []
    selected_reference_queries = 0
    for m in [6, 10, 12, 20, 40]:
        for _ in range(4):
            c = rng.uniform(0.0, 1000.0, size=(m, 2))
            v = selected_nearest_neighbor_distances(c)
            d = squareform(pdist(c))
            np.fill_diagonal(d, np.inf)
            brute = np.min(d, axis=1)
            kd_geo = float(np.exp(np.mean(np.log(selected_nearest_neighbor_distances_ckdtree_reference(c)))))
            selected_reference_queries += 1
            np.testing.assert_allclose(v, brute, rtol=1e-13, atol=1e-13)
            np.testing.assert_allclose(exact_geo_msd(c), kd_geo, rtol=1e-13, atol=1e-13)
            np.testing.assert_allclose(exact_geo_msd(c), float(np.exp(np.mean(np.log(brute)))), rtol=1e-13, atol=1e-13)
            np.testing.assert_allclose(minimum_selected_separation(c), float(np.min(brute)), rtol=1e-13, atol=1e-13)
            selected_distance_cases.append(m)
    benchmark_summary = run_selected_set_development_benchmark(rng, RUN_DEVELOPMENT_BENCHMARKS)
    if RUN_DEVELOPMENT_BENCHMARKS:
        assert benchmark_summary["development_selected_set_benchmark_run"]
        assert benchmark_summary["evaluations"] == DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS
    else:
        assert benchmark_summary["development_selected_set_benchmark_run"] is False
        assert benchmark_summary["evaluations"] == 0
        assert selected_reference_queries < DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS
    development_probe = run_selected_set_development_benchmark(rng, True, evaluations=3)
    assert development_probe["development_selected_set_benchmark_run"] and development_probe["evaluations"] == 3
    V210_VALIDATION_RESULTS["selected_set_distance_equality"] = {
        "randomized_selected_sets": int(len(selected_distance_cases)),
        "selected_set_sizes": sorted(set(int(x) for x in selected_distance_cases)),
        "ckdtree_reference_geoMSD_checks": int(selected_reference_queries),
        "normal_initialization_avoids_10000_query_development_benchmark": bool(
            (not RUN_DEVELOPMENT_BENCHMARKS) and selected_reference_queries < DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS
        ),
    }
    V210_VALIDATION_RESULTS["selected_set_distance_benchmark"] = benchmark_summary
    V210_VALIDATION_RESULTS["development_benchmark_flag_behavior"] = {
        "normal_development_benchmark_run": bool(benchmark_summary["development_selected_set_benchmark_run"]),
        "normal_development_benchmark_evaluations": int(benchmark_summary["evaluations"]),
        "explicit_development_probe_run": bool(development_probe["development_selected_set_benchmark_run"]),
        "explicit_development_probe_evaluations": int(development_probe["evaluations"]),
        "default_evaluations_when_enabled": int(DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS),
    }

    latent = rng.normal(size=(4000, 3))
    mixed = latent @ np.array([[1.0, 0.8, -0.4], [0.2, 1.4, 0.7], [-0.5, 0.1, 1.1]])
    xs = StandardScaler().fit_transform(mixed)
    estimator = PCA(n_components=3, svd_solver="full").fit(xs)
    whitened = estimator.transform(xs) / np.sqrt(estimator.explained_variance_)
    np.testing.assert_allclose(np.mean(whitened, axis=0), 0.0, atol=1e-12)
    np.testing.assert_allclose(np.std(whitened, axis=0, ddof=1), 1.0, atol=1e-12)
    np.testing.assert_allclose(np.corrcoef(whitened, rowvar=False), np.eye(3), atol=1e-12)

    support = np.array([[0.0, 0.0], [5.0, 0.0], [0.0, 5.0], [5.0, 5.0], [2.5, 2.5]])
    selected = np.array([[0.0, 0.0], [5.0, 5.0]])
    brute_sbad = np.mean(np.min(np.sqrt(((support[:, None, :] - selected[None, :, :]) ** 2).sum(axis=2)), axis=1))
    np.testing.assert_allclose(exact_sbad(support, selected), brute_sbad, rtol=1e-13, atol=1e-13)

    coords_candidates = np.array([[0., 0.], [5., 0.], [0., 5.], [5., 5.], [2., 2.], [8., 8.]])
    selected_idx = np.array([0, 3, 4])
    cache = SupportDistanceCache(support, coords_candidates, 4)
    state = SBADNearestState(cache, selected_idx, len(support))
    for prefix in [3, len(support)]:
        prefix_state = SBADNearestState(cache, selected_idx, prefix)
        direct = exact_sbad(support[:prefix], coords_candidates[selected_idx])
        np.testing.assert_allclose(prefix_state.sbad, direct, rtol=1e-7, atol=1e-7)
    for pos in range(len(selected_idx)):
        for candidate in range(len(coords_candidates)):
            if candidate in set(np.delete(selected_idx, pos)):
                continue
            proposal = selected_idx.copy()
            proposal[pos] = candidate
            np.testing.assert_allclose(state.evaluate_swap(pos, candidate), exact_sbad(support, coords_candidates[proposal]), rtol=1e-7, atol=1e-7)

    cfg_single = RSSDConfig()
    cfg_single.CANDIDATE_SATURATION_MAX_K = 12
    cfg_single.PC_CANDIDATE_TOLERANCE = 10.0
    cfg_single.MAX_PC_CANDIDATE_TOLERANCE = 10.0
    target = np.array([0.0, 0.0])
    nhood_scores = rng.normal(0.0, 0.5, size=(60, 2))
    nhood_coords = rng.uniform(0.0, 100.0, size=(60, 2))
    nhood_indices = np.arange(60, dtype=int)
    ti = pd.DataFrame({"target_instance_id": ["T001"], "base_target_id": ["center"], "target_type": ["center"], "target_PC1": [0.0], "target_PC2": [0.0]})
    seqs, _, rep = build_nested_candidate_sequences(ti, nhood_indices, nhood_scores, nhood_coords, cfg_single)
    ref = candidate_sequence_bruteforce_reference(target, nhood_indices, nhood_scores, nhood_coords, cfg_single, cfg_single.CANDIDATE_SATURATION_MAX_K)
    key = tuple(np.round(target, 12))
    np.testing.assert_array_equal(seqs[key]["indices"], ref["indices"])

    cfg_prog = RSSDConfig()
    cfg_prog.CANDIDATE_SATURATION_START_K = 3
    cfg_prog.CANDIDATE_SATURATION_MAX_K = 24
    cfg_prog.CANDIDATE_SATURATION_GROWTH_FACTOR = 2
    cfg_prog.PC_CANDIDATE_TOLERANCE = 0.08
    cfg_prog.CANDIDATE_TOLERANCE_GROWTH = 2.0
    cfg_prog.MAX_PC_CANDIDATE_TOLERANCE = 1.5
    shell_scores = np.vstack([
        rng.normal(0.0, 0.02, size=(3, 2)),
        rng.normal(0.0, 0.16, size=(5, 2)),
        rng.normal(0.0, 0.35, size=(8, 2)),
        rng.normal(0.0, 0.75, size=(16, 2)),
    ])
    shell_coords = rng.uniform(0.0, 500.0, size=(len(shell_scores), 2))
    shell_indices = np.arange(len(shell_scores), dtype=int)
    seqs2, _, rep2 = build_nested_candidate_sequences(ti, shell_indices, shell_scores, shell_coords, cfg_prog)
    ref2 = candidate_sequence_bruteforce_reference(target, shell_indices, shell_scores, shell_coords, cfg_prog, cfg_prog.CANDIDATE_SATURATION_MAX_K)
    np.testing.assert_array_equal(seqs2[key]["indices"], ref2["indices"])
    for k in [3, 6, 12, 24]:
        prefix = seqs2[key]["indices"][:min(k, len(seqs2[key]["indices"]))]
        np.testing.assert_array_equal(prefix, seqs2[key]["indices"][:len(prefix)])
    assert np.all(np.diff(seqs2[key]["expansion_stages"]) >= 0)

    cfg_bench = RSSDConfig()
    cfg_bench.CANDIDATE_SATURATION_MAX_K = 64
    cfg_bench.PC_CANDIDATE_TOLERANCE = 0.30
    cfg_bench.MAX_PC_CANDIDATE_TOLERANCE = 1.20
    cfg_bench.CANDIDATE_TOLERANCE_GROWTH = 2.0
    bench_n = 1500
    bench_scores = rng.normal(0.0, 0.5, size=(bench_n, 2))
    bench_coords2 = rng.uniform(0.0, 1000.0, size=(bench_n, 2))
    bench_indices = np.arange(bench_n, dtype=int)
    t0 = time.perf_counter()
    ref_bench = candidate_sequence_bruteforce_reference(target, bench_indices, bench_scores, bench_coords2, cfg_bench, cfg_bench.CANDIDATE_SATURATION_MAX_K)
    old_candidate_elapsed = time.perf_counter() - t0
    t0 = time.perf_counter()
    seq_bench, _, _ = build_nested_candidate_sequences(ti, bench_indices, bench_scores, bench_coords2, cfg_bench)
    new_candidate_elapsed = time.perf_counter() - t0
    candidate_equal = bool(np.array_equal(seq_bench[key]["indices"], ref_bench["indices"]))
    assert candidate_equal
    V210_VALIDATION_RESULTS["candidate_sequence_benchmark"] = {
        "neighborhood_size": int(bench_n),
        "Kmax": int(cfg_bench.CANDIDATE_SATURATION_MAX_K),
        "number_of_tolerance_expansions": int(np.max(seq_bench[key]["expansion_stages"])) if len(seq_bench[key]["expansion_stages"]) else 0,
        "old_recompute_seconds": float(old_candidate_elapsed),
        "new_incremental_seconds": float(new_candidate_elapsed),
        "speedup_ratio": float(old_candidate_elapsed / max(new_candidate_elapsed, 1e-12)),
        "exact_candidate_sequence_equality": candidate_equal,
    }

    pools = [np.array([0, 1, 2]), np.array([2, 3, 4]), np.array([1, 4, 5])]
    dists = [np.array([0.1, 0.2, 0.3]), np.array([0.2, 0.1, 0.4]), np.array([0.3, 0.2, 0.1])]
    minimum = assignment_from_costs(pools, dists)
    bank = OptimizationStartBank(minimum, pools, dists, seed, 3, "ad_reference", 8)
    assert bank.base_random_keys(5) == bank.base_random_keys(5)
    warm = np.array([1, 2, 5])
    before = bank.base_random_keys(5)
    _ = bank.starts(5, warm_starts=[warm])
    after = bank.base_random_keys(5)
    assert before == after
    base_start = np.asarray(bank.base_starts[1], dtype=int)
    perm_5000 = coordinate_exchange_permutation_preview(seed, "ad_reference", 3, base_start, len(base_start), 5)
    perm_40000 = coordinate_exchange_permutation_preview(seed, "ad_reference", 3, base_start, len(base_start), 5)
    assert perm_5000 == perm_40000
    assert perm_5000 == coordinate_exchange_permutation_preview(seed, "ad_reference", 3, base_start, len(base_start), 5)
    assert coordinate_exchange_start_seed(seed, "ad_reference", 3, base_start) == coordinate_exchange_start_seed(seed, "ad_reference", 3, base_start.copy())
    V210_VALIDATION_RESULTS["per_start_rng"] = {"same_assignment_same_schedule": True, "warm_start_insertion_independent": True, "support_size_independent": True}

    support_cfg = RSSDConfig()
    support_cfg.SUPPORT_GAP_ANCHORS = 3
    support_cfg.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR = 3
    support_coords = np.array([[0., 0.], [10., 0.], [0., 10.], [10., 10.], [5., 5.]])
    coords_support_test = np.array([[0., 0.], [10., 10.], [0., 10.], [10., 0.], [5., 0.], [5., 10.], [8., 5.]])
    core = np.array([0, 1])
    cand_idx = np.arange(len(coords_support_test), dtype=int)
    cache2 = SupportDistanceCache(support_coords, coords_support_test, 4)
    initial_state = SBADNearestState(cache2, core, len(support_coords))
    final_sel, support_rows, coverage = add_spatial_support_sites(core, 1, support_coords, cand_idx, coords_support_test, support_cfg, cache=cache2, support_prefix_size=len(support_coords))
    assert len(support_rows) == 1
    row = support_rows.iloc[0]
    bounded = np.array([int(x) for x in str(row["bounded_candidate_analysis_indices"]).split(";") if str(x)], dtype=int)
    assert len(bounded) >= 1
    brute = []
    for cand in bounded:
        proposal = np.array([*core.tolist(), int(cand)], dtype=int)
        brute_s = exact_sbad(support_coords, coords_support_test[proposal])
        brute_min = minimum_selected_separation(coords_support_test[proposal])
        brute.append((brute_s, -brute_min, int(cand)))
    brute.sort()
    assert int(row["analysis_index"]) == brute[0][2]
    assert float(row["SBAD_after_addition"]) <= float(row["SBAD_before_addition"]) + 1e-12
    assert int(row["unique_support_candidates_evaluated"]) >= 1

    rows_confirm, meta_confirm = support_confirmation_state_machine_preview(
        [5000, 10000, 20000, 40000], [False, True, True, True], 2
    )
    assert [r["support_size"] for r in rows_confirm] == [5000, 10000, 20000, 40000]
    assert rows_confirm[1]["stable_stage"] and rows_confirm[2]["stable_stage"]
    assert not rows_confirm[1]["confirmation_stage"] and not rows_confirm[2]["confirmation_stage"]
    assert rows_confirm[3]["confirmation_stage"] and rows_confirm[3]["stable_stage"]
    assert meta_confirm["ad_support_resolution_stable"] and meta_confirm["final_support_size"] == 40000

    rows_reset, meta_reset = support_confirmation_state_machine_preview(
        [5000, 10000, 20000, 40000, 80000], [False, True, True, False, True], 2
    )
    assert rows_reset[3]["confirmation_stage"] and not rows_reset[3]["stable_stage"]
    assert rows_reset[3]["support_resolution_status"] == "confirmation_failed_reset"
    assert rows_reset[4]["confirmation_stage"] is False
    assert meta_reset["ad_support_resolution_stable"] is False

    rows_max, meta_max = support_confirmation_state_machine_preview(
        [5000, 10000, 20000], [False, True, True], 2
    )
    assert rows_max[-1]["support_resolution_status"] == "stable_at_max_without_separate_confirmation"
    assert rows_max[-1]["stable_stage"] and not rows_max[-1]["confirmation_stage"]
    assert meta_max["ad_support_resolution_stable"] is False

    np.testing.assert_allclose(focused_ad_refinement_improvement(100.0, 99.0), 0.01, rtol=1e-13, atol=1e-13)
    previous_stage_sbad_star = 95.0
    assert previous_stage_sbad_star == 95.0
    np.testing.assert_allclose(focused_ad_refinement_improvement(100.0, 99.0), 0.01, rtol=1e-13, atol=1e-13)
    assert focused_ad_refinement_improvement(100.0, 100.5) == 0.0

    engine_source = globals().get("_ESAP_RSSD_ENGINE_SOURCE", "")
    screen_start = engine_source.index("def screen_support_resolution")
    screen_end = engine_source.find("\ndef ", screen_start + 1)
    screen_source = engine_source[screen_start:screen_end if screen_end != -1 else len(engine_source)]
    assert 'optimization_phase="final"' not in screen_source
    assert screen_source.count('optimization_phase="support_screen"') == 2
    V210_VALIDATION_RESULTS["support_resolution_confirmation"] = {
        "separate_confirmation_prefix_evaluated": bool(rows_confirm[-1]["support_size"] == 40000),
        "confirmation_stage_only_on_confirmation_row": bool(
            rows_confirm[3]["confirmation_stage"] and not rows_confirm[1]["confirmation_stage"] and not rows_confirm[2]["confirmation_stage"]
        ),
        "confirmation_failure_resets_provisional_stability": bool(
            rows_reset[3]["support_resolution_status"] == "confirmation_failed_reset" and not rows_reset[4]["confirmation_stage"]
        ),
        "final_support_size_equals_confirmation_prefix": bool(meta_confirm["final_support_size"] == 40000),
        "stable_at_max_without_separate_confirmation_status": bool(rows_max[-1]["support_resolution_status"] == "stable_at_max_without_separate_confirmation"),
        "support_screen_has_no_final_optimizer_phase": True,
    }
    V210_VALIDATION_RESULTS["focused_ad_refinement"] = {
        "pre_100_post_99_improvement": float(focused_ad_refinement_improvement(100.0, 99.0)),
        "previous_stage_95_pre_100_post_99_improvement": float(focused_ad_refinement_improvement(100.0, 99.0)),
        "no_better_design_improvement": float(focused_ad_refinement_improvement(100.0, 100.5)),
    }

    base = make_base_ccd(2, 1.0, 2)
    tmp = RSSDConfig(N_SAMPLES=12, N_DESIGN_COMPONENTS=2, SAMPLE_BUDGET_MODE="rssd_with_support")
    inst, _, rep_budget = allocate_sample_budget(base, 12, tmp, 2)
    assert len(inst) == 10 and rep_budget["spatial_support_sites"] == 2
    tmp.N_SAMPLES = 20
    inst, _, rep_budget = allocate_sample_budget(base, 20, tmp, 2)
    assert len(inst) == 18 and rep_budget["spatial_support_sites"] == 2
    tmp.N_SAMPLES = 10
    inst, _, rep_budget = allocate_sample_budget(base, 10, tmp, 2)
    assert len(inst) == 10 and rep_budget["spatial_support_sites"] == 0

    support_seq, report = build_spatially_balanced_support_sequence(rng.uniform(size=(250, 2)), np.arange(250), 64)
    assert len(support_seq) <= 64 and support_seq["support_rank"].is_monotonic_increasing and report["uses_only_xy"] and not report["uses_pc_scores"]

    duplicate_info = duplicate_top_level_definition_report(globals().get("_ESAP_RSSD_ENGINE_SOURCE", ""))
    assert duplicate_info["duplicate_top_level_definition_count"] == 0
    V210_VALIDATION_RESULTS["duplicate_top_level_definitions"] = duplicate_info
    print("Unit validations passed: exact vectorized selected-set distances, PCA standardization, exact SBAD/cache, incremental candidate sequences, support-resolution confirmation, true focused refinement, per-start RNG, support-site exact ranking, development benchmark flag, and duplicate-definition AST check.")


_ESAP_RSSD_ENGINE_SOURCE = '# @title Initialize ESAP RSSD v2.10 engine { display-mode: "form" }\n# Core dependencies available in a standard Google Colab runtime.\nfrom __future__ import annotations\n\nimport ast\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport sys\nimport time\nimport warnings\nfrom collections import OrderedDict\nfrom dataclasses import asdict, dataclass, field\nfrom importlib import metadata as importlib_metadata\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom IPython.display import display\nfrom scipy.optimize import linear_sum_assignment\nfrom scipy.spatial import cKDTree\nfrom scipy.spatial.distance import pdist, squareform\nfrom scipy.stats import chi2, skew\nfrom sklearn.decomposition import IncrementalPCA, PCA\nfrom sklearn.preprocessing import PowerTransformer, StandardScaler\n\nNOTEBOOK_VERSION = "2.10.0"\nnp.set_printoptions(precision=4, suppress=True)\n\n\n# v2.10 default configuration\n@dataclass\nclass RSSDConfig:\n    # Input. The defaults make the notebook executable without external files.\n    INPUT_FILE: Optional[str] = None\n    EXCEL_SHEET_NAME: Any = 0\n    USE_SYNTHETIC_DEMO: bool = True\n    SYNTHETIC_ROWS: int = 6000\n\n    # Required column roles. Change these for a real file.\n    ID_COLUMN: str = "sample_id"\n    X_COLUMN: str = "x"\n    Y_COLUMN: str = "y"\n    FEATURE_COLUMNS: List[str] = field(\n        default_factory=lambda: ["sensor_1", "sensor_2", "sensor_3", "sensor_4"]\n    )\n    FEATURE_TRANSFORMS: Dict[str, str] = field(default_factory=dict)\n\n    # Response-surface design and sample budget.\n    N_SAMPLES: int = 20\n    N_DESIGN_COMPONENTS: int = 2\n    DESIGN_COVERAGE: float = 0.80\n    CENTER_REPLICATES: int = 2\n    SAMPLE_BUDGET_MODE: str = "rssd_with_support"  # rssd_with_support, ccd_exact, or balanced_target_replication\n    SUPPORT_SITE_MODE: str = "legacy_inspired_auto"  # legacy_inspired_auto, manual, or none\n    N_SUPPORT_SITES: int = 0\n\n    # Statistical screening and candidate discovery.\n    OUTLIER_COVERAGE: float = 0.999\n    CANDIDATE_EXPLORATION_MODE: str = "adaptive"\n    CANDIDATES_PER_TARGET: int = 3\n    CANDIDATE_SATURATION_START_K: int = 3\n    CANDIDATE_SATURATION_GROWTH_FACTOR: int = 2\n    CANDIDATE_SATURATION_MAX_K: int = 64\n    CANDIDATE_SATURATION_STABLE_STAGES_REQUIRED: int = 2\n    CANDIDATE_STAGE_EXPLORATION_STARTS: int = 5\n    CANDIDATE_SATURATION_AD_REL_TOL: float = 0.005\n    CANDIDATE_SATURATION_GEOMSD_REL_TOL: float = 0.01\n    CANDIDATE_SATURATION_MINSEP_REL_TOL: float = 0.02\n    PC_CANDIDATE_TOLERANCE: float = 0.15\n    MAX_PC_CANDIDATE_TOLERANCE: float = 1.50\n    CANDIDATE_TOLERANCE_GROWTH: float = 1.5\n\n    # Optimizer. N_OPTIMIZER_STARTS is a maximum when stable-search early stopping is enabled.\n    N_OPTIMIZER_STARTS: int = 50\n    MIN_OPTIMIZER_STARTS: int = 12\n    OPTIMIZER_NO_IMPROVEMENT_PATIENCE: int = 12\n    EARLY_STOP_ON_STABLE_SEARCH: bool = True\n    VERIFY_REPRODUCIBILITY_BY_RERUN: bool = False\n    OPTIMIZER_INITIALIZATION_MODE: str = "adaptive"\n    OPTIMIZER_START_MODE: str = "adaptive"\n    OPTIMIZER_START_BATCH_SIZE: int = 10\n    OPTIMIZER_MIN_STARTS: int = 20\n    OPTIMIZER_MAX_STARTS: int = 150\n    OPTIMIZER_BEST_IMPROVEMENT_REL_TOL: float = 0.001\n    OPTIMIZER_NEAR_BEST_REL_TOL: float = 0.01\n    OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION: float = 0.80\n    OPTIMIZER_STABLE_BATCHES_REQUIRED: int = 2\n    MAX_OPTIMIZER_SWEEPS: int = 100\n    AD_SUPPORT_MODE: str = "adaptive"\n    AD_SUPPORT_START_SIZE: int = 5000\n    AD_SUPPORT_GROWTH_FACTOR: int = 2\n    AD_SUPPORT_MAX_SIZE: int = 40000\n    AD_SUPPORT_FIXED_SIZE: int = 20000\n    AD_SUPPORT_REL_TOL: float = 0.005\n    AD_SUPPORT_RANK_CORRELATION_MIN: float = 0.995\n    AD_SUPPORT_STABLE_STAGES_REQUIRED: int = 2\n    AD_SUPPORT_RANK_PANEL_SIZE: int = 20\n    SUPPORT_STAGE_INITIAL_STARTS: int = 10\n    SUPPORT_STAGE_REFINEMENT_STARTS: int = 3\n    AD_SUPPORT_PRACTICAL_EQUIV_REL_TOL: float = 0.0025\n    AD_SUPPORT_DECISIVE_PAIR_AGREEMENT_MIN: float = 0.95\n    AD_SUPPORT_NEAR_BEST_REL_TOL: float = 0.005\n    SUPPORT_HYBRID_GEOMSD_REL_TOL: float = 0.01\n    SUPPORT_HYBRID_MINSEP_REL_TOL: float = 0.02\n    SUPPORT_HYBRID_COVERAGE_RATIO_ABS_TOL: float = 0.005\n    SUPPORT_GAP_ANCHORS: int = 500\n    SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR: int = 5\n    AD_COVERAGE_ENVELOPE_REL_TOL: float = 0.05\n    AD_DISTANCE_CACHE_MAX_MIB: int = 512\n    RUN_REFERENCE_DESIGNS: bool = False\n    RANDOM_SEED: int = 20250717\n    GEOMSD_TIE_REL_TOL: float = 1e-6\n    PCA_TIE_TOL: float = 1e-12\n\n    # Memory behavior.\n    MEMORY_MODE: str = "auto"  # full, auto, or incremental\n    MAX_WORKING_MEMORY_FRACTION: float = 0.45\n    INCREMENTAL_BATCH_SIZE: int = 50000\n    NUMERIC_DTYPE: str = "float32"\n\n    # Optional approximation for extreme candidate-discovery workloads.\n    ALLOW_APPROXIMATE_PREFILTER: bool = False\n    APPROX_PREFILTER_TRIGGER_ROWS: int = 750000\n    APPROX_PREFILTER_MAX_ROWS: int = 250000\n    APPROX_PREFILTER_BINS_PER_PC: int = 8\n\n    # Diagnostics and testing.\n    PLOT_MAX_POINTS: int = 50000\n    SURVEY_COLOR: str = "C0"\n    TARGET_COLOR: str = "C3"\n    OUTLIER_COLOR: str = "C1"\n    CANDIDATE_COLOR: str = "C2"\n    SELECTED_COLOR: str = "C3"\n    FOOTPRINT_COLOR: str = "0.5"\n    SUPPORT_COLOR: str = "C4"\n    DIAGNOSTIC_CHUNK_SIZE: int = 100000\n    RUN_SPACING_DIAGNOSTIC: bool = True\n    SPACING_DIAGNOSTIC_MAX_POINTS: int = 6000\n    SPACING_DIAGNOSTIC_NEIGHBORS: int = 24\n    SPACING_DIAGNOSTIC_RANDOM_PAIRS: int = 120000\n    SPACING_DIAGNOSTIC_BINS: int = 16\n    SPACING_TARGET_SEMIVARIANCE_FRACTION: float = 0.90\n    SPACING_CAUTION_RATIO: float = 1.00\n    SPACING_WARNING_RATIO: float = 0.75\n    SPACING_MIN_PAIRS_PER_BIN: int = 250\n    SPACING_RANGE_STABILITY_BINS: int = 3\n    SPACING_PLATEAU_TAIL_BINS: int = 4\n    SPACING_PLATEAU_REL_SLOPE_TOL: float = 0.05\n\n\ncfg = RSSDConfig()\n\n# Development timing benchmarks are opt-in and are not part of normal field-run initialization.\nRUN_DEVELOPMENT_BENCHMARKS = False\nDEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS = 10000\n\n# Explicit transform examples for a real run:\n# cfg.FEATURE_TRANSFORMS = {"ECa_vertical": "log", "NDVI": "yeo-johnson"}\n# cfg.USE_SYNTHETIC_DEMO = False\n# cfg.INPUT_FILE = "/content/my_survey.csv"  # or None for Colab upload\n\n\n\n\n# @title Load internal data and PCA functions (run; no editing required) { display-mode: "form" }\ndef make_synthetic_survey(n: int, seed: int) -> pd.DataFrame:\n    """Create correlated, spatially structured proxy variables for a runnable demonstration."""\n    rng = np.random.default_rng(seed)\n    x = rng.uniform(0.0, 2400.0, n)\n    y = rng.uniform(0.0, 1600.0, n)\n    z1 = 0.9 * np.sin(x / 310.0) + 0.7 * np.cos(y / 240.0) + rng.normal(0, 0.35, n)\n    z2 = 0.75 * z1 + 0.5 * np.sin((x + y) / 420.0) + rng.normal(0, 0.3, n)\n    z3 = -0.35 * z1 + 0.9 * np.cos(x / 520.0) + rng.normal(0, 0.45, n)\n    z4 = 0.45 * z2 + 0.5 * z3 + rng.normal(0, 0.4, n)\n    return pd.DataFrame(\n        {\n            "sample_id": np.arange(1, n + 1, dtype=np.int64),\n            "x": x,\n            "y": y,\n            "sensor_1": z1,\n            "sensor_2": z2,\n            "sensor_3": z3,\n            "sensor_4": z4,\n        }\n    )\n\n\ndef load_survey(config: RSSDConfig) -> Tuple[pd.DataFrame, str]:\n    """Load CSV/XLS/XLSX input or the deterministic synthetic demonstration."""\n    if config.USE_SYNTHETIC_DEMO:\n        return make_synthetic_survey(config.SYNTHETIC_ROWS, config.RANDOM_SEED), "synthetic_demo"\n\n    input_path = config.INPUT_FILE\n    if input_path is None:\n        try:\n            from google.colab import files  # type: ignore\n        except ImportError as exc:\n            raise ValueError(\n                "Set cfg.INPUT_FILE to a CSV/XLS/XLSX path outside Colab, or enable the synthetic demo."\n            ) from exc\n        uploaded = files.upload()\n        if len(uploaded) != 1:\n            raise ValueError("Upload exactly one CSV or Excel survey file.")\n        input_path = next(iter(uploaded))\n\n    path = Path(input_path)\n    suffix = path.suffix.lower()\n    if suffix == ".csv":\n        df = pd.read_csv(path)\n    elif suffix in {".xls", ".xlsx", ".xlsm"}:\n        try:\n            df = pd.read_excel(path, sheet_name=config.EXCEL_SHEET_NAME)\n        except ImportError as exc:\n            raise ImportError("Excel input requires openpyxl (xlsx/xlsm) or xlrd (xls).") from exc\n    else:\n        raise ValueError(f"Unsupported input type {suffix!r}; use CSV, XLS, XLSX, or XLSM.")\n    return df, str(path)\n\n\n\n\ndef coordinates_look_geographic(x: np.ndarray, y: np.ndarray, x_name: str, y_name: str) -> bool:\n    """Conservatively detect decimal-degree longitude/latitude coordinates."""\n    xn, yn = x_name.lower(), y_name.lower()\n    name_signal = ("lon" in xn or "longitude" in xn) and ("lat" in yn or "latitude" in yn)\n    bounded = np.mean((x >= -180) & (x <= 180) & (y >= -90) & (y <= 90)) >= 0.99\n    finite = np.isfinite(x) & np.isfinite(y)\n    if not finite.any():\n        return False\n    xf, yf = x[finite], y[finite]\n    decimal_signal = np.mean(np.abs(xf - np.round(xf)) > 1e-6) > 0.8 and np.mean(\n        np.abs(yf - np.round(yf)) > 1e-6\n    ) > 0.8\n    typical_lonlat = (\n        np.nanmedian(np.abs(xf)) >= 20\n        and np.nanmedian(np.abs(yf)) >= 10\n        and np.ptp(xf) <= 20\n        and np.ptp(yf) <= 20\n    )\n    return bool(name_signal or (bounded and decimal_signal and typical_lonlat))\n\n\ndef validate_and_index_data(\n    df: pd.DataFrame, config: RSSDConfig\n) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, Dict[str, Any]]:\n    """Validate required data and return valid-row and coordinate-eligibility masks."""\n    required = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]\n    missing_columns = [c for c in required if c not in df.columns]\n    if missing_columns:\n        raise KeyError(f"Missing required columns: {missing_columns}")\n    if len(set(config.FEATURE_COLUMNS)) != len(config.FEATURE_COLUMNS):\n        raise ValueError("FEATURE_COLUMNS contains duplicates.")\n    if config.N_DESIGN_COMPONENTS > len(config.FEATURE_COLUMNS):\n        raise ValueError("N_DESIGN_COMPONENTS cannot exceed the number of feature columns.")\n\n    duplicate_id_count = int(df[config.ID_COLUMN].duplicated(keep=False).sum())\n    if duplicate_id_count:\n        raise ValueError(\n            f"Found {duplicate_id_count} rows with duplicated IDs. IDs must be unique before RSSD selection."\n        )\n\n    numeric = df[[config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]].apply(\n        pd.to_numeric, errors="coerce"\n    )\n    arr = numeric.to_numpy(dtype=np.float64, copy=False)\n    finite_mask = np.isfinite(arr).all(axis=1)\n    missing_nonfinite_counts = {\n        c: int((~np.isfinite(numeric[c].to_numpy(dtype=np.float64))).sum())\n        for c in numeric.columns\n    }\n    n_invalid = int((~finite_mask).sum())\n    if finite_mask.sum() < max(config.N_SAMPLES, 20):\n        raise ValueError("Too few complete finite rows remain for the requested design.")\n\n    valid_features = numeric.loc[finite_mask, config.FEATURE_COLUMNS]\n    zero_variance = [c for c in config.FEATURE_COLUMNS if float(valid_features[c].var(ddof=0)) <= 0.0]\n    if zero_variance:\n        raise ValueError(f"Zero-variance feature columns are not usable: {zero_variance}")\n\n    x = numeric.loc[finite_mask, config.X_COLUMN].to_numpy(dtype=np.float64)\n    y = numeric.loc[finite_mask, config.Y_COLUMN].to_numpy(dtype=np.float64)\n    if coordinates_look_geographic(x, y, config.X_COLUMN, config.Y_COLUMN):\n        raise ValueError(\n            "Coordinates strongly resemble longitude/latitude in decimal degrees. Reproject to an appropriate linear-unit CRS (for example, the relevant UTM zone) before computing geoMSD."\n        )\n\n    duplicated_coordinates_all = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep=False).to_numpy()\n    later_coordinate_duplicate = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep="first").to_numpy()\n    coordinate_eligible = finite_mask & ~later_coordinate_duplicate\n\n    out = df.copy()\n    out["rssd_complete_finite"] = finite_mask\n    out["rssd_duplicate_coordinate"] = duplicated_coordinates_all\n    out["rssd_coordinate_candidate_eligible"] = coordinate_eligible\n    report = {\n        "input_rows": int(len(df)),\n        "complete_finite_rows": int(finite_mask.sum()),\n        "invalid_rows_excluded": n_invalid,\n        "duplicated_id_rows": duplicate_id_count,\n        "duplicated_coordinate_rows": int(duplicated_coordinates_all.sum()),\n        "later_duplicate_coordinate_rows_excluded_from_candidates": int(later_coordinate_duplicate.sum()),\n        "missing_or_nonfinite_by_column": missing_nonfinite_counts,\n        "duplicate_coordinate_handling": (\n            "All records are retained; among exact coordinate duplicates, only the first valid record is candidate-eligible."\n        ),\n    }\n    return out, finite_mask, coordinate_eligible, report\n\n\ndef transform_features(\n    values: np.ndarray, feature_names: Sequence[str], transforms: Mapping[str, str]\n) -> Tuple[np.ndarray, Dict[str, Dict[str, Any]]]:\n    """Apply explicit per-feature transformations without automatic choices."""\n    transformed = np.asarray(values, dtype=np.float64).copy()\n    details: Dict[str, Dict[str, Any]] = {}\n    allowed = {"none", "log", "yeo-johnson"}\n    unknown_keys = sorted(set(transforms) - set(feature_names))\n    if unknown_keys:\n        raise KeyError(f"FEATURE_TRANSFORMS contains unknown features: {unknown_keys}")\n    for j, name in enumerate(feature_names):\n        method = transforms.get(name, "none").lower()\n        if method not in allowed:\n            raise ValueError(f"Unsupported transform {method!r} for {name}; choose {sorted(allowed)}.")\n        if method == "none":\n            details[name] = {"method": "none"}\n        elif method == "log":\n            if np.any(transformed[:, j] <= 0):\n                raise ValueError(f"Log transformation requires strictly positive values in {name!r}.")\n            transformed[:, j] = np.log(transformed[:, j])\n            details[name] = {"method": "log", "base": "natural"}\n        else:\n            pt = PowerTransformer(method="yeo-johnson", standardize=False)\n            transformed[:, [j]] = pt.fit_transform(transformed[:, [j]])\n            details[name] = {"method": "yeo-johnson", "lambda": float(pt.lambdas_[0])}\n    return transformed.astype(np.float32, copy=False), details\n\n\ndef available_memory_bytes() -> int:\n    """Return available RAM when psutil is present, otherwise use a conservative Colab fallback."""\n    try:\n        import psutil\n\n        return int(psutil.virtual_memory().available)\n    except ImportError:\n        return 8 * 1024**3\n\n\ndef choose_pca_mode(n_rows: int, n_features: int, config: RSSDConfig) -> Tuple[str, Dict[str, float]]:\n    """Choose full or incremental PCA from an explicit working-memory estimate."""\n    # Approximate transformed, standardized, work, and score arrays plus estimator overhead.\n    estimated = int(n_rows * n_features * (4 + 4 + 8 + 4) + n_rows * config.N_DESIGN_COMPONENTS * 4)\n    available = available_memory_bytes()\n    safe = int(config.MAX_WORKING_MEMORY_FRACTION * available)\n    if config.MEMORY_MODE == "auto":\n        mode = "full" if estimated <= safe else "incremental"\n    else:\n        mode = config.MEMORY_MODE\n    return mode, {\n        "estimated_working_bytes": estimated,\n        "available_memory_bytes": available,\n        "safe_memory_bytes": safe,\n        "estimated_working_mib": estimated / 1024**2,\n    }\n\n\ndef fit_standardized_pca(\n    transformed: np.ndarray, config: RSSDConfig\n) -> Tuple[np.ndarray, StandardScaler, Any, str, Dict[str, float]]:\n    """Standardize features, fit PCA, and divide retained scores by sqrt(explained variance)."""\n    n, f = transformed.shape\n    mode, memory_report = choose_pca_mode(n, f, config)\n    scaler = StandardScaler(copy=True)\n    batch = max(config.INCREMENTAL_BATCH_SIZE, f + 1)\n\n    def batch_slices() -> Iterable[Tuple[int, int]]:\n        """Yield batches while merging a final batch smaller than the feature count."""\n        start = 0\n        while start < n:\n            end = min(start + batch, n)\n            if 0 < n - end < f:\n                end = n\n            yield start, end\n            start = end\n\n    if mode == "full":\n        standardized = scaler.fit_transform(transformed).astype(np.float32, copy=False)\n        estimator: Any = PCA(n_components=f, svd_solver="full", random_state=config.RANDOM_SEED)\n        estimator.fit(standardized)\n        retained_raw = (standardized - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T\n    else:\n        for start, end in batch_slices():\n            scaler.partial_fit(transformed[start:end])\n        estimator = IncrementalPCA(n_components=f, batch_size=batch)\n        for start, end in batch_slices():\n            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)\n            estimator.partial_fit(xb)\n        retained_raw = np.empty((n, config.N_DESIGN_COMPONENTS), dtype=np.float32)\n        for start, end in batch_slices():\n            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)\n            retained_raw[start:end] = (\n                (xb - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T\n            )\n\n    scales = np.sqrt(np.asarray(estimator.explained_variance_[: config.N_DESIGN_COMPONENTS]))\n    if np.any(scales <= 0):\n        raise ValueError("PCA produced a nonpositive explained variance in a retained component.")\n    design_scores = (retained_raw / scales).astype(np.float32, copy=False)\n    return design_scores, scaler, estimator, mode, memory_report\n\n\ndef pca_validation_table(scores: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray]:\n    """Return means, sample standard deviations, and the correlation matrix."""\n    names = [f"PC{i + 1}" for i in range(scores.shape[1])]\n    table = pd.DataFrame(\n        {"mean": np.mean(scores, axis=0), "sample_sd": np.std(scores, axis=0, ddof=1)}, index=names\n    )\n    corr = np.corrcoef(scores, rowvar=False)\n    corr = np.atleast_2d(corr)\n    return table, corr\n\n\n# @title Load internal spatial-design functions (run; no editing required) { display-mode: "form" }\n\ndef selected_pairwise_distance_matrix(coords: np.ndarray) -> np.ndarray:\n    """Return the exact small m by m selected-site Euclidean distance matrix."""\n    coords = np.asarray(coords, dtype=np.float64)\n    if coords.ndim != 2 or coords.shape[1] != 2:\n        raise ValueError("Selected coordinates must be an m by 2 array.")\n    if coords.shape[0] < 2:\n        raise ValueError("At least two selected coordinate pairs are required.")\n    delta = coords[:, None, :] - coords[None, :, :]\n    d2 = np.einsum("ijk,ijk->ij", delta, delta, dtype=np.float64)\n    np.fill_diagonal(d2, np.inf)\n    return np.sqrt(d2, dtype=np.float64)\n\n\ndef selected_nearest_neighbor_distances_ckdtree_reference(coords: np.ndarray, workers: int = -1) -> np.ndarray:\n    """Validation-only reference for the previous tiny selected-set cKDTree implementation."""\n    coords = np.asarray(coords, dtype=np.float64)\n    if coords.ndim != 2 or coords.shape[0] < 2:\n        raise ValueError("At least two selected coordinate pairs are required.")\n    distances, _ = cKDTree(coords).query(coords, k=2, workers=workers)\n    return np.asarray(distances[:, 1], dtype=np.float64)\n\n\ndef selected_nearest_neighbor_distances(coords: np.ndarray) -> np.ndarray:\n    """Calculate exact nearest selected-site distances with a direct small m by m array."""\n    return np.min(selected_pairwise_distance_matrix(coords), axis=1)\n\n\ndef exact_geo_msd(coords: np.ndarray) -> float:\n    """Compute exp(mean(log(d_j))) for exact nearest-selected-neighbor distances d_j."""\n    d = selected_nearest_neighbor_distances(coords)\n    if np.any(d <= 0):\n        return 0.0\n    return float(np.exp(np.mean(np.log(d))))\n\n\ndef make_base_ccd(p: int, radius: float, center_replicates: int) -> pd.DataFrame:\n    """Construct cube, axial, and replicated-center targets on common outer radius R."""\n    if p > 4:\n        raise ValueError("p > 4 requires a separately specified hybrid or small composite design.")\n    rows: List[Dict[str, Any]] = []\n    cube_level = radius / math.sqrt(p)\n    for number in range(2**p):\n        signs = np.array([1.0 if (number >> j) & 1 else -1.0 for j in range(p)])\n        row = {"base_target_id": f"cube_{number + 1:02d}", "target_type": "cube"}\n        row.update({f"target_PC{j + 1}": float(signs[j] * cube_level) for j in range(p)})\n        rows.append(row)\n    for axis in range(p):\n        for sign, label in [(-1.0, "minus"), (1.0, "plus")]:\n            values = np.zeros(p)\n            values[axis] = sign * radius\n            row = {\n                "base_target_id": f"axial_PC{axis + 1}_{label}",\n                "target_type": "axial",\n            }\n            row.update({f"target_PC{j + 1}": float(values[j]) for j in range(p)})\n            rows.append(row)\n    for c in range(center_replicates):\n        row = {"base_target_id": f"center_{c + 1:02d}", "target_type": "center"}\n        row.update({f"target_PC{j + 1}": 0.0 for j in range(p)})\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\n\n\ndef approximate_pca_prefilter(\n    eligible_indices: np.ndarray,\n    scores: np.ndarray,\n    targets: np.ndarray,\n    config: RSSDConfig,\n) -> Tuple[np.ndarray, bool, Dict[str, Any]]:\n    """Optional reproducible occupancy prefilter that explicitly preserves tails and target neighborhoods."""\n    eligible_indices = np.asarray(eligible_indices, dtype=int)\n    if (\n        not config.ALLOW_APPROXIMATE_PREFILTER\n        or len(eligible_indices) <= config.APPROX_PREFILTER_TRIGGER_ROWS\n    ):\n        return eligible_indices, False, {"reason": "disabled_or_below_trigger"}\n\n    rng = np.random.default_rng(config.RANDOM_SEED)\n    z = scores[eligible_indices]\n    p = z.shape[1]\n    bins = config.APPROX_PREFILTER_BINS_PER_PC\n    bin_ids = np.zeros((len(z), p), dtype=np.int16)\n    for j in range(p):\n        edges = np.quantile(z[:, j], np.linspace(0, 1, bins + 1)[1:-1])\n        bin_ids[:, j] = np.digitize(z[:, j], np.unique(edges), right=False)\n    multipliers = (bins ** np.arange(p)).astype(np.int64)\n    codes = (bin_ids.astype(np.int64) * multipliers).sum(axis=1)\n    occupied = np.unique(codes)\n    per_bin_cap = max(1, math.ceil(config.APPROX_PREFILTER_MAX_ROWS / len(occupied)))\n    kept_local: List[np.ndarray] = []\n    for code_value in occupied:\n        members = np.flatnonzero(codes == code_value)\n        if len(members) > per_bin_cap:\n            members = np.sort(rng.choice(members, size=per_bin_cap, replace=False))\n        kept_local.append(members)\n\n    # Explicitly preserve univariate tails.\n    tail_n = min(1000, len(z))\n    tail_local = []\n    for j in range(p):\n        order = np.argsort(z[:, j], kind="stable")\n        tail_local.extend([order[:tail_n], order[-tail_n:]])\n\n    # Explicitly preserve every observation in each initial target neighborhood.\n    tree = cKDTree(z)\n    target_local: List[int] = []\n    for target in np.unique(targets, axis=0):\n        target_local.extend(tree.query_ball_point(target, r=config.PC_CANDIDATE_TOLERANCE))\n\n    local = np.unique(\n        np.concatenate([*kept_local, *tail_local, np.asarray(target_local, dtype=int)])\n    )\n    kept = eligible_indices[local]\n    report = {\n        "input_eligible_rows": int(len(eligible_indices)),\n        "prefilter_rows": int(len(kept)),\n        "occupied_pca_strata": int(len(occupied)),\n        "per_stratum_cap": int(per_bin_cap),\n        "target_neighborhood_rows_explicitly_preserved": int(len(np.unique(target_local))),\n    }\n    return kept, True, report\n\n\n\n\n\n\ndef assignment_from_costs(\n    pools: Sequence[np.ndarray],\n    pool_distances: Sequence[np.ndarray],\n    rng: Optional[np.random.Generator] = None,\n) -> Optional[np.ndarray]:\n    """Use a minimum-cost bipartite assignment; random costs produce reproducible alternative starts."""\n    union = np.unique(np.concatenate(pools))\n    if len(union) < len(pools):\n        return None\n    column = {int(index): j for j, index in enumerate(union)}\n    prohibitive = 1e12\n    cost = np.full((len(pools), len(union)), prohibitive, dtype=np.float64)\n    for i, (pool, distances) in enumerate(zip(pools, pool_distances)):\n        values = distances if rng is None else rng.random(len(pool))\n        for index, value in zip(pool, values):\n            cost[i, column[int(index)]] = float(value)\n    rows, cols = linear_sum_assignment(cost)\n    if len(rows) != len(pools) or np.any(cost[rows, cols] >= prohibitive):\n        return None\n    selected = np.empty(len(pools), dtype=int)\n    selected[rows] = union[cols]\n    return selected\n\n\ndef matching_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray) -> Tuple[float, float]:\n    """Return mean and maximum Euclidean target mismatch in standardized PC units."""\n    distances = np.linalg.norm(scores[selected] - targets, axis=1)\n    return float(np.mean(distances)), float(np.max(distances))\n\n\ndef design_rank(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray) -> Tuple[float, float, float]:\n    """Return geoMSD, mean mismatch, and maximum mismatch for lexicographic comparison."""\n    mean_pc, max_pc = matching_metrics(selected, targets, scores)\n    return exact_geo_msd(coords[selected]), mean_pc, max_pc\n\n\ndef rank_is_better(\n    candidate: Tuple[float, float, float],\n    incumbent: Tuple[float, float, float],\n    config: RSSDConfig,\n) -> bool:\n    """Compare valid designs: geoMSD first, then mean and maximum PCA mismatch only on ties."""\n    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate[0]), abs(incumbent[0]))\n    if candidate[0] > incumbent[0] + geo_tol:\n        return True\n    if abs(candidate[0] - incumbent[0]) <= geo_tol:\n        if candidate[1] < incumbent[1] - config.PCA_TIE_TOL:\n            return True\n        if abs(candidate[1] - incumbent[1]) <= config.PCA_TIE_TOL:\n            return candidate[2] < incumbent[2] - config.PCA_TIE_TOL\n    return False\n\n\n\ndef candidate_pool_summary_table(\n    target_instances: pd.DataFrame,\n    pools: Sequence[np.ndarray],\n    pool_distances: Sequence[np.ndarray],\n    target_tolerances: Sequence[float],\n    config: RSSDConfig,\n) -> pd.DataFrame:\n    """Summarize how broad each target-specific candidate search actually was."""\n    records: List[Dict[str, Any]] = []\n    for position, (pool, distances, tolerance) in enumerate(\n        zip(pools, pool_distances, target_tolerances)\n    ):\n        distances = np.asarray(distances, dtype=float)\n        row = target_instances.iloc[position]\n        records.append(\n            {\n                "target_position": int(position),\n                "target_instance_id": row["target_instance_id"],\n                "base_target_id": row["base_target_id"],\n                "target_type": row["target_type"],\n                "candidate_pool_size": int(len(pool)),\n                "unique_candidate_pool_size": int(len(np.unique(pool))),\n                "minimum_pca_target_distance": float(np.min(distances)) if len(distances) else None,\n                "median_pca_target_distance": float(np.median(distances)) if len(distances) else None,\n                "maximum_pca_target_distance": float(np.max(distances)) if len(distances) else None,\n                "final_tolerance_used": float(tolerance),\n                "tolerance_ratio_to_configured_max": float(tolerance / config.MAX_PC_CANDIDATE_TOLERANCE)\n                if config.MAX_PC_CANDIDATE_TOLERANCE > 0\n                else None,\n                "nearest_k_fallback_implied": bool(tolerance > config.MAX_PC_CANDIDATE_TOLERANCE * (1.0 + 1e-12)),\n            }\n        )\n    return pd.DataFrame(records)\n\n\ndef search_sufficiency_diagnostics(\n    optimizer_summary: pd.DataFrame,\n    candidate_pool_report: pd.DataFrame,\n    assignment_attempts: int,\n    config: RSSDConfig,\n) -> Dict[str, Any]:\n    """Describe whether the bounded candidate/exchange search appears saturated."""\n    if optimizer_summary.empty:\n        return {"status": "not_evaluated", "recommendation": "No optimizer starts were recorded."}\n\n    best_geo = float(optimizer_summary["final_geoMSD"].max())\n    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(best_geo))\n    near_best = optimizer_summary["final_geoMSD"] >= best_geo - geo_tol\n    best_rows = optimizer_summary.loc[near_best]\n    starts_run = int(optimizer_summary["start_number"].max())\n    first_best_start = int(best_rows["start_number"].iloc[0])\n    last_best_start = int(best_rows["start_number"].iloc[-1])\n    starts_after_last_best = starts_run - last_best_start\n    stop_values = optimizer_summary.get("stop_after_this_start", pd.Series(dtype=object)).replace("", np.nan).dropna()\n    stop_reason = str(stop_values.iloc[-1]) if len(stop_values) else "configured_max_starts_completed"\n\n    min_pool = int(candidate_pool_report["candidate_pool_size"].min()) if len(candidate_pool_report) else 0\n    median_pool = float(candidate_pool_report["candidate_pool_size"].median()) if len(candidate_pool_report) else 0.0\n    maxed_tolerance_targets = int(\n        (candidate_pool_report["tolerance_ratio_to_configured_max"] >= 0.999).sum()\n    ) if len(candidate_pool_report) else 0\n    fallback_targets = int(candidate_pool_report["nearest_k_fallback_implied"].sum()) if len(candidate_pool_report) else 0\n    low_pool_targets = int(\n        (candidate_pool_report["candidate_pool_size"] < max(3, config.CANDIDATES_PER_TARGET)).sum()\n    ) if len(candidate_pool_report) else 0\n\n    flags: List[str] = []\n    if assignment_attempts > 1:\n        flags.append("candidate pools had to be enlarged before a unique assignment existed")\n    if fallback_targets:\n        flags.append("one or more targets required nearest-k fallback beyond the configured PCA tolerance")\n    if maxed_tolerance_targets:\n        flags.append("one or more targets reached the configured maximum PCA tolerance")\n    if low_pool_targets:\n        flags.append("one or more targets retained fewer candidates than the requested pool size")\n\n    if flags:\n        status = "candidate_space_constrained"\n        recommendation = "Review target tolerances, candidate counts, forced locations, and PCA feature choices before treating the search as saturated."\n    elif stop_reason == "stable_no_improvement" or starts_after_last_best >= config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE:\n        status = "plausibly_saturated_local_search"\n        recommendation = "The best design survived the configured no-improvement patience; additional starts are optional unless the field plan is high stakes."\n    elif starts_run >= config.N_OPTIMIZER_STARTS and starts_after_last_best < max(3, config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE // 2):\n        status = "late_improvement_possible"\n        recommendation = "The best design appeared late; increase the maximum starts or candidate pools if runtime allows."\n    else:\n        status = "review_optimizer_trace"\n        recommendation = "Inspect the convergence table; more starts may be useful if geoMSD is still improving materially."\n\n    return {\n        "status": status,\n        "recommendation": recommendation,\n        "best_geoMSD": best_geo,\n        "starts_run": starts_run,\n        "configured_max_starts": int(config.N_OPTIMIZER_STARTS),\n        "first_best_start": first_best_start,\n        "last_best_start": last_best_start,\n        "starts_after_last_best": int(starts_after_last_best),\n        "near_best_start_count": int(near_best.sum()),\n        "optimizer_stop_reason": stop_reason,\n        "assignment_attempts": int(assignment_attempts),\n        "minimum_candidate_pool_size": min_pool,\n        "median_candidate_pool_size": median_pool,\n        "targets_at_maximum_tolerance": maxed_tolerance_targets,\n        "targets_with_nearest_k_fallback": fallback_targets,\n        "targets_below_requested_pool_size": low_pool_targets,\n        "constraint_flags": flags,\n        "interpretation": "This is a bounded-search diagnostic, not a mathematical proof of global optimality.",\n    }\n\n\ndef coordinate_exchange(\n    start: np.ndarray,\n    pools: Sequence[np.ndarray],\n    targets: np.ndarray,\n    scores: np.ndarray,\n    coords: np.ndarray,\n    config: RSSDConfig,\n    rng: np.random.Generator,\n) -> Tuple[np.ndarray, Dict[str, Any]]:\n    """Optimize one site at a time; no Cartesian product of candidate pools is generated."""\n    selected = np.asarray(start, dtype=int).copy()\n    if len(np.unique(selected)) != len(selected):\n        raise ValueError("Coordinate exchange requires a unique starting assignment.")\n    initial_rank = design_rank(selected, targets, scores, coords)\n    current_rank = initial_rank\n    accepted = 0\n    sweeps = 0\n    for sweep in range(1, config.MAX_OPTIMIZER_SWEEPS + 1):\n        accepted_this_sweep = 0\n        for position in rng.permutation(len(selected)):\n            used_elsewhere = set(np.delete(selected, position).tolist())\n            best_index = int(selected[position])\n            best_rank = current_rank\n            for candidate_index in pools[position]:\n                candidate_index = int(candidate_index)\n                if candidate_index == selected[position] or candidate_index in used_elsewhere:\n                    continue\n                proposal = selected.copy()\n                proposal[position] = candidate_index\n                proposal_rank = design_rank(proposal, targets, scores, coords)\n                if rank_is_better(proposal_rank, best_rank, config):\n                    best_index, best_rank = candidate_index, proposal_rank\n            if best_index != selected[position]:\n                selected[position] = best_index\n                current_rank = best_rank\n                accepted += 1\n                accepted_this_sweep += 1\n        sweeps = sweep\n        if accepted_this_sweep == 0:\n            break\n    return selected, {\n        "initial_geoMSD": initial_rank[0],\n        "final_geoMSD": current_rank[0],\n        "accepted_swaps": accepted,\n        "sweeps": sweeps,\n        "final_mean_pca_target_distance": current_rank[1],\n        "final_max_pca_target_distance": current_rank[2],\n    }\n\n\ndef optimize_multiple_starts(\n    minimum_distance_start: np.ndarray,\n    pools: Sequence[np.ndarray],\n    pool_distances: Sequence[np.ndarray],\n    targets: np.ndarray,\n    scores: np.ndarray,\n    coords: np.ndarray,\n    config: RSSDConfig,\n) -> Tuple[np.ndarray, pd.DataFrame]:\n    """Run minimum-mismatch and reproducible random starts with optional stable-search stopping."""\n    rng = np.random.default_rng(config.RANDOM_SEED)\n    summaries: List[Dict[str, Any]] = []\n    best: Optional[np.ndarray] = None\n    best_rank: Optional[Tuple[float, float, float]] = None\n    starts_since_best = 0\n    stop_reason = "configured_max_starts_completed"\n    for start_number in range(1, config.N_OPTIMIZER_STARTS + 1):\n        start = minimum_distance_start.copy() if start_number == 1 else assignment_from_costs(\n            pools, pool_distances, rng\n        )\n        if start is None:\n            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")\n        optimized, summary = coordinate_exchange(start, pools, targets, scores, coords, config, rng)\n        summary["start_number"] = start_number\n        summary["start_type"] = "minimum_pca_distance" if start_number == 1 else "random_unique_assignment"\n        rank = design_rank(optimized, targets, scores, coords)\n        new_best = best_rank is None or rank_is_better(rank, best_rank, config)\n        if new_best:\n            best, best_rank = optimized.copy(), rank\n            starts_since_best = 0\n        else:\n            starts_since_best += 1\n        summary["new_best_design"] = bool(new_best)\n        summary["best_so_far_geoMSD"] = float(best_rank[0]) if best_rank is not None else None\n        summary["starts_since_best"] = int(starts_since_best)\n        summary["stop_after_this_start"] = ""\n        summaries.append(summary)\n        if (\n            config.EARLY_STOP_ON_STABLE_SEARCH\n            and start_number >= config.MIN_OPTIMIZER_STARTS\n            and starts_since_best >= config.OPTIMIZER_NO_IMPROVEMENT_PATIENCE\n        ):\n            stop_reason = "stable_no_improvement"\n            summaries[-1]["stop_after_this_start"] = stop_reason\n            break\n    if best is None:\n        raise RuntimeError("Optimizer produced no design.")\n    summary_df = pd.DataFrame(summaries)\n    summary_df["optimizer_stop_reason"] = stop_reason\n    return best, summary_df\n\n\n# @title Load diagnostics and acceptance tests (run; no editing required) { display-mode: "form" }\ndef second_order_matrix(scores: np.ndarray) -> Tuple[np.ndarray, List[str]]:\n    """Build intercept, linear, squared, and pairwise-interaction columns."""\n    z = np.asarray(scores, dtype=np.float64)\n    n, p = z.shape\n    columns = [np.ones(n)]\n    names = ["Intercept"]\n    for j in range(p):\n        columns.append(z[:, j])\n        names.append(f"PC{j + 1}")\n    for j in range(p):\n        columns.append(z[:, j] ** 2)\n        names.append(f"PC{j + 1}^2")\n    for j in range(p):\n        for k in range(j + 1, p):\n            columns.append(z[:, j] * z[:, k])\n            names.append(f"PC{j + 1}:PC{k + 1}")\n    return np.column_stack(columns), names\n\n\ndef regression_design_diagnostics(\n    selected_scores: np.ndarray,\n    population_scores: np.ndarray,\n    chunk_size: int,\n) -> Tuple[Dict[str, Any], np.ndarray]:\n    """Calculate rank, condition, leverage, and chunked average relative prediction variance."""\n    x, names = second_order_matrix(selected_scores)\n    rank = int(np.linalg.matrix_rank(x))\n    n_columns = x.shape[1]\n    condition = float(np.linalg.cond(x))\n    result: Dict[str, Any] = {\n        "model_terms": names,\n        "matrix_rows": int(x.shape[0]),\n        "matrix_columns": int(n_columns),\n        "matrix_rank": rank,\n        "full_rank": rank == n_columns,\n        "condition_number": condition,\n    }\n    if rank != n_columns:\n        result.update(\n            {\n                "maximum_leverage": None,\n                "average_relative_prediction_variance": None,\n                "warning": "Rank deficient: inverse-based leverage and avePVar were not calculated.",\n            }\n        )\n        return result, np.full(len(selected_scores), np.nan)\n\n    xtx_inv = np.linalg.inv(x.T @ x)\n    leverage = np.einsum("ij,jk,ik->i", x, xtx_inv, x)\n    total = 0.0\n    count = 0\n    for start in range(0, len(population_scores), chunk_size):\n        xp, _ = second_order_matrix(population_scores[start : start + chunk_size])\n        h = np.einsum("ij,jk,ik->i", xp, xtx_inv, xp)\n        total += float(np.sum(1.0 + h))\n        count += len(h)\n    result.update(\n        {\n            "maximum_leverage": float(np.max(leverage)),\n            "average_relative_prediction_variance": total / count,\n        }\n    )\n    return result, leverage\n\n\n\n\n\ndef json_ready(value: Any) -> Any:\n    """Recursively convert NumPy/Pandas values to JSON-safe Python values."""\n    if isinstance(value, dict):\n        return {str(k): json_ready(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return None if not np.isfinite(value) else float(value)\n    if isinstance(value, (pd.Timestamp,)):\n        return value.isoformat()\n    return value\n\n\ndef package_versions() -> Dict[str, str]:\n    """Record software versions needed to reproduce a run."""\n    packages = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "openpyxl"]\n    versions = {"python": platform.python_version()}\n    for package in packages:\n        try:\n            versions[package] = importlib_metadata.version(package)\n        except importlib_metadata.PackageNotFoundError:\n            versions[package] = "not_installed"\n    return versions\n\n\n\n\n\n\n\n\n\n\n# --- ESAP RSSD v2.10 targeted overrides ---\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n# V210_OVERRIDES\n\n\n\n\n\n\n# --- ESAP RSSD v2.10 overrides inserted below ---\n\n\n@dataclass\nclass RSSDRunResult:\n    config: RSSDConfig\n    selected_sites: pd.DataFrame\n    candidate_sites: pd.DataFrame\n    target_instances: pd.DataFrame\n    support_sequence: pd.DataFrame\n    ad_support_resolution: pd.DataFrame\n    candidate_saturation: pd.DataFrame\n    optimizer_stability: pd.DataFrame\n    spatial_support_sites: pd.DataFrame\n    field_coverage_distances: pd.DataFrame\n    proxy_spatial_scale: pd.DataFrame\n    pca_validation: pd.DataFrame\n    feature_skewness: pd.DataFrame\n    metadata: Dict[str, Any]\n    run_summary: str\n    diagnostic_data: Dict[str, Any] = field(default_factory=dict)\n\n\n\ndef validate_config(config: RSSDConfig) -> None:\n    if config.MEMORY_MODE not in {"full", "auto", "incremental"}:\n        raise ValueError("MEMORY_MODE must be \'full\', \'auto\', or \'incremental\'.")\n    if config.SAMPLE_BUDGET_MODE not in {"rssd_with_support", "ccd_exact", "balanced_target_replication"}:\n        raise ValueError("SAMPLE_BUDGET_MODE must be rssd_with_support, ccd_exact, or balanced_target_replication.")\n    if config.SUPPORT_SITE_MODE not in {"legacy_inspired_auto", "manual", "none"}:\n        raise ValueError("SUPPORT_SITE_MODE must be legacy_inspired_auto, manual, or none.")\n    if config.AD_SUPPORT_MODE not in {"adaptive", "fixed"}:\n        raise ValueError("AD_SUPPORT_MODE must be adaptive or fixed.")\n    if config.OPTIMIZER_START_MODE not in {"adaptive", "fixed"}:\n        raise ValueError("OPTIMIZER_START_MODE must be adaptive or fixed.")\n    if not (1 <= config.N_DESIGN_COMPONENTS <= 4):\n        raise ValueError("This version supports 1-4 design PCs.")\n    if config.N_DESIGN_COMPONENTS == 4:\n        warnings.warn("A full 4-D central composite design contains many targets; Lesch notes that hybrid or small composite designs may be preferable.")\n    for name, value in {"OUTLIER_COVERAGE": config.OUTLIER_COVERAGE, "DESIGN_COVERAGE": config.DESIGN_COVERAGE}.items():\n        if not 0.0 < value < 1.0:\n            raise ValueError(f"{name} must lie strictly between 0 and 1.")\n    if config.CANDIDATES_PER_TARGET < 2:\n        raise ValueError("CANDIDATES_PER_TARGET must be at least 2.")\n    if int(config.CANDIDATE_SATURATION_START_K) < 1 or int(config.CANDIDATE_SATURATION_MAX_K) < int(config.CANDIDATE_SATURATION_START_K):\n        raise ValueError("Candidate saturation K settings are inconsistent.")\n    if int(config.CANDIDATE_SATURATION_GROWTH_FACTOR) < 2:\n        raise ValueError("CANDIDATE_SATURATION_GROWTH_FACTOR must be at least 2.")\n    if int(config.OPTIMIZER_START_BATCH_SIZE) < 1 or int(config.OPTIMIZER_MIN_STARTS) < 1 or int(config.OPTIMIZER_MAX_STARTS) < int(config.OPTIMIZER_MIN_STARTS):\n        raise ValueError("Batched optimizer start settings are inconsistent.")\n    if config.N_OPTIMIZER_STARTS < 1 or not (1 <= config.MIN_OPTIMIZER_STARTS <= config.N_OPTIMIZER_STARTS):\n        raise ValueError("Legacy optimizer start settings are inconsistent.")\n    if config.AD_SUPPORT_START_SIZE < 1 or config.AD_SUPPORT_MAX_SIZE < 1 or config.AD_SUPPORT_GROWTH_FACTOR < 2:\n        raise ValueError("AD support-size settings are inconsistent.")\n    if not (0.0 <= config.AD_COVERAGE_ENVELOPE_REL_TOL < 1.0):\n        raise ValueError("AD_COVERAGE_ENVELOPE_REL_TOL must be nonnegative and less than 1.")\n    if int(config.SUPPORT_GAP_ANCHORS) < 1:\n        raise ValueError("SUPPORT_GAP_ANCHORS must be at least 1.")\n    if int(config.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR) < 1:\n        raise ValueError("SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR must be at least 1.")\n    if config.SPACING_DIAGNOSTIC_MAX_POINTS < 10:\n        raise ValueError("SPACING_DIAGNOSTIC_MAX_POINTS must be at least 10.")\n    if config.SPACING_DIAGNOSTIC_NEIGHBORS < 1:\n        raise ValueError("SPACING_DIAGNOSTIC_NEIGHBORS must be positive.")\n    if config.SPACING_DIAGNOSTIC_RANDOM_PAIRS < 100:\n        raise ValueError("SPACING_DIAGNOSTIC_RANDOM_PAIRS must be at least 100.")\n    if config.SPACING_DIAGNOSTIC_BINS < 4:\n        raise ValueError("SPACING_DIAGNOSTIC_BINS must be at least 4.")\n    if int(config.SPACING_MIN_PAIRS_PER_BIN) < 1:\n        raise ValueError("SPACING_MIN_PAIRS_PER_BIN must be at least 1.")\n    if int(config.SPACING_RANGE_STABILITY_BINS) < 1 or int(config.SPACING_PLATEAU_TAIL_BINS) < 2:\n        raise ValueError("Proxy spatial-scale reliability settings are inconsistent.")\n\n\ndef allocate_response_surface_instances(base_targets: pd.DataFrame, response_surface_count: int, p: int) -> Tuple[pd.DataFrame, pd.Series]:\n    base_n = len(base_targets)\n    if response_surface_count < base_n:\n        raise ValueError(f"The full base CCD requires {base_n} response-surface sites; requested {response_surface_count}.")\n    counts = np.ones(base_n, dtype=int)\n    extras = response_surface_count - base_n\n    if extras > 0:\n        counts += extras // base_n\n        counts[: extras % base_n] += 1\n    rows: List[Dict[str, Any]] = []\n    target_cols = [f"target_PC{j + 1}" for j in range(p)]\n    for i, base in base_targets.iterrows():\n        for replicate in range(1, int(counts[i]) + 1):\n            row = {"target_instance_id": f"T{len(rows) + 1:03d}", "base_target_id": base["base_target_id"], "target_type": base["target_type"], "target_replicate_number": replicate, "sample_role": "response_surface"}\n            row.update({c: float(base[c]) for c in target_cols})\n            rows.append(row)\n    instances = pd.DataFrame(rows)\n    return instances, instances.groupby("base_target_id", sort=False).size().rename("replication_count")\n\n\ndef _legacy_inspired_desired_support_count(n_samples: int) -> int:\n    if n_samples <= 6:\n        return 1\n    if n_samples <= 20:\n        return 2\n    return max(3, int(math.ceil(n_samples / 10.0)))\n\n\ndef allocate_sample_budget(base_targets: pd.DataFrame, n_samples: int, config: RSSDConfig, p: int) -> Tuple[pd.DataFrame, pd.Series, Dict[str, Any]]:\n    base_n = len(base_targets)\n    if config.SAMPLE_BUDGET_MODE == "ccd_exact":\n        response_count, support_count = base_n, 0\n        if n_samples != base_n:\n            print(f"ccd_exact mode uses {base_n} response-surface samples; configured N_SAMPLES={n_samples} is ignored.")\n    elif config.SAMPLE_BUDGET_MODE == "balanced_target_replication":\n        response_count, support_count = n_samples, 0\n    else:\n        if n_samples < base_n:\n            raise ValueError(f"rssd_with_support requires N_SAMPLES >= {base_n}, the base CCD size.")\n        desired = 0 if config.SUPPORT_SITE_MODE == "none" else (max(0, int(config.N_SUPPORT_SITES)) if config.SUPPORT_SITE_MODE == "manual" else _legacy_inspired_desired_support_count(n_samples))\n        available_extra_sites = n_samples - base_n\n        support_count = min(desired, max(0, available_extra_sites))\n        response_count = n_samples - support_count\n        if desired > support_count:\n            print(f"Spatial support sites capped at {support_count} because the full {base_n}-site response-surface core must be preserved.")\n    target_instances, replication = allocate_response_surface_instances(base_targets, response_count, p)\n    report = {"sample_budget_mode": config.SAMPLE_BUDGET_MODE, "support_site_mode": config.SUPPORT_SITE_MODE, "requested_total_samples": int(n_samples), "base_CCD_target_count": int(base_n), "response_surface_sites": int(response_count), "spatial_support_sites": int(support_count), "full_response_surface_core_preserved": bool(response_count >= base_n), "support_allocation_label": "legacy-inspired automatic support allocation" if config.SUPPORT_SITE_MODE == "legacy_inspired_auto" else config.SUPPORT_SITE_MODE}\n    return target_instances, replication, report\n\n\ndef allocate_target_instances(base_targets: pd.DataFrame, n_samples: int, mode: str, p: int) -> Tuple[pd.DataFrame, pd.Series]:\n    temp = RSSDConfig(N_SAMPLES=n_samples, SAMPLE_BUDGET_MODE=mode)\n    if mode == "rssd_with_support":\n        temp.SUPPORT_SITE_MODE = "none"\n    instances, replication, _ = allocate_sample_budget(base_targets, n_samples, temp, p)\n    return instances, replication\n\n\ndef _nearest_observed_coordinate_to_center(coords: np.ndarray, ids: np.ndarray, indices: np.ndarray, center: np.ndarray) -> int:\n    d2 = np.einsum("ij,ij->i", coords[indices] - center, coords[indices] - center)\n    best = np.flatnonzero(d2 == np.min(d2))\n    return int(indices[int(best[np.argmin(ids[indices[best]])])])\n\n\ndef _leaf_bbox(coords: np.ndarray, indices: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:\n    subset = coords[indices]\n    lo = np.min(subset, axis=0); hi = np.max(subset, axis=0); span = hi - lo\n    return lo, hi, span, float(np.sqrt(np.sum(span * span)))\n\n\ndef build_spatially_balanced_support_sequence(domain_coords: np.ndarray, stable_ids: Optional[np.ndarray] = None, max_size: int = 40000) -> Tuple[pd.DataFrame, Dict[str, Any]]:\n    import heapq\n    coords_in = np.asarray(domain_coords, dtype=np.float64)\n    finite = np.isfinite(coords_in).all(axis=1)\n    coords_in = coords_in[finite]\n    ids_in = (np.arange(len(domain_coords), dtype=np.int64) if stable_ids is None else np.asarray(stable_ids, dtype=np.int64))[finite]\n    if len(coords_in) == 0:\n        raise ValueError("No finite coordinates are available for the spatial support sequence.")\n    order = np.lexsort((ids_in, coords_in[:, 1], coords_in[:, 0]))\n    sorted_coords = coords_in[order]; sorted_ids = ids_in[order]\n    unique_mask = np.ones(len(sorted_coords), dtype=bool)\n    if len(sorted_coords) > 1:\n        unique_mask[1:] = np.any(np.diff(sorted_coords, axis=0) != 0.0, axis=1)\n    coords = sorted_coords[unique_mask]; ids = sorted_ids[unique_mask]\n    max_len = min(int(max_size), len(coords))\n    all_indices = np.arange(len(coords), dtype=np.int64)\n    lo, hi, span, diagonal = _leaf_bbox(coords, all_indices)\n    root_rep = _nearest_observed_coordinate_to_center(coords, ids, all_indices, 0.5 * (lo + hi))\n    rows = [{"support_rank": 1, "analysis_index": int(ids[root_rep]), "X": float(coords[root_rep, 0]), "Y": float(coords[root_rep, 1])}]\n    leaves = {0: {"indices": all_indices, "representative": int(root_rep), "lo": lo, "hi": hi, "span": span, "diagonal": diagonal}}\n    heap: List[Tuple[float, int]] = []\n    if len(all_indices) > 1 and diagonal > 0:\n        heapq.heappush(heap, (-diagonal, 0))\n    leaf_counter = 0\n    while len(rows) < max_len and heap:\n        _, leaf_id = heapq.heappop(heap)\n        leaf = leaves.pop(leaf_id, None)\n        if leaf is None:\n            continue\n        indices = leaf["indices"]; lo = leaf["lo"]; hi = leaf["hi"]; span = leaf["span"]; diagonal = leaf["diagonal"]\n        if len(indices) <= 1 or diagonal <= 0 or np.max(span) <= 0:\n            continue\n        axis = int(np.argmax(span)); midpoint = float(0.5 * (lo[axis] + hi[axis]))\n        left_indices = indices[coords[indices, axis] <= midpoint]; right_indices = indices[coords[indices, axis] > midpoint]\n        if len(left_indices) == 0 or len(right_indices) == 0:\n            continue\n        rep = int(leaf["representative"])\n        for child_indices in (left_indices, right_indices):\n            child_lo, child_hi, child_span, child_diag = _leaf_bbox(coords, child_indices)\n            append_rep = not np.any(child_indices == rep)\n            child_rep = rep if not append_rep else _nearest_observed_coordinate_to_center(coords, ids, child_indices, 0.5 * (child_lo + child_hi))\n            leaf_counter += 1\n            leaves[leaf_counter] = {"indices": child_indices, "representative": int(child_rep), "lo": child_lo, "hi": child_hi, "span": child_span, "diagonal": child_diag}\n            if len(child_indices) > 1 and child_diag > 0 and np.max(child_span) > 0:\n                heapq.heappush(heap, (-float(child_diag), leaf_counter))\n            if append_rep and len(rows) < max_len:\n                rows.append({"support_rank": len(rows) + 1, "analysis_index": int(ids[child_rep]), "X": float(coords[child_rep, 0]), "Y": float(coords[child_rep, 1])})\n    table = pd.DataFrame(rows)\n    report = {"spatial_domain_rows": int(len(domain_coords)), "finite_spatial_domain_rows": int(np.sum(finite)), "unique_spatial_domain_coordinates": int(len(coords)), "support_sequence_maximum_length": int(len(table)), "algorithm": "deterministic recursive occupied-space bisection", "uses_only_xy": True, "uses_pc_scores": False, "uses_row_count_as_geographic_weight": False}\n    return table, report\n\n\ndef exact_sbad(support_coords: np.ndarray, selected_coords: np.ndarray) -> float:\n    support = np.asarray(support_coords, dtype=np.float64); selected = np.asarray(selected_coords, dtype=np.float64)\n    if len(support) == 0 or len(selected) == 0:\n        return float("nan")\n    d, _ = cKDTree(selected).query(support, k=1, workers=-1)\n    return float(np.mean(np.asarray(d, dtype=np.float64)))\n\n\n\n\n\n\n\n\ndef minimum_selected_separation(coords: np.ndarray) -> float:\n    coords = np.asarray(coords, dtype=np.float64)\n    return float(np.min(selected_nearest_neighbor_distances(coords))) if len(coords) >= 2 else float("inf")\n\n\ndef run_selected_set_development_benchmark(rng: np.random.Generator, enabled: bool, evaluations: Optional[int] = None) -> Dict[str, Any]:\n    """Optionally run the slow old-versus-new selected-set timing benchmark."""\n    configured_evaluations = int(DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS if evaluations is None else evaluations)\n    if not bool(enabled):\n        return {\n            "development_selected_set_benchmark_run": False,\n            "configured_evaluations": int(configured_evaluations),\n            "evaluations": 0,\n            "selected_sites": 20,\n            "old_ckdtree_workers_minus_1_seconds": None,\n            "new_vectorized_seconds": None,\n            "speedup_ratio": None,\n            "geoMSD_equality": None,\n        }\n\n    bench_coords = rng.uniform(0.0, 1000.0, size=(20, 2))\n    old_total = 0.0\n    new_total = 0.0\n    t0 = time.perf_counter()\n    for _ in range(configured_evaluations):\n        old_total += float(np.sum(selected_nearest_neighbor_distances_ckdtree_reference(bench_coords, workers=-1)))\n    old_elapsed = time.perf_counter() - t0\n    t0 = time.perf_counter()\n    for _ in range(configured_evaluations):\n        new_total += float(np.sum(selected_nearest_neighbor_distances(bench_coords)))\n    new_elapsed = time.perf_counter() - t0\n    np.testing.assert_allclose(old_total, new_total, rtol=1e-12, atol=1e-8)\n    return {\n        "development_selected_set_benchmark_run": True,\n        "configured_evaluations": int(configured_evaluations),\n        "evaluations": int(configured_evaluations),\n        "selected_sites": 20,\n        "old_ckdtree_workers_minus_1_seconds": float(old_elapsed),\n        "new_vectorized_seconds": float(new_elapsed),\n        "speedup_ratio": float(old_elapsed / max(new_elapsed, 1e-12)),\n        "geoMSD_equality": True,\n    }\n\n\ndef design_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, sbad: Optional[float] = None) -> Dict[str, float]:\n    mean_pc, max_pc = matching_metrics(selected, targets, scores)\n    selected_coords = coords[selected]\n    return {"SBAD": float(sbad) if sbad is not None else float("nan"), "geoMSD": exact_geo_msd(selected_coords), "minimum_separation": minimum_selected_separation(selected_coords), "mean_pca_target_distance": float(mean_pc), "max_pca_target_distance": float(max_pc)}\n\n\ndef _tol_equal(a: float, b: float, rel_tol: float = 1e-9, abs_tol: float = 1e-12) -> bool:\n    return abs(a - b) <= max(abs_tol, rel_tol * max(1.0, abs(a), abs(b)))\n\n\ndef ad_reference_is_better(candidate: Dict[str, float], incumbent: Dict[str, float], config: RSSDConfig) -> bool:\n    if candidate["SBAD"] < incumbent["SBAD"] - config.PCA_TIE_TOL:\n        return True\n    if _tol_equal(candidate["SBAD"], incumbent["SBAD"], abs_tol=config.PCA_TIE_TOL):\n        geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate["geoMSD"]), abs(incumbent["geoMSD"]))\n        if candidate["geoMSD"] > incumbent["geoMSD"] + geo_tol:\n            return True\n        if abs(candidate["geoMSD"] - incumbent["geoMSD"]) <= geo_tol:\n            if candidate["minimum_separation"] > incumbent["minimum_separation"] + config.PCA_TIE_TOL:\n                return True\n            if _tol_equal(candidate["minimum_separation"], incumbent["minimum_separation"], abs_tol=config.PCA_TIE_TOL):\n                if candidate["mean_pca_target_distance"] < incumbent["mean_pca_target_distance"] - config.PCA_TIE_TOL:\n                    return True\n                if _tol_equal(candidate["mean_pca_target_distance"], incumbent["mean_pca_target_distance"], abs_tol=config.PCA_TIE_TOL):\n                    return candidate["max_pca_target_distance"] < incumbent["max_pca_target_distance"] - config.PCA_TIE_TOL\n    return False\n\n\ndef hybrid_is_better(candidate: Dict[str, float], incumbent: Dict[str, float], config: RSSDConfig, sbad_limit: float) -> bool:\n    cand_feasible = candidate["SBAD"] <= sbad_limit + config.PCA_TIE_TOL\n    inc_feasible = incumbent["SBAD"] <= sbad_limit + config.PCA_TIE_TOL\n    if cand_feasible and not inc_feasible:\n        return True\n    if inc_feasible and not cand_feasible:\n        return False\n    if not cand_feasible and not inc_feasible:\n        return ad_reference_is_better(candidate, incumbent, config)\n    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate["geoMSD"]), abs(incumbent["geoMSD"]))\n    if candidate["geoMSD"] > incumbent["geoMSD"] + geo_tol:\n        return True\n    if abs(candidate["geoMSD"] - incumbent["geoMSD"]) <= geo_tol:\n        if candidate["minimum_separation"] > incumbent["minimum_separation"] + config.PCA_TIE_TOL:\n            return True\n        if _tol_equal(candidate["minimum_separation"], incumbent["minimum_separation"], abs_tol=config.PCA_TIE_TOL):\n            if candidate["mean_pca_target_distance"] < incumbent["mean_pca_target_distance"] - config.PCA_TIE_TOL:\n                return True\n            if _tol_equal(candidate["mean_pca_target_distance"], incumbent["mean_pca_target_distance"], abs_tol=config.PCA_TIE_TOL):\n                return candidate["max_pca_target_distance"] < incumbent["max_pca_target_distance"] - config.PCA_TIE_TOL\n    return False\n\n\ndef candidate_pool_for_target_incremental(target: np.ndarray, pc_tree: cKDTree, search_scores: np.ndarray, search_coords: np.ndarray, search_global_indices: np.ndarray, desired: int, config: RSSDConfig) -> Tuple[np.ndarray, np.ndarray, float, int, bool]:\n    tolerance = float(config.PC_CANDIDATE_TOLERANCE)\n    local = pc_tree.query_ball_point(target, r=tolerance)\n    expansions = 0\n    while len(local) < desired and tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:\n        tolerance = min(config.MAX_PC_CANDIDATE_TOLERANCE, tolerance * config.CANDIDATE_TOLERANCE_GROWTH)\n        local = pc_tree.query_ball_point(target, r=tolerance); expansions += 1\n    nearest_k_fallback = False\n    if len(local) < desired:\n        k = min(max(desired, 1), len(search_scores))\n        nearest_d, nearest_i = pc_tree.query(target, k=k)\n        local = np.atleast_1d(nearest_i).astype(int).tolist()\n        tolerance = max(tolerance, float(np.max(np.atleast_1d(nearest_d))))\n        nearest_k_fallback = True\n    if len(local) == 0:\n        raise RuntimeError("No candidate observations were found for a response-surface target.")\n    local_arr = np.asarray(local, dtype=int)\n    pc_dist = np.linalg.norm(search_scores[local_arr] - target, axis=1)\n    stable = np.lexsort((search_global_indices[local_arr], pc_dist))\n    local_arr = local_arr[stable]; pc_dist = pc_dist[stable]\n    selected_positions = [0]\n    remaining = np.ones(len(local_arr), dtype=bool); remaining[0] = False\n    min_geo = np.linalg.norm(search_coords[local_arr] - search_coords[local_arr[0]], axis=1); min_geo[0] = -np.inf\n    while len(selected_positions) < min(desired, len(local_arr)):\n        rem_pos = np.flatnonzero(remaining)\n        order = np.lexsort((search_global_indices[local_arr[rem_pos]], pc_dist[rem_pos], -min_geo[rem_pos]))\n        chosen = int(rem_pos[order[0]])\n        selected_positions.append(chosen); remaining[chosen] = False\n        new_dist = np.linalg.norm(search_coords[local_arr] - search_coords[local_arr[chosen]], axis=1)\n        min_geo = np.minimum(min_geo, new_dist); min_geo[~remaining] = -np.inf\n    positions = np.asarray(selected_positions, dtype=int)\n    return search_global_indices[local_arr[positions]], pc_dist[positions], float(tolerance), int(expansions), bool(nearest_k_fallback)\n\n\ndef candidate_pool_for_target(target, pc_tree, search_scores, search_coords, search_global_indices, desired, config):\n    indices, distances, tolerance, expansions, _ = candidate_pool_for_target_incremental(target, pc_tree, search_scores, search_coords, search_global_indices, desired, config)\n    return indices, distances, tolerance, expansions\n\n\ndef discover_candidate_pools(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig, pool_multiplier: int = 1, requested_k: Optional[int] = None) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:\n    p = config.N_DESIGN_COMPONENTS\n    target_cols = [f"target_PC{j + 1}" for j in range(p)]\n    targets = target_instances[target_cols].to_numpy(dtype=np.float64)\n    search_scores = scores[search_indices]; search_coords = coords[search_indices]; pc_tree = cKDTree(search_scores)\n    target_keys = [tuple(row) for row in np.round(targets, 12)]\n    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}\n    unique_sequences: Dict[Tuple[float, ...], Tuple[np.ndarray, np.ndarray, float, int, bool]] = {}\n    pools: List[np.ndarray] = []; pool_distances: List[np.ndarray] = []; tolerances: List[float] = []; records: List[Dict[str, Any]] = []\n    base_k = int(requested_k if requested_k is not None else config.CANDIDATES_PER_TARGET * pool_multiplier)\n    for t, target in enumerate(targets):\n        key = target_keys[t]; desired = max(base_k, multiplicities[key])\n        if key not in unique_sequences or len(unique_sequences[key][0]) < desired:\n            unique_sequences[key] = candidate_pool_for_target_incremental(target, pc_tree, search_scores, search_coords, search_indices, desired, config)\n        indices, distances, tolerance, expansions, fallback = unique_sequences[key]\n        pools.append(indices[:desired]); pool_distances.append(distances[:desired]); tolerances.append(float(tolerance))\n        for rank, (index, distance) in enumerate(zip(indices[:desired], distances[:desired]), start=1):\n            row = {"target_position": t, "target_instance_id": target_instances.iloc[t]["target_instance_id"], "base_target_id": target_instances.iloc[t]["base_target_id"], "target_type": target_instances.iloc[t]["target_type"], "sample_role": target_instances.iloc[t].get("sample_role", "response_surface"), "candidate_analysis_index": int(index), "candidate_rank": rank, "pca_target_distance": float(distance), "final_tolerance_used": float(tolerance), "tolerance_expansions": int(expansions), "nearest_k_fallback_implied": bool(fallback), "candidate_sequence_shared_for_replicated_target": bool(multiplicities[key] > 1)}\n            row.update({c: float(target_instances.iloc[t][c]) for c in target_cols})\n            records.append(row)\n    return pools, pool_distances, pd.DataFrame(records), tolerances\n\n\ndef discover_candidate_pools_saturated(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float], pd.DataFrame, int, int]:\n    if config.CANDIDATE_EXPLORATION_MODE == "fixed":\n        k_values = [int(config.CANDIDATES_PER_TARGET)]\n    else:\n        k = max(int(config.CANDIDATE_SATURATION_START_K), int(config.CANDIDATES_PER_TARGET)); k_values = []\n        while k <= int(config.CANDIDATE_SATURATION_MAX_K):\n            k_values.append(k); k *= int(config.CANDIDATE_SATURATION_GROWTH_FACTOR)\n        if k_values[-1] != int(config.CANDIDATE_SATURATION_MAX_K):\n            k_values = sorted(set(k_values + [int(config.CANDIDATE_SATURATION_MAX_K)]))\n    previous_unique = None; final = None; records = []; assignment_attempts = 0\n    for stage, k in enumerate(k_values, start=1):\n        pools, distances, associations, tolerances = discover_candidate_pools(target_instances, search_indices, scores, coords, config, requested_k=int(k))\n        assignment_possible = assignment_from_costs(pools, distances) is not None\n        if assignment_possible and assignment_attempts == 0:\n            assignment_attempts = stage\n        unique_candidates = int(len(np.unique(np.concatenate(pools)))) if pools else 0\n        added = None if previous_unique is None else unique_candidates - previous_unique\n        stable = bool(previous_unique is not None and added == 0 and assignment_possible)\n        records.append({"stage": int(stage), "requested_K": int(k), "candidate_pools_nested": True, "unique_candidate_observations": unique_candidates, "assignment_possible": bool(assignment_possible), "targets_below_requested_K": int(sum(len(pool) < k for pool in pools)), "added_unique_candidates_from_previous": added, "stable_stage": stable, "confirmation_stage": stable, "maximum_K_reached": bool(k == max(k_values))})\n        final = (pools, distances, associations, tolerances); previous_unique = unique_candidates\n        if stable and stage < len(k_values):\n            break\n    if final is None or assignment_from_costs(final[0], final[1]) is None:\n        raise RuntimeError("No unique target-to-site assignment was possible after adaptive candidate expansion.")\n    table = pd.DataFrame(records)\n    return final[0], final[1], final[2], final[3], table, max(1, assignment_attempts), int(table["requested_K"].iloc[-1])\n\n\n\ndef _start_designs(minimum_distance_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], rng: np.random.Generator, warm_starts: Optional[Sequence[np.ndarray]] = None) -> Iterable[Tuple[str, np.ndarray]]:\n    seen: set[Tuple[int, ...]] = set()\n    for label, design in [("minimum_pca_distance", minimum_distance_start), *[("warm_start", w) for w in (warm_starts or [])]]:\n        if design is None:\n            continue\n        arr = np.asarray(design, dtype=int); key = tuple(arr.tolist())\n        if len(np.unique(arr)) == len(arr) and key not in seen:\n            seen.add(key); yield label, arr.copy()\n    while True:\n        arr = assignment_from_costs(pools, pool_distances, rng)\n        if arr is None:\n            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")\n        key = tuple(arr.tolist())\n        if key not in seen:\n            seen.add(key); yield "random_unique_assignment", arr.copy()\n\n\n\n\ndef _panel_sort_key(record: Dict[str, Any], objective_type: str) -> Tuple[Any, ...]:\n    m = record["metrics"]\n    return (m["SBAD"], -m["geoMSD"], -m["minimum_separation"], m["mean_pca_target_distance"], m["max_pca_target_distance"]) if objective_type == "ad_reference" else (-m["geoMSD"], -m["minimum_separation"], m["mean_pca_target_distance"], m["max_pca_target_distance"], m["SBAD"])\n\n\n\n\ndef _support_sizes(config: RSSDConfig, max_available: int) -> List[int]:\n    if config.AD_SUPPORT_MODE == "fixed":\n        return [min(int(config.AD_SUPPORT_FIXED_SIZE), max_available)]\n    sizes = []; size = min(int(config.AD_SUPPORT_START_SIZE), max_available); limit = min(int(config.AD_SUPPORT_MAX_SIZE), max_available)\n    while True:\n        sizes.append(size)\n        if size >= limit:\n            break\n        size = min(size * int(config.AD_SUPPORT_GROWTH_FACTOR), limit)\n        if size == sizes[-1]:\n            break\n    return sorted(set(int(s) for s in sizes))\n\n\ndef export_selected_sites_kmz(selected_sites: pd.DataFrame, config: RSSDConfig, source_epsg: int, output_path: str = "ESAP_RSSD_selected_sites.kmz") -> Dict[str, Any]:\n    import html, zipfile\n    try:\n        from pyproj import CRS, Transformer\n    except ImportError:\n        return {"created": False, "reason": "pyproj is not installed"}\n    if not source_epsg or int(source_epsg) <= 0:\n        raise ValueError("A valid projected source EPSG code is required for KMZ export.")\n    source_crs = CRS.from_epsg(int(source_epsg))\n    if not source_crs.is_projected:\n        raise ValueError("KMZ source EPSG must describe a projected CRS.")\n    transformer = Transformer.from_crs(source_crs, CRS.from_epsg(4326), always_xy=True)\n    longitude, latitude = transformer.transform(pd.to_numeric(selected_sites[config.X_COLUMN]).to_numpy(float), pd.to_numeric(selected_sites[config.Y_COLUMN]).to_numpy(float))\n    placemarks: List[str] = []\n    for position, (_, row) in enumerate(selected_sites.iterrows(), start=1):\n        site_id = str(row.get(config.ID_COLUMN, position)); role = str(row.get("sample_role", "response_surface")); role_label = "Spatial support site" if role == "spatial_support" else "Response-surface site"\n        label = f"{position:02d} - {site_id}"\n        details = {"Sample order": position, "Sample ID": site_id, "Sample role": role_label, "Target instance": row.get("target_instance_id", ""), "Target type": row.get("target_type", ""), "PCA target distance": row.get("pca_target_distance", ""), "Nearest selected neighbor": row.get("nearest_selected_geographic_neighbor_distance", ""), "Projected X": row.get(config.X_COLUMN, ""), "Projected Y": row.get(config.Y_COLUMN, ""), "Source EPSG": int(source_epsg)}\n        description_rows = "".join(f"<tr><th align=\'left\'>{html.escape(str(k))}</th><td>{html.escape(str(v))}</td></tr>" for k, v in details.items())\n        placemarks.append("<Placemark>" f"<name>{html.escape(label)}</name><styleUrl>#rssdSample</styleUrl>" f"<description><![CDATA[<table>{description_rows}</table>]]></description>" f"<ExtendedData><Data name=\'sample_id\'><value>{html.escape(site_id)}</value></Data><Data name=\'sample_role\'><value>{html.escape(role_label)}</value></Data></ExtendedData>" f"<Point><coordinates>{float(longitude[position - 1]):.10f},{float(latitude[position - 1]):.10f},0</coordinates></Point></Placemark>")\n    kml = "<?xml version=\'1.0\' encoding=\'UTF-8\'?><kml xmlns=\'http://www.opengis.net/kml/2.2\'><Document><name>ESAP RSSD v2.10 selected sites</name><Style id=\'rssdSample\'><IconStyle><color>ff0000ff</color><scale>1.1</scale><Icon><href>http://maps.google.com/mapfiles/kml/pushpin/red-pushpin.png</href></Icon></IconStyle><LabelStyle><color>ff000000</color><scale>0.9</scale></LabelStyle></Style>" + "".join(placemarks) + "</Document></kml>"\n    destination = Path(output_path)\n    with zipfile.ZipFile(destination, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:\n        archive.writestr("doc.kml", kml.encode("utf-8"))\n    return {"created": True, "file": str(destination), "source_epsg": int(source_epsg), "output_crs": "EPSG:4326", "placemark_count": int(len(selected_sites))}\n\n\n\n\n\n\n\ndef _save_or_show_figure(fig: Any, name: str, output_dir: Optional[Path], show: bool, saved: List[str]) -> None:\n    fig.tight_layout()\n    if output_dir is not None:\n        output_dir.mkdir(parents=True, exist_ok=True)\n        path = output_dir / name\n        fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")\n        saved.append(str(path))\n    if show:\n        plt.show()\n    else:\n        plt.close(fig)\n\n\ndef create_esap_rssd_figures(result: RSSDRunResult, output_dir: Optional[Any] = None, show: bool = True) -> List[str]:\n    """Create live and bundle figures for v2.10 maps, legacy diagnostics, and SBAD diagnostics."""\n    cfg_local = result.config\n    output_path = Path(output_dir) if output_dir is not None else None\n    saved: List[str] = []\n    selected = result.selected_sites.copy()\n    candidates = result.candidate_sites.copy()\n    support = result.support_sequence.copy()\n    coverage = result.field_coverage_distances.copy()\n    diag = result.diagnostic_data or {}\n    coords_all = diag.get("coords")\n    scores_all = diag.get("design_scores")\n    eligible = diag.get("eligible_indices")\n    original_features = diag.get("original_features")\n    xcol, ycol = cfg_local.X_COLUMN, cfg_local.Y_COLUMN\n    p = int(cfg_local.N_DESIGN_COMPONENTS)\n\n    # 01: full survey/support footprint and final selected sites.\n    fig, ax = plt.subplots(figsize=(9, 7))\n    if coords_all is not None:\n        ax.scatter(coords_all[:, 0], coords_all[:, 1], s=2, alpha=0.10, color=cfg_local.FOOTPRINT_COLOR, rasterized=True, label="Survey footprint")\n    if len(support):\n        final_support = support[support.get("used_in_final_support_prefix", False).astype(bool)] if "used_in_final_support_prefix" in support else support\n        ax.scatter(final_support["X"], final_support["Y"], s=8, alpha=0.20, color=cfg_local.SUPPORT_COLOR, label="SBAD support prefix")\n    for role, marker, color, label in [("response_surface", "x", cfg_local.SELECTED_COLOR, "Response-surface sites"), ("spatial_support", "^", cfg_local.SUPPORT_COLOR, "Spatial support sites")]:\n        part = selected[selected["sample_role"] == role]\n        if len(part):\n            ax.scatter(part[xcol], part[ycol], marker=marker, s=90 if marker == "x" else 70, color=color, edgecolor="black" if marker != "x" else None, label=label)\n    if len(selected) <= 60:\n        for _, row in selected.iterrows():\n            ax.annotate(str(row["sample_order"]), (row[xcol], row[ycol]), xytext=(3, 3), textcoords="offset points", fontsize=7)\n    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Final ESAP RSSD v2.10 field map")\n    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")\n    _save_or_show_figure(fig, "01_final_field_map.png", output_path, show, saved)\n\n    # 02: candidates and selected sites.\n    fig, ax = plt.subplots(figsize=(9, 7))\n    if len(candidates):\n        unique_candidates = candidates.drop_duplicates("candidate_analysis_index")\n        ax.scatter(unique_candidates[xcol], unique_candidates[ycol], s=12, alpha=0.35, color=cfg_local.CANDIDATE_COLOR, label="Candidate observations")\n    ax.scatter(selected[xcol], selected[ycol], marker="x", s=80, color=cfg_local.SELECTED_COLOR, label="Final selected")\n    ax.set_xlabel(f"{xcol} (projected units)"); ax.set_ylabel(f"{ycol} (projected units)"); ax.set_title("Candidate pool and final selected locations")\n    ax.set_aspect("equal", adjustable="box"); ax.legend(loc="best")\n    _save_or_show_figure(fig, "02_candidates_and_selected_map.png", output_path, show, saved)\n\n    # 03: response-surface target matches in PC space.\n    response = selected[selected["sample_role"] == "response_surface"].copy()\n    target_cols = [f"target_PC{j + 1}" for j in range(p)]\n    selected_pc_cols = [f"selected_PC{j + 1}" for j in range(p)]\n    for col in target_cols + selected_pc_cols:\n        if col in response:\n            response[col] = pd.to_numeric(response[col], errors="coerce")\n    fig, ax = plt.subplots(figsize=(8, 6 if p > 1 else 4.5))\n    if p == 1:\n        ax.scatter(response["target_PC1"], np.zeros(len(response)), marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")\n        ax.scatter(response["selected_PC1"], np.zeros(len(response)), s=40, color=cfg_local.SELECTED_COLOR, label="Selected")\n        for _, row in response.iterrows():\n            ax.plot([row["target_PC1"], row["selected_PC1"]], [0, 0], linewidth=0.8, alpha=0.55)\n        ax.set_yticks([]); ax.set_xlabel("Standardized PC1")\n    else:\n        ax.scatter(response["target_PC1"], response["target_PC2"], marker="x", s=80, color=cfg_local.TARGET_COLOR, label="Targets")\n        ax.scatter(response["selected_PC1"], response["selected_PC2"], s=42, color=cfg_local.SELECTED_COLOR, label="Selected response-surface sites")\n        for _, row in response.iterrows():\n            ax.plot([row["target_PC1"], row["selected_PC1"]], [row["target_PC2"], row["selected_PC2"]], linewidth=0.8, alpha=0.55)\n        ax.set_xlabel("Standardized PC1"); ax.set_ylabel("Standardized PC2"); ax.set_aspect("equal", adjustable="box")\n    ax.set_title("Response-surface target matches"); ax.legend(loc="best")\n    _save_or_show_figure(fig, "03_response_surface_target_matches.png", output_path, show, saved)\n\n    # 04: selected versus whole eligible PC boxplots, restored from v2.8 style.\n    if scores_all is not None and eligible is not None:\n        fig, axes = plt.subplots(1, p, figsize=(4.8 * p, 5), squeeze=False)\n        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)\n        for j in range(p):\n            axes[0, j].boxplot([scores_all[eligible, j], scores_all[selected_indices, j]], patch_artist=True)\n            axes[0, j].set_xticks([1, 2], ["Whole eligible", "Selected"])\n            axes[0, j].set_title(f"Standardized PC{j + 1}")\n        fig.suptitle("PC distributions: whole eligible survey vs selected")\n        _save_or_show_figure(fig, "04_pc_boxplots_whole_vs_selected.png", output_path, show, saved)\n\n    # 05: original feature boxplots, restored from v2.8 style.\n    if original_features is not None and eligible is not None:\n        feature_names = list(diag.get("feature_columns", cfg_local.FEATURE_COLUMNS))\n        n_features = len(feature_names); ncols = min(3, n_features); nrows = math.ceil(n_features / ncols)\n        selected_indices = np.asarray(diag.get("selected_indices"), dtype=int)\n        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows), squeeze=False)\n        for j, feature in enumerate(feature_names):\n            axes.flat[j].boxplot([original_features[eligible, j], original_features[selected_indices, j]], patch_artist=True)\n            axes.flat[j].set_xticks([1, 2], ["Whole eligible", "Selected"])\n            axes.flat[j].set_title(str(feature))\n        for ax in axes.flat[n_features:]:\n            ax.set_visible(False)\n        fig.suptitle("Sensor features: whole eligible survey vs selected")\n        _save_or_show_figure(fig, "05_feature_boxplots_whole_vs_selected.png", output_path, show, saved)\n\n    # 06: SBAD support-resolution diagnostics.\n    ad = result.ad_support_resolution.copy()\n    if len(ad):\n        fig, ax1 = plt.subplots(figsize=(8.5, 5.2))\n        sbad_star_col = "SBAD_star" if "SBAD_star" in ad.columns else "SBAD_star_screen"\n        ax1.plot(ad["support_size"], ad[sbad_star_col], marker="o", label="AD-reference SBAD*")\n        ax1.plot(ad["support_size"], ad["hybrid_core_SBAD"], marker="o", label="Hybrid core SBAD")\n        ax1.set_xlabel("Nested support prefix size"); ax1.set_ylabel("SBAD (projected units)")\n        ax2 = ax1.twinx()\n        ax2.plot(ad["support_size"], ad["core_coverage_ratio"], marker="s", linestyle="--", color="black", label="Core coverage ratio")\n        ax2.axhline(1.0 + cfg_local.AD_COVERAGE_ENVELOPE_REL_TOL, linestyle=":", color="black", linewidth=1)\n        ax2.set_ylabel("SBAD / SBAD*")\n        lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()\n        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")\n        ax1.set_title("Adaptive SBAD support-resolution diagnostics")\n        _save_or_show_figure(fig, "06_sbad_support_resolution.png", output_path, show, saved)\n\n    # 07: final field coverage distance distribution.\n    if len(coverage):\n        d = pd.to_numeric(coverage["nearest_final_site_distance"], errors="coerce").dropna().to_numpy(float)\n        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))\n        ax1.hist(d, bins=30, color=cfg_local.SUPPORT_COLOR, alpha=0.8)\n        ax1.axvline(np.mean(d), linestyle="--", color="black", label=f"SBAD={np.mean(d):.3g}")\n        ax1.set_xlabel("Nearest final-site distance"); ax1.set_ylabel("Support coordinates"); ax1.set_title("Field coverage distances"); ax1.legend()\n        sorted_d = np.sort(d); ax2.plot(sorted_d, np.linspace(0, 1, len(sorted_d), endpoint=True))\n        ax2.set_xlabel("Nearest final-site distance"); ax2.set_ylabel("Cumulative fraction"); ax2.set_title("Coverage-distance CDF")\n        _save_or_show_figure(fig, "07_field_coverage_distance_distribution.png", output_path, show, saved)\n\n    # 08: spatial support additions.\n    additions = result.spatial_support_sites.copy()\n    if len(additions):\n        fig, ax = plt.subplots(figsize=(7.5, 4.8))\n        x = additions["support_addition_order"].to_numpy(int)\n        ax.plot(x - 1, additions["SBAD_before_addition"], marker="o", label="Before addition")\n        ax.plot(x, additions["SBAD_after_addition"], marker="o", label="After addition")\n        ax.set_xlabel("Spatial support addition step"); ax.set_ylabel("Final-design SBAD")\n        ax.set_title("Sequential spatial support-site SBAD reduction"); ax.legend()\n        _save_or_show_figure(fig, "08_spatial_support_sbad_reduction.png", output_path, show, saved)\n\n    # 09: nearest selected-neighbor distances.\n    if "nearest_selected_geographic_neighbor_distance" in selected:\n        nn = pd.to_numeric(selected["nearest_selected_geographic_neighbor_distance"], errors="coerce").dropna().to_numpy(float)\n        fig, ax = plt.subplots(figsize=(8, 5))\n        ax.hist(nn, bins=min(15, max(5, len(nn) // 2)), color=cfg_local.SELECTED_COLOR, alpha=0.85)\n        ax.axvline(float(np.exp(np.mean(np.log(nn[nn > 0])))), linestyle="--", color="black", label="geoMSD")\n        ax.set_xlabel("Nearest selected-neighbor distance"); ax.set_ylabel("Selected sites"); ax.set_title("Selected-site nearest-neighbor distances"); ax.legend()\n        _save_or_show_figure(fig, "09_nearest_neighbor_distances.png", output_path, show, saved)\n\n    # 10: optimizer stability traces.\n    opt = result.optimizer_stability.copy()\n    if len(opt):\n        final_support = opt["support_size"].max()\n        opt_final = opt[opt["support_size"] == final_support]\n        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))\n        for objective, part in opt_final.groupby("objective_type"):\n            ax1.plot(part["start_number"], part["final_SBAD"], marker=".", label=objective)\n            ax2.plot(part["start_number"], part["final_geoMSD"], marker=".", label=objective)\n        ax1.set_xlabel("Optimizer start"); ax1.set_ylabel("Final SBAD"); ax1.set_title("Optimizer SBAD trace")\n        ax2.set_xlabel("Optimizer start"); ax2.set_ylabel("Final geoMSD"); ax2.set_title("Optimizer geoMSD trace")\n        ax1.legend(); ax2.legend()\n        _save_or_show_figure(fig, "10_optimizer_stability_traces.png", output_path, show, saved)\n\n    # 11: proxy spatial-scale diagnostic.\n    proxy = result.proxy_spatial_scale.copy()\n    if len(proxy):\n        fig, ax = plt.subplots(figsize=(8.5, 5.2))\n        for pc_name, part in proxy.groupby("PC"):\n            ax.plot(part["distance_median"], part["fraction_of_far_distance_semivariance"], marker="o", label=pc_name)\n        ax.axhline(cfg_local.SPACING_TARGET_SEMIVARIANCE_FRACTION, linestyle="--", color="black", linewidth=1, label="Target fraction")\n        ax.set_xlabel("Pair distance (projected units)"); ax.set_ylabel("Fraction of far-distance semivariance")\n        ax.set_title("Per-PC proxy spatial-scale diagnostic")\n        ax.legend(loc="best")\n        _save_or_show_figure(fig, "11_proxy_spatial_scale.png", output_path, show, saved)\n\n    return saved\n\n\n# --- ESAP RSSD v2.10 targeted overrides ---\n\n\nclass SupportDistanceCache:\n    """Shared lazy cache of maximum-support-to-candidate distance vectors."""\n    def __init__(self, support_coords: np.ndarray, coords: np.ndarray, max_mib: int = 512):\n        self.support_coords = np.asarray(support_coords, dtype=np.float64)\n        self.coords = np.asarray(coords, dtype=np.float64)\n        self.max_bytes = int(max_mib) * 1024 * 1024\n        self.cache: "OrderedDict[int, np.ndarray]" = OrderedDict()\n        self.bytes = 0\n        self.cache_hits = 0\n        self.cache_misses = 0\n        self.cache_evictions = 0\n        self.peak_cached_vectors = 0\n        self.peak_cache_bytes = 0\n    def get(self, candidate_index: int) -> np.ndarray:\n        key = int(candidate_index)\n        if key in self.cache:\n            self.cache_hits += 1\n            value = self.cache.pop(key)\n            self.cache[key] = value\n            return value\n        self.cache_misses += 1\n        delta = self.support_coords - self.coords[key]\n        dist = np.sqrt(np.einsum("ij,ij->i", delta, delta)).astype(np.float32, copy=False)\n        self.cache[key] = dist\n        self.bytes += dist.nbytes\n        while self.bytes > self.max_bytes and len(self.cache) > 1:\n            _, old = self.cache.popitem(last=False)\n            self.bytes -= old.nbytes\n            self.cache_evictions += 1\n        self.peak_cached_vectors = max(self.peak_cached_vectors, len(self.cache))\n        self.peak_cache_bytes = max(self.peak_cache_bytes, self.bytes)\n        return dist\n    def get_prefix(self, candidate_index: int, prefix_size: int) -> np.ndarray:\n        return self.get(int(candidate_index))[:int(prefix_size)]\n    def snapshot(self) -> Dict[str, Any]:\n        return {\n            "cache_hits": int(self.cache_hits),\n            "cache_misses": int(self.cache_misses),\n            "cache_evictions": int(self.cache_evictions),\n            "peak_cached_vectors": int(self.peak_cached_vectors),\n            "peak_cache_mib": float(self.peak_cache_bytes / (1024 * 1024)),\n            "current_cached_vectors": int(len(self.cache)),\n            "current_cache_mib": float(self.bytes / (1024 * 1024)),\n        }\n\n\nclass SBADNearestState:\n    """Nearest/second-nearest fixed-prefix support distances backed by the shared cache."""\n    def __init__(self, cache: SupportDistanceCache, selected: np.ndarray, prefix_size: int):\n        self.cache = cache\n        self.selected = np.asarray(selected, dtype=int).copy()\n        self.prefix_size = int(prefix_size)\n        self.recompute()\n    def recompute(self) -> None:\n        dmat = np.vstack([self.cache.get_prefix(int(idx), self.prefix_size).astype(np.float64, copy=False) for idx in self.selected]).T\n        if dmat.shape[1] == 1:\n            self.nearest_distance = dmat[:, 0]\n            self.second_distance = np.full(self.prefix_size, np.inf, dtype=np.float64)\n            self.nearest_position = np.zeros(self.prefix_size, dtype=int)\n        else:\n            order = np.argpartition(dmat, kth=1, axis=1)[:, :2]\n            first = order[np.arange(self.prefix_size), np.argmin(dmat[np.arange(self.prefix_size)[:, None], order], axis=1)]\n            nearest_vals = dmat[np.arange(self.prefix_size), first]\n            dmat2 = dmat.copy()\n            dmat2[np.arange(self.prefix_size), first] = np.inf\n            second = np.min(dmat2, axis=1)\n            self.nearest_distance = nearest_vals\n            self.second_distance = second\n            self.nearest_position = first.astype(int)\n        self.sbad = float(np.mean(self.nearest_distance, dtype=np.float64))\n    def evaluate_swap(self, position: int, candidate_index: int) -> float:\n        candidate_distance = self.cache.get_prefix(int(candidate_index), self.prefix_size).astype(np.float64, copy=False)\n        baseline = np.where(self.nearest_position == int(position), self.second_distance, self.nearest_distance)\n        return float(np.mean(np.minimum(baseline, candidate_distance), dtype=np.float64))\n\n\ndef exact_sbad_cached(cache: SupportDistanceCache, selected: np.ndarray, prefix_size: int) -> float:\n    return SBADNearestState(cache, np.asarray(selected, dtype=int), int(prefix_size)).sbad\n\n\ndef _objective_seed_offset(objective_type: str) -> int:\n    return {"ad_reference": 101, "hybrid": 211, "candidate_ad": 307, "candidate_hybrid": 401}.get(str(objective_type), 503)\n\n\nclass OptimizationStartBank:\n    """Reusable start assignments independent of SBAD support size."""\n    def __init__(self, minimum_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], random_seed: int, candidate_k: int, objective_type: str, max_starts: int):\n        self.minimum_start = np.asarray(minimum_start, dtype=int).copy()\n        self.pools = pools\n        self.pool_distances = pool_distances\n        self.random_seed = int(random_seed)\n        self.candidate_k = int(candidate_k)\n        self.objective_type = str(objective_type)\n        self.max_starts = int(max_starts)\n        self.base_starts: List[np.ndarray] = []\n        self._generate()\n    def _generate(self) -> None:\n        rng = np.random.default_rng(self.random_seed + 1009 * self.candidate_k + _objective_seed_offset(self.objective_type))\n        seen: set[Tuple[int, ...]] = set()\n        def add(arr: np.ndarray) -> None:\n            key = tuple(int(x) for x in arr.tolist())\n            if len(np.unique(arr)) == len(arr) and key not in seen:\n                seen.add(key); self.base_starts.append(np.asarray(arr, dtype=int).copy())\n        add(self.minimum_start)\n        attempts = 0\n        while len(self.base_starts) < self.max_starts and attempts < self.max_starts * 50:\n            attempts += 1\n            arr = assignment_from_costs(self.pools, self.pool_distances, rng)\n            if arr is not None:\n                add(arr)\n    def starts(self, requested: int, warm_starts: Optional[Sequence[np.ndarray]] = None) -> List[Tuple[str, np.ndarray]]:\n        out: List[Tuple[str, np.ndarray]] = []\n        seen: set[Tuple[int, ...]] = set()\n        for w in warm_starts or []:\n            if w is None: continue\n            arr = np.asarray(w, dtype=int)\n            key = tuple(int(x) for x in arr.tolist())\n            if len(np.unique(arr)) == len(arr) and key not in seen:\n                seen.add(key); out.append(("warm_start", arr.copy()))\n        for i, arr in enumerate(self.base_starts):\n            key = tuple(int(x) for x in arr.tolist())\n            if key not in seen:\n                seen.add(key); out.append(("minimum_pca_distance" if i == 0 else "random_unique_assignment", arr.copy()))\n            if len(out) >= int(requested):\n                break\n        return out[:int(requested)]\n    def base_random_keys(self, n: int) -> List[Tuple[int, ...]]:\n        return [tuple(int(x) for x in arr.tolist()) for arr in self.base_starts[:int(n)]]\n\n\ndef design_metrics_cached(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, prefix_size: int, sbad: Optional[float] = None) -> Dict[str, float]:\n    mean_pc, max_pc = matching_metrics(selected, targets, scores)\n    selected_coords = coords[np.asarray(selected, dtype=int)]\n    return {\n        "SBAD": float(sbad) if sbad is not None else exact_sbad_cached(cache, selected, prefix_size),\n        "geoMSD": exact_geo_msd(selected_coords),\n        "minimum_separation": minimum_selected_separation(selected_coords),\n        "mean_pca_target_distance": float(mean_pc),\n        "max_pca_target_distance": float(max_pc),\n    }\n\n\ndef _relative_change(new: Optional[float], old: Optional[float]) -> Optional[float]:\n    if new is None or old is None or not np.isfinite(new) or not np.isfinite(old):\n        return None\n    return float(abs(float(new) - float(old)) / max(1e-12, abs(float(old))))\n\n\ndef focused_ad_refinement_improvement(pre_refinement_best_sbad: Optional[float], post_refinement_best_sbad: Optional[float], numerical_epsilon: float = 1e-12) -> float:\n    """Relative improvement from focused AD refinement at a fixed support prefix."""\n    if pre_refinement_best_sbad is None or post_refinement_best_sbad is None:\n        return 0.0\n    pre = float(pre_refinement_best_sbad)\n    post = float(post_refinement_best_sbad)\n    if not np.isfinite(pre) or not np.isfinite(post):\n        return 0.0\n    return float(max(0.0, (pre - post) / max(abs(pre), float(numerical_epsilon))))\n\n\ndef support_confirmation_transition(stable_stage: bool, stable_count: int, confirmation_pending: bool, has_larger_prefix: bool, required_stable_stages: int) -> Dict[str, Any]:\n    """Advance the adaptive SBAD support-resolution confirmation state machine."""\n    required = max(1, int(required_stable_stages))\n    confirmation_stage = bool(confirmation_pending)\n    stable_count_after = int(stable_count)\n    confirmation_pending_next = False\n    stop = False\n    resolution_stable = False\n\n    if confirmation_stage:\n        if stable_stage:\n            stable_count_after += 1\n            stop = True\n            resolution_stable = True\n            status = "confirmed"\n        else:\n            stable_count_after = 0\n            status = "confirmation_failed_reset" if has_larger_prefix else "confirmation_failed_at_max"\n            stop = not bool(has_larger_prefix)\n        return {\n            "confirmation_stage": True,\n            "stable_count": int(stable_count_after),\n            "confirmation_pending_next": bool(confirmation_pending_next),\n            "stop": bool(stop),\n            "ad_support_resolution_stable": bool(resolution_stable),\n            "support_resolution_status": status,\n        }\n\n    stable_count_after = stable_count_after + 1 if stable_stage else 0\n    if stable_count_after >= required:\n        if has_larger_prefix:\n            confirmation_pending_next = True\n            status = "provisional_stability_reached"\n        else:\n            stop = True\n            status = "stable_at_max_without_separate_confirmation"\n    elif not has_larger_prefix:\n        stop = True\n        status = "support_limit_reached_without_provisional_stability"\n    else:\n        status = "expanding"\n\n    return {\n        "confirmation_stage": False,\n        "stable_count": int(stable_count_after),\n        "confirmation_pending_next": bool(confirmation_pending_next),\n        "stop": bool(stop),\n        "ad_support_resolution_stable": False,\n        "support_resolution_status": status,\n    }\n\n\ndef support_confirmation_state_machine_preview(sizes: Sequence[int], stable_flags: Sequence[bool], required_stable_stages: int) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:\n    """Small deterministic preview used by unit validations for the support-confirmation state machine."""\n    stable_count = 0\n    confirmation_pending = False\n    rows: List[Dict[str, Any]] = []\n    final_size = int(sizes[-1]) if sizes else 0\n    final_stable = False\n    status = "not_started"\n    for i, size in enumerate(sizes):\n        transition = support_confirmation_transition(\n            bool(stable_flags[i]),\n            stable_count,\n            confirmation_pending,\n            i < len(sizes) - 1,\n            required_stable_stages,\n        )\n        stable_count = int(transition["stable_count"])\n        confirmation_pending = bool(transition["confirmation_pending_next"])\n        status = str(transition["support_resolution_status"])\n        final_size = int(size)\n        final_stable = bool(transition["ad_support_resolution_stable"])\n        rows.append({\n            "support_size": int(size),\n            "stable_stage": bool(stable_flags[i]),\n            "confirmation_stage": bool(transition["confirmation_stage"]),\n            "provisional_stable_count": int(stable_count),\n            "support_resolution_status": status,\n        })\n        if transition["stop"]:\n            break\n    return rows, {\n        "final_support_size": int(final_size),\n        "ad_support_resolution_stable": bool(final_stable),\n        "support_resolution_status": status,\n    }\n\n\ndef _site_set_overlap(a: Optional[np.ndarray], b: Optional[np.ndarray]) -> Optional[float]:\n    if a is None or b is None: return None\n    sa, sb = set(map(int, np.asarray(a).tolist())), set(map(int, np.asarray(b).tolist()))\n    return float(len(sa & sb) / max(1, len(sa | sb)))\n\n\n\n\ndef _candidate_k_values(config: RSSDConfig) -> List[int]:\n    if config.CANDIDATE_EXPLORATION_MODE == "fixed":\n        return [int(config.CANDIDATES_PER_TARGET)]\n    vals=[]; k=max(int(config.CANDIDATE_SATURATION_START_K), int(config.CANDIDATES_PER_TARGET))\n    while k <= int(config.CANDIDATE_SATURATION_MAX_K):\n        vals.append(k); k *= int(config.CANDIDATE_SATURATION_GROWTH_FACTOR)\n    if vals[-1] != int(config.CANDIDATE_SATURATION_MAX_K):\n        vals=sorted(set(vals+[int(config.CANDIDATE_SATURATION_MAX_K)]))\n    return vals\n\n\n\ndef candidate_sequence_bruteforce_reference(target: np.ndarray, search_indices: np.ndarray, search_scores: np.ndarray, search_coords: np.ndarray, config: RSSDConfig, kmax: int) -> Dict[str, Any]:\n    """Validation-only brute-force greedy candidate sequence with preserved tolerance shells."""\n    target = np.asarray(target, dtype=np.float64)\n    pc_tree = cKDTree(search_scores)\n    retained_local: List[int] = []\n    retained_set: set[int] = set()\n    available_set: set[int] = set()\n    available_tol: Dict[int, float] = {}\n    available_stage: Dict[int, int] = {}\n    added_tol: Dict[int, float] = {}\n    added_stage: Dict[int, int] = {}\n    tolerance = float(config.PC_CANDIDATE_TOLERANCE)\n    expansion_stage = 0\n    fallback_used = False\n\n    def add_available(local_indices: np.ndarray, tol: float, stage: int) -> None:\n        for value in np.asarray(local_indices, dtype=int).tolist():\n            value = int(value)\n            if value not in retained_set and value not in available_set:\n                available_set.add(value)\n                available_tol[value] = float(tol)\n                available_stage[value] = int(stage)\n\n    add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)\n    while len(retained_local) < int(kmax):\n        choices = np.array(sorted(available_set - retained_set), dtype=int)\n        while len(choices) == 0:\n            if tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:\n                tolerance = min(float(config.MAX_PC_CANDIDATE_TOLERANCE), tolerance * float(config.CANDIDATE_TOLERANCE_GROWTH))\n                expansion_stage += 1\n                add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)\n                choices = np.array(sorted(available_set - retained_set), dtype=int)\n                continue\n            kq = min(len(search_scores), max(int(kmax), len(retained_local) + 1))\n            _, nearest_i = pc_tree.query(target, k=kq)\n            add_available(np.atleast_1d(nearest_i).astype(int), tolerance, expansion_stage)\n            fallback_used = True\n            choices = np.array(sorted(available_set - retained_set), dtype=int)\n            if len(choices) == 0:\n                break\n        if len(choices) == 0:\n            break\n        pc_dist = np.linalg.norm(search_scores[choices] - target, axis=1)\n        if not retained_local:\n            order = np.lexsort((search_indices[choices], pc_dist))\n        else:\n            retained_coords = search_coords[np.asarray(retained_local, dtype=int)]\n            d = np.sqrt(((search_coords[choices, None, :] - retained_coords[None, :, :]) ** 2).sum(axis=2))\n            min_geo = np.min(d, axis=1)\n            order = np.lexsort((search_indices[choices], pc_dist, -min_geo))\n        chosen = int(choices[int(order[0])])\n        retained_local.append(chosen)\n        retained_set.add(chosen)\n        added_tol[chosen] = float(available_tol.get(chosen, tolerance))\n        added_stage[chosen] = int(available_stage.get(chosen, expansion_stage))\n    return {\n        "indices": search_indices[np.asarray(retained_local, dtype=int)].astype(int),\n        "distances": np.linalg.norm(search_scores[np.asarray(retained_local, dtype=int)] - target, axis=1).astype(float) if retained_local else np.array([], dtype=float),\n        "tolerances": np.array([added_tol[i] for i in retained_local], dtype=float),\n        "expansion_stages": np.array([added_stage[i] for i in retained_local], dtype=int),\n        "fallback_used": bool(fallback_used),\n    }\n\n\ndef build_nested_candidate_sequences(target_instances: pd.DataFrame, search_indices: np.ndarray, scores: np.ndarray, coords: np.ndarray, config: RSSDConfig) -> Tuple[Dict[Tuple[float, ...], Dict[str, Any]], pd.DataFrame, Dict[str, Any]]:\n    """Build one exact incremental min_geo candidate sequence per unique target coordinate."""\n    p = config.N_DESIGN_COMPONENTS\n    target_cols = [f"target_PC{j + 1}" for j in range(p)]\n    targets = target_instances[target_cols].to_numpy(dtype=np.float64)\n    search_indices = np.asarray(search_indices, dtype=int)\n    search_scores = scores[search_indices]\n    search_coords = coords[search_indices]\n    pc_tree = cKDTree(search_scores)\n    target_keys = [tuple(row) for row in np.round(targets, 12)]\n    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}\n    kmax = max(int(config.CANDIDATE_SATURATION_MAX_K), max(multiplicities.values()))\n    sequences: Dict[Tuple[float, ...], Dict[str, Any]] = {}\n    records: List[Dict[str, Any]] = []\n\n    for key in sorted(multiplicities.keys()):\n        target = np.asarray(key, dtype=np.float64)\n        retained_local: List[int] = []\n        retained_set: set[int] = set()\n        available_set: set[int] = set()\n        available_tol: Dict[int, float] = {}\n        available_stage: Dict[int, int] = {}\n        min_geo: Dict[int, float] = {}\n        added_tol: Dict[int, float] = {}\n        added_stage: Dict[int, int] = {}\n        tolerance = float(config.PC_CANDIDATE_TOLERANCE)\n        expansion_stage = 0\n        fallback_used = False\n\n        def add_available(local_indices: np.ndarray, tol: float, stage: int) -> None:\n            new_values: List[int] = []\n            for value in np.asarray(local_indices, dtype=int).tolist():\n                value = int(value)\n                if value not in retained_set and value not in available_set:\n                    available_set.add(value)\n                    available_tol[value] = float(tol)\n                    available_stage[value] = int(stage)\n                    new_values.append(value)\n            if new_values and retained_local:\n                new_arr = np.asarray(new_values, dtype=int)\n                retained_coords = search_coords[np.asarray(retained_local, dtype=int)]\n                d = np.sqrt(((search_coords[new_arr, None, :] - retained_coords[None, :, :]) ** 2).sum(axis=2))\n                vals = np.min(d, axis=1)\n                for local_idx, value in zip(new_arr.tolist(), vals.tolist()):\n                    min_geo[int(local_idx)] = float(value)\n            elif new_values:\n                for local_idx in new_values:\n                    min_geo[int(local_idx)] = float("inf")\n\n        add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)\n        while len(retained_local) < kmax:\n            choices = np.array(sorted(available_set - retained_set), dtype=int)\n            while len(choices) == 0:\n                if tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:\n                    tolerance = min(float(config.MAX_PC_CANDIDATE_TOLERANCE), tolerance * float(config.CANDIDATE_TOLERANCE_GROWTH))\n                    expansion_stage += 1\n                    add_available(np.asarray(pc_tree.query_ball_point(target, r=tolerance), dtype=int), tolerance, expansion_stage)\n                    choices = np.array(sorted(available_set - retained_set), dtype=int)\n                    continue\n                kq = min(len(search_scores), max(kmax, len(retained_local) + 1))\n                _, nearest_i = pc_tree.query(target, k=kq)\n                add_available(np.atleast_1d(nearest_i).astype(int), tolerance, expansion_stage)\n                fallback_used = True\n                choices = np.array(sorted(available_set - retained_set), dtype=int)\n                if len(choices) == 0:\n                    break\n            if len(choices) == 0:\n                break\n            pc_dist_choices = np.linalg.norm(search_scores[choices] - target, axis=1)\n            if not retained_local:\n                order = np.lexsort((search_indices[choices], pc_dist_choices))\n            else:\n                current_min_geo = np.array([min_geo[int(i)] for i in choices], dtype=np.float64)\n                order = np.lexsort((search_indices[choices], pc_dist_choices, -current_min_geo))\n            chosen = int(choices[int(order[0])])\n            retained_local.append(chosen)\n            retained_set.add(chosen)\n            added_tol[chosen] = float(available_tol.get(chosen, tolerance))\n            added_stage[chosen] = int(available_stage.get(chosen, expansion_stage))\n            remaining = np.array(sorted(available_set - retained_set), dtype=int)\n            if len(remaining):\n                d_new = np.sqrt(np.sum((search_coords[remaining] - search_coords[chosen]) ** 2, axis=1))\n                for local_idx, dist_value in zip(remaining.tolist(), d_new.tolist()):\n                    local_idx = int(local_idx)\n                    min_geo[local_idx] = float(min(min_geo.get(local_idx, float("inf")), dist_value))\n\n        retained_arr = np.asarray(retained_local, dtype=int)\n        indices = search_indices[retained_arr]\n        distances = np.linalg.norm(search_scores[retained_arr] - target, axis=1) if len(retained_arr) else np.array([], dtype=float)\n        seq = {\n            "target": target,\n            "indices": indices.astype(int),\n            "distances": distances.astype(float),\n            "tolerances": np.array([added_tol[i] for i in retained_local], dtype=float),\n            "expansion_stages": np.array([added_stage[i] for i in retained_local], dtype=int),\n            "fallback_used": bool(fallback_used),\n            "multiplicity": int(multiplicities[key]),\n        }\n        sequences[key] = seq\n        for rank, (idx, dist, tol, stage) in enumerate(zip(seq["indices"], seq["distances"], seq["tolerances"], seq["expansion_stages"]), start=1):\n            records.append({\n                "target_key": str(key),\n                "candidate_analysis_index": int(idx),\n                "candidate_sequence_rank": int(rank),\n                "pca_target_distance": float(dist),\n                "candidate_added_at_tolerance": float(tol),\n                "candidate_tolerance_expansion_stage": int(stage),\n            })\n\n    k_values = _candidate_k_values(config)\n    nested = True\n    for seq in sequences.values():\n        prev: Optional[np.ndarray] = None\n        for k in k_values:\n            cur = seq["indices"][:min(k, len(seq["indices"]))]\n            if prev is not None and not np.array_equal(prev, cur[:len(prev)]):\n                nested = False\n            prev = cur.copy()\n    report = {"candidate_sequences_verified_nested": bool(nested), "unique_base_target_sequences": int(len(sequences)), "candidate_sequence_max_K": int(kmax)}\n    return sequences, pd.DataFrame(records), report\n\n\ndef candidate_pools_from_sequences(target_instances: pd.DataFrame, sequences: Dict[Tuple[float, ...], Dict[str, Any]], requested_k: int, config: RSSDConfig) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:\n    p = config.N_DESIGN_COMPONENTS\n    target_cols = [f"target_PC{j + 1}" for j in range(p)]\n    targets = target_instances[target_cols].to_numpy(dtype=np.float64)\n    target_keys = [tuple(row) for row in np.round(targets, 12)]\n    pools=[]; distances=[]; tolerances=[]; records=[]\n    for pos, key in enumerate(target_keys):\n        seq=sequences[key]\n        k=max(int(requested_k), int(seq.get("multiplicity", 1)))\n        idx=seq["indices"][:k]; dist=seq["distances"][:k]\n        pools.append(idx.astype(int)); distances.append(dist.astype(float)); tolerances.append(float(np.max(seq["tolerances"][:len(idx)])) if len(idx) else np.nan)\n        for rank, (candidate_idx, candidate_dist) in enumerate(zip(idx, dist), start=1):\n            row={"target_position": int(pos), "target_instance_id": target_instances.iloc[pos]["target_instance_id"], "base_target_id": target_instances.iloc[pos]["base_target_id"], "target_type": target_instances.iloc[pos]["target_type"], "sample_role": target_instances.iloc[pos].get("sample_role", "response_surface"), "candidate_analysis_index": int(candidate_idx), "candidate_rank": int(rank), "candidate_sequence_rank": int(rank), "pca_target_distance": float(candidate_dist), "final_tolerance_used": tolerances[-1], "candidate_added_at_tolerance": float(seq["tolerances"][rank-1]), "candidate_tolerance_expansion_stage": int(seq["expansion_stages"][rank-1])}\n            row.update({c: float(target_instances.iloc[pos][c]) for c in target_cols})\n            records.append(row)\n    return pools, distances, pd.DataFrame(records), tolerances\n\n\ndef _candidate_pool_overlap(pools: Sequence[np.ndarray]) -> int:\n    flat=np.concatenate(pools) if pools else np.array([], dtype=int)\n    return int(len(flat)-len(np.unique(flat)))\n\n\ndef screen_candidate_saturation(target_instances: pd.DataFrame, sequences: Dict[Tuple[float, ...], Dict[str, Any]], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, support_prefix_size: int, config: RSSDConfig) -> Tuple[int, List[np.ndarray], List[np.ndarray], pd.DataFrame, pd.DataFrame, Dict[str, Any]]:\n    records=[]; all_opt=[]; previous=None; stable_count=0; selected_k=None; final_pools=None; final_dists=None; final_assoc=None; warm_ad=[]; warm_hybrid=[]\n    for stage, k in enumerate(_candidate_k_values(config), start=1):\n        stage_start=time.perf_counter()\n        pools, pool_dists, assoc, tolerances = candidate_pools_from_sequences(target_instances, sequences, int(k), config)\n        assignment=assignment_from_costs(pools, pool_dists)\n        possible=assignment is not None\n        if not possible:\n            records.append({"stage": stage, "requested_K": int(k), "candidate_sequences_verified_nested": True, "unique_candidate_observations": int(len(np.unique(np.concatenate(pools))) if pools else 0), "assignment_possible": False, "targets_below_requested_K": int(sum(len(pool)<k for pool in pools)), "targets_requiring_tolerance_expansion": int(sum(np.nan_to_num(tolerances)>config.PC_CANDIDATE_TOLERANCE)), "maximum_target_tolerance": float(np.nanmax(tolerances)), "SBAD_star": None, "hybrid_core_SBAD": None, "core_coverage_ratio": None, "hybrid_geoMSD": None, "hybrid_minimum_separation": None, "mean_pca_target_distance": None, "maximum_pca_target_distance": None, "relative_change_in_SBAD_star": None, "relative_change_in_hybrid_geoMSD": None, "relative_change_in_hybrid_minimum_separation": None, "AD_exploration_starts": 0, "hybrid_exploration_starts": 0, "stage_runtime_seconds": time.perf_counter()-stage_start, "stable_stage": False, "confirmation_stage": False, "maximum_K_reached": bool(k==_candidate_k_values(config)[-1]), "candidate_space_status": "assignment_not_possible"})\n            continue\n        ad_best, ad_table, ad_panel = optimize_multiple_starts_sbad(assignment, pools, pool_dists, targets, scores, coords, cache, int(support_prefix_size), config, "ad_reference", candidate_k=int(k), optimization_phase="candidate_screen", start_limit=int(config.CANDIDATE_STAGE_EXPLORATION_STARTS), warm_starts=warm_ad, adaptive=False)\n        sbad_star=exact_sbad_cached(cache, ad_best, int(support_prefix_size)); sbad_limit=(1+config.AD_COVERAGE_ENVELOPE_REL_TOL)*sbad_star\n        hybrid_best, hy_table, hy_panel = optimize_multiple_starts_sbad(assignment, pools, pool_dists, targets, scores, coords, cache, int(support_prefix_size), config, "hybrid", sbad_limit=sbad_limit, candidate_k=int(k), optimization_phase="candidate_screen", start_limit=int(config.CANDIDATE_STAGE_EXPLORATION_STARTS), warm_starts=[ad_best,*warm_hybrid], adaptive=False)\n        hmet=design_metrics_cached(hybrid_best, targets, scores, coords, cache, int(support_prefix_size))\n        all_opt.extend([ad_table, hy_table])\n        rel_sbad=_relative_change(sbad_star, previous["SBAD_star"] if previous else None)\n        rel_geo=_relative_change(hmet["geoMSD"], previous["hybrid_geoMSD"] if previous else None)\n        rel_min=_relative_change(hmet["minimum_separation"], previous["hybrid_minimum_separation"] if previous else None)\n        stable=bool(previous is not None and (rel_sbad is not None and rel_sbad <= config.CANDIDATE_SATURATION_AD_REL_TOL) and (rel_geo is not None and rel_geo <= config.CANDIDATE_SATURATION_GEOMSD_REL_TOL) and (rel_min is not None and rel_min <= config.CANDIDATE_SATURATION_MINSEP_REL_TOL))\n        stable_count = stable_count + 1 if stable else 0\n        confirmation=bool(stable_count >= int(config.CANDIDATE_SATURATION_STABLE_STAGES_REQUIRED))\n        status="objective_saturated" if confirmation else ("candidate_limit_reached_without_objective_saturation" if k==_candidate_k_values(config)[-1] else "expanding")\n        rec={"stage": stage, "requested_K": int(k), "candidate_sequences_verified_nested": True, "unique_candidate_observations": int(len(np.unique(np.concatenate(pools))) if pools else 0), "assignment_possible": True, "targets_below_requested_K": int(sum(len(pool)<k for pool in pools)), "targets_requiring_tolerance_expansion": int(sum(np.nan_to_num(tolerances)>config.PC_CANDIDATE_TOLERANCE)), "maximum_target_tolerance": float(np.nanmax(tolerances)), "SBAD_star": float(sbad_star), "hybrid_core_SBAD": float(hmet["SBAD"]), "core_coverage_ratio": float(hmet["SBAD"]/sbad_star) if sbad_star>0 else np.nan, "hybrid_geoMSD": float(hmet["geoMSD"]), "hybrid_minimum_separation": float(hmet["minimum_separation"]), "mean_pca_target_distance": float(hmet["mean_pca_target_distance"]), "maximum_pca_target_distance": float(hmet["max_pca_target_distance"]), "candidate_pool_overlap": _candidate_pool_overlap(pools), "relative_change_in_SBAD_star": rel_sbad, "relative_change_in_hybrid_geoMSD": rel_geo, "relative_change_in_hybrid_minimum_separation": rel_min, "AD_exploration_starts": int(len(ad_table)), "hybrid_exploration_starts": int(len(hy_table)), "stage_runtime_seconds": float(time.perf_counter()-stage_start), "stable_stage": bool(stable), "confirmation_stage": bool(confirmation), "maximum_K_reached": bool(k==_candidate_k_values(config)[-1]), "candidate_space_status": status}\n        records.append(rec)\n        selected_k=int(k); final_pools=pools; final_dists=pool_dists; final_assoc=assoc; previous=rec; warm_ad=[ad_best]; warm_hybrid=[hybrid_best]\n        if confirmation:\n            break\n    if selected_k is None or final_pools is None or final_dists is None or final_assoc is None:\n        raise RuntimeError("Candidate saturation screening did not produce feasible candidate pools.")\n    return selected_k, final_pools, final_dists, final_assoc, pd.DataFrame(records), {"candidate_space_saturated": bool(records[-1].get("candidate_space_status") == "objective_saturated"), "candidate_space_status": records[-1].get("candidate_space_status"), "optimizer_rows": _concat_preserving_columns(all_opt)}\n\n\n\n\ndef coordinate_exchange_sbad(start: np.ndarray, pools: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, prefix_size: int, config: RSSDConfig, rng: np.random.Generator, objective_type: str, sbad_limit: Optional[float] = None) -> Tuple[np.ndarray, Dict[str, Any]]:\n    selected=np.asarray(start,dtype=int).copy()\n    if len(np.unique(selected)) != len(selected): raise ValueError("Coordinate exchange requires a unique starting assignment.")\n    state=SBADNearestState(cache, selected, int(prefix_size))\n    current=design_metrics_cached(selected, targets, scores, coords, cache, int(prefix_size), state.sbad); initial=dict(current)\n    accepted=0; sweeps=0\n    for sweep in range(1, int(config.MAX_OPTIMIZER_SWEEPS)+1):\n        accepted_this=0\n        for position in rng.permutation(len(selected)):\n            used=set(np.delete(selected, position).tolist()); best_idx=int(selected[position]); best_metrics=current\n            for candidate_index in pools[position]:\n                candidate_index=int(candidate_index)\n                if candidate_index == selected[position] or candidate_index in used: continue\n                proposal=selected.copy(); proposal[position]=candidate_index\n                psbad=state.evaluate_swap(int(position), candidate_index)\n                pmet=design_metrics_cached(proposal, targets, scores, coords, cache, int(prefix_size), psbad)\n                better=ad_reference_is_better(pmet,best_metrics,config) if objective_type=="ad_reference" else hybrid_is_better(pmet,best_metrics,config,float(sbad_limit))\n                if better: best_idx=candidate_index; best_metrics=pmet\n            if best_idx != selected[position]:\n                selected[position]=best_idx; state=SBADNearestState(cache, selected, int(prefix_size)); current=design_metrics_cached(selected, targets, scores, coords, cache, int(prefix_size), state.sbad); accepted+=1; accepted_this+=1\n        sweeps=sweep\n        if accepted_this==0: break\n    return selected, {"objective_type": objective_type, "initial_SBAD": initial["SBAD"], "final_SBAD": current["SBAD"], "initial_geoMSD": initial["geoMSD"], "final_geoMSD": current["geoMSD"], "initial_minimum_separation": initial["minimum_separation"], "final_minimum_separation": current["minimum_separation"], "accepted_swaps": int(accepted), "sweeps": int(sweeps), "coordinate_exchange_converged": bool(sweeps < int(config.MAX_OPTIMIZER_SWEEPS)), "final_mean_pca_target_distance": current["mean_pca_target_distance"], "final_max_pca_target_distance": current["max_pca_target_distance"]}\n\n\ndef _metric_better(metrics: Dict[str,float], best: Optional[Dict[str,float]], objective_type: str, config: RSSDConfig, sbad_limit: Optional[float]) -> bool:\n    if best is None: return True\n    return ad_reference_is_better(metrics,best,config) if objective_type=="ad_reference" else hybrid_is_better(metrics,best,config,float(sbad_limit))\n\n\ndef _near_best(metrics: Dict[str,float], best: Dict[str,float], objective_type: str, config: RSSDConfig, sbad_limit: Optional[float]) -> bool:\n    tol=float(config.OPTIMIZER_NEAR_BEST_REL_TOL)\n    if objective_type=="ad_reference": return metrics["SBAD"] <= best["SBAD"]*(1+tol)\n    if metrics["SBAD"] > float(sbad_limit) + config.PCA_TIE_TOL: return False\n    return metrics["geoMSD"] >= best["geoMSD"]*(1-tol)\n\n\ndef _best_objective_value(metrics: Dict[str,float], objective_type: str) -> float:\n    return float(metrics["SBAD"] if objective_type=="ad_reference" else metrics["geoMSD"])\n\n\n\ndef coordinate_exchange_start_seed(random_seed: int, objective_type: str, candidate_k: int, start: np.ndarray) -> int:\n    """Stable per-start visitation seed independent of support size and list position."""\n    arr = np.ascontiguousarray(np.asarray(start, dtype=np.int64))\n    digest = hashlib.blake2b(digest_size=8)\n    digest.update(arr.tobytes())\n    digest.update(str(objective_type).encode("utf-8"))\n    digest.update(np.asarray([int(candidate_k)], dtype=np.int64).tobytes())\n    seed_offset = int.from_bytes(digest.digest(), byteorder="little", signed=False)\n    return int((int(random_seed) + seed_offset) % (2**32 - 1))\n\n\ndef coordinate_exchange_permutation_preview(random_seed: int, objective_type: str, candidate_k: int, start: np.ndarray, n_positions: int, n_sweeps: int = 4) -> List[Tuple[int, ...]]:\n    rng = np.random.default_rng(coordinate_exchange_start_seed(random_seed, objective_type, candidate_k, start))\n    return [tuple(int(x) for x in rng.permutation(int(n_positions)).tolist()) for _ in range(int(n_sweeps))]\n\n\ndef optimize_multiple_starts_sbad(minimum_distance_start: np.ndarray, pools: Sequence[np.ndarray], pool_distances: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, prefix_size: int, config: RSSDConfig, objective_type: str, sbad_limit: Optional[float] = None, warm_starts: Optional[Sequence[np.ndarray]] = None, candidate_k: Optional[int] = None, optimization_phase: str = "final", start_limit: Optional[int] = None, adaptive: bool = True, start_bank: Optional[OptimizationStartBank] = None) -> Tuple[np.ndarray, pd.DataFrame, List[Dict[str, Any]]]:\n    candidate_k = int(candidate_k if candidate_k is not None else max(len(p) for p in pools))\n    max_starts = int(start_limit if start_limit is not None else (config.OPTIMIZER_MAX_STARTS if adaptive else config.N_OPTIMIZER_STARTS))\n    batch_size = max(1, min(int(config.OPTIMIZER_START_BATCH_SIZE), max_starts))\n    min_starts = min(max_starts, int(config.OPTIMIZER_MIN_STARTS if adaptive else max_starts))\n    if start_bank is None:\n        start_bank = OptimizationStartBank(minimum_distance_start, pools, pool_distances, config.RANDOM_SEED, candidate_k, objective_type, max_starts + len(warm_starts or []))\n    starts = start_bank.starts(max_starts, warm_starts)\n    rows = []\n    panel_by_key = {}\n    best = None\n    best_metrics = None\n    previous_batch_best = None\n    stable_batches = 0\n    stop_reason = "configured_max_starts_completed"\n    cumulative = 0\n    batch_number = 0\n    for batch_start in range(0, len(starts), batch_size):\n        batch_number += 1\n        batch = starts[batch_start:batch_start + batch_size]\n        batch_metrics = []\n        for label, start in batch:\n            cumulative += 1\n            start_seed = coordinate_exchange_start_seed(config.RANDOM_SEED, objective_type, candidate_k, start)\n            start_rng = np.random.default_rng(start_seed)\n            opt, summary = coordinate_exchange_sbad(start, pools, targets, scores, coords, cache, int(prefix_size), config, start_rng, objective_type, sbad_limit)\n            metrics = design_metrics_cached(opt, targets, scores, coords, cache, int(prefix_size), summary["final_SBAD"])\n            key = tuple(int(x) for x in opt.tolist())\n            panel_by_key[key] = {"design_key": key, "selected": opt.copy(), "metrics": metrics, "objective_type": objective_type}\n            if _metric_better(metrics, best_metrics, objective_type, config, sbad_limit):\n                best = opt.copy()\n                best_metrics = metrics\n            batch_metrics.append(metrics)\n            rows.append({\n                **summary,\n                "optimization_phase": optimization_phase,\n                "support_size": int(prefix_size),\n                "candidate_K": int(candidate_k),\n                "start_number": int(cumulative),\n                "start_type": label,\n                "start_fingerprint_seed": int(start_seed),\n                "batch_number": int(batch_number),\n            })\n        current_value = _best_objective_value(best_metrics, objective_type)\n        if previous_batch_best is None:\n            rel_improve = None\n        elif objective_type == "ad_reference":\n            rel_improve = max(0.0, (previous_batch_best - current_value) / max(1e-12, abs(previous_batch_best)))\n        else:\n            rel_improve = max(0.0, (current_value - previous_batch_best) / max(1e-12, abs(previous_batch_best)))\n        near_fraction = float(np.mean([_near_best(m, best_metrics, objective_type, config, sbad_limit) for m in batch_metrics])) if batch_metrics else 0.0\n        stable_batch = bool(cumulative >= min_starts and rel_improve is not None and rel_improve <= config.OPTIMIZER_BEST_IMPROVEMENT_REL_TOL and near_fraction >= config.OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION)\n        stable_batches = stable_batches + 1 if stable_batch else 0\n        init_stable = bool(adaptive and stable_batches >= int(config.OPTIMIZER_STABLE_BATCHES_REQUIRED))\n        for r in rows[-len(batch):]:\n            r.update({\n                "starts_in_batch": int(len(batch)),\n                "cumulative_starts": int(cumulative),\n                "best_SBAD": float(best_metrics["SBAD"]),\n                "best_geoMSD": float(best_metrics["geoMSD"]),\n                "best_minimum_separation": float(best_metrics["minimum_separation"]),\n                "relative_best_objective_improvement": rel_improve,\n                "recent_near_best_fraction": near_fraction,\n                "stable_batch": bool(stable_batch),\n                "optimizer_initialization_stable": bool(init_stable),\n                "stop_reason": "",\n            })\n        previous_batch_best = current_value\n        if init_stable:\n            stop_reason = "optimizer_initialization_stable"\n            rows[-1]["stop_reason"] = stop_reason\n            break\n        if cumulative >= max_starts:\n            break\n    if best is None:\n        raise RuntimeError("Optimizer produced no design.")\n    if rows and not rows[-1].get("stop_reason"):\n        rows[-1]["stop_reason"] = stop_reason\n    panel = sorted(panel_by_key.values(), key=lambda r: _panel_sort_key(r, objective_type))[:int(config.AD_SUPPORT_RANK_PANEL_SIZE)]\n    return best, pd.DataFrame(rows), panel\n\n\ndef practical_support_stability(prev_designs: Sequence[np.ndarray], curr_designs: Sequence[np.ndarray], prev_prefix: int, curr_prefix: int, cache: SupportDistanceCache, config: RSSDConfig) -> Dict[str, Any]:\n    union={tuple(d.tolist()):np.asarray(d,dtype=int) for d in prev_designs}; union.update({tuple(d.tolist()):np.asarray(d,dtype=int) for d in curr_designs})\n    keys=list(union.keys()); designs=[union[k] for k in keys]\n    prev_vals=np.array([exact_sbad_cached(cache,d,prev_prefix) for d in designs], dtype=float); curr_vals=np.array([exact_sbad_cached(cache,d,curr_prefix) for d in designs], dtype=float)\n    best_prev=float(np.min(prev_vals)); best_curr=float(np.min(curr_vals)); decisive=0; agree=0; equiv=0\n    for i in range(len(designs)):\n        for j in range(i+1,len(designs)):\n            prev_dec=abs(prev_vals[i]-prev_vals[j])/max(1e-12,best_prev) > config.AD_SUPPORT_PRACTICAL_EQUIV_REL_TOL\n            curr_dec=abs(curr_vals[i]-curr_vals[j])/max(1e-12,best_curr) > config.AD_SUPPORT_PRACTICAL_EQUIV_REL_TOL\n            if prev_dec and curr_dec:\n                decisive += 1\n                agree += int(np.sign(prev_vals[i]-prev_vals[j]) == np.sign(curr_vals[i]-curr_vals[j]))\n            else:\n                equiv += 1\n    prev_near={keys[i] for i,v in enumerate(prev_vals) if v <= best_prev*(1+config.AD_SUPPORT_NEAR_BEST_REL_TOL)}; curr_near={keys[i] for i,v in enumerate(curr_vals) if v <= best_curr*(1+config.AD_SUPPORT_NEAR_BEST_REL_TOL)}\n    overlap=len(prev_near & curr_near); union_near=len(prev_near | curr_near)\n    return {"design_panel_size": int(len(designs)), "practically_equivalent_pair_count": int(equiv), "decisive_pair_count": int(decisive), "decisive_pair_order_agreement": float(agree/decisive) if decisive else None, "near_best_set_size_previous": int(len(prev_near)), "near_best_set_size_current": int(len(curr_near)), "near_best_set_overlap": int(overlap), "near_best_set_jaccard": float(overlap/union_near) if union_near else None}\n\n\ndef screen_support_resolution(minimum_assignment: np.ndarray, pools: Sequence[np.ndarray], pool_dists: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, config: RSSDConfig, candidate_k: int) -> Tuple[int, pd.DataFrame, pd.DataFrame, Dict[str, Any], List[np.ndarray]]:\n    sizes = _support_sizes(config, len(cache.support_coords))\n    records: List[Dict[str, Any]] = []\n    opt_tables: List[pd.DataFrame] = []\n    prev_panel: List[np.ndarray] = []\n    prev_size: Optional[int] = None\n    prev_sbad: Optional[float] = None\n    prev_hmet: Optional[Dict[str, float]] = None\n    prev_hybrid: Optional[np.ndarray] = None\n    stable_count = 0\n    confirmation_pending = False\n    final_size = int(sizes[-1])\n    final_status = "not_started"\n    final_stable = bool(config.AD_SUPPORT_MODE == "fixed")\n    panel: List[np.ndarray] = []\n\n    for si, size in enumerate(sizes):\n        stage_start = time.perf_counter()\n        size = int(size)\n        initial = si == 0\n        has_larger_prefix = si < len(sizes) - 1\n\n        pre_refinement_best_sbad: Optional[float] = None\n        best_re_evaluated_panel: List[np.ndarray] = []\n        if prev_panel:\n            re_evaluated = sorted(\n                ((exact_sbad_cached(cache, np.asarray(d, dtype=int), size), np.asarray(d, dtype=int).copy()) for d in prev_panel),\n                key=lambda x: (x[0], tuple(int(v) for v in x[1].tolist())),\n            )\n            pre_refinement_best_sbad = float(re_evaluated[0][0])\n            best_re_evaluated_panel = [d for _, d in re_evaluated[:max(1, int(config.SUPPORT_STAGE_REFINEMENT_STARTS))]]\n\n        starts = int(config.SUPPORT_STAGE_INITIAL_STARTS if initial else max(1, config.SUPPORT_STAGE_REFINEMENT_STARTS))\n        ad_best, ad_tab, ad_panel = optimize_multiple_starts_sbad(\n            minimum_assignment, pools, pool_dists, targets, scores, coords, cache, size, config,\n            "ad_reference", candidate_k=candidate_k, optimization_phase="support_screen",\n            start_limit=starts, warm_starts=best_re_evaluated_panel, adaptive=False,\n        )\n        sbad_star = exact_sbad_cached(cache, ad_best, size)\n        post_refinement_best_sbad = float(sbad_star)\n        focused_improve = focused_ad_refinement_improvement(pre_refinement_best_sbad, post_refinement_best_sbad)\n        limit = (1 + config.AD_COVERAGE_ENVELOPE_REL_TOL) * sbad_star\n\n        hybrid_warm_starts = [ad_best, *best_re_evaluated_panel]\n        if prev_hybrid is not None:\n            hybrid_warm_starts.append(prev_hybrid)\n        hy_best, hy_tab, hy_panel = optimize_multiple_starts_sbad(\n            minimum_assignment, pools, pool_dists, targets, scores, coords, cache, size, config,\n            "hybrid", sbad_limit=limit, candidate_k=candidate_k, optimization_phase="support_screen",\n            start_limit=starts, warm_starts=hybrid_warm_starts, adaptive=False,\n        )\n        hmet = design_metrics_cached(hy_best, targets, scores, coords, cache, size)\n        opt_tables.extend([ad_tab, hy_tab])\n\n        panel_by: Dict[Tuple[int, ...], np.ndarray] = {}\n        for d in [*prev_panel, *best_re_evaluated_panel]:\n            arr = np.asarray(d, dtype=int)\n            panel_by[tuple(int(v) for v in arr.tolist())] = arr.copy()\n        for row in ad_panel + hy_panel:\n            arr = np.asarray(row["selected"], dtype=int)\n            panel_by[tuple(int(v) for v in arr.tolist())] = arr.copy()\n        panel_candidates = list(panel_by.values())\n        panel = sorted(\n            panel_candidates,\n            key=lambda d: (exact_sbad_cached(cache, d, size), tuple(int(v) for v in d.tolist())),\n        )[:int(config.AD_SUPPORT_RANK_PANEL_SIZE)]\n\n        stab = {\n            "design_panel_size": len(panel),\n            "practically_equivalent_pair_count": None,\n            "decisive_pair_count": None,\n            "decisive_pair_order_agreement": None,\n            "near_best_set_size_previous": None,\n            "near_best_set_size_current": None,\n            "near_best_set_overlap": None,\n            "near_best_set_jaccard": None,\n        }\n        rel_ad = rel_hsbad = rel_hgeo = rel_hmin = rel_cov = None\n        site_overlap = None\n        stable = False\n\n        if prev_size is not None and prev_sbad is not None and prev_hmet is not None and prev_hybrid is not None:\n            stab = practical_support_stability(prev_panel, panel, int(prev_size), size, cache, config)\n            rel_ad = _relative_change(sbad_star, prev_sbad)\n            rel_hsbad = _relative_change(hmet["SBAD"], prev_hmet["SBAD"])\n            rel_hgeo = _relative_change(hmet["geoMSD"], prev_hmet["geoMSD"])\n            rel_hmin = _relative_change(hmet["minimum_separation"], prev_hmet["minimum_separation"])\n            prev_ratio = prev_hmet["SBAD"] / prev_sbad if prev_sbad else np.nan\n            curr_ratio = hmet["SBAD"] / sbad_star if sbad_star else np.nan\n            rel_cov = abs(curr_ratio - prev_ratio) if np.isfinite(prev_ratio) and np.isfinite(curr_ratio) else None\n            site_overlap = _site_set_overlap(prev_hybrid, hy_best)\n            decisive_ok = (\n                stab["decisive_pair_count"] == 0\n                or (\n                    stab["decisive_pair_order_agreement"] is not None\n                    and stab["decisive_pair_order_agreement"] >= config.AD_SUPPORT_DECISIVE_PAIR_AGREEMENT_MIN\n                )\n            )\n            stable = bool(\n                rel_ad is not None\n                and rel_ad <= config.AD_SUPPORT_REL_TOL\n                and decisive_ok\n                and focused_improve <= config.AD_SUPPORT_REL_TOL\n                and (rel_hsbad is None or rel_hsbad <= config.AD_SUPPORT_REL_TOL)\n                and (rel_hgeo is None or rel_hgeo <= config.SUPPORT_HYBRID_GEOMSD_REL_TOL)\n                and (rel_hmin is None or rel_hmin <= config.SUPPORT_HYBRID_MINSEP_REL_TOL)\n                and (rel_cov is None or rel_cov <= config.SUPPORT_HYBRID_COVERAGE_RATIO_ABS_TOL)\n            )\n\n        if config.AD_SUPPORT_MODE == "fixed":\n            transition = {\n                "confirmation_stage": False,\n                "stable_count": 0,\n                "confirmation_pending_next": False,\n                "stop": True,\n                "ad_support_resolution_stable": True,\n                "support_resolution_status": "fixed_support_size",\n            }\n        elif prev_size is None:\n            transition = {\n                "confirmation_stage": False,\n                "stable_count": stable_count,\n                "confirmation_pending_next": False,\n                "stop": not has_larger_prefix,\n                "ad_support_resolution_stable": False,\n                "support_resolution_status": "initial_support_screen" if has_larger_prefix else "support_limit_reached_without_provisional_stability",\n            }\n        else:\n            transition = support_confirmation_transition(\n                stable, stable_count, confirmation_pending, has_larger_prefix, config.AD_SUPPORT_STABLE_STAGES_REQUIRED\n            )\n\n        stable_count = int(transition["stable_count"])\n        confirmation_pending = bool(transition["confirmation_pending_next"])\n        final_status = str(transition["support_resolution_status"])\n        final_stable = bool(transition["ad_support_resolution_stable"])\n\n        rec = {\n            "support_size": int(size),\n            "SBAD_star_screen": float(sbad_star),\n            "pre_refinement_best_SBAD": pre_refinement_best_sbad,\n            "post_refinement_best_SBAD": post_refinement_best_sbad,\n            "focused_AD_refinement_improvement": float(focused_improve),\n            "relative_change_in_best_AD_reference_SBAD": rel_ad,\n            **stab,\n            "hybrid_core_SBAD": float(hmet["SBAD"]),\n            "core_coverage_ratio": float(hmet["SBAD"] / sbad_star) if sbad_star else np.nan,\n            "hybrid_geoMSD": float(hmet["geoMSD"]),\n            "hybrid_minimum_separation": float(hmet["minimum_separation"]),\n            "relative_change_in_hybrid_SBAD": rel_hsbad,\n            "relative_change_in_hybrid_geoMSD": rel_hgeo,\n            "relative_change_in_hybrid_minimum_separation": rel_hmin,\n            "relative_change_in_core_coverage_ratio": rel_cov,\n            "site_set_overlap_with_previous": site_overlap,\n            "stage_runtime_seconds": float(time.perf_counter() - stage_start),\n            "stable_stage": bool(stable),\n            "confirmation_stage": bool(transition["confirmation_stage"]),\n            "provisional_stable_count": int(stable_count),\n            "confirmation_pending_next": bool(confirmation_pending),\n            "support_resolution_status": final_status,\n            "stable_at_max_without_separate_confirmation": bool(final_status == "stable_at_max_without_separate_confirmation"),\n            "maximum_support_size_reached": bool(size == sizes[-1]),\n        }\n        records.append(rec)\n\n        final_size = int(size)\n        prev_panel = [np.asarray(d, dtype=int).copy() for d in panel]\n        prev_size = int(size)\n        prev_sbad = float(sbad_star)\n        prev_hmet = dict(hmet)\n        prev_hybrid = hy_best.copy()\n\n        if transition["stop"]:\n            break\n\n    meta = {\n        "ad_support_resolution_stable": bool(final_stable),\n        "final_support_size": int(final_size),\n        "support_screening_only": True,\n        "support_resolution_status": final_status,\n        "max_support_resolution_reached": bool(final_size == sizes[-1]),\n    }\n    return int(final_size), pd.DataFrame(records), _concat_preserving_columns(opt_tables), meta, panel\n\n\ndef final_core_optimization(minimum_assignment: np.ndarray, pools: Sequence[np.ndarray], pool_dists: Sequence[np.ndarray], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, final_B: int, config: RSSDConfig, candidate_k: int, warm_panel: Optional[Sequence[np.ndarray]]=None) -> Tuple[np.ndarray,np.ndarray,pd.DataFrame,Dict[str,Any]]:\n    ad_best, ad_tab, _=optimize_multiple_starts_sbad(minimum_assignment,pools,pool_dists,targets,scores,coords,cache,int(final_B),config,"ad_reference",candidate_k=candidate_k,optimization_phase="final",warm_starts=warm_panel,adaptive=(config.OPTIMIZER_START_MODE=="adaptive"))\n    sbad_star=exact_sbad_cached(cache,ad_best,int(final_B)); limit=(1+config.AD_COVERAGE_ENVELOPE_REL_TOL)*sbad_star\n    hy_best, hy_tab, _=optimize_multiple_starts_sbad(minimum_assignment,pools,pool_dists,targets,scores,coords,cache,int(final_B),config,"hybrid",sbad_limit=limit,candidate_k=candidate_k,optimization_phase="final",warm_starts=[ad_best,*(warm_panel or [])],adaptive=(config.OPTIMIZER_START_MODE=="adaptive"))\n    hmet=design_metrics_cached(hy_best,targets,scores,coords,cache,int(final_B))\n    meta={"SBAD_star":float(sbad_star),"SBAD_limit":float(limit),"hybrid_core_SBAD":float(hmet["SBAD"]),"core_coverage_ratio":float(hmet["SBAD"]/sbad_star) if sbad_star else np.nan,"hybrid_geoMSD":float(hmet["geoMSD"]),"hybrid_minimum_separation":float(hmet["minimum_separation"]),"AD_reference_starts":int(len(ad_tab)),"hybrid_starts":int(len(hy_tab)),"AD_initialization_stable":bool(ad_tab["optimizer_initialization_stable"].iloc[-1]) if len(ad_tab) else False,"hybrid_initialization_stable":bool(hy_tab["optimizer_initialization_stable"].iloc[-1]) if len(hy_tab) else False}\n    return hy_best, ad_best, _concat_preserving_columns([ad_tab, hy_tab]), meta\n\n\n\n\ndef confirm_candidate_saturation_final_B(selected_k: int, target_instances: pd.DataFrame, sequences: Dict[Tuple[float,...],Dict[str,Any]], targets: np.ndarray, scores: np.ndarray, coords: np.ndarray, cache: SupportDistanceCache, final_B: int, config: RSSDConfig, candidate_table: pd.DataFrame) -> Tuple[int,List[np.ndarray],List[np.ndarray],pd.DataFrame,pd.DataFrame,pd.DataFrame,Dict[str,Any]]:\n    k_values=_candidate_k_values(config); opt_rows=[]; records=[]; current_k=int(selected_k); status="final_B_confirmed_at_screened_K"\n    start_idx=k_values.index(current_k) if current_k in k_values else 0\n    previous=None; warm_ad=[]; warm_hybrid=[]\n    for k in k_values[start_idx:]:\n        stage_start=time.perf_counter(); pools,dists,assoc,tols=candidate_pools_from_sequences(target_instances,sequences,int(k),config); assignment=assignment_from_costs(pools,dists)\n        if assignment is None: continue\n        ad_best,ad_tab,_=optimize_multiple_starts_sbad(assignment,pools,dists,targets,scores,coords,cache,int(final_B),config,"ad_reference",candidate_k=int(k),optimization_phase="final_B_candidate_confirm",start_limit=max(3,int(config.CANDIDATE_STAGE_EXPLORATION_STARTS)),warm_starts=warm_ad,adaptive=False)\n        sbad_star=exact_sbad_cached(cache,ad_best,int(final_B)); limit=(1+config.AD_COVERAGE_ENVELOPE_REL_TOL)*sbad_star\n        hy_best,hy_tab,_=optimize_multiple_starts_sbad(assignment,pools,dists,targets,scores,coords,cache,int(final_B),config,"hybrid",sbad_limit=limit,candidate_k=int(k),optimization_phase="final_B_candidate_confirm",start_limit=max(3,int(config.CANDIDATE_STAGE_EXPLORATION_STARTS)),warm_starts=[ad_best,*warm_hybrid],adaptive=False)\n        hmet=design_metrics_cached(hy_best,targets,scores,coords,cache,int(final_B)); opt_rows.extend([ad_tab,hy_tab])\n        rel_sbad=_relative_change(sbad_star,previous["SBAD_star"] if previous else None); rel_geo=_relative_change(hmet["geoMSD"],previous["hybrid_geoMSD"] if previous else None); rel_min=_relative_change(hmet["minimum_separation"],previous["hybrid_minimum_separation"] if previous else None)\n        stable=bool(previous is not None and rel_sbad is not None and rel_sbad<=config.CANDIDATE_SATURATION_AD_REL_TOL and rel_geo is not None and rel_geo<=config.CANDIDATE_SATURATION_GEOMSD_REL_TOL and rel_min is not None and rel_min<=config.CANDIDATE_SATURATION_MINSEP_REL_TOL)\n        status="objective_saturated_at_final_B" if stable else ("candidate_limit_reached_without_objective_saturation" if k==k_values[-1] else "expanding_at_final_B")\n        records.append({"stage": int((candidate_table["stage"].max() if len(candidate_table) else 0)+len(records)+1), "requested_K":int(k), "candidate_sequences_verified_nested":True, "unique_candidate_observations":int(len(np.unique(np.concatenate(pools))) if pools else 0), "assignment_possible":True, "targets_below_requested_K":int(sum(len(pool)<k for pool in pools)), "targets_requiring_tolerance_expansion":int(sum(np.nan_to_num(tols)>config.PC_CANDIDATE_TOLERANCE)), "maximum_target_tolerance":float(np.nanmax(tols)), "SBAD_star":float(sbad_star), "hybrid_core_SBAD":float(hmet["SBAD"]), "core_coverage_ratio":float(hmet["SBAD"]/sbad_star) if sbad_star else np.nan, "hybrid_geoMSD":float(hmet["geoMSD"]), "hybrid_minimum_separation":float(hmet["minimum_separation"]), "mean_pca_target_distance":float(hmet["mean_pca_target_distance"]), "maximum_pca_target_distance":float(hmet["max_pca_target_distance"]), "relative_change_in_SBAD_star":rel_sbad, "relative_change_in_hybrid_geoMSD":rel_geo, "relative_change_in_hybrid_minimum_separation":rel_min, "AD_exploration_starts":len(ad_tab), "hybrid_exploration_starts":len(hy_tab), "stage_runtime_seconds":float(time.perf_counter()-stage_start), "stable_stage":bool(stable), "confirmation_stage":bool(stable), "maximum_K_reached":bool(k==k_values[-1]), "candidate_space_status":status})\n        current_k=int(k); previous=records[-1]; warm_ad=[ad_best]; warm_hybrid=[hy_best]\n        if stable: break\n    pools,dists,assoc,tols=candidate_pools_from_sequences(target_instances,sequences,current_k,config)\n    combined=_concat_preserving_columns([candidate_table, pd.DataFrame(records)]) if records else candidate_table\n    meta={"final_candidate_K":int(current_k),"candidate_space_status":status,"candidate_space_saturated":bool("saturated" in status)}\n    return current_k,pools,dists,assoc,combined,_concat_preserving_columns(opt_rows),meta\n\n\n\ndef bounded_support_site_candidates(state: SBADNearestState, candidate_tree: cKDTree, candidate_indices: np.ndarray, selected: Sequence[int], cache: SupportDistanceCache, support_prefix_size: int, config: RSSDConfig) -> Tuple[np.ndarray, np.ndarray]:\n    gap_order = np.argsort(-state.nearest_distance, kind="stable")[:min(int(config.SUPPORT_GAP_ANCHORS), int(support_prefix_size))]\n    bounded: List[int] = []\n    for support_pos in gap_order:\n        k = min(len(candidate_indices), int(config.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR))\n        _, local = candidate_tree.query(cache.support_coords[int(support_pos)], k=k)\n        bounded.extend(candidate_indices[np.atleast_1d(local).astype(int)].tolist())\n    selected_set = set(map(int, selected))\n    bounded_arr = np.array(sorted(set(map(int, bounded)) - selected_set), dtype=int)\n    return bounded_arr, np.asarray(gap_order, dtype=int)\n\n\ndef add_spatial_support_sites(core_selected: np.ndarray, n_support_sites: int, support_coords: np.ndarray, candidate_indices: np.ndarray, coords: np.ndarray, config: RSSDConfig, cache: Optional[SupportDistanceCache] = None, support_prefix_size: Optional[int] = None) -> Tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:\n    selected = list(map(int, np.asarray(core_selected, dtype=int).tolist()))\n    records = []\n    candidate_indices = np.asarray(candidate_indices, dtype=int)\n    support_prefix_size = int(support_prefix_size or len(support_coords))\n    local_cache = cache if cache is not None else SupportDistanceCache(support_coords, coords, config.AD_DISTANCE_CACHE_MAX_MIB)\n    candidate_tree = cKDTree(coords[candidate_indices])\n    state = SBADNearestState(local_cache, np.asarray(selected, dtype=int), support_prefix_size)\n    for step in range(1, int(n_support_sites) + 1):\n        before = state.sbad\n        before_hits = local_cache.cache_hits\n        before_misses = local_cache.cache_misses\n        bounded, gap_order = bounded_support_site_candidates(state, candidate_tree, candidate_indices, selected, local_cache, support_prefix_size, config)\n        if len(bounded) == 0:\n            break\n        best = None\n        best_sbad = np.inf\n        best_minsep = -np.inf\n        best_rank = None\n        ranked = []\n        for cand in bounded:\n            cand = int(cand)\n            cand_dist = local_cache.get_prefix(cand, support_prefix_size).astype(np.float64, copy=False)\n            proposal_nearest = np.minimum(state.nearest_distance, cand_dist)\n            psbad = float(np.mean(proposal_nearest, dtype=np.float64))\n            pcoords = coords[np.asarray([*selected, cand], dtype=int)]\n            pmin = minimum_selected_separation(pcoords)\n            ranked.append((psbad, -pmin, cand, pmin))\n        ranked.sort(key=lambda item: (item[0], item[1], item[2]))\n        if ranked:\n            best_sbad, neg_minsep, best, best_minsep = ranked[0]\n            best = int(best)\n            best_rank = 1\n        if best is None:\n            break\n        selected.append(best)\n        state = SBADNearestState(local_cache, np.asarray(selected, dtype=int), support_prefix_size)\n        records.append({\n            "support_addition_order": int(step),\n            "analysis_index": int(best),\n            "gap_anchors_considered": int(len(gap_order)),\n            "unique_support_candidates_evaluated": int(len(bounded)),\n            "bounded_candidate_analysis_indices": ";".join(str(int(x)) for x in bounded.tolist()),\n            "selected_candidate_rank_by_exact_SBAD": int(best_rank),\n            "SBAD_before_addition": float(before),\n            "best_candidate_SBAD_after": float(best_sbad),\n            "SBAD_after_addition": float(state.sbad),\n            "SBAD_reduction": float(before - state.sbad),\n            "minimum_separation_after_addition": float(best_minsep),\n            "cache_hits": int(local_cache.cache_hits - before_hits),\n            "cache_misses": int(local_cache.cache_misses - before_misses),\n            "sample_role": "spatial_support",\n        })\n    final_selected = np.asarray(selected, dtype=int)\n    final_state = SBADNearestState(local_cache, final_selected, support_prefix_size)\n    dmat = np.vstack([local_cache.get_prefix(int(idx), support_prefix_size) for idx in final_selected]).T\n    nearest_site = np.argmin(dmat, axis=1)\n    coverage = pd.DataFrame({\n        "support_rank": np.arange(1, support_prefix_size + 1, dtype=int),\n        "support_X": local_cache.support_coords[:support_prefix_size, 0],\n        "support_Y": local_cache.support_coords[:support_prefix_size, 1],\n        "nearest_final_site_position": nearest_site + 1,\n        "nearest_final_site_analysis_index": final_selected[nearest_site],\n        "nearest_final_site_distance": final_state.nearest_distance,\n    })\n    return final_selected, pd.DataFrame(records), coverage\n\n\ndef proxy_spacing_diagnostic(eligible_coords: np.ndarray, eligible_scores: np.ndarray, selected_nearest_distances: np.ndarray, selected_geo_msd: float, config: RSSDConfig, support_coords: Optional[np.ndarray] = None, support_scores: Optional[np.ndarray] = None) -> Tuple[pd.DataFrame, Dict[str, Any]]:\n    if not config.RUN_SPACING_DIAGNOSTIC: return pd.DataFrame(), {"status":"disabled"}\n    coords_arr=np.asarray(support_coords if support_coords is not None else eligible_coords,dtype=np.float64); scores_arr=np.asarray(support_scores if support_scores is not None else eligible_scores,dtype=np.float64); selected_nn=np.asarray(selected_nearest_distances,dtype=np.float64)\n    n=len(coords_arr)\n    if n<3: return pd.DataFrame(), {"status":"not_estimable","interpretation":"Too few support points."}\n    rng=np.random.default_rng(config.RANDOM_SEED+280); sample_n=min(int(config.SPACING_DIAGNOSTIC_MAX_POINTS),n); sample_coords=coords_arr[:sample_n]; sample_scores=scores_arr[:sample_n]\n    k=min(sample_n,int(config.SPACING_DIAGNOSTIC_NEIGHBORS)+1); _,nbr=cKDTree(sample_coords).query(sample_coords,k=k,workers=-1); nbr=np.atleast_2d(nbr); left=[np.repeat(np.arange(sample_n),nbr.shape[1]-1)]; right=[nbr[:,1:].reshape(-1)]\n    a=rng.integers(0,sample_n,size=min(int(config.SPACING_DIAGNOSTIC_RANDOM_PAIRS),max(1000,sample_n*25)),endpoint=False); b=rng.integers(0,sample_n,size=len(a),endpoint=False); same=a==b\n    while np.any(same): b[same]=rng.integers(0,sample_n,size=int(np.sum(same)),endpoint=False); same=a==b\n    left.append(a); right.append(b); left=np.concatenate(left); right=np.concatenate(right)\n    delta=sample_coords[left]-sample_coords[right]; distances=np.sqrt(np.einsum("ij,ij->i",delta,delta)); valid=np.isfinite(distances)&(distances>0); distances=distances[valid]; left=left[valid]; right=right[valid]\n    edges=np.unique(np.quantile(distances,np.linspace(0,1,int(config.SPACING_DIAGNOSTIC_BINS)+1)))\n    if len(edges)<3: edges=np.unique(np.linspace(float(np.min(distances)),float(np.max(distances)),int(config.SPACING_DIAGNOSTIC_BINS)+1))\n    bin_id=np.digitize(distances,edges[1:-1],right=False); records=[]; pc_ranges={}\n    for pc in range(sample_scores.shape[1]):\n        semivar=0.5*(sample_scores[left,pc]-sample_scores[right,pc])**2; bins=[]\n        for b_id in range(len(edges)-1):\n            mask=bin_id==b_id; count=int(np.sum(mask))\n            if count==0: continue\n            bins.append({"PC":f"PC{pc+1}","bin":b_id+1,"pair_count":count,"distance_min":float(np.min(distances[mask])),"distance_median":float(np.median(distances[mask])),"distance_max":float(np.max(distances[mask])),"mean_proxy_semivariance":float(np.mean(semivar[mask]))})\n        tab=pd.DataFrame(bins)\n        if len(tab)==0: pc_ranges[f"PC{pc+1}"]={"proxy_range":None,"range_reliable":False,"reason":"no_valid_bins"}; continue\n        tab["interpreted_bin"]=tab["pair_count"]>=int(config.SPACING_MIN_PAIRS_PER_BIN); valid_tab=tab[tab["interpreted_bin"]].copy()\n        if len(valid_tab)<max(3,int(config.SPACING_PLATEAU_TAIL_BINS)):\n            reason="insufficient_interpretable_bins"; proxy_range=None; reliable=False; ref=np.nan\n        else:\n            valid_tab["smoothed_semivariance"]=valid_tab["mean_proxy_semivariance"].rolling(3,center=True,min_periods=1).median()\n            tail=valid_tab.tail(int(config.SPACING_PLATEAU_TAIL_BINS)); ref=float(tail["smoothed_semivariance"].mean()); threshold=float(config.SPACING_TARGET_SEMIVARIANCE_FRACTION*ref)\n            above=valid_tab["smoothed_semivariance"]>=threshold; run=above.rolling(int(config.SPACING_RANGE_STABILITY_BINS),min_periods=int(config.SPACING_RANGE_STABILITY_BINS)).sum()>=int(config.SPACING_RANGE_STABILITY_BINS)\n            reached_idx=np.flatnonzero(run.to_numpy())\n            slope=float((tail["smoothed_semivariance"].iloc[-1]-tail["smoothed_semivariance"].iloc[0])/max(1e-12,abs(ref)))\n            diffs=np.diff(tail["smoothed_semivariance"].to_numpy()); oscillatory=bool(np.any(diffs>0) and np.any(diffs<0) and (np.max(tail["smoothed_semivariance"])-np.min(tail["smoothed_semivariance"]))/max(1e-12,abs(ref))>config.SPACING_PLATEAU_REL_SLOPE_TOL)\n            if len(reached_idx)==0: reason="threshold_not_reached"; reliable=False; proxy_range=None\n            elif slope>config.SPACING_PLATEAU_REL_SLOPE_TOL: reason="semivariance_tail_still_increasing"; reliable=False; proxy_range=None\n            elif oscillatory: reason="semivariance_tail_unstable_or_oscillatory"; reliable=False; proxy_range=None\n            else: reason="threshold_reached_and_tail_plateaued"; reliable=True; proxy_range=float(valid_tab.iloc[int(reached_idx[0])]["distance_median"])\n        tab["far_distance_semivariance_reference"]=ref; tab["fraction_of_far_distance_semivariance"]=tab["mean_proxy_semivariance"]/ref if np.isfinite(ref) and ref>0 else np.nan; tab["range_reliable"]=bool(reliable); tab["range_failure_reason"]=reason; records.extend(tab.to_dict(orient="records")); pc_ranges[f"PC{pc+1}"]={"proxy_range":proxy_range,"range_reliable":bool(reliable),"reason":reason}\n    reliable_ranges=[v["proxy_range"] for v in pc_ranges.values() if v["range_reliable"] and v["proxy_range"] is not None]\n    summary={"status":"reliable_proxy_range_estimated" if reliable_ranges else "proxy_range_unreliable","support_sequence_used":True,"simple_random_raw_row_sample_used":False,"sampled_support_points":int(sample_n),"sampled_pairs":int(len(distances)),"per_pc_range_reliability":pc_ranges,"proxy_decorrelation_distance":float(np.nanmax(reliable_ranges)) if reliable_ranges else None,"selected_absolute_minimum_spacing":float(np.min(selected_nn)),"selected_geoMSD":float(selected_geo_msd),"selected_median_nearest_neighbor_spacing":float(np.median(selected_nn)),"interpretation":"This diagnostic never changes the hybrid design objective."}\n    return pd.DataFrame(records), summary\n\n\n# V210_OVERRIDES\n\n\n\n\n# --- ESAP RSSD v2.10 active orchestration override ---\n\n\ndef _concat_preserving_columns(tables: Sequence[pd.DataFrame]) -> pd.DataFrame:\n    """Concat nonempty frames without pandas empty/all-NA dtype FutureWarnings."""\n    valid = [table for table in tables if isinstance(table, pd.DataFrame) and len(table)]\n    if not valid:\n        return pd.DataFrame()\n    columns: List[Any] = []\n    for table in valid:\n        for col in table.columns:\n            if col not in columns:\n                columns.append(col)\n    cleaned = []\n    for table in valid:\n        part = table.dropna(axis=1, how="all")\n        if len(part.columns):\n            cleaned.append(part)\n    if cleaned:\n        out = pd.concat(cleaned, ignore_index=True)\n    else:\n        out = pd.DataFrame(index=range(sum(len(table) for table in valid)))\n    for col in columns:\n        if col not in out.columns:\n            out[col] = pd.NA\n    return out.reindex(columns=columns)\n\n\ndef _concat_diagnostic_tables(tables: Sequence[pd.DataFrame]) -> pd.DataFrame:\n    return _concat_preserving_columns(tables)\n\n\ndef _add_candidate_saturation_aliases(table: pd.DataFrame) -> pd.DataFrame:\n    out = table.copy()\n    if len(out) == 0:\n        return out\n    if "starts" not in out:\n        ad = pd.to_numeric(out.get("AD_exploration_starts", 0), errors="coerce").fillna(0)\n        hy = pd.to_numeric(out.get("hybrid_exploration_starts", 0), errors="coerce").fillna(0)\n        out["starts"] = (ad + hy).astype(int)\n    if "runtime" not in out and "stage_runtime_seconds" in out:\n        out["runtime"] = out["stage_runtime_seconds"]\n    if "stable" not in out and "stable_stage" in out:\n        out["stable"] = out["stable_stage"]\n    if "confirmation" not in out and "confirmation_stage" in out:\n        out["confirmation"] = out["confirmation_stage"]\n    if "max" not in out and "maximum_K_reached" in out:\n        out["max"] = out["maximum_K_reached"]\n    if "status" not in out and "candidate_space_status" in out:\n        out["status"] = out["candidate_space_status"]\n    preferred = [\n        "stage", "requested_K", "candidate_sequences_verified_nested", "unique_candidate_observations",\n        "assignment_possible", "targets_below_requested_K", "targets_requiring_tolerance_expansion",\n        "maximum_target_tolerance", "SBAD_star", "hybrid_core_SBAD", "core_coverage_ratio",\n        "hybrid_geoMSD", "hybrid_minimum_separation", "mean_pca_target_distance",\n        "maximum_pca_target_distance", "relative_change_in_SBAD_star",\n        "relative_change_in_hybrid_geoMSD", "relative_change_in_hybrid_minimum_separation",\n        "starts", "runtime", "stable", "confirmation", "max", "status",\n    ]\n    return out[[c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]]\n\n\ndef _add_support_resolution_aliases(table: pd.DataFrame) -> pd.DataFrame:\n    out = table.copy()\n    if len(out) == 0:\n        return out\n    if "runtime" not in out and "stage_runtime_seconds" in out:\n        out["runtime"] = out["stage_runtime_seconds"]\n    preferred = [\n        "support_size", "SBAD_star_screen", "relative_change_in_best_AD_reference_SBAD",\n        "design_panel_size", "practically_equivalent_pair_count", "decisive_pair_count",\n        "decisive_pair_order_agreement", "near_best_set_size_previous", "near_best_set_size_current",\n        "near_best_set_overlap", "near_best_set_jaccard", "focused_AD_refinement_improvement",\n        "hybrid_core_SBAD", "core_coverage_ratio", "hybrid_geoMSD", "hybrid_minimum_separation",\n        "relative_change_in_hybrid_SBAD", "relative_change_in_hybrid_geoMSD",\n        "relative_change_in_hybrid_minimum_separation", "relative_change_in_core_coverage_ratio",\n        "site_set_overlap_with_previous", "runtime", "stable_stage", "confirmation_stage",\n    ]\n    return out[[c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]]\n\n\ndef _optimizer_phase_summary(table: pd.DataFrame, phase: str, objective: str) -> Dict[str, Any]:\n    if not isinstance(table, pd.DataFrame) or len(table) == 0:\n        return {"starts": 0, "initialization_stable": False, "stop_reason": "not_run"}\n    part = table[(table.get("optimization_phase") == phase) & (table.get("objective_type") == objective)]\n    if len(part) == 0:\n        return {"starts": 0, "initialization_stable": False, "stop_reason": "not_run"}\n    stop_values = part.get("stop_reason", pd.Series(dtype=object)).replace("", np.nan).dropna()\n    return {\n        "starts": int(pd.to_numeric(part["start_number"], errors="coerce").max()),\n        "initialization_stable": bool(part.get("optimizer_initialization_stable", pd.Series([False])).iloc[-1]),\n        "stop_reason": str(stop_values.iloc[-1]) if len(stop_values) else "completed",\n    }\n\n\ndef run_esap_rssd(config: RSSDConfig, source_df: Optional[pd.DataFrame] = None, workflow_state: Optional[Dict[str, Any]] = None) -> RSSDRunResult:\n    run_started = time.perf_counter()\n    validate_config(config)\n    workflow_state = workflow_state or {}\n    stage_times: Dict[str, float] = {}\n\n    print("1/10 Validating survey data")\n    t0 = time.perf_counter()\n    if source_df is None:\n        source_df_raw, input_label = load_survey(config)\n    else:\n        source_df_raw = source_df.copy()\n        input_label = str(workflow_state.get("input_label", "provided_dataframe"))\n    source_df, valid_mask, coordinate_eligible_mask, validation_report = validate_and_index_data(source_df_raw, config)\n    valid_source_positions = np.flatnonzero(valid_mask)\n    analysis_to_source = valid_source_positions.copy()\n    analysis_coordinate_eligible = coordinate_eligible_mask[analysis_to_source]\n    coords = source_df.iloc[analysis_to_source][[config.X_COLUMN, config.Y_COLUMN]].to_numpy(dtype=np.float64)\n    original_features = source_df.iloc[analysis_to_source][config.FEATURE_COLUMNS].to_numpy(dtype=np.float64)\n    transformed_features, transformation_details = transform_features(original_features, config.FEATURE_COLUMNS, config.FEATURE_TRANSFORMS)\n    skewness_summary = pd.DataFrame(\n        {\n            "configured_transform": [config.FEATURE_TRANSFORMS.get(c, "none") for c in config.FEATURE_COLUMNS],\n            "raw_skewness": skew(original_features, axis=0, bias=False),\n            "transformed_skewness": skew(transformed_features, axis=0, bias=False),\n        },\n        index=config.FEATURE_COLUMNS,\n    )\n    stage_times["validate_transform"] = time.perf_counter() - t0\n\n    print("2/10 Standardizing sensor variables and fitting PCA")\n    t0 = time.perf_counter()\n    design_scores, feature_scaler, pca_estimator, pca_mode, memory_report = fit_standardized_pca(transformed_features, config)\n    pca_validation, pca_correlation = pca_validation_table(design_scores)\n    stage_times["pca"] = time.perf_counter() - t0\n\n    print("3/10 Constructing the response-surface design")\n    t0 = time.perf_counter()\n    p_dim = int(config.N_DESIGN_COMPONENTS)\n    pc_radius = np.linalg.norm(design_scores, axis=1)\n    design_radius = float(np.quantile(pc_radius, config.DESIGN_COVERAGE))\n    pca_d2 = np.einsum("ij,ij->i", design_scores, design_scores)\n    outlier_threshold_d2 = float(chi2.ppf(config.OUTLIER_COVERAGE, df=p_dim))\n    pca_outlier = pca_d2 > outlier_threshold_d2\n    candidate_eligible = analysis_coordinate_eligible & ~pca_outlier\n    if int(candidate_eligible.sum()) < min(config.N_SAMPLES, len(candidate_eligible)):\n        raise ValueError("Outlier and duplicate-coordinate masks leave too few candidate-eligible observations.")\n    source_df["pca_space_outlier_flag"] = False\n    source_df.iloc[analysis_to_source, source_df.columns.get_loc("pca_space_outlier_flag")] = pca_outlier\n    base_targets = make_base_ccd(p_dim, design_radius, config.CENTER_REPLICATES)\n    target_instances, target_replication_counts, sample_budget_report = allocate_sample_budget(base_targets, config.N_SAMPLES, config, p_dim)\n    target_cols = [f"target_PC{j + 1}" for j in range(p_dim)]\n    target_array = target_instances[target_cols].to_numpy(dtype=np.float64)\n    stage_times["response_surface_design"] = time.perf_counter() - t0\n\n    print("4/10 Building spatially balanced geographic support")\n    t0 = time.perf_counter()\n    support_max_request = int(config.AD_SUPPORT_FIXED_SIZE if config.AD_SUPPORT_MODE == "fixed" else config.AD_SUPPORT_MAX_SIZE)\n    support_sequence, support_report = build_spatially_balanced_support_sequence(coords, np.arange(len(coords), dtype=np.int64), support_max_request)\n    support_sizes = _support_sizes(config, len(support_sequence))\n    max_support_size = int(max(support_sizes))\n    support_all = support_sequence[["X", "Y"]].to_numpy(dtype=np.float64)[:max_support_size]\n    support_cache = SupportDistanceCache(support_all, coords, config.AD_DISTANCE_CACHE_MAX_MIB)\n    support_report.update({\n        "candidate_eligible_rows": int(candidate_eligible.sum()),\n        "maximum_cached_support_prefix": int(max_support_size),\n        "support_sizes_screened": [int(x) for x in support_sizes],\n    })\n    stage_times["support_sequence"] = time.perf_counter() - t0\n\n    print("5/10 Building nested base-target candidate sequences")\n    t0 = time.perf_counter()\n    eligible_indices = np.flatnonzero(candidate_eligible)\n    search_indices, approximate_prefilter_used, prefilter_report = approximate_pca_prefilter(eligible_indices, design_scores, target_array, config)\n    sequences, candidate_sequence_records, sequence_report = build_nested_candidate_sequences(target_instances, search_indices, design_scores, coords, config)\n    stage_times["candidate_sequence_build"] = time.perf_counter() - t0\n\n    print("6/10 Screening candidate-space saturation")\n    t0 = time.perf_counter()\n    screen_support_size = int(support_sizes[0])\n    selected_k, candidate_pools, candidate_pool_distances, candidate_associations, candidate_saturation, candidate_screen_meta = screen_candidate_saturation(\n        target_instances, sequences, target_array, design_scores, coords, support_cache, screen_support_size, config\n    )\n    candidate_optimizer_rows = candidate_screen_meta.pop("optimizer_rows", pd.DataFrame())\n    candidate_saturation = _add_candidate_saturation_aliases(candidate_saturation)\n    stage_times["candidate_saturation_screen"] = time.perf_counter() - t0\n\n    print("7/10 Stabilizing SBAD geographic support resolution")\n    t0 = time.perf_counter()\n    minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)\n    if minimum_assignment is None:\n        raise RuntimeError("No unique target-to-site assignment was possible after v2.10 candidate screening.")\n    final_B, ad_support_resolution, support_optimizer_rows, support_screen_meta, warm_panel = screen_support_resolution(\n        minimum_assignment, candidate_pools, candidate_pool_distances, target_array, design_scores, coords, support_cache, config, int(selected_k)\n    )\n    ad_support_resolution = _add_support_resolution_aliases(ad_support_resolution)\n    stage_times["support_resolution_screen"] = time.perf_counter() - t0\n\n    print("8/10 Confirming candidate saturation and optimizing the final hybrid RSSD core")\n    t0 = time.perf_counter()\n    final_k, candidate_pools, candidate_pool_distances, candidate_associations, candidate_saturation, confirm_optimizer_rows, final_k_meta = confirm_candidate_saturation_final_B(\n        int(selected_k), target_instances, sequences, target_array, design_scores, coords, support_cache, int(final_B), config, candidate_saturation\n    )\n    candidate_saturation = _add_candidate_saturation_aliases(candidate_saturation)\n    minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)\n    if minimum_assignment is None:\n        raise RuntimeError("Final-B candidate confirmation left no unique target-to-site assignment.")\n    _, _, _, final_tolerances = candidate_pools_from_sequences(target_instances, sequences, int(final_k), config)\n    core_selected, ad_reference_selected, final_optimizer_rows, final_meta = final_core_optimization(\n        minimum_assignment, candidate_pools, candidate_pool_distances, target_array, design_scores, coords, support_cache, int(final_B), config, int(final_k), warm_panel\n    )\n    optimizer_stability = _concat_diagnostic_tables([candidate_optimizer_rows, support_optimizer_rows, confirm_optimizer_rows, final_optimizer_rows])\n    support_opt_report = {\n        **support_screen_meta,\n        **final_meta,\n        **final_k_meta,\n        "screened_candidate_K": int(selected_k),\n        "final_candidate_K": int(final_k),\n        "final_support_size": int(final_B),\n        "support_cache": support_cache.snapshot(),\n    }\n    stage_times["final_candidate_confirmation_and_optimization"] = time.perf_counter() - t0\n\n    print("9/10 Adding spatial support sites")\n    t0 = time.perf_counter()\n    final_support_coords = support_all[: int(final_B)]\n    support_sequence = support_sequence.copy()\n    support_sequence["used_in_final_support_prefix"] = support_sequence["support_rank"] <= int(final_B)\n    final_selected, spatial_support_sites, field_coverage_distances = add_spatial_support_sites(\n        core_selected,\n        int(sample_budget_report["spatial_support_sites"]),\n        final_support_coords,\n        eligible_indices,\n        coords,\n        config,\n        cache=support_cache,\n        support_prefix_size=int(final_B),\n    )\n    stage_times["spatial_support_site_addition"] = time.perf_counter() - t0\n\n    print("10/10 Calculating diagnostics and writing outputs")\n    t0 = time.perf_counter()\n    selected_coords = coords[final_selected].astype(np.float64)\n    selected_scores = design_scores[final_selected].astype(np.float64)\n    nearest_selected = selected_nearest_neighbor_distances(selected_coords)\n    final_geo_msd = exact_geo_msd(selected_coords)\n    final_sbad = exact_sbad_cached(support_cache, final_selected, int(final_B))\n    response_surface_count = len(core_selected)\n    core_sbad = exact_sbad_cached(support_cache, core_selected, int(final_B))\n    core_geo_msd = exact_geo_msd(coords[core_selected])\n    core_min_sep = minimum_selected_separation(coords[core_selected])\n    mean_target_distance, max_target_distance = matching_metrics(core_selected, target_array, design_scores)\n    regression_diagnostics, leverage = regression_design_diagnostics(selected_scores, design_scores[eligible_indices], config.DIAGNOSTIC_CHUNK_SIZE)\n    support_analysis_idx = support_sequence.loc[support_sequence["used_in_final_support_prefix"], "analysis_index"].to_numpy(dtype=int)\n    support_scores = design_scores[support_analysis_idx]\n    proxy_spatial_scale, spacing_diagnostic = proxy_spacing_diagnostic(\n        coords[eligible_indices],\n        design_scores[eligible_indices],\n        nearest_selected,\n        final_geo_msd,\n        config,\n        support_coords=final_support_coords,\n        support_scores=support_scores,\n    )\n\n    selected_source_rows = analysis_to_source[final_selected]\n    base_columns = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]\n    selected_export = source_df.iloc[selected_source_rows][base_columns].reset_index(drop=True).copy()\n    selected_export.insert(0, "sample_order", np.arange(1, len(selected_export) + 1, dtype=int))\n    selected_export["analysis_index"] = final_selected\n    selected_export["sample_role"] = ["response_surface"] * response_surface_count + ["spatial_support"] * (len(final_selected) - response_surface_count)\n    for j in range(p_dim):\n        selected_export[f"selected_PC{j + 1}"] = selected_scores[:, j]\n        selected_export[f"target_PC{j + 1}"] = None\n    for column in ["target_instance_id", "base_target_id", "target_type", "target_replicate_number", "pca_target_distance"]:\n        selected_export[column] = None\n    for i in range(response_surface_count):\n        tr = target_instances.iloc[i]\n        selected_export.loc[i, "target_instance_id"] = tr["target_instance_id"]\n        selected_export.loc[i, "base_target_id"] = tr["base_target_id"]\n        selected_export.loc[i, "target_type"] = tr["target_type"]\n        selected_export.loc[i, "target_replicate_number"] = tr["target_replicate_number"]\n        for j in range(p_dim):\n            selected_export.loc[i, f"target_PC{j + 1}"] = float(tr[f"target_PC{j + 1}"])\n        selected_export.loc[i, "pca_target_distance"] = float(np.linalg.norm(selected_scores[i] - target_array[i]))\n    selected_export.loc[selected_export["sample_role"] == "spatial_support", "target_type"] = "support"\n    selected_export["nearest_selected_geographic_neighbor_distance"] = nearest_selected\n    if len(leverage) == len(selected_export):\n        selected_export["second_order_model_leverage"] = leverage\n\n    candidate_export = candidate_associations.copy()\n    if len(candidate_export):\n        cand_idx = candidate_export["candidate_analysis_index"].to_numpy(dtype=int)\n        candidate_export[config.ID_COLUMN] = source_df.iloc[analysis_to_source[cand_idx]][config.ID_COLUMN].to_numpy()\n        candidate_export[config.X_COLUMN] = coords[cand_idx, 0]\n        candidate_export[config.Y_COLUMN] = coords[cand_idx, 1]\n    candidate_pool_report = candidate_pool_summary_table(target_instances, candidate_pools, candidate_pool_distances, final_tolerances, config)\n\n    output_tables = {\n        "ESAP_RSSD_spatial_support_sequence.csv": support_sequence,\n        "ESAP_RSSD_ad_support_resolution.csv": ad_support_resolution,\n        "ESAP_RSSD_candidate_saturation.csv": candidate_saturation,\n        "ESAP_RSSD_optimizer_stability.csv": optimizer_stability,\n        "ESAP_RSSD_spatial_support_sites.csv": spatial_support_sites,\n        "ESAP_RSSD_field_coverage_distances.csv": field_coverage_distances,\n        "ESAP_RSSD_proxy_spatial_scale.csv": proxy_spatial_scale,\n        "ESAP_RSSD_selected_sites.csv": selected_export,\n        "ESAP_RSSD_candidate_sites.csv": candidate_export,\n        "ESAP_RSSD_candidate_pool_report.csv": candidate_pool_report,\n        "ESAP_RSSD_candidate_sequences.csv": candidate_sequence_records,\n        "ESAP_RSSD_target_instances.csv": target_instances,\n        "ESAP_RSSD_pca_standardization_validation.csv": pca_validation,\n        "ESAP_RSSD_feature_skewness.csv": skewness_summary,\n    }\n    for filename, table in output_tables.items():\n        table.to_csv(filename, index=False)\n\n    kmz_export_metadata = {"created": False, "reason": "source projected EPSG was not supplied"}\n    source_epsg = workflow_state.get("coordinate_epsg")\n    if source_epsg:\n        try:\n            kmz_export_metadata = export_selected_sites_kmz(selected_export, config, int(source_epsg))\n        except Exception as exc:\n            kmz_export_metadata = {"created": False, "reason": str(exc)}\n\n    total_runtime = time.perf_counter() - run_started\n    final_ad_summary = _optimizer_phase_summary(optimizer_stability, "final", "ad_reference")\n    final_hybrid_summary = _optimizer_phase_summary(optimizer_stability, "final", "hybrid")\n    support_stable = bool(support_opt_report.get("ad_support_resolution_stable", False))\n    candidate_status = str(support_opt_report.get("candidate_space_status", "unknown"))\n    candidate_saturated = bool(support_opt_report.get("candidate_space_saturated", False))\n    support_site_count = len(final_selected) - response_surface_count\n    support_sbad_reduction = float(core_sbad - final_sbad)\n    cache_stats = support_cache.snapshot()\n    support_opt_report["support_cache"] = cache_stats\n\n    run_summary = f"""# ESAP RSSD v2.10 run summary\n\n{int(config.N_SAMPLES)} calibration locations were requested. The selected design used {response_surface_count} PCA response-surface sites and {support_site_count} spatial support sites.\n\nFinal SBAD geographic support prefix: {int(final_B):,}. {\'Support resolution met the practical-equivalence stability rule.\' if support_stable else \'Support resolution reached the configured limit before practical-equivalence confirmation; review ESAP_RSSD_ad_support_resolution.csv.\'}\n\nFinal candidate K: {int(final_k)}. Candidate-space status: `{candidate_status}`; objective saturation = {candidate_saturated}.\n\nFull final optimizer starts: AD-reference {final_ad_summary[\'starts\']} starts (stable={final_ad_summary[\'initialization_stable\']}, stop={final_ad_summary[\'stop_reason\']}), hybrid {final_hybrid_summary[\'starts\']} starts (stable={final_hybrid_summary[\'initialization_stable\']}, stop={final_hybrid_summary[\'stop_reason\']}).\n\nSBAD_star = {support_opt_report[\'SBAD_star\']:.6g}; SBAD_limit = {support_opt_report[\'SBAD_limit\']:.6g}. The final hybrid response-surface core achieved SBAD = {core_sbad:.6g}, coverage ratio = {core_sbad / support_opt_report[\'SBAD_star\']:.6g}, exact geoMSD = {core_geo_msd:.6g}, and minimum separation = {core_min_sep:.6g}.\n\n{support_site_count} spatial support site(s) were added after the response-surface core, reducing final-design SBAD by {support_sbad_reduction:.6g} to {final_sbad:.6g}.\n\nThe proxy spatial-scale diagnostic status was `{spacing_diagnostic.get(\'status\')}`. It is diagnostic only and does not alter the hybrid optimizer.\n\nShared support-distance cache: {cache_stats[\'cache_hits\']:,} hits, {cache_stats[\'cache_misses\']:,} misses, {cache_stats[\'cache_evictions\']:,} evictions, peak {cache_stats[\'peak_cached_vectors\']:,} vectors / {cache_stats[\'peak_cache_mib\']:.2f} MiB.\n\nTotal runtime: {total_runtime:.1f} seconds.\n\nESAP 2 separates geographic support from sensor observation density. Adaptive diagnostics should vary one source of uncertainty at a time.\n"""\n    Path("ESAP_RSSD_run_summary.md").write_text(run_summary, encoding="utf-8")\n\n    metadata = {\n        "notebook_version": NOTEBOOK_VERSION,\n        "input_label": input_label,\n        "configuration": asdict(config),\n        "validation_report": validation_report,\n        "transformation_details": transformation_details,\n        "pca_mode": pca_mode,\n        "pca_explained_variance_ratio": np.asarray(pca_estimator.explained_variance_ratio_).tolist(),\n        "memory_report": memory_report,\n        "sample_budget": sample_budget_report,\n        "support_sequence": support_report,\n        "support_optimization": support_opt_report,\n        "candidate_prefilter": {"used": approximate_prefilter_used, **prefilter_report},\n        "candidate_sequence_report": sequence_report,\n        "candidate_screening": candidate_screen_meta,\n        "candidate_final_K": int(final_k),\n        "regression_design_diagnostics": regression_diagnostics,\n        "spacing_diagnostic": spacing_diagnostic,\n        "kmz_export": kmz_export_metadata,\n        "stage_runtime_seconds": stage_times,\n        "total_runtime_seconds": total_runtime,\n        "package_versions": package_versions(),\n    }\n    diagnostic_data = {\n        "coords": coords,\n        "design_scores": design_scores,\n        "eligible_indices": eligible_indices,\n        "pca_outlier": pca_outlier,\n        "target_array": target_array,\n        "selected_indices": final_selected,\n        "core_selected_indices": core_selected,\n        "ad_reference_indices": ad_reference_selected,\n        "original_features": original_features,\n        "feature_columns": list(config.FEATURE_COLUMNS),\n        "candidate_indices": np.unique(candidate_export["candidate_analysis_index"].to_numpy(dtype=int)) if len(candidate_export) else np.array([], dtype=int),\n    }\n    Path("ESAP_RSSD_run_metadata.json").write_text(json.dumps(json_ready(metadata), indent=2, allow_nan=False), encoding="utf-8")\n    stage_times["diagnostics_and_exports"] = time.perf_counter() - t0\n    print(f"Completed ESAP RSSD v2.10 in {total_runtime:.1f} seconds.")\n    print(run_summary)\n    return RSSDRunResult(config, selected_export, candidate_export, target_instances, support_sequence, ad_support_resolution, candidate_saturation, optimizer_stability, spatial_support_sites, field_coverage_distances, proxy_spatial_scale, pca_validation, skewness_summary, metadata, run_summary, diagnostic_data)\n\ndef duplicate_top_level_definition_report(source: str) -> Dict[str, Any]:\n    if not source:\n        return {"duplicate_top_level_definition_count": 0, "duplicate_top_level_definition_names": []}\n    tree = ast.parse(source)\n    names: List[str] = []\n    for node in tree.body:\n        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\n            names.append(node.name)\n    counts = {name: names.count(name) for name in sorted(set(names))}\n    duplicates = [name for name, count in counts.items() if count > 1]\n    return {"duplicate_top_level_definition_count": int(len(duplicates)), "duplicate_top_level_definition_names": duplicates}\n\n\n\ndef run_unit_validations(seed: int) -> None:\n    global V210_VALIDATION_RESULTS\n    V210_VALIDATION_RESULTS = {}\n    coords_small = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0], [3.0, 4.0], [8.0, 1.0]])\n    kd_value = float(np.exp(np.mean(np.log(selected_nearest_neighbor_distances_ckdtree_reference(coords_small)))))\n    vec_value = exact_geo_msd(coords_small)\n    dense = squareform(pdist(coords_small))\n    np.fill_diagonal(dense, np.inf)\n    brute_value = float(np.exp(np.mean(np.log(np.min(dense, axis=1)))))\n    np.testing.assert_allclose(vec_value, brute_value, rtol=1e-13, atol=1e-13)\n    np.testing.assert_allclose(vec_value, kd_value, rtol=1e-13, atol=1e-13)\n\n    rng = np.random.default_rng(seed)\n    selected_distance_cases = []\n    selected_reference_queries = 0\n    for m in [6, 10, 12, 20, 40]:\n        for _ in range(4):\n            c = rng.uniform(0.0, 1000.0, size=(m, 2))\n            v = selected_nearest_neighbor_distances(c)\n            d = squareform(pdist(c))\n            np.fill_diagonal(d, np.inf)\n            brute = np.min(d, axis=1)\n            kd_geo = float(np.exp(np.mean(np.log(selected_nearest_neighbor_distances_ckdtree_reference(c)))))\n            selected_reference_queries += 1\n            np.testing.assert_allclose(v, brute, rtol=1e-13, atol=1e-13)\n            np.testing.assert_allclose(exact_geo_msd(c), kd_geo, rtol=1e-13, atol=1e-13)\n            np.testing.assert_allclose(exact_geo_msd(c), float(np.exp(np.mean(np.log(brute)))), rtol=1e-13, atol=1e-13)\n            np.testing.assert_allclose(minimum_selected_separation(c), float(np.min(brute)), rtol=1e-13, atol=1e-13)\n            selected_distance_cases.append(m)\n    benchmark_summary = run_selected_set_development_benchmark(rng, RUN_DEVELOPMENT_BENCHMARKS)\n    if RUN_DEVELOPMENT_BENCHMARKS:\n        assert benchmark_summary["development_selected_set_benchmark_run"]\n        assert benchmark_summary["evaluations"] == DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS\n    else:\n        assert benchmark_summary["development_selected_set_benchmark_run"] is False\n        assert benchmark_summary["evaluations"] == 0\n        assert selected_reference_queries < DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS\n    development_probe = run_selected_set_development_benchmark(rng, True, evaluations=3)\n    assert development_probe["development_selected_set_benchmark_run"] and development_probe["evaluations"] == 3\n    V210_VALIDATION_RESULTS["selected_set_distance_equality"] = {\n        "randomized_selected_sets": int(len(selected_distance_cases)),\n        "selected_set_sizes": sorted(set(int(x) for x in selected_distance_cases)),\n        "ckdtree_reference_geoMSD_checks": int(selected_reference_queries),\n        "normal_initialization_avoids_10000_query_development_benchmark": bool(\n            (not RUN_DEVELOPMENT_BENCHMARKS) and selected_reference_queries < DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS\n        ),\n    }\n    V210_VALIDATION_RESULTS["selected_set_distance_benchmark"] = benchmark_summary\n    V210_VALIDATION_RESULTS["development_benchmark_flag_behavior"] = {\n        "normal_development_benchmark_run": bool(benchmark_summary["development_selected_set_benchmark_run"]),\n        "normal_development_benchmark_evaluations": int(benchmark_summary["evaluations"]),\n        "explicit_development_probe_run": bool(development_probe["development_selected_set_benchmark_run"]),\n        "explicit_development_probe_evaluations": int(development_probe["evaluations"]),\n        "default_evaluations_when_enabled": int(DEVELOPMENT_SELECTED_SET_BENCHMARK_EVALUATIONS),\n    }\n\n    latent = rng.normal(size=(4000, 3))\n    mixed = latent @ np.array([[1.0, 0.8, -0.4], [0.2, 1.4, 0.7], [-0.5, 0.1, 1.1]])\n    xs = StandardScaler().fit_transform(mixed)\n    estimator = PCA(n_components=3, svd_solver="full").fit(xs)\n    whitened = estimator.transform(xs) / np.sqrt(estimator.explained_variance_)\n    np.testing.assert_allclose(np.mean(whitened, axis=0), 0.0, atol=1e-12)\n    np.testing.assert_allclose(np.std(whitened, axis=0, ddof=1), 1.0, atol=1e-12)\n    np.testing.assert_allclose(np.corrcoef(whitened, rowvar=False), np.eye(3), atol=1e-12)\n\n    support = np.array([[0.0, 0.0], [5.0, 0.0], [0.0, 5.0], [5.0, 5.0], [2.5, 2.5]])\n    selected = np.array([[0.0, 0.0], [5.0, 5.0]])\n    brute_sbad = np.mean(np.min(np.sqrt(((support[:, None, :] - selected[None, :, :]) ** 2).sum(axis=2)), axis=1))\n    np.testing.assert_allclose(exact_sbad(support, selected), brute_sbad, rtol=1e-13, atol=1e-13)\n\n    coords_candidates = np.array([[0., 0.], [5., 0.], [0., 5.], [5., 5.], [2., 2.], [8., 8.]])\n    selected_idx = np.array([0, 3, 4])\n    cache = SupportDistanceCache(support, coords_candidates, 4)\n    state = SBADNearestState(cache, selected_idx, len(support))\n    for prefix in [3, len(support)]:\n        prefix_state = SBADNearestState(cache, selected_idx, prefix)\n        direct = exact_sbad(support[:prefix], coords_candidates[selected_idx])\n        np.testing.assert_allclose(prefix_state.sbad, direct, rtol=1e-7, atol=1e-7)\n    for pos in range(len(selected_idx)):\n        for candidate in range(len(coords_candidates)):\n            if candidate in set(np.delete(selected_idx, pos)):\n                continue\n            proposal = selected_idx.copy()\n            proposal[pos] = candidate\n            np.testing.assert_allclose(state.evaluate_swap(pos, candidate), exact_sbad(support, coords_candidates[proposal]), rtol=1e-7, atol=1e-7)\n\n    cfg_single = RSSDConfig()\n    cfg_single.CANDIDATE_SATURATION_MAX_K = 12\n    cfg_single.PC_CANDIDATE_TOLERANCE = 10.0\n    cfg_single.MAX_PC_CANDIDATE_TOLERANCE = 10.0\n    target = np.array([0.0, 0.0])\n    nhood_scores = rng.normal(0.0, 0.5, size=(60, 2))\n    nhood_coords = rng.uniform(0.0, 100.0, size=(60, 2))\n    nhood_indices = np.arange(60, dtype=int)\n    ti = pd.DataFrame({"target_instance_id": ["T001"], "base_target_id": ["center"], "target_type": ["center"], "target_PC1": [0.0], "target_PC2": [0.0]})\n    seqs, _, rep = build_nested_candidate_sequences(ti, nhood_indices, nhood_scores, nhood_coords, cfg_single)\n    ref = candidate_sequence_bruteforce_reference(target, nhood_indices, nhood_scores, nhood_coords, cfg_single, cfg_single.CANDIDATE_SATURATION_MAX_K)\n    key = tuple(np.round(target, 12))\n    np.testing.assert_array_equal(seqs[key]["indices"], ref["indices"])\n\n    cfg_prog = RSSDConfig()\n    cfg_prog.CANDIDATE_SATURATION_START_K = 3\n    cfg_prog.CANDIDATE_SATURATION_MAX_K = 24\n    cfg_prog.CANDIDATE_SATURATION_GROWTH_FACTOR = 2\n    cfg_prog.PC_CANDIDATE_TOLERANCE = 0.08\n    cfg_prog.CANDIDATE_TOLERANCE_GROWTH = 2.0\n    cfg_prog.MAX_PC_CANDIDATE_TOLERANCE = 1.5\n    shell_scores = np.vstack([\n        rng.normal(0.0, 0.02, size=(3, 2)),\n        rng.normal(0.0, 0.16, size=(5, 2)),\n        rng.normal(0.0, 0.35, size=(8, 2)),\n        rng.normal(0.0, 0.75, size=(16, 2)),\n    ])\n    shell_coords = rng.uniform(0.0, 500.0, size=(len(shell_scores), 2))\n    shell_indices = np.arange(len(shell_scores), dtype=int)\n    seqs2, _, rep2 = build_nested_candidate_sequences(ti, shell_indices, shell_scores, shell_coords, cfg_prog)\n    ref2 = candidate_sequence_bruteforce_reference(target, shell_indices, shell_scores, shell_coords, cfg_prog, cfg_prog.CANDIDATE_SATURATION_MAX_K)\n    np.testing.assert_array_equal(seqs2[key]["indices"], ref2["indices"])\n    for k in [3, 6, 12, 24]:\n        prefix = seqs2[key]["indices"][:min(k, len(seqs2[key]["indices"]))]\n        np.testing.assert_array_equal(prefix, seqs2[key]["indices"][:len(prefix)])\n    assert np.all(np.diff(seqs2[key]["expansion_stages"]) >= 0)\n\n    cfg_bench = RSSDConfig()\n    cfg_bench.CANDIDATE_SATURATION_MAX_K = 64\n    cfg_bench.PC_CANDIDATE_TOLERANCE = 0.30\n    cfg_bench.MAX_PC_CANDIDATE_TOLERANCE = 1.20\n    cfg_bench.CANDIDATE_TOLERANCE_GROWTH = 2.0\n    bench_n = 1500\n    bench_scores = rng.normal(0.0, 0.5, size=(bench_n, 2))\n    bench_coords2 = rng.uniform(0.0, 1000.0, size=(bench_n, 2))\n    bench_indices = np.arange(bench_n, dtype=int)\n    t0 = time.perf_counter()\n    ref_bench = candidate_sequence_bruteforce_reference(target, bench_indices, bench_scores, bench_coords2, cfg_bench, cfg_bench.CANDIDATE_SATURATION_MAX_K)\n    old_candidate_elapsed = time.perf_counter() - t0\n    t0 = time.perf_counter()\n    seq_bench, _, _ = build_nested_candidate_sequences(ti, bench_indices, bench_scores, bench_coords2, cfg_bench)\n    new_candidate_elapsed = time.perf_counter() - t0\n    candidate_equal = bool(np.array_equal(seq_bench[key]["indices"], ref_bench["indices"]))\n    assert candidate_equal\n    V210_VALIDATION_RESULTS["candidate_sequence_benchmark"] = {\n        "neighborhood_size": int(bench_n),\n        "Kmax": int(cfg_bench.CANDIDATE_SATURATION_MAX_K),\n        "number_of_tolerance_expansions": int(np.max(seq_bench[key]["expansion_stages"])) if len(seq_bench[key]["expansion_stages"]) else 0,\n        "old_recompute_seconds": float(old_candidate_elapsed),\n        "new_incremental_seconds": float(new_candidate_elapsed),\n        "speedup_ratio": float(old_candidate_elapsed / max(new_candidate_elapsed, 1e-12)),\n        "exact_candidate_sequence_equality": candidate_equal,\n    }\n\n    pools = [np.array([0, 1, 2]), np.array([2, 3, 4]), np.array([1, 4, 5])]\n    dists = [np.array([0.1, 0.2, 0.3]), np.array([0.2, 0.1, 0.4]), np.array([0.3, 0.2, 0.1])]\n    minimum = assignment_from_costs(pools, dists)\n    bank = OptimizationStartBank(minimum, pools, dists, seed, 3, "ad_reference", 8)\n    assert bank.base_random_keys(5) == bank.base_random_keys(5)\n    warm = np.array([1, 2, 5])\n    before = bank.base_random_keys(5)\n    _ = bank.starts(5, warm_starts=[warm])\n    after = bank.base_random_keys(5)\n    assert before == after\n    base_start = np.asarray(bank.base_starts[1], dtype=int)\n    perm_5000 = coordinate_exchange_permutation_preview(seed, "ad_reference", 3, base_start, len(base_start), 5)\n    perm_40000 = coordinate_exchange_permutation_preview(seed, "ad_reference", 3, base_start, len(base_start), 5)\n    assert perm_5000 == perm_40000\n    assert perm_5000 == coordinate_exchange_permutation_preview(seed, "ad_reference", 3, base_start, len(base_start), 5)\n    assert coordinate_exchange_start_seed(seed, "ad_reference", 3, base_start) == coordinate_exchange_start_seed(seed, "ad_reference", 3, base_start.copy())\n    V210_VALIDATION_RESULTS["per_start_rng"] = {"same_assignment_same_schedule": True, "warm_start_insertion_independent": True, "support_size_independent": True}\n\n    support_cfg = RSSDConfig()\n    support_cfg.SUPPORT_GAP_ANCHORS = 3\n    support_cfg.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR = 3\n    support_coords = np.array([[0., 0.], [10., 0.], [0., 10.], [10., 10.], [5., 5.]])\n    coords_support_test = np.array([[0., 0.], [10., 10.], [0., 10.], [10., 0.], [5., 0.], [5., 10.], [8., 5.]])\n    core = np.array([0, 1])\n    cand_idx = np.arange(len(coords_support_test), dtype=int)\n    cache2 = SupportDistanceCache(support_coords, coords_support_test, 4)\n    initial_state = SBADNearestState(cache2, core, len(support_coords))\n    final_sel, support_rows, coverage = add_spatial_support_sites(core, 1, support_coords, cand_idx, coords_support_test, support_cfg, cache=cache2, support_prefix_size=len(support_coords))\n    assert len(support_rows) == 1\n    row = support_rows.iloc[0]\n    bounded = np.array([int(x) for x in str(row["bounded_candidate_analysis_indices"]).split(";") if str(x)], dtype=int)\n    assert len(bounded) >= 1\n    brute = []\n    for cand in bounded:\n        proposal = np.array([*core.tolist(), int(cand)], dtype=int)\n        brute_s = exact_sbad(support_coords, coords_support_test[proposal])\n        brute_min = minimum_selected_separation(coords_support_test[proposal])\n        brute.append((brute_s, -brute_min, int(cand)))\n    brute.sort()\n    assert int(row["analysis_index"]) == brute[0][2]\n    assert float(row["SBAD_after_addition"]) <= float(row["SBAD_before_addition"]) + 1e-12\n    assert int(row["unique_support_candidates_evaluated"]) >= 1\n\n    rows_confirm, meta_confirm = support_confirmation_state_machine_preview(\n        [5000, 10000, 20000, 40000], [False, True, True, True], 2\n    )\n    assert [r["support_size"] for r in rows_confirm] == [5000, 10000, 20000, 40000]\n    assert rows_confirm[1]["stable_stage"] and rows_confirm[2]["stable_stage"]\n    assert not rows_confirm[1]["confirmation_stage"] and not rows_confirm[2]["confirmation_stage"]\n    assert rows_confirm[3]["confirmation_stage"] and rows_confirm[3]["stable_stage"]\n    assert meta_confirm["ad_support_resolution_stable"] and meta_confirm["final_support_size"] == 40000\n\n    rows_reset, meta_reset = support_confirmation_state_machine_preview(\n        [5000, 10000, 20000, 40000, 80000], [False, True, True, False, True], 2\n    )\n    assert rows_reset[3]["confirmation_stage"] and not rows_reset[3]["stable_stage"]\n    assert rows_reset[3]["support_resolution_status"] == "confirmation_failed_reset"\n    assert rows_reset[4]["confirmation_stage"] is False\n    assert meta_reset["ad_support_resolution_stable"] is False\n\n    rows_max, meta_max = support_confirmation_state_machine_preview(\n        [5000, 10000, 20000], [False, True, True], 2\n    )\n    assert rows_max[-1]["support_resolution_status"] == "stable_at_max_without_separate_confirmation"\n    assert rows_max[-1]["stable_stage"] and not rows_max[-1]["confirmation_stage"]\n    assert meta_max["ad_support_resolution_stable"] is False\n\n    np.testing.assert_allclose(focused_ad_refinement_improvement(100.0, 99.0), 0.01, rtol=1e-13, atol=1e-13)\n    previous_stage_sbad_star = 95.0\n    assert previous_stage_sbad_star == 95.0\n    np.testing.assert_allclose(focused_ad_refinement_improvement(100.0, 99.0), 0.01, rtol=1e-13, atol=1e-13)\n    assert focused_ad_refinement_improvement(100.0, 100.5) == 0.0\n\n    engine_source = globals().get("_ESAP_RSSD_ENGINE_SOURCE", "")\n    screen_start = engine_source.index("def screen_support_resolution")\n    screen_end = engine_source.find("\\ndef ", screen_start + 1)\n    screen_source = engine_source[screen_start:screen_end if screen_end != -1 else len(engine_source)]\n    assert \'optimization_phase="final"\' not in screen_source\n    assert screen_source.count(\'optimization_phase="support_screen"\') == 2\n    V210_VALIDATION_RESULTS["support_resolution_confirmation"] = {\n        "separate_confirmation_prefix_evaluated": bool(rows_confirm[-1]["support_size"] == 40000),\n        "confirmation_stage_only_on_confirmation_row": bool(\n            rows_confirm[3]["confirmation_stage"] and not rows_confirm[1]["confirmation_stage"] and not rows_confirm[2]["confirmation_stage"]\n        ),\n        "confirmation_failure_resets_provisional_stability": bool(\n            rows_reset[3]["support_resolution_status"] == "confirmation_failed_reset" and not rows_reset[4]["confirmation_stage"]\n        ),\n        "final_support_size_equals_confirmation_prefix": bool(meta_confirm["final_support_size"] == 40000),\n        "stable_at_max_without_separate_confirmation_status": bool(rows_max[-1]["support_resolution_status"] == "stable_at_max_without_separate_confirmation"),\n        "support_screen_has_no_final_optimizer_phase": True,\n    }\n    V210_VALIDATION_RESULTS["focused_ad_refinement"] = {\n        "pre_100_post_99_improvement": float(focused_ad_refinement_improvement(100.0, 99.0)),\n        "previous_stage_95_pre_100_post_99_improvement": float(focused_ad_refinement_improvement(100.0, 99.0)),\n        "no_better_design_improvement": float(focused_ad_refinement_improvement(100.0, 100.5)),\n    }\n\n    base = make_base_ccd(2, 1.0, 2)\n    tmp = RSSDConfig(N_SAMPLES=12, N_DESIGN_COMPONENTS=2, SAMPLE_BUDGET_MODE="rssd_with_support")\n    inst, _, rep_budget = allocate_sample_budget(base, 12, tmp, 2)\n    assert len(inst) == 10 and rep_budget["spatial_support_sites"] == 2\n    tmp.N_SAMPLES = 20\n    inst, _, rep_budget = allocate_sample_budget(base, 20, tmp, 2)\n    assert len(inst) == 18 and rep_budget["spatial_support_sites"] == 2\n    tmp.N_SAMPLES = 10\n    inst, _, rep_budget = allocate_sample_budget(base, 10, tmp, 2)\n    assert len(inst) == 10 and rep_budget["spatial_support_sites"] == 0\n\n    support_seq, report = build_spatially_balanced_support_sequence(rng.uniform(size=(250, 2)), np.arange(250), 64)\n    assert len(support_seq) <= 64 and support_seq["support_rank"].is_monotonic_increasing and report["uses_only_xy"] and not report["uses_pc_scores"]\n\n    duplicate_info = duplicate_top_level_definition_report(globals().get("_ESAP_RSSD_ENGINE_SOURCE", ""))\n    assert duplicate_info["duplicate_top_level_definition_count"] == 0\n    V210_VALIDATION_RESULTS["duplicate_top_level_definitions"] = duplicate_info\n    print("Unit validations passed: exact vectorized selected-set distances, PCA standardization, exact SBAD/cache, incremental candidate sequences, support-resolution confirmation, true focused refinement, per-start RNG, support-site exact ranking, development benchmark flag, and duplicate-definition AST check.")\n\n\n_ESAP_RSSD_ENGINE_SOURCE = None\nvalidate_config(cfg)\n_ESAP_RSSD_ENGINE_SOURCE = \'\'.join(globals().get(\'_ESAP_RSSD_ENGINE_SOURCE_LINES\', [])) if \'_ESAP_RSSD_ENGINE_SOURCE_LINES\' in globals() else _ESAP_RSSD_ENGINE_SOURCE\nrun_unit_validations(cfg.RANDOM_SEED)\nprint("ESAP RSSD v2.10 engine initialized successfully.")\n\n'
validate_config(cfg)
_ESAP_RSSD_ENGINE_SOURCE = ''.join(globals().get('_ESAP_RSSD_ENGINE_SOURCE_LINES', [])) if '_ESAP_RSSD_ENGINE_SOURCE_LINES' in globals() else _ESAP_RSSD_ENGINE_SOURCE
run_unit_validations(cfg.RANDOM_SEED)
print("ESAP RSSD v2.10 engine initialized successfully.")



## 1. Guided Google Colab workflow

Run the following cells in order. Each stage presents buttons, selectors, sliders, previews, or text fields. No code editing is required. After the final **Apply optimization settings** button reports that the workflow is ready, continue to **Run the scalable RSSD analysis**.


### 1.1 Upload the survey file

Choose one complete CSV or Excel survey file. Running the next cell opens Colab's native uploader, including its upload-progress and Cancel Upload controls. The same cell can instead create the synthetic demonstration or read an existing `/content/...` path.


In [ ]:
# @title 1. Load one survey file or use the synthetic demonstration { display-mode: "form" }
DATA_SOURCE = "Upload one CSV or Excel file"  # @param ["Upload one CSV or Excel file", "Use synthetic demonstration", "Read an existing Colab file path"]
EXISTING_COLAB_FILE_PATH = ""  # @param {type:"string"}

import io

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("ipywidgets is required for the later interactive controls and is included in Google Colab.") from exc

try:
    from google.colab import output as colab_output  # type: ignore

    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def responsive_widget_grid(
    children: Sequence[Any], min_width: int = 250, gap: str = "8px 12px"
) -> widgets.GridBox:
    """Wrap controls automatically and preserve complete description labels in narrow Colab outputs."""
    prepared = []
    for child in children:
        try:
            child.style.description_width = "initial"
        except (AttributeError, TypeError):
            pass
        prepared.append(child)
    return widgets.GridBox(
        children=tuple(prepared),
        layout=widgets.Layout(
            width="100%",
            grid_template_columns=f"repeat(auto-fit, minmax({min_width}px, 1fr))",
            grid_gap=gap,
            align_items="center",
        ),
    )


def use_full_description(widget: Any, width: Optional[str] = None) -> Any:
    """Apply natural label width and an optional full control width."""
    try:
        widget.style.description_width = "initial"
    except (AttributeError, TypeError):
        pass
    if width is not None:
        widget.layout.width = width
    return widget


display(
    widgets.HTML(
        "<style>.widget-label{max-width:none !important; white-space:normal !important;}"
        ".widget-inline-hbox{align-items:center !important;}</style>"
    )
)

SURVEY_DF: Optional[pd.DataFrame] = None
SURVEY_INPUT_LABEL: Optional[str] = None
WORKFLOW_CONFIG_APPLIED = False
RANGE_FILTER_AUDIT: List[Dict[str, Any]] = []
ROW_RANGE_FILTER_APPLIED = False


def _read_survey_bytes(name: str, content: bytes) -> pd.DataFrame:
    suffix = Path(name).suffix.lower()
    stream = io.BytesIO(content)
    if suffix == ".csv":
        return pd.read_csv(stream)
    if suffix in {".xls", ".xlsx", ".xlsm"}:
        return pd.read_excel(stream)
    raise ValueError(f"Unsupported file extension {suffix!r}; use CSV, XLS, XLSX, or XLSM.")


def _read_survey_path(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"No file exists at {path}.")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xls", ".xlsx", ".xlsm"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file extension {suffix!r}; use CSV, XLS, XLSX, or XLSM.")


if DATA_SOURCE == "Use synthetic demonstration":
    print("Creating the deterministic synthetic demonstration...")
    SURVEY_DF = make_synthetic_survey(cfg.SYNTHETIC_ROWS, cfg.RANDOM_SEED)
    SURVEY_INPUT_LABEL = "synthetic_demo"
    print(
        f"SYNTHETIC DATA READY: {len(SURVEY_DF):,} rows × "
        f"{SURVEY_DF.shape[1]} columns."
    )

elif DATA_SOURCE == "Read an existing Colab file path":
    if not EXISTING_COLAB_FILE_PATH.strip():
        raise ValueError("Enter an existing Colab file path, such as /content/my_survey.csv.")
    print(f"[1/3] Reading {EXISTING_COLAB_FILE_PATH}...")
    SURVEY_DF = _read_survey_path(EXISTING_COLAB_FILE_PATH.strip())
    SURVEY_INPUT_LABEL = str(Path(EXISTING_COLAB_FILE_PATH.strip()))
    print("[2/3] File parsed. Validating rows and columns...")
    if SURVEY_DF.empty or SURVEY_DF.shape[1] == 0:
        raise ValueError("The parsed file contains no data rows or columns.")
    print(
        f"[3/3] FILE LOAD SUCCESSFUL: {SURVEY_INPUT_LABEL}; "
        f"{len(SURVEY_DF):,} rows × {SURVEY_DF.shape[1]} columns."
    )

else:
    try:
        from google.colab import files as colab_files  # type: ignore
    except ImportError as exc:
        raise RuntimeError(
            "Native upload is available in Google Colab. Outside Colab, choose "
            "'Read an existing Colab file path' and provide a local path."
        ) from exc

    print("[1/3] Choose exactly one CSV or Excel file in the native Colab dialog below.")
    print("Colab displays its own byte-level upload indicator and Cancel Upload button.")
    native_uploaded = colab_files.upload()
    if not native_uploaded:
        raise RuntimeError("Upload was cancelled or no file was received. Rerun this cell to try again.")
    if len(native_uploaded) != 1:
        raise ValueError("Upload exactly one complete survey file, then rerun this cell.")
    uploaded_name = next(iter(native_uploaded))
    uploaded_content = bytes(native_uploaded[uploaded_name])
    print(
        f"[2/3] TRANSFER COMPLETE: {uploaded_name} "
        f"({len(uploaded_content) / 1024**2:,.2f} MiB). Parsing the dataframe..."
    )
    SURVEY_DF = _read_survey_bytes(uploaded_name, uploaded_content)
    if SURVEY_DF.empty or SURVEY_DF.shape[1] == 0:
        raise ValueError("The uploaded file parsed successfully but contains no data rows or columns.")
    SURVEY_INPUT_LABEL = f"uploaded:{uploaded_name}"
    print(
        f"[3/3] UPLOAD AND FILE LOAD SUCCESSFUL: {uploaded_name}; "
        f"{len(SURVEY_DF):,} rows × {SURVEY_DF.shape[1]} columns."
    )

if SURVEY_DF is not None:
    WORKFLOW_CONFIG_APPLIED = False
    display(SURVEY_DF.head(8))
    display(
        pd.DataFrame(
            {
                "dtype": SURVEY_DF.dtypes.astype(str),
                "missing": SURVEY_DF.isna().sum(),
                "unique": SURVEY_DF.nunique(dropna=True),
            }
        )
    )


### 1.2 Enter existing or required target locations (optional)

Use **+ Add location** for sites already known or previously sampled. Enter X/longitude and Y/latitude in the same coordinate system as the survey input. Later, choose whether these locations are only evaluated/overlaid or forced into the final design. Forced locations are matched to unique eligible survey observations and locked to the statistically closest response-surface targets.


In [ ]:
# @title 2. Add or remove existing target locations { display-mode: "form" }
preferred_rows: List[Tuple[widgets.Text, widgets.FloatText, widgets.FloatText, widgets.HBox]] = []
preferred_box = widgets.VBox()
add_preferred_button = widgets.Button(description="+ Add location", button_style="info")
clear_preferred_button = widgets.Button(description="Clear all", button_style="warning")


def _refresh_preferred_box() -> None:
    preferred_box.children = tuple(row[3] for row in preferred_rows)


def _remove_preferred(row: Tuple[Any, Any, Any, Any]) -> None:
    if row in preferred_rows:
        preferred_rows.remove(row)
    _refresh_preferred_box()


def _add_preferred(_: Optional[widgets.Button] = None) -> None:
    identifier = widgets.Text(value=f"existing_{len(preferred_rows) + 1}", description="Location ID:")
    x_value = widgets.FloatText(description="X / longitude:", style={"description_width": "initial"})
    y_value = widgets.FloatText(description="Y / latitude:", style={"description_width": "initial"})
    remove = widgets.Button(description="X", layout=widgets.Layout(width="38px"))
    container = widgets.VBox([
        responsive_widget_grid([identifier, x_value, y_value], min_width=220),
        remove,
    ])
    row = (identifier, x_value, y_value, container)
    remove.on_click(lambda _, current=row: _remove_preferred(current))
    preferred_rows.append(row)
    _refresh_preferred_box()


def collect_preferred_raw() -> pd.DataFrame:
    records = []
    for identifier, x_value, y_value, _ in preferred_rows:
        records.append(
            {
                "preferred_location_id": identifier.value or f"existing_{len(records) + 1}",
                "input_x": float(x_value.value),
                "input_y": float(y_value.value),
            }
        )
    return pd.DataFrame(records, columns=["preferred_location_id", "input_x", "input_y"])


add_preferred_button.on_click(_add_preferred)
clear_preferred_button.on_click(lambda _: (preferred_rows.clear(), _refresh_preferred_box()))
display(preferred_box, widgets.HBox([add_preferred_button, clear_preferred_button]))
print("No existing locations are required. Leave this list empty to skip them.")


### 1.3 Preview the first data file

This preview shows the first rows, data types, missing counts, and unique counts for the single RSSD survey file loaded in step 1.


In [ ]:
# @title 3. View the first data file and its columns { display-mode: "form" }
preview_rows = use_full_description(widgets.IntSlider(value=8, min=3, max=30, description="Preview rows:"))
preview_button = widgets.Button(description="Refresh preview", button_style="info")
preview_output = widgets.Output()


def _preview_files(_: Optional[widgets.Button] = None) -> None:
    with preview_output:
        preview_output.clear_output(wait=True)
        if SURVEY_DF is None:
            print("Upload/load the first survey file in step 1.")
            return
        print("FIRST / RSSD SURVEY FILE")
        display(SURVEY_DF.head(preview_rows.value))
        display(
            pd.DataFrame(
                {
                    "dtype": SURVEY_DF.dtypes.astype(str),
                    "missing": SURVEY_DF.isna().sum(),
                    "unique": SURVEY_DF.nunique(dropna=True),
                }
            )
        )


preview_button.on_click(_preview_files)
display(widgets.HBox([preview_rows, preview_button]), preview_output)
_preview_files()


### 1.4 Apply scientific validity ranges before outlier screening (optional)

Use this stage to remove values that are physically invalid or scientifically irrelevant before PCA and statistical outlier screening—for example NDVI below 0.1, negative EMI readings caused by metal, or sensor saturation above a known maximum. Add one or more rules; all rules are combined with AND logic. Preview the effect first, then explicitly apply it.

Applying a filter permanently drops rows from the working dataframe for this run. The uploaded source file is never modified. To undo an applied filter, rerun the upload cell. Every rule and row count is stored in the run metadata and ZIP archive.


In [ ]:
# @title 4. Define, preview, and apply raw-data range filters { display-mode: "form" }
range_filter_rows: List[Dict[str, Any]] = []
range_filter_box = widgets.VBox()
add_range_filter_button = widgets.Button(description="+ Add range rule", button_style="info")
clear_range_filters_button = widgets.Button(description="Clear rules", button_style="warning")
preview_range_filter_button = widgets.Button(description="Preview rows to drop", button_style="info")
apply_range_filter_button = widgets.Button(description="Apply filters and drop rows", button_style="danger")
drop_nonfinite_control = widgets.Checkbox(
    value=True,
    description="Drop missing/nonfinite values in filtered columns",
    indent=False,
)
filter_plot_max_control = widgets.IntSlider(
    value=50000,
    min=1000,
    max=100000,
    step=1000,
    description="Plot sample:",
    style={"description_width": "initial"},
)
range_filter_output = widgets.Output()


def _numeric_filter_columns() -> List[str]:
    if SURVEY_DF is None:
        return []
    numeric = [str(column) for column in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    # Also allow columns that are numeric strings throughout a small validation sample.
    for column in SURVEY_DF.columns:
        name = str(column)
        if name in numeric:
            continue
        sample = SURVEY_DF[column].dropna().head(1000)
        if len(sample) and pd.to_numeric(sample, errors="coerce").notna().all():
            numeric.append(name)
    return numeric


def _refresh_range_filter_box() -> None:
    range_filter_box.children = tuple(rule["container"] for rule in range_filter_rows)


def _remove_range_rule(rule: Dict[str, Any]) -> None:
    if rule in range_filter_rows:
        range_filter_rows.remove(rule)
    _refresh_range_filter_box()


def _add_range_rule(_: Optional[widgets.Button] = None) -> None:
    columns = _numeric_filter_columns()
    if not columns:
        with range_filter_output:
            range_filter_output.clear_output(wait=True)
            print("Load the survey dataframe before adding a range rule.")
        return
    preferred = next(
        (column for column in columns if column.lower() in {"ndvi", "ndre", "eca", "ec", "emi"}),
        columns[0],
    )
    column_control = widgets.Dropdown(
        options=columns,
        value=preferred,
        description="Column:",
        layout=widgets.Layout(width="260px"),
    )
    lower_control = widgets.Text(
        value="",
        placeholder="No lower bound",
        description="Lower:",
        layout=widgets.Layout(width="210px"),
    )
    upper_control = widgets.Text(
        value="",
        placeholder="No upper bound",
        description="Upper:",
        layout=widgets.Layout(width="210px"),
    )
    inclusive_control = widgets.Checkbox(value=True, description="Inclusive", indent=False)
    remove_control = widgets.Button(description="X", layout=widgets.Layout(width="38px"))
    container = widgets.VBox(
        [
            responsive_widget_grid(
                [column_control, lower_control, upper_control, inclusive_control], min_width=190
            ),
            remove_control,
        ]
    )
    rule = {
        "column": column_control,
        "lower": lower_control,
        "upper": upper_control,
        "inclusive": inclusive_control,
        "container": container,
    }
    remove_control.on_click(lambda _, current=rule: _remove_range_rule(current))
    range_filter_rows.append(rule)
    _refresh_range_filter_box()


def _parse_optional_bound(text_value: str, label: str) -> Optional[float]:
    stripped = text_value.strip()
    if stripped == "":
        return None
    value = float(stripped)
    if not np.isfinite(value):
        raise ValueError(f"{label} bound must be finite or blank.")
    return value


def _calculate_range_filter() -> Tuple[np.ndarray, List[Dict[str, Any]]]:
    if SURVEY_DF is None:
        raise ValueError("Load the survey dataframe first.")
    if not range_filter_rows:
        return np.ones(len(SURVEY_DF), dtype=bool), []
    cumulative_mask = np.ones(len(SURVEY_DF), dtype=bool)
    records: List[Dict[str, Any]] = []
    for position, rule in enumerate(range_filter_rows, start=1):
        column = str(rule["column"].value)
        lower = _parse_optional_bound(rule["lower"].value, "Lower")
        upper = _parse_optional_bound(rule["upper"].value, "Upper")
        inclusive = bool(rule["inclusive"].value)
        if lower is None and upper is None and not drop_nonfinite_control.value:
            raise ValueError(f"Rule {position} has no bound and does not filter nonfinite values.")
        if lower is not None and upper is not None and lower > upper:
            raise ValueError(f"Rule {position}: lower bound exceeds upper bound.")

        numeric = pd.to_numeric(SURVEY_DF[column], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(numeric)
        rule_mask = finite.copy() if drop_nonfinite_control.value else np.ones(len(numeric), dtype=bool)
        if lower is not None:
            comparison = numeric >= lower if inclusive else numeric > lower
            rule_mask &= comparison if drop_nonfinite_control.value else (~finite | comparison)
        if upper is not None:
            comparison = numeric <= upper if inclusive else numeric < upper
            rule_mask &= comparison if drop_nonfinite_control.value else (~finite | comparison)
        cumulative_before = int(cumulative_mask.sum())
        cumulative_mask &= rule_mask
        records.append(
            {
                "rule_number": position,
                "column": column,
                "lower_bound": lower,
                "upper_bound": upper,
                "inclusive": inclusive,
                "drop_nonfinite": bool(drop_nonfinite_control.value),
                "finite_values": int(finite.sum()),
                "rows_passing_rule_alone": int(rule_mask.sum()),
                "cumulative_rows_before_rule": cumulative_before,
                "cumulative_rows_after_rule": int(cumulative_mask.sum()),
            }
        )
    return cumulative_mask, records


def _preview_range_filters(_: Optional[widgets.Button] = None) -> None:
    with range_filter_output:
        range_filter_output.clear_output(wait=True)
        try:
            mask, records = _calculate_range_filter()
            total = len(mask)
            kept = int(mask.sum())
            dropped = total - kept
            display(
                pd.DataFrame(
                    {
                        "rows_before": [total],
                        "rows_retained": [kept],
                        "rows_dropped": [dropped],
                        "percent_retained": [100.0 * kept / total if total else np.nan],
                    }
                )
            )
            if records:
                display(pd.DataFrame(records))
                n_rules = len(records)
                ncols = min(2, n_rules)
                nrows = math.ceil(n_rules / ncols)
                fig, axes = plt.subplots(
                    nrows, ncols, figsize=(6 * ncols, 3.8 * nrows), squeeze=False
                )
                rng = np.random.default_rng(cfg.RANDOM_SEED)
                sample_n = min(filter_plot_max_control.value, total)
                sample_index = np.sort(rng.choice(total, sample_n, replace=False))
                for j, record in enumerate(records):
                    values = pd.to_numeric(
                        SURVEY_DF[record["column"]], errors="coerce"
                    ).to_numpy(dtype=float)
                    sampled = values[sample_index]
                    axes.flat[j].hist(sampled[np.isfinite(sampled)], bins=50, color=cfg.SURVEY_COLOR)
                    if record["lower_bound"] is not None:
                        axes.flat[j].axvline(
                            record["lower_bound"], color="C3", linestyle="--", label="Lower bound"
                        )
                    if record["upper_bound"] is not None:
                        axes.flat[j].axvline(
                            record["upper_bound"], color="C1", linestyle="--", label="Upper bound"
                        )
                    axes.flat[j].set_title(f"Rule {j + 1}: {record['column']}")
                    axes.flat[j].legend()
                for ax in axes.flat[n_rules:]:
                    ax.set_visible(False)
                fig.suptitle("Raw-data range-filter preview")
                fig.tight_layout()
                plt.show()
            if kept == 0:
                print("WARNING: current rules would remove every row. Adjust bounds before applying.")
            else:
                print("Preview only—no rows have been removed yet.")
        except Exception as exc:
            print(f"Range-filter preview error: {type(exc).__name__}: {exc}")


def _apply_range_filters(_: widgets.Button) -> None:
    global SURVEY_DF, RANGE_FILTER_AUDIT, ROW_RANGE_FILTER_APPLIED, WORKFLOW_CONFIG_APPLIED
    with range_filter_output:
        range_filter_output.clear_output(wait=True)
        try:
            mask, records = _calculate_range_filter()
            total = len(mask)
            kept = int(mask.sum())
            dropped = total - kept
            if kept == 0:
                raise ValueError("Current rules would remove every row; filters were not applied.")
            application = {
                "application_number": len(RANGE_FILTER_AUDIT) + 1,
                "rows_before": total,
                "rows_retained": kept,
                "rows_dropped": dropped,
                "percent_retained": 100.0 * kept / total,
                "rules": records,
            }
            # Create only the retained working dataframe; the source file remains unchanged on disk.
            SURVEY_DF = SURVEY_DF.loc[mask].reset_index(drop=True)
            RANGE_FILTER_AUDIT.append(application)
            ROW_RANGE_FILTER_APPLIED = True
            WORKFLOW_CONFIG_APPLIED = False
            gc.collect()
            print(
                f"RANGE FILTER APPLIED: retained {kept:,} of {total:,} rows; "
                f"dropped {dropped:,} ({100.0 * dropped / total:.2f}%)."
            )
            print("This working-data change is recorded. Rerun the upload cell to undo it.")
            display(pd.DataFrame(records))
            display(SURVEY_DF.head(8))
        except Exception as exc:
            print(f"Range-filter apply error: {type(exc).__name__}: {exc}")


def _clear_range_rules(_: widgets.Button) -> None:
    range_filter_rows.clear()
    _refresh_range_filter_box()
    with range_filter_output:
        range_filter_output.clear_output(wait=True)
        print("Filter rules cleared. Previously applied row drops are not undone; rerun upload to reset.")


add_range_filter_button.on_click(_add_range_rule)
clear_range_filters_button.on_click(_clear_range_rules)
preview_range_filter_button.on_click(_preview_range_filters)
apply_range_filter_button.on_click(_apply_range_filters)

display(
    widgets.VBox(
        [
            range_filter_box,
            widgets.HBox([add_range_filter_button, clear_range_filters_button]),
            responsive_widget_grid([drop_nonfinite_control, filter_plot_max_control]),
            responsive_widget_grid([preview_range_filter_button, apply_range_filter_button]),
            range_filter_output,
        ]
    )
)
print("No range filter is applied unless you click 'Apply filters and drop rows'.")


### 1.5 Coordinate system and decimal-degree to UTM conversion

Select the source X/longitude and Y/latitude columns. If they are decimal degrees, choose automatic UTM detection or a manual zone and hemisphere, then convert. Original longitude/latitude columns are retained and new `_rssd_easting` and `_rssd_northing` columns are added. No zone is hard-coded. If the input is already projected, enter its EPSG code so the final selected sites can be transformed correctly for the Google Earth KMZ export.


In [ ]:
# @title 5. Select projected coordinates or convert longitude/latitude to UTM { display-mode: "form" }
coordinate_mode = widgets.RadioButtons(
    options=[
        ("Already projected in linear units", "projected"),
        ("Decimal degrees (longitude / latitude)", "decimal_degrees"),
    ],
    value="projected",
    description="Input coordinates:",
    style={"description_width": "initial"},
)
coordinate_x_source = use_full_description(widgets.Dropdown(description="X / longitude:"))
coordinate_y_source = use_full_description(widgets.Dropdown(description="Y / latitude:"))
utm_detection = widgets.RadioButtons(
    options=[("Automatically detect from data centroid", "auto"), ("Choose manually", "manual")],
    value="auto",
    description="UTM selection:",
    style={"description_width": "initial"},
)
utm_zone = use_full_description(widgets.Dropdown(options=list(range(1, 61)), value=13, description="UTM zone:"))
utm_hemisphere = use_full_description(widgets.Dropdown(options=["N", "S"], value="N", description="Hemisphere:"))
projected_epsg_control = use_full_description(
    widgets.IntText(
        value=0,
        description="Projected EPSG for Google Earth/KMZ (0 = unknown):",
        style={"description_width": "initial"},
    ),
    width="100%",
)
apply_coordinate_system = widgets.Button(description="Apply / convert coordinates", button_style="success")
coordinate_output = widgets.Output()
COORDINATE_MODE_APPLIED: Optional[str] = None
COORDINATE_EPSG: Optional[int] = None


def _guess_column(columns: Sequence[str], candidates: Sequence[str], fallback: str) -> str:
    lookup = {str(column).lower(): str(column) for column in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    return fallback


def refresh_coordinate_options() -> None:
    if SURVEY_DF is None:
        coordinate_x_source.options = []
        coordinate_y_source.options = []
        return
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    if not numeric:
        raise ValueError("No numeric coordinate columns were found.")
    coordinate_x_source.options = numeric
    coordinate_y_source.options = numeric
    coordinate_x_source.value = _guess_column(
        numeric, ["longitude", "lon", "long", "x", "easting", "utm_x"], numeric[0]
    )
    coordinate_y_source.value = _guess_column(
        numeric,
        ["latitude", "lat", "y", "northing", "utm_y"],
        numeric[min(1, len(numeric) - 1)],
    )


def _epsg_from_zone(zone: int, hemisphere: str) -> int:
    return (32600 if hemisphere == "N" else 32700) + int(zone)


def _auto_utm(lon: np.ndarray, lat: np.ndarray) -> Tuple[int, str, int]:
    lon_center = float(np.nanmedian(lon))
    lat_center = float(np.nanmedian(lat))
    zone = int(np.floor((lon_center + 180.0) / 6.0) + 1)
    zone = min(60, max(1, zone))
    hemisphere = "N" if lat_center >= 0 else "S"
    return zone, hemisphere, _epsg_from_zone(zone, hemisphere)


def transform_lonlat_arrays(lon: np.ndarray, lat: np.ndarray, epsg: int) -> Tuple[np.ndarray, np.ndarray]:
    try:
        from pyproj import Transformer
    except ImportError:
        import subprocess

        print("pyproj is not present; installing the coordinate-conversion dependency once...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
        from pyproj import Transformer
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    easting, northing = transformer.transform(lon, lat)
    return np.asarray(easting), np.asarray(northing)


def _apply_coordinates(_: widgets.Button) -> None:
    global SURVEY_DF, COORDINATE_MODE_APPLIED, COORDINATE_EPSG, WORKFLOW_CONFIG_APPLIED
    with coordinate_output:
        coordinate_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("Load the survey file first.")
            x_col, y_col = coordinate_x_source.value, coordinate_y_source.value
            if not x_col or not y_col or x_col == y_col:
                raise ValueError("Choose two distinct numeric coordinate columns.")
            x = pd.to_numeric(SURVEY_DF[x_col], errors="coerce").to_numpy(dtype=float)
            y = pd.to_numeric(SURVEY_DF[y_col], errors="coerce").to_numpy(dtype=float)
            if coordinate_mode.value == "decimal_degrees":
                if not np.all(np.isfinite(x) & np.isfinite(y)):
                    raise ValueError("Longitude/latitude columns contain missing or nonfinite values.")
                if np.any((x < -180) | (x > 180) | (y < -90) | (y > 90)):
                    raise ValueError("Selected decimal-degree values fall outside longitude/latitude bounds.")
                if utm_detection.value == "auto":
                    zone, hemisphere, epsg = _auto_utm(x, y)
                    zone_values = np.floor((x + 180.0) / 6.0).astype(int) + 1
                    if np.unique(zone_values).size > 1:
                        warnings.warn("Survey crosses more than one UTM zone; inspect projection distortion.")
                else:
                    zone, hemisphere = int(utm_zone.value), str(utm_hemisphere.value)
                    epsg = _epsg_from_zone(zone, hemisphere)
                easting, northing = transform_lonlat_arrays(x, y, epsg)
                SURVEY_DF = SURVEY_DF.copy()
                SURVEY_DF["_rssd_easting"] = easting
                SURVEY_DF["_rssd_northing"] = northing
                cfg.X_COLUMN, cfg.Y_COLUMN = "_rssd_easting", "_rssd_northing"
                COORDINATE_EPSG = epsg
                print(f"Converted WGS84 decimal degrees to UTM zone {zone}{hemisphere} (EPSG:{epsg}).")
                print("Added columns: _rssd_easting and _rssd_northing; original coordinates retained.")
            else:
                cfg.X_COLUMN, cfg.Y_COLUMN = str(x_col), str(y_col)
                entered_epsg = int(projected_epsg_control.value)
                if entered_epsg > 0:
                    try:
                        from pyproj import CRS
                    except ImportError:
                        import subprocess

                        print("pyproj is not present; installing the CRS validation dependency once...")
                        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
                        from pyproj import CRS
                    projected_crs = CRS.from_epsg(entered_epsg)
                    if not projected_crs.is_projected:
                        raise ValueError("Enter a projected CRS EPSG code, not a geographic longitude/latitude CRS.")
                    COORDINATE_EPSG = entered_epsg
                    print(f"Using existing projected coordinates with EPSG:{entered_epsg}.")
                    print("This CRS will be used to create the Google Earth KMZ export.")
                else:
                    COORDINATE_EPSG = None
                    print("Using existing projected coordinates, but their EPSG code is unknown.")
                    print("RSSD can run, but KMZ export will be skipped until a valid EPSG code is supplied.")
            COORDINATE_MODE_APPLIED = coordinate_mode.value
            WORKFLOW_CONFIG_APPLIED = False
            display(SURVEY_DF[[cfg.X_COLUMN, cfg.Y_COLUMN]].head())
        except Exception as exc:
            print(f"Coordinate error: {type(exc).__name__}: {exc}")


apply_coordinate_system.on_click(_apply_coordinates)
refresh_coordinate_options()
display(
    widgets.VBox(
        [
            coordinate_mode,
            responsive_widget_grid([coordinate_x_source, coordinate_y_source]),
            utm_detection,
            responsive_widget_grid([utm_zone, utm_hemisphere]),
            projected_epsg_control,
            widgets.HTML(
                "<i>For projected input, enter its EPSG code so selected sites can be transformed "
                "to WGS84 for Google Earth. Leave 0 only if KMZ is not needed.</i>"
            ),
            apply_coordinate_system,
            coordinate_output,
        ]
    )
)


### 1.6 Select the sample ID column

Choose an existing unique identifier or create a sequential ID. Duplicate IDs are reported immediately and remain a fatal ambiguity for analysis.


In [ ]:
# @title 6. Select or create the ID column { display-mode: "form" }
id_column_selector = widgets.Dropdown(description="Sample ID:", style={"description_width": "initial"})
apply_id_button = widgets.Button(description="Apply ID column", button_style="success")
id_output = widgets.Output()


def refresh_id_options() -> None:
    options = ["Create sequential ID"]
    if SURVEY_DF is not None:
        options.extend([str(c) for c in SURVEY_DF.columns])
    id_column_selector.options = options


def _apply_id(_: widgets.Button) -> None:
    global SURVEY_DF, WORKFLOW_CONFIG_APPLIED
    with id_output:
        id_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("Load the survey file first.")
            if id_column_selector.value == "Create sequential ID":
                SURVEY_DF = SURVEY_DF.copy()
                SURVEY_DF["_rssd_id"] = np.arange(1, len(SURVEY_DF) + 1, dtype=np.int64)
                cfg.ID_COLUMN = "_rssd_id"
            else:
                cfg.ID_COLUMN = str(id_column_selector.value)
            duplicate_rows = int(SURVEY_DF[cfg.ID_COLUMN].duplicated(keep=False).sum())
            if duplicate_rows:
                print(f"ERROR: {duplicate_rows} rows have duplicate IDs. Choose/create a unique ID before analysis.")
            else:
                print(f"ID column set to {cfg.ID_COLUMN!r}; all {len(SURVEY_DF):,} IDs are unique.")
            WORKFLOW_CONFIG_APPLIED = False
        except Exception as exc:
            print(f"ID error: {type(exc).__name__}: {exc}")


apply_id_button.on_click(_apply_id)
refresh_id_options()
display(id_column_selector, apply_id_button, id_output)


### 1.7 Select X/easting and Y/northing; preview geographic space

Select the coordinates used by geoMSD. The preview is interactive and does not calculate a survey-wide distance matrix. If longitude/latitude were converted, choose `_rssd_easting` and `_rssd_northing`.


In [ ]:
# @title 7. Select X/Y columns and preview geographic space { display-mode: "form" }
x_column_selector = widgets.Dropdown(description="X / easting:", style={"description_width": "initial"})
y_column_selector = widgets.Dropdown(description="Y / northing:", style={"description_width": "initial"})
geo_color = widgets.Dropdown(
    options=[("Blue", "C0"), ("Green", "C2"), ("Gray", "0.5"), ("Black", "black")],
    value=cfg.SURVEY_COLOR,
    description="Point color:",
)
geo_point_size = widgets.IntSlider(value=4, min=1, max=30, description="Point size:")
apply_xy_button = widgets.Button(description="Apply X/Y", button_style="success")
geo_output = widgets.Output()


def refresh_xy_options() -> None:
    if SURVEY_DF is None:
        x_column_selector.options = []
        y_column_selector.options = []
        return
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    x_column_selector.options = numeric
    y_column_selector.options = numeric
    if cfg.X_COLUMN in numeric:
        x_column_selector.value = cfg.X_COLUMN
    if cfg.Y_COLUMN in numeric:
        y_column_selector.value = cfg.Y_COLUMN


def _plot_geospace(*_: Any) -> None:
    with geo_output:
        geo_output.clear_output(wait=True)
        if SURVEY_DF is None or not x_column_selector.value or not y_column_selector.value:
            print("Load data and choose coordinate columns.")
            return
        x = pd.to_numeric(SURVEY_DF[x_column_selector.value], errors="coerce").to_numpy(float)
        y = pd.to_numeric(SURVEY_DF[y_column_selector.value], errors="coerce").to_numpy(float)
        finite = np.isfinite(x) & np.isfinite(y)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(x[finite], y[finite], s=geo_point_size.value, alpha=0.35, color=geo_color.value, rasterized=True)
        raw_preferred = collect_preferred_raw()
        if len(raw_preferred):
            preferred_x = raw_preferred.input_x.to_numpy(float)
            preferred_y = raw_preferred.input_y.to_numpy(float)
            if COORDINATE_MODE_APPLIED == "decimal_degrees" and COORDINATE_EPSG is not None:
                preferred_x, preferred_y = transform_lonlat_arrays(
                    preferred_x, preferred_y, COORDINATE_EPSG
                )
            ax.scatter(preferred_x, preferred_y, marker="*", s=100, color="C3", label="Entered locations")
            ax.legend()
        ax.set_xlabel(str(x_column_selector.value))
        ax.set_ylabel(str(y_column_selector.value))
        ax.set_title("Geographic-space preview")
        ax.set_aspect("equal", adjustable="box")
        fig.tight_layout()
        plt.show()
        print(f"Finite coordinate rows shown: {int(finite.sum()):,} / {len(finite):,}")


def _apply_xy(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    if x_column_selector.value == y_column_selector.value:
        with geo_output:
            print("Choose distinct X and Y columns.")
        return
    cfg.X_COLUMN = str(x_column_selector.value)
    cfg.Y_COLUMN = str(y_column_selector.value)
    cfg.SURVEY_COLOR = str(geo_color.value)
    WORKFLOW_CONFIG_APPLIED = False
    _plot_geospace()


apply_xy_button.on_click(_apply_xy)
x_column_selector.observe(_plot_geospace, names="value")
y_column_selector.observe(_plot_geospace, names="value")
geo_color.observe(_plot_geospace, names="value")
geo_point_size.observe(_plot_geospace, names="value")
refresh_xy_options()
display(
    responsive_widget_grid([x_column_selector, y_column_selector]),
    responsive_widget_grid([geo_color, geo_point_size, apply_xy_button], min_width=200),
    geo_output,
)
_plot_geospace()


### 1.8 Select sensor features for PCA

Select NDVI alone, NDRE alone, both, or any other numeric proxy columns. Use Ctrl/Cmd-click for multiple selections. Coordinate and ID columns are excluded automatically.


In [ ]:
# @title 8. Select PCA features { display-mode: "form" }
feature_selector = widgets.SelectMultiple(
    description="PCA features:", rows=12, layout=widgets.Layout(width="650px"), style={"description_width": "initial"}
)
select_all_features = widgets.Button(description="Select all numeric")
clear_features = widgets.Button(description="Clear selection")
apply_features = widgets.Button(description="Apply feature selection", button_style="success")
feature_output = widgets.Output()


def refresh_feature_options() -> None:
    if SURVEY_DF is None:
        feature_selector.options = []
        return
    excluded = {cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN}
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns if str(c) not in excluded]
    feature_selector.options = numeric
    preferred = [c for c in numeric if c.lower() in {"ndvi", "ndre"}]
    feature_selector.value = tuple(preferred if preferred else numeric[: min(2, len(numeric))])


def _apply_features(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with feature_output:
        feature_output.clear_output(wait=True)
        features = [str(c) for c in feature_selector.value]
        if not features:
            print("Select at least one numeric sensor feature.")
            return
        cfg.FEATURE_COLUMNS = features
        cfg.N_DESIGN_COMPONENTS = min(cfg.N_DESIGN_COMPONENTS, len(features), 4)
        WORKFLOW_CONFIG_APPLIED = False
        display(SURVEY_DF[features].describe().T)
        print(f"Selected {len(features)} PCA feature(s): {features}")


select_all_features.on_click(lambda _: setattr(feature_selector, "value", tuple(feature_selector.options)))
clear_features.on_click(lambda _: setattr(feature_selector, "value", ()))
apply_features.on_click(_apply_features)
refresh_feature_options()
display(
    feature_selector,
    widgets.HBox([select_all_features, clear_features, apply_features]),
    feature_output,
)


### 1.9 Choose transformation/scaling and inspect feature distributions

Lesch's method requires centered, standardized sensor variables, so `StandardScaler` is always applied. The selector controls the explicit transformation before standardization. Robust or min-max scaling is not substituted because that would change the published coding step.


In [ ]:
# @title 9. Select transformation/scaling and view distributions/correlation { display-mode: "form" }
scaling_scheme = widgets.RadioButtons(
    options=[
        ("StandardScaler only (recommended)", "none"),
        ("Yeo-Johnson, then StandardScaler", "yeo-johnson"),
        ("Natural log, then StandardScaler", "log"),
        ("Choose separately for each feature", "custom"),
    ],
    value="none",
    description="Scaling scheme:",
    style={"description_width": "initial"},
)
custom_transform_box = widgets.VBox()
custom_transform_controls: Dict[str, widgets.Dropdown] = {}
apply_scaling_button = widgets.Button(description="Apply and preview", button_style="success")
scaling_output = widgets.Output()


def _refresh_custom_transforms(*_: Any) -> None:
    global custom_transform_controls
    previous = {name: control.value for name, control in custom_transform_controls.items()}
    custom_transform_controls = {}
    controls = []
    for feature in cfg.FEATURE_COLUMNS:
        control = widgets.Dropdown(
            options=["none", "log", "yeo-johnson"],
            value=previous.get(feature, cfg.FEATURE_TRANSFORMS.get(feature, "none")),
            description=feature,
            style={"description_width": "initial"},
        )
        custom_transform_controls[feature] = control
        controls.append(control)
    custom_transform_box.children = tuple(controls) if scaling_scheme.value == "custom" else ()


def _apply_scaling(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with scaling_output:
        scaling_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None or not cfg.FEATURE_COLUMNS:
                raise ValueError("Load data and apply the feature selection first.")
            if scaling_scheme.value == "custom":
                transforms = {name: control.value for name, control in custom_transform_controls.items()}
            else:
                transforms = {name: scaling_scheme.value for name in cfg.FEATURE_COLUMNS}
            values = SURVEY_DF[cfg.FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
            finite = np.isfinite(values.to_numpy(float)).all(axis=1)
            transformed, details = transform_features(
                values.loc[finite].to_numpy(float), cfg.FEATURE_COLUMNS, transforms
            )
            cfg.FEATURE_TRANSFORMS = transforms
            WORKFLOW_CONFIG_APPLIED = False
            n_features = len(cfg.FEATURE_COLUMNS)
            ncols = min(3, n_features)
            nrows = math.ceil(n_features / ncols)
            fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.4 * nrows), squeeze=False)
            for j, feature in enumerate(cfg.FEATURE_COLUMNS):
                axes.flat[j].hist(transformed[:, j], bins=40, color=cfg.SURVEY_COLOR)
                axes.flat[j].set_title(f"{feature} ({transforms[feature]})")
            for ax in axes.flat[n_features:]:
                ax.set_visible(False)
            fig.suptitle("Transformed feature distributions")
            fig.tight_layout()
            plt.show()

            corr = np.corrcoef(transformed, rowvar=False)
            corr = np.atleast_2d(corr)
            fig, ax = plt.subplots(figsize=(max(5, n_features), max(4, n_features * 0.8)))
            image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
            ax.set_xticks(range(n_features), cfg.FEATURE_COLUMNS, rotation=45, ha="right")
            ax.set_yticks(range(n_features), cfg.FEATURE_COLUMNS)
            for i in range(n_features):
                for j in range(n_features):
                    ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center")
            ax.set_title("Selected-feature correlation matrix")
            fig.colorbar(image, ax=ax, shrink=0.8)
            fig.tight_layout()
            plt.show()
            display(pd.DataFrame(details).T)
        except Exception as exc:
            print(f"Scaling/transform error: {type(exc).__name__}: {exc}")


scaling_scheme.observe(_refresh_custom_transforms, names="value")
apply_scaling_button.on_click(_apply_scaling)
_refresh_custom_transforms()
display(scaling_scheme, custom_transform_box, apply_scaling_button, scaling_output)


### 1.10 Select the number of PCA components

Calculate the PCA preview, inspect individual and cumulative explained variance, then choose one to four standardized design PCs. For NDVI-only or NDRE-only, choose one. For a joint NDVI/NDRE design, choose one or two depending on whether both independent directions are needed.


In [ ]:
# @title 10. Calculate PCA preview and select components to retain { display-mode: "form" }
calculate_pca_button = widgets.Button(description="Calculate PCA preview", button_style="info")
pca_selector = use_full_description(widgets.IntSlider(value=1, min=1, max=1, step=1, description="Design PCs:"))
pca_color = widgets.Dropdown(
    options=[("Blue", "C0"), ("Green", "C2"), ("Purple", "C4"), ("Gray", "0.5")],
    value=cfg.SURVEY_COLOR,
    description="Point color:",
)
apply_pca_button = widgets.Button(description="Apply PCA count", button_style="success")
pca_output = widgets.Output()
pca_dimension_note = widgets.HTML(
    value="Select two design PCs for the familiar two-dimensional PC1-vs-PC2 response-surface plot."
)
PCA_PREVIEW_SCORES: Optional[np.ndarray] = None
PCA_PREVIEW_ESTIMATOR: Any = None


def _calculate_pca_preview(_: Optional[widgets.Button] = None) -> None:
    global PCA_PREVIEW_SCORES, PCA_PREVIEW_ESTIMATOR, WORKFLOW_CONFIG_APPLIED
    with pca_output:
        pca_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None or not cfg.FEATURE_COLUMNS:
                raise ValueError("Load data and apply features/transforms first.")
            values = SURVEY_DF[cfg.FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
            finite = np.isfinite(values).all(axis=1)
            transformed, _ = transform_features(values[finite], cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS)
            standardized = StandardScaler().fit_transform(transformed).astype(np.float32, copy=False)
            estimator = PCA(n_components=standardized.shape[1], svd_solver="full", random_state=cfg.RANDOM_SEED)
            raw_scores = estimator.fit_transform(standardized)
            whitened = raw_scores / np.sqrt(estimator.explained_variance_)
            PCA_PREVIEW_ESTIMATOR = estimator
            PCA_PREVIEW_SCORES = whitened[:, : min(4, whitened.shape[1])].astype(np.float32)
            pca_selector.max = PCA_PREVIEW_SCORES.shape[1]
            pca_selector.value = 2 if pca_selector.max >= 2 else 1
            explained = estimator.explained_variance_ratio_
            cumulative = np.cumsum(explained)
            display(
                pd.DataFrame(
                    {
                        "component": np.arange(1, len(explained) + 1),
                        "explained_variance_ratio": explained,
                        "cumulative_explained_variance": cumulative,
                    }
                )
            )
            fig, ax1 = plt.subplots(figsize=(8, 4.5))
            numbers = np.arange(1, len(explained) + 1)
            ax1.bar(numbers, explained, color=pca_color.value, alpha=0.7)
            ax1.set_xlabel("Principal component")
            ax1.set_ylabel("Explained variance ratio")
            ax2 = ax1.twinx()
            ax2.plot(numbers, cumulative, marker="o", color=cfg.TARGET_COLOR)
            ax2.set_ylabel("Cumulative explained variance")
            ax2.set_ylim(0, 1.05)
            ax1.set_title("PCA explained variance")
            fig.tight_layout()
            plt.show()
            WORKFLOW_CONFIG_APPLIED = False
        except Exception as exc:
            print(f"PCA preview error: {type(exc).__name__}: {exc}")


def _apply_pca(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with pca_output:
        if PCA_PREVIEW_SCORES is None:
            print("Calculate the PCA preview first.")
            return
        cfg.N_DESIGN_COMPONENTS = int(pca_selector.value)
        cfg.SURVEY_COLOR = str(pca_color.value)
        WORKFLOW_CONFIG_APPLIED = False
        print(f"Retained design PCs set to {cfg.N_DESIGN_COMPONENTS}.")


def _update_pca_dimension_note(change: Optional[Mapping[str, Any]] = None) -> None:
    if pca_selector.value == 1:
        extra = (
            " Only a one-dimensional PC1 line can be drawn. Choose 2 above for the "
            "traditional PC1-vs-PC2 rifle-scope view."
            if pca_selector.max >= 2
            else " Only one PCA direction exists because one feature was selected."
        )
        pca_dimension_note.value = f"<b>One design PC selected.</b>{extra}"
    else:
        pca_dimension_note.value = (
            "<b>Two-dimensional design enabled:</b> the next step plots standardized PC1 "
            "against PC2 with CCD targets and design/tolerance rings."
        )


calculate_pca_button.on_click(_calculate_pca_preview)
apply_pca_button.on_click(_apply_pca)
pca_selector.observe(_update_pca_dimension_note, names="value")
_update_pca_dimension_note()
display(
    responsive_widget_grid([calculate_pca_button, pca_selector, pca_color, apply_pca_button], min_width=190),
    pca_dimension_note,
    pca_output,
)


### 1.11 Select outlier threshold, design scale, and sample count

The legacy IQR slider is replaced by a statistically coherent chi-square coverage threshold in standardized PCA space. The **Design coverage** slider is the scale factor: it changes the empirical radius containing the selected proportion of survey observations. Sample count is editable, subject to the full-CCD minimum displayed by the widget.


In [ ]:
# @title 11. Adjust PCA outlier coverage, design radius, sample count, and preview layers { display-mode: "form" }
outlier_coverage_control = widgets.FloatSlider(
    value=cfg.OUTLIER_COVERAGE,
    min=0.95,
    max=0.9999,
    step=0.0001,
    readout_format=".4f",
    description="Outlier coverage:",
    style={"description_width": "initial"},
)
design_coverage_control = widgets.FloatSlider(
    value=cfg.DESIGN_COVERAGE,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Design coverage / scale:",
    style={"description_width": "initial"},
)
center_replicates_control = widgets.IntSlider(value=cfg.CENTER_REPLICATES, min=1, max=10, description="Center replicates:", style={"description_width": "initial"})
budget_mode_control = widgets.Dropdown(
    options=[("RSSD with spatial support", "rssd_with_support"), ("Balanced target replication", "balanced_target_replication"), ("Exact base CCD", "ccd_exact")],
    value=cfg.SAMPLE_BUDGET_MODE,
    description="Budget mode:",
)
sample_count_control = use_full_description(widgets.BoundedIntText(value=cfg.N_SAMPLES, min=1, max=1000, description="Sample count:"))
design_layers = widgets.SelectMultiple(
    options=["All observations", "PCA outliers", "CCD targets", "Design radius", "Target tolerance rings"],
    value=("All observations", "PCA outliers", "CCD targets", "Design radius", "Target tolerance rings"),
    description="Plot layers:",
    rows=3,
    style={"description_width": "initial"},
)
target_color_control = widgets.Dropdown(
    options=[("Red", "C3"), ("Orange", "C1"), ("Black", "black"), ("Purple", "C4")],
    value=cfg.TARGET_COLOR,
    description="Target color:",
)
outlier_color_control = widgets.Dropdown(
    options=[("Orange", "C1"), ("Red", "C3"), ("Purple", "C4"), ("Black", "black")],
    value=cfg.OUTLIER_COLOR,
    description="Outlier color:",
)
design_point_size = use_full_description(widgets.IntSlider(value=5, min=1, max=30, description="Point size:"))
preview_tolerance_control = widgets.FloatSlider(
    value=cfg.PC_CANDIDATE_TOLERANCE,
    min=0.02,
    max=1.0,
    step=0.01,
    description="Target-ring radius:",
    style={"description_width": "initial"},
)
apply_design_button = widgets.Button(description="Apply design settings", button_style="success")
design_preview_output = widgets.Output()
design_minimum_label = widgets.HTML()


def _update_design_minimum(*_: Any) -> None:
    p_value = int(pca_selector.value)
    base_count = 2**p_value + 2 * p_value + int(center_replicates_control.value)
    sample_count_control.min = base_count
    if budget_mode_control.value == "ccd_exact":
        sample_count_control.value = base_count
        sample_count_control.disabled = True
    else:
        sample_count_control.disabled = False
        if sample_count_control.value < base_count:
            sample_count_control.value = base_count
    design_minimum_label.value = (
        f"<b>Full {p_value}-PC CCD minimum: {base_count} samples.</b> "
        f"Current requested count: {sample_count_control.value}."
    )


def _design_preview(*_: Any) -> None:
    with design_preview_output:
        design_preview_output.clear_output(wait=True)
        if PCA_PREVIEW_SCORES is None:
            print("Calculate the PCA preview in step 9 first.")
            return
        p_value = int(pca_selector.value)
        scores = PCA_PREVIEW_SCORES[:, :p_value]
        d2 = np.einsum("ij,ij->i", scores, scores)
        threshold_value = chi2.ppf(outlier_coverage_control.value, df=p_value)
        outliers = d2 > threshold_value
        radius_values = np.linalg.norm(scores, axis=1)
        radius = float(np.quantile(radius_values, design_coverage_control.value))
        base = make_base_ccd(p_value, radius, int(center_replicates_control.value))
        targets, replication = allocate_target_instances(
            base, int(sample_count_control.value), str(budget_mode_control.value), p_value
        )
        target_columns = [f"target_PC{i + 1}" for i in range(p_value)]
        target_values = targets[target_columns].to_numpy(float)
        rng = np.random.default_rng(cfg.RANDOM_SEED)
        plot_n = min(cfg.PLOT_MAX_POINTS, len(scores))
        plot_indices = np.sort(rng.choice(len(scores), plot_n, replace=False))
        fig, ax = plt.subplots(figsize=(8, 5.5 if p_value > 1 else 4.5))
        if p_value == 1:
            if "All observations" in design_layers.value:
                ax.scatter(scores[plot_indices, 0], np.zeros(plot_n), s=design_point_size.value, alpha=0.18, color=pca_color.value, label="Observations")
            if "PCA outliers" in design_layers.value and outliers.any():
                idx = np.flatnonzero(outliers)
                ax.scatter(scores[idx, 0], np.zeros(len(idx)), s=design_point_size.value + 5, color=outlier_color_control.value, label="Outliers")
            if "CCD targets" in design_layers.value:
                ax.scatter(target_values[:, 0], np.zeros(len(target_values)), marker="x", s=75, color=target_color_control.value, label="CCD targets")
            if "Design radius" in design_layers.value:
                ax.axvline(-radius, linestyle="--", linewidth=1, color=target_color_control.value)
                ax.axvline(radius, linestyle="--", linewidth=1, color=target_color_control.value)
            ax.set_yticks([])
            ax.set_xlabel("Standardized PC1")
            print(
                "This is a one-dimensional design because one PC is selected. "
                "Return to step 9 and select two PCs for the PC1-vs-PC2 rifle-scope view."
            )
        else:
            if "All observations" in design_layers.value:
                ax.scatter(scores[plot_indices, 0], scores[plot_indices, 1], s=design_point_size.value, alpha=0.18, color=pca_color.value, label="Observations")
            if "PCA outliers" in design_layers.value and outliers.any():
                idx = np.flatnonzero(outliers)
                ax.scatter(scores[idx, 0], scores[idx, 1], s=design_point_size.value + 5, color=outlier_color_control.value, label="Outliers")
            if "CCD targets" in design_layers.value:
                ax.scatter(target_values[:, 0], target_values[:, 1], marker="x", s=75, color=target_color_control.value, label="CCD targets")
            if "Design radius" in design_layers.value:
                ax.add_patch(
                    plt.Circle((0, 0), radius, fill=False, linestyle="--", linewidth=1.3, color=target_color_control.value, label="Design radius R")
                )
            if "Target tolerance rings" in design_layers.value:
                for target in np.unique(target_values[:, :2], axis=0):
                    ax.add_patch(
                        plt.Circle(
                            tuple(target),
                            preview_tolerance_control.value,
                            fill=False,
                            linestyle=":",
                            linewidth=0.9,
                            color=target_color_control.value,
                            alpha=0.75,
                        )
                    )
            ax.set_xlabel("Standardized PC1")
            ax.set_ylabel("Standardized PC2")
            ax.set_aspect("equal", adjustable="box")
        ax.set_title("PCA response-surface design preview" + (" (first two PCs)" if p_value > 2 else ""))
        ax.legend()
        fig.tight_layout()
        plt.show()
        display(
            pd.DataFrame(
                {
                    "design_radius_R": [radius],
                    "realized_coverage": [float(np.mean(radius_values <= radius))],
                    "masked_outliers": [int(outliers.sum())],
                    "base_targets": [len(base)],
                    "target_instances": [len(targets)],
                }
            )
        )
        display(replication.to_frame())


def _apply_design(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    cfg.N_DESIGN_COMPONENTS = int(pca_selector.value)
    cfg.OUTLIER_COVERAGE = float(outlier_coverage_control.value)
    cfg.DESIGN_COVERAGE = float(design_coverage_control.value)
    cfg.CENTER_REPLICATES = int(center_replicates_control.value)
    cfg.SAMPLE_BUDGET_MODE = str(budget_mode_control.value)
    cfg.N_SAMPLES = int(sample_count_control.value)
    cfg.TARGET_COLOR = str(target_color_control.value)
    cfg.PC_CANDIDATE_TOLERANCE = float(preview_tolerance_control.value)
    cfg.OUTLIER_COLOR = str(outlier_color_control.value)
    WORKFLOW_CONFIG_APPLIED = False
    with design_preview_output:
        print("Design settings applied. Continue to optimization settings.")


for control in [outlier_coverage_control, design_coverage_control, center_replicates_control, budget_mode_control, sample_count_control, design_layers, target_color_control, outlier_color_control, design_point_size, preview_tolerance_control]:
    control.observe(_design_preview, names="value")
center_replicates_control.observe(_update_design_minimum, names="value")
budget_mode_control.observe(_update_design_minimum, names="value")
pca_selector.observe(_update_design_minimum, names="value")
apply_design_button.on_click(_apply_design)
_update_design_minimum()
display(
    widgets.VBox(
        [
            responsive_widget_grid([outlier_coverage_control, design_coverage_control]),
            responsive_widget_grid([center_replicates_control, budget_mode_control, sample_count_control], min_width=220),
            design_minimum_label,
            responsive_widget_grid([design_layers, target_color_control, outlier_color_control, design_point_size], min_width=190),
            preview_tolerance_control,
            apply_design_button,
            design_preview_output,
        ]
    )
)
_design_preview()


### 1.12 Candidate, optimization, preferred-location, and output controls

These controls replace the legacy weighted-score slider. Candidate tolerance constrains response-surface fidelity; exact geoMSD remains the primary objective. Mean and maximum PCA mismatch are tie-breakers. Existing target locations can be overlaid/evaluated or forced into the design.


In [ ]:
# @title 12. Apply candidate and optimization settings; finalize workflow { display-mode: "form" }
support_mode_control = widgets.Dropdown(
    options=[("Legacy-inspired automatic", "legacy_inspired_auto"), ("Manual", "manual"), ("None", "none")],
    value=cfg.SUPPORT_SITE_MODE,
    description="Support-site mode:",
    style={"description_width": "initial"},
)
manual_support_control = widgets.IntSlider(value=cfg.N_SUPPORT_SITES, min=0, max=50, description="Manual support sites:", style={"description_width": "initial"})
support_gap_anchors_control = widgets.BoundedIntText(value=cfg.SUPPORT_GAP_ANCHORS, min=1, max=10000, description="Gap anchors:", style={"description_width": "initial"})
support_candidate_neighbors_control = widgets.BoundedIntText(value=cfg.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR, min=1, max=100, description="Neighbors/anchor:", style={"description_width": "initial"})
candidate_mode_control = widgets.Dropdown(options=[("Automatic saturation", "adaptive"), ("Fixed", "fixed")], value=cfg.CANDIDATE_EXPLORATION_MODE, description="Candidate exploration:", style={"description_width": "initial"})
candidates_control = widgets.IntSlider(value=cfg.CANDIDATES_PER_TARGET, min=2, max=25, description="Initial candidates/target:", style={"description_width": "initial"})
tolerance_control = widgets.FloatSlider(value=cfg.PC_CANDIDATE_TOLERANCE, min=0.02, max=1.0, step=0.01, description="Initial PC tolerance:", style={"description_width": "initial"})
max_tolerance_control = widgets.FloatSlider(value=cfg.MAX_PC_CANDIDATE_TOLERANCE, min=0.15, max=3.0, step=0.05, description="Maximum PC tolerance:", style={"description_width": "initial"})
starts_control = widgets.IntSlider(value=cfg.OPTIMIZER_MAX_STARTS, min=1, max=150, description="Max batched starts:", style={"description_width": "initial"})
min_starts_control = widgets.IntSlider(value=cfg.OPTIMIZER_MIN_STARTS, min=1, max=150, description="Minimum starts:", style={"description_width": "initial"})
batch_size_control = widgets.IntSlider(value=cfg.OPTIMIZER_START_BATCH_SIZE, min=1, max=25, description="Batch size:", style={"description_width": "initial"})
near_best_fraction_control = widgets.FloatSlider(value=cfg.OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION, min=0.25, max=1.0, step=0.05, readout_format=".2f", description="Near-best fraction:", style={"description_width": "initial"})
spacing_diagnostic_control = widgets.Checkbox(value=cfg.RUN_SPACING_DIAGNOSTIC, description="Proxy spatial scale diagnostic", indent=False)
sbad_mode_control = widgets.Dropdown(options=[("Automatic decision-stable", "adaptive"), ("Fixed", "fixed")], value=cfg.AD_SUPPORT_MODE, description="SBAD accuracy:", style={"description_width": "initial"})
coverage_envelope_control = widgets.FloatSlider(value=cfg.AD_COVERAGE_ENVELOPE_REL_TOL, min=0.0, max=0.25, step=0.01, readout_format=".2f", description="SBAD envelope:", style={"description_width": "initial"})
optimizer_mode_control = widgets.Dropdown(options=[("Automatic", "adaptive"), ("Fixed starts", "fixed")], value=cfg.OPTIMIZER_START_MODE, description="Optimizer starts:", style={"description_width": "initial"})
memory_control = widgets.Dropdown(options=["auto", "full", "incremental"], value=cfg.MEMORY_MODE, description="Memory mode:")
approximate_control = widgets.Checkbox(value=cfg.ALLOW_APPROXIMATE_PREFILTER, description="Allow approximate extreme-data prefilter", indent=False)
preferred_mode_control = widgets.Dropdown(
    options=[
        ("Ignore entered locations", "none"),
        ("Evaluate and overlay only", "evaluate_only"),
        ("Force nearest eligible survey sites into final design", "force"),
    ],
    value="evaluate_only" if preferred_rows else "none",
    description="Existing locations:",
    style={"description_width": "initial"},
)
candidate_color_control = widgets.Dropdown(options=[("Green", "C2"), ("Blue", "C0"), ("Purple", "C4"), ("Black", "black")], value=cfg.CANDIDATE_COLOR, description="Candidate color:")
selected_color_control = widgets.Dropdown(options=[("Red", "C3"), ("Orange", "C1"), ("Purple", "C4"), ("Black", "black")], value=cfg.SELECTED_COLOR, description="Selected color:")
apply_optimizer_button = widgets.Button(description="Apply optimization settings", button_style="success", icon="check")
optimizer_output = widgets.Output()
PREFERRED_LOCATIONS = pd.DataFrame(columns=["preferred_location_id", "x", "y"])
PREFERRED_MODE = "none"


def _prepare_preferred_locations() -> pd.DataFrame:
    raw = collect_preferred_raw()
    if len(raw) == 0:
        return pd.DataFrame(columns=["preferred_location_id", "x", "y"])
    if COORDINATE_MODE_APPLIED == "decimal_degrees":
        if COORDINATE_EPSG is None:
            raise ValueError("Apply decimal-degree to UTM conversion before finalizing locations.")
        x, y = transform_lonlat_arrays(
            raw.input_x.to_numpy(float), raw.input_y.to_numpy(float), COORDINATE_EPSG
        )
    else:
        x, y = raw.input_x.to_numpy(float), raw.input_y.to_numpy(float)
    return pd.DataFrame(
        {"preferred_location_id": raw.preferred_location_id.astype(str), "x": x, "y": y}
    )


def _finalize_workflow(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED, PREFERRED_LOCATIONS, PREFERRED_MODE
    with optimizer_output:
        optimizer_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("No survey dataframe is loaded.")
            required = [cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, *cfg.FEATURE_COLUMNS]
            missing = [column for column in required if column not in SURVEY_DF.columns]
            if missing:
                raise KeyError(f"Required columns not applied or missing: {missing}")
            if COORDINATE_MODE_APPLIED == "decimal_degrees" and cfg.X_COLUMN not in {"_rssd_easting"}:
                raise ValueError("Select the generated UTM easting/northing columns in step 6.")
            cfg.SUPPORT_SITE_MODE = str(support_mode_control.value)
            cfg.N_SUPPORT_SITES = int(manual_support_control.value)
            cfg.CANDIDATE_EXPLORATION_MODE = str(candidate_mode_control.value)
            cfg.CANDIDATES_PER_TARGET = int(candidates_control.value)
            cfg.SUPPORT_GAP_ANCHORS = int(support_gap_anchors_control.value)
            cfg.SUPPORT_CANDIDATE_NEIGHBORS_PER_ANCHOR = int(support_candidate_neighbors_control.value)
            cfg.PC_CANDIDATE_TOLERANCE = float(tolerance_control.value)
            cfg.MAX_PC_CANDIDATE_TOLERANCE = max(float(max_tolerance_control.value), cfg.PC_CANDIDATE_TOLERANCE)
            cfg.OPTIMIZER_MAX_STARTS = int(starts_control.value)
            cfg.OPTIMIZER_MIN_STARTS = min(int(min_starts_control.value), cfg.OPTIMIZER_MAX_STARTS)
            cfg.N_OPTIMIZER_STARTS = cfg.OPTIMIZER_MAX_STARTS
            cfg.MIN_OPTIMIZER_STARTS = cfg.OPTIMIZER_MIN_STARTS
            cfg.OPTIMIZER_START_BATCH_SIZE = int(batch_size_control.value)
            cfg.OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION = float(near_best_fraction_control.value)
            cfg.RUN_SPACING_DIAGNOSTIC = bool(spacing_diagnostic_control.value)
            cfg.AD_SUPPORT_MODE = str(sbad_mode_control.value)
            cfg.AD_COVERAGE_ENVELOPE_REL_TOL = float(coverage_envelope_control.value)
            cfg.OPTIMIZER_START_MODE = str(optimizer_mode_control.value)
            cfg.OPTIMIZER_INITIALIZATION_MODE = cfg.OPTIMIZER_START_MODE
            cfg.MEMORY_MODE = str(memory_control.value)
            cfg.ALLOW_APPROXIMATE_PREFILTER = bool(approximate_control.value)
            cfg.CANDIDATE_COLOR = str(candidate_color_control.value)
            cfg.SELECTED_COLOR = str(selected_color_control.value)
            cfg.USE_SYNTHETIC_DEMO = False
            PREFERRED_LOCATIONS = _prepare_preferred_locations()
            PREFERRED_MODE = str(preferred_mode_control.value) if len(PREFERRED_LOCATIONS) else "none"
            if len(PREFERRED_LOCATIONS) > cfg.N_SAMPLES:
                raise ValueError("Existing/forced locations cannot exceed the requested sample count.")
            validate_config(cfg)
            WORKFLOW_CONFIG_APPLIED = True
            display(
                pd.DataFrame(
                    {
                        "setting": ["input", "range-filter applications", "current working rows", "ID", "X", "Y", "features", "transforms", "design PCs", "samples", "outlier coverage", "design coverage", "candidates/target", "PC tolerance", "max batched starts", "minimum starts", "batch size", "near-best fraction", "spacing diagnostic", "existing-location mode"],
                        "value": [SURVEY_INPUT_LABEL, len(RANGE_FILTER_AUDIT), len(SURVEY_DF), cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS, cfg.N_DESIGN_COMPONENTS, cfg.N_SAMPLES, cfg.OUTLIER_COVERAGE, cfg.DESIGN_COVERAGE, cfg.CANDIDATES_PER_TARGET, cfg.PC_CANDIDATE_TOLERANCE, cfg.OPTIMIZER_MAX_STARTS, cfg.OPTIMIZER_MIN_STARTS, cfg.OPTIMIZER_START_BATCH_SIZE, cfg.OPTIMIZER_NEAR_BEST_REQUIRED_FRACTION, cfg.RUN_SPACING_DIAGNOSTIC, PREFERRED_MODE],
                    }
                )
            )
            if len(PREFERRED_LOCATIONS):
                display(PREFERRED_LOCATIONS)
            print("WORKFLOW READY. Continue to 'Run the scalable RSSD analysis' below.")
        except Exception as exc:
            WORKFLOW_CONFIG_APPLIED = False
            print(f"Finalization error: {type(exc).__name__}: {exc}")


apply_optimizer_button.on_click(_finalize_workflow)
display(
    widgets.VBox(
        [
            widgets.HTML("<b>Spatial support sites</b><br><span>What this controls: reserves a small portion of the sample budget for calibration locations selected specifically to reduce geographic coverage gaps after the response-surface regression design is built.<br>Recommended default: legacy-inspired automatic allocation.<br>When to change it: use manual or none primarily for validation against classical designs or when every sample must be tied to a theoretical response-surface target.</span>"),
            responsive_widget_grid([support_mode_control, manual_support_control], min_width=260),
            widgets.HTML("<b>Candidate exploration</b><br><span>What this controls: how many geographically distinct survey observations are made available as alternatives for each PCA response-surface condition.<br>Recommended default: automatic candidate exploration expands alternatives until the achievable SBAD-constrained design stops materially changing. Merely finding additional candidate observations does not mean the sampling design is still improving.<br>When to change it: use fixed mode for methodological comparisons or exact reruns with a fixed candidate pool.</span>"),
            responsive_widget_grid([candidate_mode_control, candidates_control, tolerance_control, max_tolerance_control], min_width=230),
            widgets.HTML("<b>SBAD accuracy</b><br><span>What this controls: the geographic support resolution used to measure field-wide coverage independently of raw sensor observation density.<br>Recommended default: automatic SBAD support resolution increases geographic support until coverage values and materially different design choices are stable. Reordering practically equivalent designs is not treated as instability.<br>When to change it: use fixed support size only for benchmarks or methodological experiments.</span>"),
            responsive_widget_grid([sbad_mode_control, coverage_envelope_control], min_width=260),
            widgets.HTML("<b>Advanced spatial support-site search</b><br><span><b>Gap anchors</b> controls the number of most poorly covered geographic support locations used to identify possible spatial support-site regions. Recommended default: 500. Increase only for unusually large or fragmented survey domains when support-site candidate search appears too geographically restricted.<br><b>Neighbors/anchor</b> controls the number of nearby candidate-eligible survey observations retrieved around each high-gap geographic support location. Recommended default: 5. Increase when sensor observations are sparse or irregular and the bounded support-site candidate set is too small.</span>"),
            responsive_widget_grid([support_gap_anchors_control, support_candidate_neighbors_control], min_width=260),
            widgets.HTML("<b>Optimizer starts</b><br><span>What this controls: automatic optimizer starts test different valid initial sampling designs in batches. The search stops when new batches are no longer finding materially better solutions and most recent starts converge near the best known result.<br>Recommended default: automatic.</span>"),
            responsive_widget_grid([optimizer_mode_control, starts_control, min_starts_control, batch_size_control, near_best_fraction_control], min_width=230),
            widgets.HTML("<b>Proxy spatial scale</b><br><span>What this controls: a pre-calibration diagnostic describing the distance over which standardized sensor predictor patterns appear spatially structured.<br>Recommended default: estimate automatically. This is not the future regression residual correlation range.</span>"),
            responsive_widget_grid([spacing_diagnostic_control], min_width=230),
            responsive_widget_grid([memory_control, approximate_control], min_width=230),
            preferred_mode_control,
            responsive_widget_grid([candidate_color_control, selected_color_control]),
            widgets.HTML("<i>Primary objective: maintain near-optimal SBAD field coverage, then maximize exact geoMSD inside that coverage envelope.</i>"),
            apply_optimizer_button,
            optimizer_output,
        ]
    )
)


## 2. Run ESAP RSSD v2.10
Run the single analysis cell below after the guided workflow reports **WORKFLOW READY**. Internal numerical stages are wrapped by `run_esap_rssd(...)`.


In [ ]:
# @title Run ESAP RSSD analysis { display-mode: "form" }
workflow_state = {}
if globals().get("WORKFLOW_CONFIG_APPLIED", False):
    source_df_for_run = SURVEY_DF.copy()
    workflow_state = {
        "input_label": str(SURVEY_INPUT_LABEL),
        "preferred_locations": globals().get("PREFERRED_LOCATIONS", pd.DataFrame(columns=["preferred_location_id", "x", "y"])),
        "preferred_mode": globals().get("PREFERRED_MODE", "none"),
        "coordinate_epsg": globals().get("SOURCE_PROJECTED_EPSG", globals().get("COORDINATE_EPSG", None)),
    }
    print("Using the dataframe and settings finalized in the sequential guided workflow.")
else:
    source_df_for_run = None
    print("Workflow was not finalized; using the configuration/default synthetic input.")

rssd_result = run_esap_rssd(cfg, source_df_for_run, workflow_state)
selected_export = rssd_result.selected_sites
candidate_export = rssd_result.candidate_sites
target_instances = rssd_result.target_instances
support_sequence = rssd_result.support_sequence
ad_support_resolution = rssd_result.ad_support_resolution
candidate_saturation = rssd_result.candidate_saturation
optimizer_stability = rssd_result.optimizer_stability
spatial_support_sites = rssd_result.spatial_support_sites
field_coverage_distances = rssd_result.field_coverage_distances
proxy_spatial_scale = rssd_result.proxy_spatial_scale
pca_validation = rssd_result.pca_validation
skewness_summary = rssd_result.feature_skewness
metadata = rssd_result.metadata
run_summary = rssd_result.run_summary

display(selected_export)
display(ad_support_resolution)
display(candidate_saturation)
if len(optimizer_stability):
    display(
        optimizer_stability.groupby(["support_size", "objective_type"], as_index=False).agg(
            starts=("start_number", "max"),
            best_SBAD=("best_SBAD", "last"),
            best_geoMSD=("best_geoMSD", "last"),
            stable=("optimizer_initialization_stable", "last"),
        )
    )

create_esap_rssd_figures(rssd_result, show=True)



## Why Moran's I Is Not Used Here

Lesch (2005) uses the Moran residual statistic to assess spatial autocorrelation after a calibration response has been measured and a regression model has been fitted. At the RSSD site-selection stage, no target response or fitted calibration residuals exist. Standardized PCA scores are coded sensor covariates, not regression residuals. Consequently, Moran's I is neither calculated nor optimized here. It belongs in the later calibration and model-validation workflow, alongside other residual diagnostics.


## 3. Download a complete RSSD v2.10 run bundle
Create a timestamped ZIP containing the v2.10 selected sites, support sequence, SBAD support-resolution table, candidate-saturation table, optimizer-stability table, metadata, summary, and optional KMZ.


In [ ]:
# @title Create and download complete RSSD v2.10 run bundle { display-mode: "form" }
CREATE_RUN_BUNDLE = False  # @param {type:"boolean"}

if CREATE_RUN_BUNDLE:
    import re, shutil
    from datetime import datetime, timezone
    if "rssd_result" not in globals():
        raise RuntimeError("Run the ESAP RSSD analysis successfully before creating the bundle.")
    bundle_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    input_label = str(rssd_result.metadata.get("input_label", "rssd_input"))
    input_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(str(input_label).replace("uploaded:", "")).stem or "rssd_input")
    bundle_dir = Path(f"ESAP_RSSD_v2_10_{input_stem}_{bundle_timestamp}"); bundle_dir.mkdir(parents=False, exist_ok=False)
    figures_dir = bundle_dir / "figures"
    saved_figures = create_esap_rssd_figures(rssd_result, output_dir=figures_dir, show=False)
    for filename, table in {
        "ESAP_RSSD_selected_sites.csv": rssd_result.selected_sites,
        "ESAP_RSSD_candidate_sites.csv": rssd_result.candidate_sites,
        "ESAP_RSSD_target_instances.csv": rssd_result.target_instances,
        "ESAP_RSSD_spatial_support_sequence.csv": rssd_result.support_sequence,
        "ESAP_RSSD_ad_support_resolution.csv": rssd_result.ad_support_resolution,
        "ESAP_RSSD_candidate_saturation.csv": rssd_result.candidate_saturation,
        "ESAP_RSSD_optimizer_stability.csv": rssd_result.optimizer_stability,
        "ESAP_RSSD_spatial_support_sites.csv": rssd_result.spatial_support_sites,
        "ESAP_RSSD_field_coverage_distances.csv": rssd_result.field_coverage_distances,
        "ESAP_RSSD_proxy_spatial_scale.csv": rssd_result.proxy_spatial_scale,
        "ESAP_RSSD_pca_standardization_validation.csv": rssd_result.pca_validation,
        "ESAP_RSSD_feature_skewness.csv": rssd_result.feature_skewness,
    }.items():
        table.to_csv(bundle_dir / filename, index=False)
    (bundle_dir / "ESAP_RSSD_run_metadata.json").write_text(json.dumps(json_ready(rssd_result.metadata), indent=2, allow_nan=False), encoding="utf-8")
    (bundle_dir / "ESAP_RSSD_run_summary.md").write_text(rssd_result.run_summary, encoding="utf-8")
    (bundle_dir / "ESAP_RSSD_configuration.json").write_text(json.dumps(json_ready(asdict(rssd_result.config)), indent=2, allow_nan=False), encoding="utf-8")
    kmz_meta = rssd_result.metadata.get("kmz_export", {})
    if kmz_meta.get("created") and Path(str(kmz_meta.get("file"))).exists():
        shutil.copy2(Path(str(kmz_meta["file"])), bundle_dir / "ESAP_RSSD_selected_sites.kmz")
    manifest = f"""# ESAP RSSD v2.10 complete run bundle\n\n- Notebook version: {NOTEBOOK_VERSION}\n- Bundle created (UTC): {bundle_timestamp}\n- Selected samples: {len(rssd_result.selected_sites):,}\n- Response-surface sites: {(rssd_result.selected_sites.sample_role == 'response_surface').sum():,}\n- Spatial support sites: {(rssd_result.selected_sites.sample_role == 'spatial_support').sum():,}\n- Final SBAD support size: {rssd_result.metadata.get('support_optimization', {}).get('final_support_size')}\n- Random seed: {rssd_result.config.RANDOM_SEED}\n- Figures: {len(saved_figures)} PNG files\n- Raw input dataset is not copied into this bundle.\n"""
    (bundle_dir / "README.md").write_text(manifest, encoding="utf-8")
    zip_path = Path(shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir))
    print(f"Complete run bundle created: {zip_path.resolve()}")
    try:
        from google.colab import files as colab_files  # type: ignore
        colab_files.download(str(zip_path))
    except ImportError:
        print("Outside Colab: use the ZIP path printed above.")
else:
    print("Run bundle not created. Check CREATE_RUN_BUNDLE and rerun this final cell after analysis.")

